In [ ]:
# ============================================================
# BLOCK 00
# Dataset Loading + Initial Dataset Inspection
# ============================================================

import os
import pandas as pd
import numpy as np

# ============================================================
# Dataset Path
# ============================================================

DATASET_PATH = "/kaggle/input/datasets/valeriamaciel/e-coli-resistance-dataset"

# ============================================================
# Output Directory
# ============================================================

OUTPUT_DIR = "outputs/block00"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# Show All Files
# ============================================================

print("=" * 60)
print("FILES IN DATASET")
print("=" * 60)

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        print(os.path.join(root, file))

# ============================================================
# Automatically Find CSV File
# ============================================================

csv_files = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, file))

csv_files = sorted(csv_files)

print("\n")
print("=" * 60)
print("CSV FILES FOUND")
print("=" * 60)

for f in csv_files:
    print(f)

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file found inside the dataset directory.")

# ============================================================
# Load Dataset
# ============================================================

df = pd.read_csv(csv_files[0], low_memory=False)

# ============================================================
# Basic Information
# ============================================================

print("\n")
print("=" * 60)
print("DATASET SHAPE")
print("=" * 60)

print(df.shape)

print("\n")
print("=" * 60)
print("FIRST 5 ROWS")
print("=" * 60)

print(df.head())

print("\n")
print("=" * 60)
print("COLUMN NAMES")
print("=" * 60)

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

print("\n")
print("=" * 60)
print("DATASET INFO")
print("=" * 60)

df.info()

print("\n")
print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing_values = df.isnull().sum().sort_values(ascending=False)
print(missing_values)

print("\n")
print("=" * 60)
print("DUPLICATE ROWS")
print("=" * 60)

duplicate_rows = df.duplicated().sum()
print(duplicate_rows)

# ============================================================
# Unique Values Per Column
# ============================================================

print("\n")
print("=" * 60)
print("UNIQUE VALUE COUNT")
print("=" * 60)

for col in df.columns:
    print(f"{col}: {df[col].nunique()}")

# ============================================================
# Sample Values
# ============================================================

print("\n")
print("=" * 60)
print("SAMPLE VALUES")
print("=" * 60)

for col in df.columns:
    print(f"\nCOLUMN: {col}")
    print(df[col].dropna().unique()[:10])

# ============================================================
# Statistical Summary
# ============================================================

print("\n")
print("=" * 60)
print("NUMERICAL SUMMARY")
print("=" * 60)

print(df.describe())

print("\n")
print("=" * 60)
print("CATEGORICAL SUMMARY")
print("=" * 60)

print(df.describe(include="object"))

# ============================================================
# Dataset Memory Usage
# ============================================================

memory_usage = df.memory_usage(deep=True).sum() / (1024 ** 2)

print("\n")
print("=" * 60)
print("DATASET MEMORY USAGE")
print("=" * 60)

print(f"{memory_usage:.2f} MB")

# ============================================================
# Column Summary
# ============================================================

column_summary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Missing (%)": (df.isnull().mean() * 100).round(2).values,
    "Unique Values": [df[col].nunique() for col in df.columns]
})

column_summary.to_csv(
    os.path.join(OUTPUT_DIR, "column_summary.csv"),
    index=False
)

# ============================================================
# Dataset Health Report
# ============================================================

health_report = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate Rows",
        "Duplicate (%)",
        "Total Missing Values",
        "Memory Usage (MB)"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        duplicate_rows,
        round((duplicate_rows / len(df)) * 100, 4),
        int(df.isnull().sum().sum()),
        round(memory_usage, 2)
    ]
})

health_report.to_csv(
    os.path.join(OUTPUT_DIR, "dataset_health_report.csv"),
    index=False
)

# ============================================================
# Completed
# ============================================================

print("\n")
print("=" * 60)
print("FILES SAVED")
print("=" * 60)
print(os.path.join(OUTPUT_DIR, "column_summary.csv"))
print(os.path.join(OUTPUT_DIR, "dataset_health_report.csv"))

print("\n")
print("=" * 60)
print("BLOCK 00 COMPLETED SUCCESSFULLY")
print("=" * 60)

In [ ]:
genome_counts = df.groupby("Genome ID")["Antibiotic"].nunique()

print(genome_counts.describe())

In [ ]:
# ============================================================
# BLOCK 01
# Global Configuration + Dataset Loading + Dataset Audit
# ============================================================

# ============================================================
# 1. Imports
# ============================================================

import os
import gc
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# 2. Global Configuration
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# 3. Output Directories
# ============================================================

OUTPUT_DIR = "outputs"
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("=" * 60)
print("OUTPUT DIRECTORY READY")
print("=" * 60)

# ============================================================
# 4. Dataset Path
# ============================================================

DATASET_PATH = "/kaggle/input/datasets/valeriamaciel/e-coli-resistance-dataset"

# ============================================================
# 5. Find CSV Files
# ============================================================

csv_files = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, file))

csv_files = sorted(csv_files)

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file found in the dataset directory.")

print("\n")
print("=" * 60)
print("CSV FILES FOUND")
print("=" * 60)

for f in csv_files:
    print(f)

# ============================================================
# 6. Load Dataset
# ============================================================

df = pd.read_csv(
    csv_files[0],
    low_memory=False
)

# ============================================================
# 7. Dataset Shape
# ============================================================

print("\n")
print("=" * 60)
print("DATASET SHAPE")
print("=" * 60)

print(df.shape)

# ============================================================
# 8. First Rows
# ============================================================

print("\n")
print("=" * 60)
print("HEAD")
print("=" * 60)

print(df.head())

# ============================================================
# 9. Column Names
# ============================================================

print("\n")
print("=" * 60)
print("COLUMN NAMES")
print("=" * 60)

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

# ============================================================
# 10. Dataset Information
# ============================================================

print("\n")
print("=" * 60)
print("DATASET INFO")
print("=" * 60)

df.info()

# ============================================================
# 11. Data Types
# ============================================================

print("\n")
print("=" * 60)
print("DATA TYPES")
print("=" * 60)

print(df.dtypes)

# ============================================================
# 12. Missing Value Analysis
# ============================================================

missing_df = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().mean() * 100).round(2)
})

missing_df = missing_df.sort_values(
    by="Missing_Count",
    ascending=False
)

print("\n")
print("=" * 60)
print("MISSING VALUE REPORT")
print("=" * 60)

print(missing_df)

missing_df.to_csv(
    os.path.join(REPORT_DIR, "missing_value_report.csv")
)

# ============================================================
# 13. Duplicate Analysis
# ============================================================

duplicate_count = df.duplicated().sum()

duplicate_percentage = round(
    duplicate_count / len(df) * 100,
    2
)

print("\n")
print("=" * 60)
print("DUPLICATE REPORT")
print("=" * 60)

print(f"Duplicate Rows : {duplicate_count}")
print(f"Duplicate %    : {duplicate_percentage}")

# ============================================================
# 14. Numerical Summary
# ============================================================

print("\n")
print("=" * 60)
print("NUMERICAL SUMMARY")
print("=" * 60)

print(df.describe())

# ============================================================
# 15. Categorical Summary
# ============================================================

print("\n")
print("=" * 60)
print("CATEGORICAL SUMMARY")
print("=" * 60)

print(df.describe(include="object"))

# ============================================================
# 16. Memory Usage
# ============================================================

memory_usage = (
    df.memory_usage(deep=True).sum() /
    (1024 ** 2)
)

print("\n")
print("=" * 60)
print("MEMORY USAGE")
print("=" * 60)

print(f"{memory_usage:.2f} MB")

# ============================================================
# 17. Dataset Audit Report
# ============================================================

audit_report = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate Rows",
        "Duplicate Percentage",
        "Total Missing Values",
        "Missing Percentage",
        "Memory Usage (MB)"
    ],
    "Value": [
        len(df),
        len(df.columns),
        duplicate_count,
        duplicate_percentage,
        int(df.isnull().sum().sum()),
        round(df.isnull().mean().mean() * 100, 2),
        round(memory_usage, 2)
    ]
})

audit_report.to_csv(
    os.path.join(REPORT_DIR, "dataset_audit_report.csv"),
    index=False
)

# ============================================================
# BLOCK COMPLETED
# ============================================================

print("\n")
print("=" * 60)
print("BLOCK 01 COMPLETED SUCCESSFULLY")
print("=" * 60)

print("Generated Files:")
print("- reports/dataset_audit_report.csv")
print("- reports/missing_value_report.csv")

In [ ]:
# ============================================================
# BLOCK 02
# Data Cleaning & Standardization
# PART 1
# ============================================================

print("=" * 60)
print("BLOCK 02 STARTED")
print("=" * 60)

# ============================================================
# Create Working Copy
# ============================================================

df_clean = df.copy()

print(f"\nOriginal Shape : {df_clean.shape}")

# ============================================================
# Output Directory
# ============================================================

DATASET_DIR = os.path.join(OUTPUT_DIR, "datasets")
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ============================================================
# 1. Drop Useless Columns
# ============================================================

drop_columns = [
    "Computational Method",
    "Computational Method Version",
    "Computational Method Performance",
    "Source",
    "Laboratory Typing Method Version"
]

existing_drop_cols = [
    col
    for col in drop_columns
    if col in df_clean.columns
]

df_clean.drop(
    columns=existing_drop_cols,
    inplace=True
)

print("\nDropped Columns")
print("-" * 40)

for col in existing_drop_cols:
    print(col)

# ============================================================
# 2. Remove Exact Duplicates
# ============================================================

rows_before = len(df_clean)

df_clean = (
    df_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

rows_after = len(df_clean)

duplicates_removed = rows_before - rows_after

print("\nDuplicate Removal")
print("-" * 40)
print("Rows Before :", rows_before)
print("Rows After  :", rows_after)
print("Removed     :", duplicates_removed)

# ============================================================
# 3. Genome ID Cleaning
# ============================================================

df_clean["Genome ID"] = (
    df_clean["Genome ID"]
    .astype(str)
    .str.strip()
)

# ============================================================
# 4. Genome Name Cleaning
# ============================================================

df_clean["Genome Name"] = (
    df_clean["Genome Name"]
    .astype(str)
    .str.strip()
)

# ============================================================
# 5. Antibiotic Cleaning
# ============================================================

df_clean["Antibiotic"] = (
    df_clean["Antibiotic"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ============================================================
# 6. Resistant Phenotype Cleaning
# ============================================================

df_clean["Resistant Phenotype"] = (
    df_clean["Resistant Phenotype"]
    .astype(str)
    .str.strip()
    .str.lower()
)

phenotype_mapping = {

    "resistant":
        "Resistant",

    "susceptible":
        "Susceptible",

    "intermediate":
        "Intermediate",

    "nonsusceptible":
        "Nonsusceptible",

    "susceptible-dose dependent":
        "Susceptible-dose dependent",

    "nan":
        np.nan

}

df_clean["Resistant Phenotype"] = (
    df_clean["Resistant Phenotype"]
    .replace(phenotype_mapping)
)

print("\n")
print("=" * 60)
print("PHENOTYPE DISTRIBUTION")
print("=" * 60)

print(
    df_clean["Resistant Phenotype"]
    .value_counts(dropna=False)
)

# ============================================================
# 7. Measurement Sign Cleaning
# ============================================================

df_clean["Measurement Sign"] = (
    df_clean["Measurement Sign"]
    .astype(str)
    .str.strip()
)

df_clean["Measurement Sign"] = (
    df_clean["Measurement Sign"]
    .replace({
        "nan": np.nan,
        "": np.nan
    })
)

print("\n")
print("=" * 60)
print("MEASUREMENT SIGN")
print("=" * 60)

print(
    df_clean["Measurement Sign"]
    .value_counts(dropna=False)
)

# ============================================================
# 8. Measurement Cleaning
# ============================================================

df_clean["Measurement"] = (
    df_clean["Measurement"]
    .astype(str)
    .str.strip()
)

df_clean["Measurement"] = (
    df_clean["Measurement"]
    .replace({
        "nan": np.nan,
        "": np.nan
    })
)

# ============================================================
# 9. Measurement Value Cleaning
# ============================================================

df_clean["Measurement Value"] = (
    df_clean["Measurement Value"]
    .astype(str)
    .str.strip()
)

df_clean["Measurement Value"] = (
    df_clean["Measurement Value"]
    .replace({
        "nan": np.nan,
        "": np.nan
    })
)

# ============================================================
# PART 1 COMPLETED
# ============================================================

print("\nPART 1 COMPLETED")

In [ ]:
# ============================================================
# BLOCK 02
# Data Cleaning & Standardization
# PART 2
# ============================================================

# ============================================================
# 10. MIC Numeric Extraction
# ============================================================

def extract_numeric_mic(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if "/" in value:
        value = value.split("/")[0]

    value = (
        value.replace("<=", "")
             .replace(">=", "")
             .replace("<", "")
             .replace(">", "")
             .replace("=", "")
             .strip()
    )

    try:
        return float(value)
    except ValueError:
        return np.nan


df_clean["MIC_Numeric"] = (
    df_clean["Measurement Value"]
    .apply(extract_numeric_mic)
)

# ============================================================
# 11. Measurement Unit Standardization
# ============================================================

df_clean["Measurement Unit"] = (
    df_clean["Measurement Unit"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df_clean["Measurement Unit"] = (
    df_clean["Measurement Unit"]
    .replace({
        "nan": np.nan,
        "": np.nan,
        "mg/l": "mg/L",
        "mm": "mm"
    })
)

print("\n")
print("=" * 60)
print("MEASUREMENT UNIT DISTRIBUTION")
print("=" * 60)

print(
    df_clean["Measurement Unit"]
    .value_counts(dropna=False)
)

# ============================================================
# 12. Testing Standard Standardization
# ============================================================

df_clean["Testing Standard"] = (
    df_clean["Testing Standard"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_clean["Testing Standard"] = (
    df_clean["Testing Standard"]
    .replace({
        "NAN": np.nan,
        "MISSING": np.nan,
        "CLSI, EUCAST": "CLSI/EUCAST",
        "EUCAST,CLSI": "CLSI/EUCAST",
        "EUCAST AND CLSI": "CLSI/EUCAST"
    })
)

# ============================================================
# 13. Laboratory Typing Method Standardization
# ============================================================

df_clean["Laboratory Typing Method"] = (
    df_clean["Laboratory Typing Method"]
    .astype(str)
    .str.strip()
    .str.title()
)

df_clean["Laboratory Typing Method"] = (
    df_clean["Laboratory Typing Method"]
    .replace({
        "Nan": np.nan
    })
)

# ============================================================
# 14. Missing Value Report
# ============================================================

missing_after = pd.DataFrame({
    "Missing_Count": df_clean.isnull().sum(),
    "Missing_Percentage": (
        df_clean.isnull().mean() * 100
    ).round(2)
})

missing_after = (
    missing_after
    .sort_values(
        by="Missing_Count",
        ascending=False
    )
)

print("\n")
print("=" * 60)
print("MISSING VALUE REPORT")
print("=" * 60)

print(missing_after)

# ============================================================
# 15. MIC Summary
# ============================================================

print("\n")
print("=" * 60)
print("MIC NUMERIC SUMMARY")
print("=" * 60)

print(
    df_clean["MIC_Numeric"]
    .describe()
)

# ============================================================
# 16. Cleaning Report
# ============================================================

cleaning_report = pd.DataFrame({
    "Metric": [
        "Rows Before",
        "Rows After",
        "Columns Before",
        "Columns After",
        "Dropped Columns",
        "Duplicates Removed",
        "Missing Values After Cleaning"
    ],
    "Value": [
        rows_before,
        rows_after,
        len(df.columns),
        len(df_clean.columns),
        len(existing_drop_cols),
        duplicates_removed,
        int(df_clean.isnull().sum().sum())
    ]
})

# ============================================================
# 17. Save Outputs
# ============================================================

df_clean.to_csv(
    os.path.join(DATASET_DIR, "clean_dataset.csv"),
    index=False
)

missing_after.to_csv(
    os.path.join(REPORT_DIR, "missing_after_cleaning.csv")
)

cleaning_report.to_csv(
    os.path.join(REPORT_DIR, "cleaning_report.csv"),
    index=False
)

# ============================================================
# 18. Final Dataset Information
# ============================================================

print("\n")
print("=" * 60)
print("FINAL DATASET SHAPE")
print("=" * 60)

print(df_clean.shape)

print("\n")
print("=" * 60)
print("FILES GENERATED")
print("=" * 60)

print("datasets/clean_dataset.csv")
print("reports/missing_after_cleaning.csv")
print("reports/cleaning_report.csv")

# ============================================================
# BLOCK 02 COMPLETED
# ============================================================

print("\n")
print("=" * 60)
print("BLOCK 02 COMPLETED SUCCESSFULLY")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 03
# Comprehensive EDA + Resistance Trend Analysis
# PART 1
# ============================================================

print("=" * 60)
print("BLOCK 03 STARTED")
print("=" * 60)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# ============================================================
# Output Directories
# ============================================================

FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures", "EDA")
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")

os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ============================================================
# Figure Configuration
# ============================================================

FIG_DPI = 300

# ============================================================
# 1. Dataset Statistics
# ============================================================

print("\n")
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

num_rows = len(df_clean)
num_cols = len(df_clean.columns)
num_genomes = df_clean["Genome ID"].nunique()
num_antibiotics = df_clean["Antibiotic"].nunique()

print(f"Rows               : {num_rows:,}")
print(f"Columns            : {num_cols}")
print(f"Unique Genomes     : {num_genomes:,}")
print(f"Unique Antibiotics : {num_antibiotics}")

# ============================================================
# 2. Phenotype Distribution
# ============================================================

print("\n")
print("=" * 60)
print("PHENOTYPE DISTRIBUTION")
print("=" * 60)

phenotype_counts = (
    df_clean["Resistant Phenotype"]
    .fillna("Missing")
    .value_counts()
)

print(phenotype_counts)

phenotype_counts.to_csv(
    os.path.join(REPORT_DIR,
    "phenotype_distribution.csv")
)

plt.figure(figsize=(8,5))

sns.countplot(
    data=df_clean,
    x=df_clean["Resistant Phenotype"].fillna("Missing"),
    order=phenotype_counts.index
)

plt.title("Phenotype Distribution")
plt.xlabel("Phenotype")
plt.ylabel("Count")
plt.xticks(rotation=20)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "figure_02_phenotype_distribution.png"
    ),
    dpi=FIG_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ============================================================
# 3. Antibiotic Distribution
# ============================================================

print("\n")
print("=" * 60)
print("TOP ANTIBIOTICS")
print("=" * 60)

top_antibiotics = (
    df_clean["Antibiotic"]
    .value_counts()
    .head(15)
)

print(top_antibiotics)

top_antibiotics.to_csv(
    os.path.join(
        REPORT_DIR,
        "top_antibiotics.csv"
    )
)

plt.figure(figsize=(11,6))

sns.barplot(
    x=top_antibiotics.values,
    y=top_antibiotics.index
)

plt.title("Top 15 Antibiotics")
plt.xlabel("Number of Records")
plt.ylabel("Antibiotic")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "figure_03_antibiotic_distribution.png"
    ),
    dpi=FIG_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ============================================================
# 4. Genome Coverage Analysis
# ============================================================

print("\n")
print("=" * 60)
print("GENOME COVERAGE ANALYSIS")
print("=" * 60)

genome_coverage = (
    df_clean
    .groupby("Genome ID")["Antibiotic"]
    .nunique()
)

print(genome_coverage.describe())

genome_coverage.describe().to_csv(
    os.path.join(
        REPORT_DIR,
        "genome_coverage_statistics.csv"
    )
)

plt.figure(figsize=(8,5))

sns.histplot(
    genome_coverage,
    bins=30,
    kde=True
)

plt.title("Antibiotics Tested Per Genome")
plt.xlabel("Number of Antibiotics")
plt.ylabel("Genome Count")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "figure_04_genome_coverage.png"
    ),
    dpi=FIG_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("\n")
print("=" * 60)
print("BLOCK 03 PART 1 COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 03
# Comprehensive EDA + Resistance Trend Analysis
# PART 2
# ============================================================

# ============================================================
# 5. MIC Distribution Analysis
# ============================================================

if "MIC_Numeric" in df_clean.columns:

    mic_data = df_clean["MIC_Numeric"].dropna()

    # Remove unrealistic outliers for visualization only
    mic_plot = mic_data[
        (mic_data >= 0) &
        (mic_data <= mic_data.quantile(0.99))
    ]

    print("\n")
    print("=" * 60)
    print("MIC STATISTICS")
    print("=" * 60)

    print(mic_data.describe())

    mic_data.describe().to_csv(
        os.path.join(
            REPORT_DIR,
            "mic_statistics.csv"
        )
    )

    plt.figure(figsize=(8,5))

    sns.histplot(
        mic_plot,
        bins=30,
        kde=True
    )

    plt.title("MIC Distribution (99th Percentile)")
    plt.xlabel("MIC")
    plt.ylabel("Frequency")

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            FIGURE_DIR,
            "figure_05_mic_distribution.png"
        ),
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

# ============================================================
# 6. Measurement Unit Distribution
# ============================================================

print("\n")
print("=" * 60)
print("MEASUREMENT UNIT DISTRIBUTION")
print("=" * 60)

measurement_unit = (
    df_clean["Measurement Unit"]
    .fillna("Missing")
    .value_counts()
)

print(measurement_unit)

measurement_unit.to_csv(
    os.path.join(
        REPORT_DIR,
        "measurement_unit_distribution.csv"
    )
)

# ============================================================
# 7. Top Resistant Antibiotics
# ============================================================

resistant_df = df_clean[
    df_clean["Resistant Phenotype"] == "Resistant"
]

print("\n")
print("=" * 60)
print("TOP RESISTANT ANTIBIOTICS")
print("=" * 60)

top_resistant = (
    resistant_df["Antibiotic"]
    .value_counts()
    .head(10)
)

print(top_resistant)

top_resistant.to_csv(
    os.path.join(
        REPORT_DIR,
        "top_resistant_antibiotics.csv"
    )
)

plt.figure(figsize=(10,5))

sns.barplot(
    x=top_resistant.values,
    y=top_resistant.index
)

plt.title("Top Resistant Antibiotics")
plt.xlabel("Number of Resistant Isolates")
plt.ylabel("Antibiotic")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "figure_06_top_resistant_antibiotics.png"
    ),
    dpi=FIG_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ============================================================
# 8. Top Susceptible Antibiotics
# ============================================================

susceptible_df = df_clean[
    df_clean["Resistant Phenotype"] == "Susceptible"
]

print("\n")
print("=" * 60)
print("TOP SUSCEPTIBLE ANTIBIOTICS")
print("=" * 60)

top_susceptible = (
    susceptible_df["Antibiotic"]
    .value_counts()
    .head(10)
)

print(top_susceptible)

top_susceptible.to_csv(
    os.path.join(
        REPORT_DIR,
        "top_susceptible_antibiotics.csv"
    )
)

plt.figure(figsize=(10,5))

sns.barplot(
    x=top_susceptible.values,
    y=top_susceptible.index
)

plt.title("Top Susceptible Antibiotics")
plt.xlabel("Number of Susceptible Isolates")
plt.ylabel("Antibiotic")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "figure_07_top_susceptible_antibiotics.png"
    ),
    dpi=FIG_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("\n")
print("=" * 60)
print("BLOCK 03 PART 2 COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 03
# Comprehensive EDA + Resistance Trend Analysis
# PART 3
# ============================================================

# ============================================================
# 9. Testing Standard Analysis
# ============================================================

print("\n")
print("=" * 60)
print("TESTING STANDARD DISTRIBUTION")
print("=" * 60)

testing_standard = (
    df_clean["Testing Standard"]
    .fillna("Missing")
    .value_counts()
)

print(testing_standard)

testing_standard.to_csv(
    os.path.join(
        REPORT_DIR,
        "testing_standard_distribution.csv"
    )
)

plt.figure(figsize=(9,5))

sns.barplot(
    x=testing_standard.values,
    y=testing_standard.index
)

plt.title("Testing Standard Distribution")
plt.xlabel("Count")
plt.ylabel("Testing Standard")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "figure_08_testing_standard_distribution.png"
    ),
    dpi=FIG_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ============================================================
# 10. Laboratory Typing Method
# ============================================================

print("\n")
print("=" * 60)
print("LABORATORY TYPING METHOD")
print("=" * 60)

lab_method = (
    df_clean["Laboratory Typing Method"]
    .fillna("Missing")
    .value_counts()
)

print(lab_method)

lab_method.to_csv(
    os.path.join(
        REPORT_DIR,
        "laboratory_method_distribution.csv"
    )
)

plt.figure(figsize=(9,5))

sns.barplot(
    x=lab_method.values,
    y=lab_method.index
)

plt.title("Laboratory Typing Method")
plt.xlabel("Count")
plt.ylabel("Method")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "figure_09_laboratory_method.png"
    ),
    dpi=FIG_DPI,
    bbox_inches="tight"
)

plt.show()
plt.close()

# ============================================================
# 11. Resistance Frequency Analysis
# ============================================================

print("\n")
print("=" * 60)
print("RESISTANCE FREQUENCY")
print("=" * 60)

resistance_frequency = (
    df_clean["Resistant Phenotype"]
    .fillna("Missing")
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(resistance_frequency)

resistance_frequency.to_csv(
    os.path.join(
        REPORT_DIR,
        "resistance_frequency.csv"
    )
)

# ============================================================
# 12. EDA Summary
# ============================================================

eda_summary = pd.DataFrame({

    "Metric":[

        "Rows",
        "Columns",
        "Unique Genomes",
        "Unique Antibiotics",
        "Duplicate Removed",
        "MIC Available",
        "Missing Phenotype"

    ],

    "Value":[

        len(df_clean),
        len(df_clean.columns),
        df_clean["Genome ID"].nunique(),
        df_clean["Antibiotic"].nunique(),
        duplicates_removed,
        df_clean["MIC_Numeric"].notna().sum(),
        df_clean["Resistant Phenotype"].isna().sum()

    ]

})

eda_summary.to_csv(

    os.path.join(
        REPORT_DIR,
        "eda_summary.csv"
    ),

    index=False

)

# ============================================================
# 13. Generated Files
# ============================================================

print("\n")
print("=" * 60)
print("GENERATED FILES")
print("=" * 60)

print("Reports")
print("- phenotype_distribution.csv")
print("- top_antibiotics.csv")
print("- genome_coverage_statistics.csv")
print("- mic_statistics.csv")
print("- measurement_unit_distribution.csv")
print("- top_resistant_antibiotics.csv")
print("- top_susceptible_antibiotics.csv")
print("- testing_standard_distribution.csv")
print("- laboratory_method_distribution.csv")
print("- resistance_frequency.csv")
print("- eda_summary.csv")

print("\nFigures")
print("- figure_02_phenotype_distribution.png")
print("- figure_03_antibiotic_distribution.png")
print("- figure_04_genome_coverage.png")
print("- figure_05_mic_distribution.png")
print("- figure_06_top_resistant_antibiotics.png")
print("- figure_07_top_susceptible_antibiotics.png")
print("- figure_08_testing_standard_distribution.png")
print("- figure_09_laboratory_method.png")

# ============================================================
# BLOCK 03 COMPLETED
# ============================================================

print("\n")
print("=" * 60)
print("BLOCK 03 COMPLETED SUCCESSFULLY")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 04
# Genome-Level Resistance Profile Construction
# PART 1 (FINAL)
# ============================================================

print("=" * 60)
print("BLOCK 04 STARTED")
print("=" * 60)

# ============================================================
# Output Directory
# ============================================================

GENOME_DIR = os.path.join(
    OUTPUT_DIR,
    "genome_level"
)

os.makedirs(
    GENOME_DIR,
    exist_ok=True
)

# ============================================================
# Create Working Copy
# ============================================================

df_genome = df_clean.copy()

print("\n")
print("=" * 60)
print("GENOME DATASET OVERVIEW")
print("=" * 60)

print(f"Rows                : {len(df_genome):,}")
print(f"Columns             : {df_genome.shape[1]}")
print(f"Unique Genomes      : {df_genome['Genome ID'].nunique():,}")
print(f"Unique Antibiotics  : {df_genome['Antibiotic'].nunique()}")

# ============================================================
# Genome Coverage Analysis
# ============================================================

genome_coverage = (

    df_genome

    .groupby("Genome ID")["Antibiotic"]

    .nunique()

)

print("\n")
print("=" * 60)
print("GENOME COVERAGE STATISTICS")
print("=" * 60)

print(genome_coverage.describe())

genome_coverage.describe().to_csv(

    os.path.join(
        GENOME_DIR,
        "genome_coverage_statistics.csv"
    )

)

# ============================================================
# Coverage Distribution Figure
# ============================================================

plt.figure(figsize=(8,5))

sns.histplot(
    genome_coverage,
    bins=30,
    kde=True
)

plt.title("Unique Antibiotics Tested Per Genome")
plt.xlabel("Number of Antibiotics")
plt.ylabel("Genome Count")

plt.tight_layout()

plt.savefig(

    os.path.join(
        GENOME_DIR,
        "genome_coverage_distribution.png"
    ),

    dpi=300,
    bbox_inches="tight"

)

plt.show()
plt.close()

# ============================================================
# Antibiotics Per Genome Table
# ============================================================

coverage_table = (

    genome_coverage

    .reset_index()

    .rename(
        columns={
            "Antibiotic":"Antibiotic_Count"
        }
    )

)

coverage_table.to_csv(

    os.path.join(
        GENOME_DIR,
        "antibiotics_per_genome.csv"
    ),

    index=False

)

# ============================================================
# Genome × Antibiotic Profile Matrix
# (Phenotype Matrix - No Binary Encoding)
# ============================================================

print("\n")
print("=" * 60)
print("CREATING GENOME × ANTIBIOTIC PROFILE")
print("=" * 60)

genome_profile = pd.pivot_table(

    df_genome,

    index="Genome ID",

    columns="Antibiotic",

    values="Resistant Phenotype",

    aggfunc="first",

    observed=False

)

print("Profile Matrix Shape :", genome_profile.shape)

print(
    "\nGenome Preserved :",
    genome_profile.shape[0]
)

print(
    "Antibiotics Preserved :",
    genome_profile.shape[1]
)

genome_profile = genome_profile.reset_index()

print("\n")
print("=" * 60)
print("PROFILE MATRIX PREVIEW")
print("=" * 60)

print(genome_profile.head())

print("\n")
print("=" * 60)
print("BLOCK 04 PART 1 COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 04
# Genome-Level Resistance Profile Construction
# PART 1 (FINAL JOURNAL VERSION)
# ============================================================

print("=" * 60)
print("BLOCK 04 STARTED")
print("=" * 60)

# ============================================================
# Output Directory
# ============================================================

GENOME_DIR = os.path.join(
    OUTPUT_DIR,
    "genome_level"
)

os.makedirs(
    GENOME_DIR,
    exist_ok=True
)

# ============================================================
# Working Copy
# ============================================================

df_genome = df_clean.copy()

# ============================================================
# Dataset Overview
# ============================================================

print("\n")
print("=" * 60)
print("GENOME DATASET OVERVIEW")
print("=" * 60)

total_records = len(df_genome)
total_genomes = df_genome["Genome ID"].nunique()
total_antibiotics = df_genome["Antibiotic"].nunique()

print(f"Total Records      : {total_records:,}")
print(f"Unique Genomes     : {total_genomes:,}")
print(f"Unique Antibiotics : {total_antibiotics}")

# ============================================================
# Genome Coverage
# ============================================================

genome_coverage = (

    df_genome

    .groupby("Genome ID")["Antibiotic"]

    .nunique()

)

print("\n")
print("=" * 60)
print("GENOME COVERAGE")
print("=" * 60)

print(genome_coverage.describe())

coverage_summary = (
    genome_coverage
    .describe()
    .to_frame(name="Value")
)

coverage_summary.to_csv(

    os.path.join(
        GENOME_DIR,
        "genome_coverage_statistics.csv"
    )

)

coverage_table = (

    genome_coverage

    .reset_index()

    .rename(
        columns={
            "Antibiotic":"Antibiotics_Tested"
        }
    )

)

coverage_table.to_csv(

    os.path.join(
        GENOME_DIR,
        "antibiotics_per_genome.csv"
    ),

    index=False

)

# ============================================================
# Coverage Distribution
# ============================================================

plt.figure(figsize=(8,5))

sns.histplot(

    genome_coverage,

    bins=30,

    kde=True

)

plt.title(
    "Genome Coverage"
)

plt.xlabel(
    "Unique Antibiotics Tested"
)

plt.ylabel(
    "Genome Count"
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        GENOME_DIR,
        "genome_coverage_distribution.png"
    ),

    dpi=300,

    bbox_inches="tight"

)

plt.show()
plt.close()

# ============================================================
# Qualified Genome Detection
# ============================================================

print("\n")
print("=" * 60)
print("QUALIFIED GENOME DETECTION")
print("=" * 60)

qualified_records = df_genome[

    df_genome["Resistant Phenotype"]
    .notna()

].copy()

qualified_genomes = (

    qualified_records["Genome ID"]

    .drop_duplicates()

    .sort_values()

)

qualified_genome_table = pd.DataFrame({

    "Genome ID":qualified_genomes

})

qualified_genome_table.to_csv(

    os.path.join(
        GENOME_DIR,
        "qualified_genomes.csv"
    ),

    index=False

)

print(
    "Qualified Genomes :",
    len(qualified_genome_table)
)

print(
    "Removed Genomes   :",
    total_genomes - len(qualified_genome_table)
)

print("\n")
print("=" * 60)
print("BLOCK 04 PART 1 COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 04
# Genome-Level Resistance Profile Construction
# PART 2 (FINAL JOURNAL VERSION)
# ============================================================

# ============================================================
# Qualified Genome Coverage Statistics
# ============================================================

qualified_df = df_genome[
    df_genome["Genome ID"].isin(
        qualified_genome_table["Genome ID"]
    )
].copy()

qualified_coverage = (

    qualified_df

    .groupby("Genome ID")["Antibiotic"]

    .nunique()

)

print("\n")
print("=" * 60)
print("QUALIFIED GENOME COVERAGE")
print("=" * 60)

print(
    qualified_coverage.describe()
)

qualified_coverage.describe().to_csv(

    os.path.join(
        GENOME_DIR,
        "qualified_genome_coverage_statistics.csv"
    )

)

# ============================================================
# Coverage Distribution (Qualified Genomes)
# ============================================================

plt.figure(figsize=(8,5))

sns.histplot(

    qualified_coverage,

    bins=30,

    kde=True

)

plt.title(
    "Qualified Genome Coverage"
)

plt.xlabel(
    "Unique Antibiotics Tested"
)

plt.ylabel(
    "Genome Count"
)

plt.tight_layout()

plt.savefig(

    os.path.join(

        GENOME_DIR,

        "qualified_genome_coverage_distribution.png"

    ),

    dpi=300,

    bbox_inches="tight"

)

plt.show()
plt.close()

# ============================================================
# Qualified Genome Summary
# ============================================================

qualified_summary = pd.DataFrame({

    "Metric":[

        "Total Records",

        "Total Genomes",

        "Qualified Genomes",

        "Removed Genomes",

        "Unique Antibiotics",

        "Average Antibiotics per Qualified Genome",

        "Median Antibiotics per Qualified Genome",

        "Maximum Antibiotics per Genome",

        "Minimum Antibiotics per Genome"

    ],

    "Value":[

        len(df_genome),

        total_genomes,

        len(qualified_genome_table),

        total_genomes - len(qualified_genome_table),

        total_antibiotics,

        round(
            qualified_coverage.mean(),
            2
        ),

        round(
            qualified_coverage.median(),
            2
        ),

        int(
            qualified_coverage.max()
        ),

        int(
            qualified_coverage.min()
        )

    ]

})

qualified_summary.to_csv(

    os.path.join(
        GENOME_DIR,
        "qualified_genome_summary.csv"
    ),

    index=False

)

# ============================================================
# Coverage Validation
# ============================================================

validation_table = pd.DataFrame({

    "Validation":[

        "Genome IDs Unique",

        "Genome IDs Missing",

        "Duplicate Genome IDs",

        "Coverage Table Rows",

        "Qualified Genome Rows"

    ],

    "Result":[

        qualified_df["Genome ID"].nunique(),

        qualified_df["Genome ID"].isna().sum(),

        qualified_df["Genome ID"].duplicated().sum(),

        len(qualified_coverage),

        len(qualified_genome_table)

    ]

})

validation_table.to_csv(

    os.path.join(
        GENOME_DIR,
        "genome_validation_report.csv"
    ),

    index=False

)

# ============================================================
# Generated Files
# ============================================================

print("\n")
print("=" * 60)
print("GENERATED FILES")
print("=" * 60)

print("genome_coverage_statistics.csv")
print("antibiotics_per_genome.csv")
print("qualified_genomes.csv")
print("qualified_genome_summary.csv")
print("qualified_genome_coverage_statistics.csv")
print("genome_validation_report.csv")
print("genome_coverage_distribution.png")
print("qualified_genome_coverage_distribution.png")

# ============================================================
# BLOCK 04 COMPLETED
# ============================================================

print("\n")
print("=" * 60)
print("BLOCK 04 COMPLETED SUCCESSFULLY")
print("=" * 60)

In [ ]:
# ============================================================
# Export Complete Antibiotic List
# ============================================================

antibiotic_list = (
    pd.DataFrame(
        sorted(df_clean["Antibiotic"].dropna().unique()),
        columns=["Antibiotic"]
    )
)

print("Total Antibiotics:", len(antibiotic_list))

display(antibiotic_list)

antibiotic_list.to_csv(
    "all_antibiotics.csv",
    index=False
)

In [ ]:
print("\n".join(sorted(df_clean["Antibiotic"].dropna().unique())))

In [ ]:
# ============================================================
# BLOCK 05
# Antibiotic Class Mapping + MDR Label Generation
# PART 1 (FINAL)
# ============================================================

print("=" * 60)
print("BLOCK 05 STARTED")
print("=" * 60)

# ============================================================
# Output Directory
# ============================================================

MDR_DIR = os.path.join(
    OUTPUT_DIR,
    "mdr"
)

os.makedirs(
    MDR_DIR,
    exist_ok=True
)

# ============================================================
# Qualified Dataset
# ============================================================

qualified_df = df_clean[
    df_clean["Genome ID"].isin(
        qualified_genome_table["Genome ID"]
    )
].copy()

print("\n")
print("=" * 60)
print("QUALIFIED DATASET")
print("=" * 60)

print(f"Rows              : {len(qualified_df):,}")
print(f"Unique Genomes    : {qualified_df['Genome ID'].nunique():,}")
print(f"Unique Antibiotics: {qualified_df['Antibiotic'].nunique()}")

# ============================================================
# Antibiotic Name Standardization
# ============================================================

name_corrections = {

    "cefalexin":"cephalexin",

    "cefalotin":"cephalothin",

    "sulfamethoxazole/trimethoprim":
        "trimethoprim/sulfamethoxazole",

    "cefotaxime/clavulanic acidâ":
        "cefotaxime/clavulanic acid"

}

qualified_df["Antibiotic"] = (

    qualified_df["Antibiotic"]

    .replace(name_corrections)

)

# ============================================================
# Remove Invalid Antibiotics
# ============================================================

invalid_antibiotics = [

    "publicatiion"

]

qualified_df = qualified_df[
    ~qualified_df["Antibiotic"]
    .isin(invalid_antibiotics)
].copy()

print("\n")
print("=" * 60)
print("DATASET AFTER STANDARDIZATION")
print("=" * 60)

print(f"Rows              : {len(qualified_df):,}")
print(f"Unique Antibiotics: {qualified_df['Antibiotic'].nunique()}")

# ============================================================
# Resistance Encoding
# ============================================================

encoding = {

    "Resistant":1,

    "Nonsusceptible":1,

    "Susceptible":0,

    "Intermediate":0,

    "Susceptible-dose dependent":0

}

qualified_df["Resistance_Binary"] = (

    qualified_df["Resistant Phenotype"]

    .map(encoding)

)

print("\n")
print("=" * 60)
print("RESISTANCE ENCODING")
print("=" * 60)

print(

    qualified_df["Resistance_Binary"]

    .value_counts(dropna=False)

)

# ============================================================
# Antibiotic Class Mapping
# ============================================================

antibiotic_class_map = {

    # Penicillins
    "ampicillin":"Penicillin",
    "amoxicillin":"Penicillin",
    "penicillin":"Penicillin",
    "carbenicillin":"Penicillin",
    "piperacillin":"Penicillin",
    "ticarcillin":"Penicillin",
    "temocillin":"Penicillin",

    # Penicillin Combination
    "ampicillin/sulbactam":"Penicillin Combination",
    "amoxicillin/clavulanic acid":"Penicillin Combination",
    "piperacillin/tazobactam":"Penicillin Combination",
    "ticarcillin/clavulanic acid":"Penicillin Combination",

    # Sulbactam Combination
    "cefoperazone/sulbactam":"Cephalosporin Combination",
    "ceftolozane/tazobactam":"Cephalosporin Combination",
    "ceftazidime/avibactam":"Cephalosporin Combination",
    "cefepime_taniborbactam":"Cephalosporin Combination",

    # Cephalosporins
    "cefazolin":"Cephalosporin",
    "cephalexin":"Cephalosporin",
    "cephalothin":"Cephalosporin",
    "cefotaxime":"Cephalosporin",
    "ceftriaxone":"Cephalosporin",
    "cefuroxime":"Cephalosporin",
    "cefepime":"Cephalosporin",
    "ceftazidime":"Cephalosporin",
    "cefotetan":"Cephalosporin",
    "cefoxitin":"Cephalosporin",
    "cefpodoxime":"Cephalosporin",
    "cefmetazole":"Cephalosporin",
    "cefquinome":"Cephalosporin",
    "ceftaroline":"Cephalosporin",
    "ceftiofur":"Cephalosporin",
    "ceftibuten":"Cephalosporin",
    "cefozopran":"Cephalosporin"

}

print("\n")
print("=" * 60)
print("BLOCK 05 PART 1 COMPLETED")
print("=" * 60)

In [ ]:
print(sorted(qualified_df["Antibiotic"].unique()))

In [ ]:
# ============================================================
# BLOCK 05
# PART 2A
# Complete Clinical Antibiotic Class Mapping
# ============================================================

# ============================================================
# Remove Non-Antibiotic Entry
# ============================================================

qualified_df = qualified_df[
    qualified_df["Antibiotic"] != "beta-lactam"
].copy()

print("\n")
print("=" * 60)
print("AFTER REMOVING NON-ANTIBIOTIC ENTRY")
print("=" * 60)

print(
    "Unique Antibiotics :",
    qualified_df["Antibiotic"].nunique()
)

# ============================================================
# Complete Clinical Mapping
# ============================================================

antibiotic_class_map = {

    # --------------------------------------------------------
    # Penicillins
    # --------------------------------------------------------

    "amoxicillin":"Penicillin",
    "ampicillin":"Penicillin",
    "carbenicillin":"Penicillin",
    "penicillin":"Penicillin",
    "piperacillin":"Penicillin",
    "ticarcillin":"Penicillin",
    "temocillin":"Penicillin",

    # --------------------------------------------------------
    # Beta-lactam Combination
    # --------------------------------------------------------

    "amoxicillin/clavulanic acid":
        "Beta-Lactam Combination",

    "ampicillin/sulbactam":
        "Beta-Lactam Combination",

    "piperacillin/tazobactam":
        "Beta-Lactam Combination",

    "ticarcillin/clavulanic acid":
        "Beta-Lactam Combination",

    "cefoperazone/sulbactam":
        "Beta-Lactam Combination",

    "ceftazidime/avibactam":
        "Beta-Lactam Combination",

    "ceftolozane/tazobactam":
        "Beta-Lactam Combination",

    "cefepime_taniborbactam":
        "Beta-Lactam Combination",

    "cefotaxime/clavulanic acid":
        "Beta-Lactam Combination",

    "cefpodoxime_clavulanic_acid":
        "Beta-Lactam Combination",

    "ceftazidime/clavulanic acid":
        "Beta-Lactam Combination",

    # --------------------------------------------------------
    # Cephalosporins
    # --------------------------------------------------------

    "cefazolin":"Cephalosporin",
    "cefepime":"Cephalosporin",
    "cefotaxime":"Cephalosporin",
    "cefotetan":"Cephalosporin",
    "cefoxitin":"Cephalosporin",
    "cefpodoxime":"Cephalosporin",
    "ceftaroline":"Cephalosporin",
    "ceftazidime":"Cephalosporin",
    "ceftiofur":"Cephalosporin",
    "ceftriaxone":"Cephalosporin",
    "cefuroxime":"Cephalosporin",
    "cephalexin":"Cephalosporin",
    "cephalothin":"Cephalosporin",

    # --------------------------------------------------------
    # Carbapenems
    # --------------------------------------------------------

    "imipenem":"Carbapenem",
    "meropenem":"Carbapenem",
    "ertapenem":"Carbapenem",
    "doripenem":"Carbapenem",

    # --------------------------------------------------------
    # Monobactam
    # --------------------------------------------------------

    "aztreonam":"Monobactam",

    # --------------------------------------------------------
    # Aminoglycosides
    # --------------------------------------------------------

    "amikacin":"Aminoglycoside",
    "gentamicin":"Aminoglycoside",
    "kanamycin":"Aminoglycoside",
    "neomycin":"Aminoglycoside",
    "netilmicin":"Aminoglycoside",
    "streptomycin":"Aminoglycoside",
    "spectinomycin":"Aminoglycoside",
    "tobramycin":"Aminoglycoside"

}

qualified_df["Antibiotic_Class"] = (
    qualified_df["Antibiotic"]
    .map(antibiotic_class_map)
)

print("\n")
print("=" * 60)
print("CURRENTLY MAPPED")
print("=" * 60)

print(
    qualified_df["Antibiotic_Class"]
    .value_counts(dropna=False)
)

print("\n")
print("=" * 60)
print("BLOCK 05 PART 2A COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 05
# PART 2B
# Remaining Clinical Class Mapping + Validation
# ============================================================

remaining_mapping = {

    # Fluoroquinolones
    "ciprofloxacin": "Fluoroquinolone",
    "levofloxacin": "Fluoroquinolone",
    "moxifloxacin": "Fluoroquinolone",
    "norfloxacin": "Fluoroquinolone",
    "ofloxacin": "Fluoroquinolone",
    "delafloxacin": "Fluoroquinolone",
    "nalidixic acid": "Quinolone",

    # Tetracyclines
    "doxycycline": "Tetracycline",
    "minocycline": "Tetracycline",
    "oxytetracycline": "Tetracycline",
    "tetracycline": "Tetracycline",
    "tigecycline": "Glycylcycline",

    # Phenicols
    "chloramphenicol": "Phenicol",

    # Phosphonic Acid
    "fosfomycin": "Phosphonic Acid",

    # Nitrofurans
    "nitrofurantoin": "Nitrofuran",

    # Polymyxins
    "colistin": "Polymyxin",
    "polymyxin b": "Polymyxin",

    # Sulfonamides
    "sulfamethoxazole": "Sulfonamide",
    "sulfisoxazole": "Sulfonamide",

    # Trimethoprim
    "trimethoprim": "Trimethoprim",
    "trimethoprim/sulfamethoxazole": "Folate Pathway Inhibitor",

    # Macrolides
    "azithromycin": "Macrolide",
    "clarithromycin": "Macrolide",
    "erythromycin": "Macrolide",

    # Lincosamides
    "clindamycin": "Lincosamide",
    "lincomycin": "Lincosamide",

    # Glycopeptides
    "vancomycin": "Glycopeptide",
    "teicoplanin": "Glycopeptide",

    # Oxazolidinones
    "linezolid": "Oxazolidinone",

    # Lipopeptides
    "daptomycin": "Lipopeptide",

    # Rifamycins
    "rifampin": "Rifamycin",

    # Miscellaneous
    "metronidazole": "Nitroimidazole",
    "oxacillin": "Penicillin",
    "sulbactam": "Beta-Lactamase Inhibitor"
}

# Merge previous mapping with remaining mapping
antibiotic_class_map.update(remaining_mapping)

# Apply complete mapping
qualified_df["Antibiotic_Class"] = (
    qualified_df["Antibiotic"]
    .map(antibiotic_class_map)
)

print("\n")
print("=" * 60)
print("FINAL CLASS DISTRIBUTION")
print("=" * 60)

print(
    qualified_df["Antibiotic_Class"]
    .value_counts(dropna=False)
)

# ============================================================
# Unmapped Antibiotics
# ============================================================

unmapped = sorted(
    qualified_df.loc[
        qualified_df["Antibiotic_Class"].isna(),
        "Antibiotic"
    ].unique()
)

print("\n")
print("=" * 60)
print("UNMAPPED ANTIBIOTICS")
print("=" * 60)

print("Count :", len(unmapped))

for antibiotic in unmapped:
    print("-", antibiotic)

pd.DataFrame({
    "Antibiotic": unmapped
}).to_csv(
    os.path.join(
        MDR_DIR,
        "unmapped_antibiotics.csv"
    ),
    index=False
)

print("\n")
print("=" * 60)
print("BLOCK 05 PART 2B COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 05
# PART 3
# Genome-Level MDR Dataset Generation
# ============================================================

print("\n")
print("=" * 60)
print("CREATING GENOME-LEVEL MDR DATASET")
print("=" * 60)

# ============================================================
# Resistant Records Only
# ============================================================

resistant_df = qualified_df[
    qualified_df["Resistance_Binary"] == 1
].copy()

# ============================================================
# Resistant Class Count Per Genome
# ============================================================

genome_class_count = (

    resistant_df

    .groupby("Genome ID")["Antibiotic_Class"]

    .nunique()

    .reset_index()

)

genome_class_count.columns = [

    "Genome ID",

    "Resistant_Class_Count"

]

# ============================================================
# Genome × Antibiotic Matrix
# ============================================================

genome_matrix = pd.pivot_table(

    qualified_df,

    index="Genome ID",

    columns="Antibiotic",

    values="Resistance_Binary",

    aggfunc="max",

    fill_value=0,

    observed=False

)

genome_matrix = genome_matrix.reset_index()

print(
    "Genome Matrix Shape :",
    genome_matrix.shape
)

# ============================================================
# Merge Resistant Class Count
# ============================================================

genome_mdr = genome_matrix.merge(

    genome_class_count,

    on="Genome ID",

    how="left"

)

genome_mdr["Resistant_Class_Count"] = (

    genome_mdr["Resistant_Class_Count"]

    .fillna(0)

    .astype(int)

)

# ============================================================
# MDR Label
# ============================================================

genome_mdr["MDR_Label"] = np.where(

    genome_mdr["Resistant_Class_Count"] >= 3,

    1,

    0

)

# ============================================================
# MDR Distribution
# ============================================================

print("\n")
print("=" * 60)
print("MDR DISTRIBUTION")
print("=" * 60)

print(
    genome_mdr["MDR_Label"]
    .value_counts()
)

# ============================================================
# Resistant Class Statistics
# ============================================================

print("\n")
print("=" * 60)
print("RESISTANT CLASS STATISTICS")
print("=" * 60)

print(
    genome_mdr["Resistant_Class_Count"]
    .describe()
)

# ============================================================
# Top MDR Genomes
# ============================================================

print("\n")
print("=" * 60)
print("TOP MDR GENOMES")
print("=" * 60)

print(

    genome_mdr[

        [

            "Genome ID",

            "Resistant_Class_Count",

            "MDR_Label"

        ]

    ]

    .sort_values(

        by="Resistant_Class_Count",

        ascending=False

    )

    .head(10)

)

# ============================================================
# MDR Statistics Table
# ============================================================

mdr_statistics = pd.DataFrame({

    "Metric":[

        "Total Genomes",

        "MDR Genomes",

        "Non-MDR Genomes",

        "MDR Percentage",

        "Average Resistant Classes",

        "Maximum Resistant Classes"

    ],

    "Value":[

        len(genome_mdr),

        int(
            (genome_mdr["MDR_Label"] == 1).sum()
        ),

        int(
            (genome_mdr["MDR_Label"] == 0).sum()
        ),

        round(
            genome_mdr["MDR_Label"].mean()*100,
            2
        ),

        round(
            genome_mdr["Resistant_Class_Count"].mean(),
            2
        ),

        int(
            genome_mdr["Resistant_Class_Count"].max()
        )

    ]

})

# ============================================================
# Save Outputs
# ============================================================

genome_mdr.to_csv(

    os.path.join(

        MDR_DIR,

        "genome_mdr_dataset.csv"

    ),

    index=False

)

mdr_statistics.to_csv(

    os.path.join(

        MDR_DIR,

        "mdr_statistics.csv"

    ),

    index=False

)

# ============================================================
# Validation
# ============================================================

print("\n")
print("=" * 60)
print("VALIDATION")
print("=" * 60)

print(
    "Genome Matrix Rows :",
    len(genome_matrix)
)

print(
    "Genome MDR Rows    :",
    len(genome_mdr)
)

print(
    "Unique Genomes     :",
    genome_mdr["Genome ID"].nunique()
)

# ============================================================
# Generated Files
# ============================================================

print("\n")
print("=" * 60)
print("GENERATED FILES")
print("=" * 60)

print("genome_mdr_dataset.csv")
print("mdr_statistics.csv")
print("unmapped_antibiotics.csv")

# ============================================================
# BLOCK 05 COMPLETED
# ============================================================

print("\n")
print("=" * 60)
print("BLOCK 05 COMPLETED SUCCESSFULLY")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 06
# Resistance Severity Modeling
# PART 1 (FINAL JOURNAL VERSION)
# ============================================================

print("=" * 60)
print("BLOCK 06 STARTED")
print("=" * 60)

# ============================================================
# Output Directory
# ============================================================

SEVERITY_DIR = os.path.join(
    OUTPUT_DIR,
    "severity"
)

os.makedirs(
    SEVERITY_DIR,
    exist_ok=True
)

# ============================================================
# Working Copy
# ============================================================

severity_df = genome_mdr.copy()

# ============================================================
# Resistant Antibiotic Count
# ============================================================

antibiotic_columns = [

    col

    for col in severity_df.columns

    if col not in [

        "Genome ID",

        "Resistant_Class_Count",

        "MDR_Label"

    ]

]

severity_df["Resistant_Antibiotic_Count"] = (

    severity_df[
        antibiotic_columns
    ]

    .sum(axis=1)

)

print("\n")
print("=" * 60)
print("RESISTANT ANTIBIOTIC COUNT")
print("=" * 60)

print(

    severity_df[
        "Resistant_Antibiotic_Count"
    ].describe()

)

# ============================================================
# Resistant Class Count
# ============================================================

print("\n")
print("=" * 60)
print("RESISTANT CLASS COUNT")
print("=" * 60)

print(

    severity_df[
        "Resistant_Class_Count"
    ].describe()

)

# ============================================================
# Data-Driven Severity Index
# ============================================================

severity_df["Severity_Index"] = (

    severity_df["Resistant_Antibiotic_Count"]

    +

    severity_df["Resistant_Class_Count"]

)

print("\n")
print("=" * 60)
print("SEVERITY INDEX")
print("=" * 60)

print(

    severity_df[
        "Severity_Index"
    ].describe()

)

# ============================================================
# Percentile Thresholds
# ============================================================

q1 = severity_df["Severity_Index"].quantile(0.25)
q2 = severity_df["Severity_Index"].quantile(0.50)
q3 = severity_df["Severity_Index"].quantile(0.75)

print("\n")
print("=" * 60)
print("SEVERITY THRESHOLDS")
print("=" * 60)

print(f"25th Percentile : {q1}")
print(f"50th Percentile : {q2}")
print(f"75th Percentile : {q3}")

# ============================================================
# Severity Label Function
# ============================================================

def assign_severity(index):

    if index <= q1:
        return "Low"

    elif index <= q2:
        return "Moderate"

    elif index <= q3:
        return "High"

    else:
        return "Critical"

# ============================================================
# Generate Severity Labels
# ============================================================

severity_df["Severity_Label"] = (

    severity_df["Severity_Index"]

    .apply(assign_severity)

)

print("\n")
print("=" * 60)
print("SEVERITY DISTRIBUTION")
print("=" * 60)

print(

    severity_df[
        "Severity_Label"
    ].value_counts()

)

print("\n")
print("=" * 60)
print("BLOCK 06 PART 1 COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 06
# Resistance Severity Modeling
# PART 2 (FINAL JOURNAL VERSION)
# ============================================================

# ============================================================
# Severity Percentage
# ============================================================

print("\n")
print("=" * 60)
print("SEVERITY PERCENTAGE")
print("=" * 60)

severity_percentage = round(

    severity_df["Severity_Label"]

    .value_counts(normalize=True)

    .sort_index()

    * 100,

    2

)

print(severity_percentage)

severity_percentage.to_csv(

    os.path.join(
        SEVERITY_DIR,
        "severity_percentage.csv"
    )

)

# ============================================================
# Severity Statistics
# ============================================================

severity_statistics = pd.DataFrame({

    "Metric":[

        "Total Genomes",

        "Low",

        "Moderate",

        "High",

        "Critical",

        "25th Percentile",

        "50th Percentile",

        "75th Percentile",

        "Maximum Severity Index",

        "Average Severity Index"

    ],

    "Value":[

        len(severity_df),

        (severity_df["Severity_Label"]=="Low").sum(),

        (severity_df["Severity_Label"]=="Moderate").sum(),

        (severity_df["Severity_Label"]=="High").sum(),

        (severity_df["Severity_Label"]=="Critical").sum(),

        q1,

        q2,

        q3,

        severity_df["Severity_Index"].max(),

        round(
            severity_df["Severity_Index"].mean(),
            2
        )

    ]

})

severity_statistics.to_csv(

    os.path.join(

        SEVERITY_DIR,

        "severity_statistics.csv"

    ),

    index=False

)

# ============================================================
# Top Critical Genomes
# ============================================================

critical_genomes = (

    severity_df

    .sort_values(

        by="Severity_Index",

        ascending=False

    )

)

critical_genomes.head(20).to_csv(

    os.path.join(

        SEVERITY_DIR,

        "top_critical_genomes.csv"

    ),

    index=False

)

print("\n")
print("=" * 60)
print("TOP CRITICAL GENOMES")
print("=" * 60)

print(

    critical_genomes[

        [

            "Genome ID",

            "Resistant_Antibiotic_Count",

            "Resistant_Class_Count",

            "Severity_Index",

            "Severity_Label",

            "MDR_Label"

        ]

    ].head(10)

)

# ============================================================
# Severity Distribution Figure
# ============================================================

plt.figure(figsize=(8,5))

sns.countplot(

    data=severity_df,

    x="Severity_Label",

    order=[

        "Low",

        "Moderate",

        "High",

        "Critical"

    ]

)

plt.title(
    "Resistance Severity Distribution"
)

plt.xlabel(
    "Severity Class"
)

plt.ylabel(
    "Genome Count"
)

plt.tight_layout()

plt.savefig(

    os.path.join(

        SEVERITY_DIR,

        "severity_distribution.png"

    ),

    dpi=300,

    bbox_inches="tight"

)

plt.show()
plt.close()

# ============================================================
# Severity Index Distribution
# ============================================================

plt.figure(figsize=(8,5))

sns.histplot(

    severity_df["Severity_Index"],

    bins=30,

    kde=True

)

plt.title(
    "Severity Index Distribution"
)

plt.xlabel(
    "Severity Index"
)

plt.ylabel(
    "Frequency"
)

plt.tight_layout()

plt.savefig(

    os.path.join(

        SEVERITY_DIR,

        "severity_index_distribution.png"

    ),

    dpi=300,

    bbox_inches="tight"

)

plt.show()
plt.close()

# ============================================================
# Validation
# ============================================================

validation_report = pd.DataFrame({

    "Validation":[

        "Total Genomes",

        "Unique Genomes",

        "Missing Severity Label",

        "Missing Severity Index"

    ],

    "Result":[

        len(severity_df),

        severity_df["Genome ID"].nunique(),

        severity_df["Severity_Label"].isna().sum(),

        severity_df["Severity_Index"].isna().sum()

    ]

})

validation_report.to_csv(

    os.path.join(

        SEVERITY_DIR,

        "severity_validation_report.csv"

    ),

    index=False

)

# ============================================================
# Save Final Dataset
# ============================================================

severity_df.to_csv(

    os.path.join(

        SEVERITY_DIR,

        "genome_severity_dataset.csv"

    ),

    index=False

)

# ============================================================
# Generated Files
# ============================================================

print("\n")
print("=" * 60)
print("GENERATED FILES")
print("=" * 60)

print("genome_severity_dataset.csv")
print("severity_statistics.csv")
print("severity_percentage.csv")
print("top_critical_genomes.csv")
print("severity_validation_report.csv")
print("severity_distribution.png")
print("severity_index_distribution.png")

# ============================================================
# BLOCK 06 COMPLETED
# ============================================================

print("\n")
print("=" * 60)
print("BLOCK 06 COMPLETED SUCCESSFULLY")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 07
# Advanced Feature Engineering
# PART 1 (FINAL JOURNAL VERSION)
# ============================================================

print("=" * 60)
print("BLOCK 07 STARTED")
print("=" * 60)

# ============================================================
# Output Directory
# ============================================================

FEATURE_DIR = os.path.join(
    OUTPUT_DIR,
    "features"
)

os.makedirs(
    FEATURE_DIR,
    exist_ok=True
)

# ============================================================
# Working Copy
# ============================================================

feature_df = qualified_df.copy()

# ============================================================
# Genome Coverage Features
# ============================================================

print("\n")
print("=" * 60)
print("GENOME COVERAGE FEATURES")
print("=" * 60)

coverage_features = (

    feature_df

    .groupby("Genome ID")

    .agg(

        Antibiotics_Tested=("Antibiotic", "nunique"),

        Total_Records=("Antibiotic", "count")

    )

    .reset_index()

)

coverage_features["Coverage_Ratio"] = (

    coverage_features["Antibiotics_Tested"]

    /

    qualified_df["Antibiotic"].nunique()

)

print(coverage_features.head())

# ============================================================
# MIC Features
# ============================================================

print("\n")
print("=" * 60)
print("MIC FEATURES")
print("=" * 60)

mic_features = (

    feature_df

    .groupby("Genome ID")

    .agg(

        Mean_MIC=("MIC_Numeric", "mean"),

        Median_MIC=("MIC_Numeric", "median"),

        Max_MIC=("MIC_Numeric", "max"),

        Min_MIC=("MIC_Numeric", "min"),

        Std_MIC=("MIC_Numeric", "std")

    )

    .reset_index()

)

mic_features["MIC_Range"] = (

    mic_features["Max_MIC"]

    -

    mic_features["Min_MIC"]

)

mic_features["Log_Mean_MIC"] = np.log1p(

    mic_features["Mean_MIC"]

)

print(mic_features.head())

# ============================================================
# Merge Coverage + MIC
# ============================================================

feature_matrix = coverage_features.merge(

    mic_features,

    on="Genome ID",

    how="left"

)

# ============================================================
# Missing MIC Features
# ============================================================

mic_columns = [

    "Mean_MIC",

    "Median_MIC",

    "Max_MIC",

    "Min_MIC",

    "Std_MIC",

    "MIC_Range",

    "Log_Mean_MIC"

]

feature_matrix[mic_columns] = (

    feature_matrix[mic_columns]

    .fillna(0)

)

# ============================================================
# Feature Matrix Preview
# ============================================================

print("\n")
print("=" * 60)
print("FEATURE MATRIX")
print("=" * 60)

print(feature_matrix.head())

print("\n")
print("Shape :", feature_matrix.shape)

print("\n")
print("=" * 60)
print("BLOCK 07 PART 1 COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 07
# PART 2A
# Laboratory Feature Engineering
# ============================================================

print("\n")
print("=" * 60)
print("LABORATORY FEATURE ENGINEERING")
print("=" * 60)

# ============================================================
# CLSI Count
# ============================================================

clsi_count = (

    qualified_df[
        qualified_df["Testing Standard"] == "CLSI"
    ]

    .groupby("Genome ID")

    .size()

    .reset_index(name="CLSI_Count")

)

# ============================================================
# EUCAST Count
# ============================================================

eucast_count = (

    qualified_df[
        qualified_df["Testing Standard"] == "EUCAST"
    ]

    .groupby("Genome ID")

    .size()

    .reset_index(name="EUCAST_Count")

)

# ============================================================
# Broth Dilution Count
# ============================================================

broth_count = (

    qualified_df[
        qualified_df["Laboratory Typing Method"] == "Broth Dilution"
    ]

    .groupby("Genome ID")

    .size()

    .reset_index(name="Broth_Dilution_Count")

)

# ============================================================
# Disk Diffusion Count
# ============================================================

disk_count = (

    qualified_df[
        qualified_df["Laboratory Typing Method"] == "Disk Diffusion"
    ]

    .groupby("Genome ID")

    .size()

    .reset_index(name="Disk_Diffusion_Count")

)

# ============================================================
# Agar Dilution Count
# ============================================================

agar_count = (

    qualified_df[
        qualified_df["Laboratory Typing Method"] == "Agar Dilution"
    ]

    .groupby("Genome ID")

    .size()

    .reset_index(name="Agar_Dilution_Count")

)

# ============================================================
# MIC Method Count
# ============================================================

mic_method_count = (

    qualified_df[
        qualified_df["Laboratory Typing Method"] == "Mic"
    ]

    .groupby("Genome ID")

    .size()

    .reset_index(name="MIC_Method_Count")

)

# ============================================================
# Merge Laboratory Features
# ============================================================

lab_features = coverage_features[["Genome ID"]].copy()

for temp_df in [

    clsi_count,
    eucast_count,
    broth_count,
    disk_count,
    agar_count,
    mic_method_count

]:

    lab_features = lab_features.merge(

        temp_df,

        on="Genome ID",

        how="left"

    )

lab_features = lab_features.fillna(0)

feature_matrix = feature_matrix.merge(

    lab_features,

    on="Genome ID",

    how="left"

)

lab_columns = [

    "CLSI_Count",
    "EUCAST_Count",
    "Broth_Dilution_Count",
    "Disk_Diffusion_Count",
    "Agar_Dilution_Count",
    "MIC_Method_Count"

]

feature_matrix[lab_columns] = (

    feature_matrix[lab_columns]

    .fillna(0)

    .astype(int)

)

print(feature_matrix[lab_columns].describe())

print("\n")
print("=" * 60)
print("BLOCK 07 PART 2A COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 07
# PART 2B
# Missingness Feature Engineering
# ============================================================

print("\n")
print("=" * 60)
print("MISSINGNESS FEATURE ENGINEERING")
print("=" * 60)

# ============================================================
# Missing MIC Ratio
# ============================================================

missing_features = (

    qualified_df

    .groupby("Genome ID")

    .agg(

        Missing_MIC_Ratio=(

            "MIC_Numeric",

            lambda x: x.isna().mean()

        ),

        Missing_Testing_Standard_Ratio=(

            "Testing Standard",

            lambda x: x.isna().mean()

        ),

        Missing_Laboratory_Method_Ratio=(

            "Laboratory Typing Method",

            lambda x: x.isna().mean()

        )

    )

    .reset_index()

)

# ============================================================
# Merge Missingness Features
# ============================================================

feature_matrix = feature_matrix.merge(

    missing_features,

    on="Genome ID",

    how="left"

)

missing_cols = [

    "Missing_MIC_Ratio",

    "Missing_Testing_Standard_Ratio",

    "Missing_Laboratory_Method_Ratio"

]

feature_matrix[missing_cols] = (

    feature_matrix[missing_cols]

    .fillna(0)

)

# ============================================================
# Summary
# ============================================================

print("\n")
print("=" * 60)
print("MISSING FEATURE SUMMARY")
print("=" * 60)

print(

    feature_matrix[missing_cols]

    .describe()

)

# ============================================================
# Missing Value Check
# ============================================================

print("\n")
print("=" * 60)
print("FEATURE MATRIX MISSING VALUES")
print("=" * 60)

missing_report = (

    feature_matrix

    .isna()

    .sum()

    .sort_values(

        ascending=False

    )

)

print(missing_report)

print("\n")
print("=" * 60)
print("BLOCK 07 PART 2B COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 07
# PART 2C
# Feature Quality Assessment
# ============================================================

print("\n")
print("=" * 60)
print("FEATURE QUALITY ASSESSMENT")
print("=" * 60)

# ============================================================
# Constant Feature Detection
# ============================================================

constant_features = [

    col

    for col in feature_matrix.columns

    if feature_matrix[col].nunique() <= 1

]

print("\nConstant Features")

print("-" * 40)

if len(constant_features) == 0:

    print("No Constant Features Found")

else:

    print(constant_features)

# ============================================================
# Low Variance Detection
# ============================================================

numeric_cols = feature_matrix.select_dtypes(

    include=["number"]

).columns

variance_table = pd.DataFrame({

    "Feature": numeric_cols,

    "Variance": [

        feature_matrix[col].var()

        for col in numeric_cols

    ]

})

low_variance = variance_table[

    variance_table["Variance"] < 0.01

]

print("\n")
print("=" * 60)
print("LOW VARIANCE FEATURES")
print("=" * 60)

if len(low_variance) == 0:

    print("No Low Variance Features")

else:

    print(low_variance)

# ============================================================
# Feature Summary
# ============================================================

feature_summary = pd.DataFrame({

    "Feature": feature_matrix.columns,

    "Missing": feature_matrix.isnull().sum().values,

    "Unique": [

        feature_matrix[col].nunique()

        for col in feature_matrix.columns

    ],

    "Datatype": feature_matrix.dtypes.values

})

print("\n")
print("=" * 60)
print("FEATURE SUMMARY")
print("=" * 60)

print(feature_summary)

# ============================================================
# Save Reports
# ============================================================

variance_table.to_csv(

    os.path.join(

        FEATURE_DIR,

        "feature_variance.csv"

    ),

    index=False

)

feature_summary.to_csv(

    os.path.join(

        FEATURE_DIR,

        "feature_summary.csv"

    ),

    index=False

)

# ============================================================
# BLOCK 07 PART 2C COMPLETED
# ============================================================

print("\n")
print("=" * 60)
print("BLOCK 07 PART 2C COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 07
# PART 3
# Correlation Analysis + Final Export + Validation
# FINAL JOURNAL VERSION
# ============================================================

print("\n")
print("=" * 60)
print("FINAL FEATURE VALIDATION")
print("=" * 60)

# ============================================================
# Correlation Matrix
# ============================================================

numeric_features = feature_matrix.select_dtypes(
    include=["number"]
)

corr_matrix = numeric_features.corr()

print("\n")
print("=" * 60)
print("CORRELATION MATRIX")
print("=" * 60)

print(corr_matrix.round(3))

# ============================================================
# Correlation Heatmap
# ============================================================

plt.figure(figsize=(14,10))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    square=True
)

plt.title(
    "Feature Correlation Matrix"
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        FEATURE_DIR,
        "feature_correlation_heatmap.png"
    ),

    dpi=300,

    bbox_inches="tight"

)

plt.show()
plt.close()

# ============================================================
# Merge Targets
# ============================================================

target_df = severity_df[

    [

        "Genome ID",

        "MDR_Label",

        "Severity_Label"

    ]

].copy()

target_df = target_df.drop_duplicates()

# ============================================================
# Validation
# ============================================================

print("\n")
print("=" * 60)
print("DATASET VALIDATION")
print("=" * 60)

print(
    "Feature Rows :",
    len(feature_matrix)
)

print(
    "Target Rows  :",
    len(target_df)
)

print(
    "Unique Feature Genomes :",
    feature_matrix["Genome ID"].nunique()
)

print(
    "Unique Target Genomes  :",
    target_df["Genome ID"].nunique()
)

assert len(feature_matrix) == len(target_df)

assert (

    feature_matrix["Genome ID"]

    .nunique()

    ==

    target_df["Genome ID"]

    .nunique()

)

print("\nValidation Passed")

# ============================================================
# Feature Validation Report
# ============================================================

validation_report = pd.DataFrame({

    "Metric":[

        "Feature Rows",

        "Feature Columns",

        "Unique Genomes",

        "Missing Values",

        "Constant Features"

    ],

    "Value":[

        len(feature_matrix),

        len(feature_matrix.columns),

        feature_matrix["Genome ID"].nunique(),

        int(feature_matrix.isnull().sum().sum()),

        len(constant_features)

    ]

})

validation_report.to_csv(

    os.path.join(

        FEATURE_DIR,

        "feature_validation_report.csv"

    ),

    index=False

)

# ============================================================
# Save Final Datasets
# ============================================================

feature_matrix.to_csv(

    os.path.join(

        FEATURE_DIR,

        "feature_dataset.csv"

    ),

    index=False

)

target_df.to_csv(

    os.path.join(

        FEATURE_DIR,

        "target_dataset.csv"

    ),

    index=False

)

# ============================================================
# Generated Files
# ============================================================

print("\n")
print("=" * 60)
print("GENERATED FILES")
print("=" * 60)

print("feature_dataset.csv")
print("target_dataset.csv")
print("feature_summary.csv")
print("feature_variance.csv")
print("feature_validation_report.csv")
print("feature_correlation_heatmap.png")

# ============================================================
# BLOCK 07 COMPLETED
# ============================================================

print("\n")
print("=" * 60)
print("BLOCK 07 COMPLETED SUCCESSFULLY")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 08
# PART 1
# Dataset Preparation + Validation + Stratified Split
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 60)
print("BLOCK 08 STARTED")
print("=" * 60)

from sklearn.model_selection import train_test_split

SPLIT_DIR = os.path.join(
    OUTPUT_DIR,
    "splits"
)

os.makedirs(
    SPLIT_DIR,
    exist_ok=True
)

# ============================================================
# Load Final Dataset
# ============================================================

model_df = feature_matrix.merge(

    target_df,

    on="Genome ID",

    how="inner"

)

print("\n")
print("=" * 60)
print("MERGED DATASET")
print("=" * 60)

print("Rows    :", len(model_df))
print("Columns :", len(model_df.columns))

# ============================================================
# Validation
# ============================================================

assert len(model_df) == feature_matrix.shape[0]

assert model_df["Genome ID"].nunique() == len(model_df)

print("Dataset Validation Passed")

# ============================================================
# Feature & Target Selection
# ============================================================

feature_columns = [

    col

    for col in model_df.columns

    if col not in [

        "Genome ID",

        "MDR_Label",

        "Severity_Label"

    ]

]

X = model_df[feature_columns]

y_mdr = model_df["MDR_Label"]

y_severity = model_df["Severity_Label"]

print("\n")
print("=" * 60)
print("FEATURE INFORMATION")
print("=" * 60)

print("Total Features :", len(feature_columns))

# ============================================================
# Missing Value Validation
# ============================================================

print("\n")
print("=" * 60)
print("MISSING VALUE VALIDATION")
print("=" * 60)

print(

    X.isnull()

    .sum()

    .sort_values(

        ascending=False

    )

)

# ============================================================
# Numeric Validation
# ============================================================

non_numeric = X.select_dtypes(

    exclude=["number"]

).columns.tolist()

print("\n")
print("=" * 60)
print("NON-NUMERIC FEATURES")
print("=" * 60)

if len(non_numeric) == 0:

    print("No Non-Numeric Features")

else:

    print(non_numeric)

# ============================================================
# Target Distribution
# ============================================================

print("\n")
print("=" * 60)
print("MDR DISTRIBUTION")
print("=" * 60)

print(y_mdr.value_counts())

print("\n")
print("=" * 60)
print("SEVERITY DISTRIBUTION")
print("=" * 60)

print(y_severity.value_counts())

# ============================================================
# Stratified Split
# ============================================================

stratify_target = (

    y_mdr.astype(str)

    +

    "_"

    +

    y_severity.astype(str)

)

X_train, X_temp, y_train, y_temp = train_test_split(

    X,

    model_df[["MDR_Label", "Severity_Label"]],

    test_size=0.30,

    random_state=SEED,

    stratify=stratify_target

)

temp_stratify = (

    y_temp["MDR_Label"].astype(str)

    +

    "_"

    +

    y_temp["Severity_Label"].astype(str)

)

X_val, X_test, y_val, y_test = train_test_split(

    X_temp,

    y_temp,

    test_size=0.50,

    random_state=SEED,

    stratify=temp_stratify

)

print("\n")
print("=" * 60)
print("DATASET SPLITS")
print("=" * 60)

print("Train      :", X_train.shape)

print("Validation :", X_val.shape)

print("Test       :", X_test.shape)

print("\n")
print("=" * 60)
print("BLOCK 08 PART 1 COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 08
# PART 2
# Feature Scaling + Dataset Transformation
# FINAL JOURNAL VERSION
# ============================================================

print("\n")
print("=" * 60)
print("FEATURE SCALING")
print("=" * 60)

from sklearn.preprocessing import StandardScaler
import joblib

# ============================================================
# Initialize Scaler
# ============================================================

scaler = StandardScaler()

# ============================================================
# Fit on Train Only
# ============================================================

X_train_scaled = pd.DataFrame(

    scaler.fit_transform(X_train),

    columns=X_train.columns,

    index=X_train.index

)

# ============================================================
# Validation Transform
# ============================================================

X_val_scaled = pd.DataFrame(

    scaler.transform(X_val),

    columns=X_val.columns,

    index=X_val.index

)

# ============================================================
# Test Transform
# ============================================================

X_test_scaled = pd.DataFrame(

    scaler.transform(X_test),

    columns=X_test.columns,

    index=X_test.index

)

# ============================================================
# Validation
# ============================================================

print("\n")
print("=" * 60)
print("SCALED DATASET SHAPES")
print("=" * 60)

print("Train      :", X_train_scaled.shape)
print("Validation :", X_val_scaled.shape)
print("Test       :", X_test_scaled.shape)

# ============================================================
# Mean / Std Check
# ============================================================

print("\n")
print("=" * 60)
print("TRAIN SCALING CHECK")
print("=" * 60)

print("Average Mean :",
      round(X_train_scaled.mean().mean(),6))

print("Average Std  :",
      round(X_train_scaled.std().mean(),6))

# ============================================================
# Save Scaler
# ============================================================

joblib.dump(

    scaler,

    os.path.join(

        SPLIT_DIR,

        "standard_scaler.pkl"

    )

)

# ============================================================
# Save Scaled Features
# ============================================================

X_train_scaled.to_csv(

    os.path.join(

        SPLIT_DIR,

        "X_train_scaled.csv"

    ),

    index=False

)

X_val_scaled.to_csv(

    os.path.join(

        SPLIT_DIR,

        "X_val_scaled.csv"

    ),

    index=False

)

X_test_scaled.to_csv(

    os.path.join(

        SPLIT_DIR,

        "X_test_scaled.csv"

    ),

    index=False

)

# ============================================================
# Scaling Report
# ============================================================

scaling_report = pd.DataFrame({

    "Dataset":[

        "Train",

        "Validation",

        "Test"

    ],

    "Rows":[

        len(X_train_scaled),

        len(X_val_scaled),

        len(X_test_scaled)

    ],

    "Columns":[

        X_train_scaled.shape[1],

        X_val_scaled.shape[1],

        X_test_scaled.shape[1]

    ]

})

scaling_report.to_csv(

    os.path.join(

        SPLIT_DIR,

        "scaling_report.csv"

    ),

    index=False

)

# ============================================================
# Generated Files
# ============================================================

print("\n")
print("=" * 60)
print("GENERATED FILES")
print("=" * 60)

print("standard_scaler.pkl")
print("X_train_scaled.csv")
print("X_val_scaled.csv")
print("X_test_scaled.csv")
print("scaling_report.csv")

print("\n")
print("=" * 60)
print("BLOCK 08 PART 2 COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 08
# PART 3
# Class Weights + Final Validation + Export
# FINAL JOURNAL VERSION
# ============================================================

print("\n")
print("=" * 60)
print("CLASS WEIGHT CALCULATION")
print("=" * 60)

from sklearn.utils.class_weight import compute_class_weight

# ============================================================
# MDR Class Weight
# ============================================================

mdr_classes = np.sort(
    y_train["MDR_Label"].unique()
)

mdr_weights = compute_class_weight(
    class_weight="balanced",
    classes=mdr_classes,
    y=y_train["MDR_Label"]
)

mdr_class_weight = {

    int(cls): float(weight)

    for cls, weight in zip(

        mdr_classes,

        mdr_weights

    )

}

print("\nMDR Class Weight")

print(mdr_class_weight)

# ============================================================
# Severity Class Weight
# ============================================================

severity_classes = np.sort(
    y_train["Severity_Label"].unique()
)

severity_weights = compute_class_weight(
    class_weight="balanced",
    classes=severity_classes,
    y=y_train["Severity_Label"]
)

severity_class_weight = {

    str(cls): float(weight)

    for cls, weight in zip(

        severity_classes,

        severity_weights

    )

}

print("\nSeverity Class Weight")

print(severity_class_weight)

# ============================================================
# Dataset Integrity
# ============================================================

print("\n")
print("=" * 60)
print("DATASET INTEGRITY")
print("=" * 60)

print("Training Samples   :", len(X_train_scaled))
print("Validation Samples :", len(X_val_scaled))
print("Test Samples       :", len(X_test_scaled))

print("\nFeatures :", X_train_scaled.shape[1])

print("\nTrain MDR")

print(
    y_train["MDR_Label"]
    .value_counts()
)

print("\nValidation MDR")

print(
    y_val["MDR_Label"]
    .value_counts()
)

print("\nTest MDR")

print(
    y_test["MDR_Label"]
    .value_counts()
)

print("\nTrain Severity")

print(
    y_train["Severity_Label"]
    .value_counts()
)

print("\nValidation Severity")

print(
    y_val["Severity_Label"]
    .value_counts()
)

print("\nTest Severity")

print(
    y_test["Severity_Label"]
    .value_counts()
)

# ============================================================
# Save Targets
# ============================================================

y_train.to_csv(

    os.path.join(

        SPLIT_DIR,

        "y_train.csv"

    ),

    index=False

)

y_val.to_csv(

    os.path.join(

        SPLIT_DIR,

        "y_validation.csv"

    ),

    index=False

)

y_test.to_csv(

    os.path.join(

        SPLIT_DIR,

        "y_test.csv"

    ),

    index=False

)

# ============================================================
# Save Feature List
# ============================================================

pd.DataFrame({

    "Feature": X_train_scaled.columns

}).to_csv(

    os.path.join(

        SPLIT_DIR,

        "feature_columns.csv"

    ),

    index=False

)

# ============================================================
# Split Summary
# ============================================================

split_summary = pd.DataFrame({

    "Metric":[

        "Training Samples",

        "Validation Samples",

        "Test Samples",

        "Number of Features"

    ],

    "Value":[

        len(X_train_scaled),

        len(X_val_scaled),

        len(X_test_scaled),

        X_train_scaled.shape[1]

    ]

})

split_summary.to_csv(

    os.path.join(

        SPLIT_DIR,

        "split_summary.csv"

    ),

    index=False

)

# ============================================================
# Validation Report
# ============================================================

validation_report = pd.DataFrame({

    "Metric":[

        "Missing Values (Train)",

        "Missing Values (Validation)",

        "Missing Values (Test)",

        "Duplicate Features"

    ],

    "Value":[

        int(X_train_scaled.isnull().sum().sum()),

        int(X_val_scaled.isnull().sum().sum()),

        int(X_test_scaled.isnull().sum().sum()),

        X_train_scaled.columns.duplicated().sum()

    ]

})

validation_report.to_csv(

    os.path.join(

        SPLIT_DIR,

        "dataset_validation_report.csv"

    ),

    index=False

)

# ============================================================
# Generated Files
# ============================================================

print("\n")
print("=" * 60)
print("GENERATED FILES")
print("=" * 60)

print("y_train.csv")
print("y_validation.csv")
print("y_test.csv")
print("feature_columns.csv")
print("split_summary.csv")
print("dataset_validation_report.csv")

print("\n")
print("=" * 60)
print("BLOCK 08 COMPLETED SUCCESSFULLY")
print("=" * 60)

In [ ]:
import xgboost
import lightgbm
import catboost

print("All Installed")

In [ ]:
# ============================================================
# BLOCK 09
# Baseline Machine Learning Models
# PART 1
# Setup + Model Definitions + Full Evaluation Functions
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 09 STARTED — BASELINE MACHINE LEARNING MODELS")
print("=" * 70)

# ============================================================
# Imports
# ============================================================

import os
import time
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc
)
from sklearn.preprocessing import label_binarize
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Optional model imports
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print("XGBoost not available:", e)

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception as e:
    HAS_LGBM = False
    print("LightGBM not available:", e)

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception as e:
    HAS_CATBOOST = False
    print("CatBoost not available:", e)

# ============================================================
# Output Directories
# ============================================================

BLOCK09_DIR = os.path.join(OUTPUT_DIR, "block09_baseline_models")

MODEL_DIR = os.path.join(BLOCK09_DIR, "models")
METRIC_DIR = os.path.join(BLOCK09_DIR, "metrics")
REPORT_DIR_09 = os.path.join(BLOCK09_DIR, "classification_reports")
PRED_DIR = os.path.join(BLOCK09_DIR, "predictions")
FIG_DIR = os.path.join(BLOCK09_DIR, "figures")

for d in [
    BLOCK09_DIR,
    MODEL_DIR,
    METRIC_DIR,
    REPORT_DIR_09,
    PRED_DIR,
    FIG_DIR
]:
    os.makedirs(d, exist_ok=True)

FIG_DPI = 300
RUN_LEARNING_CURVES = True

# ============================================================
# Target Preparation
# ============================================================

# MDR target
y_train_mdr = y_train["MDR_Label"].astype(int)
y_val_mdr = y_val["MDR_Label"].astype(int)
y_test_mdr = y_test["MDR_Label"].astype(int)

# Severity target encoding
SEVERITY_ORDER = ["Low", "Moderate", "High", "Critical"]
SEVERITY_TO_ID = {label: i for i, label in enumerate(SEVERITY_ORDER)}
ID_TO_SEVERITY = {i: label for label, i in SEVERITY_TO_ID.items()}

y_train_severity = y_train["Severity_Label"].map(SEVERITY_TO_ID).astype(int)
y_val_severity = y_val["Severity_Label"].map(SEVERITY_TO_ID).astype(int)
y_test_severity = y_test["Severity_Label"].map(SEVERITY_TO_ID).astype(int)

# Class names
MDR_CLASS_NAMES = ["Non-MDR", "MDR"]
MDR_CLASSES = [0, 1]

SEVERITY_CLASS_NAMES = SEVERITY_ORDER
SEVERITY_CLASSES = [0, 1, 2, 3]

# ============================================================
# Class Weight Fallback
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

if "mdr_class_weight" not in globals():
    mdr_classes_temp = np.sort(y_train_mdr.unique())
    mdr_weights_temp = compute_class_weight(
        class_weight="balanced",
        classes=mdr_classes_temp,
        y=y_train_mdr
    )
    mdr_class_weight = {
        int(cls): float(w)
        for cls, w in zip(mdr_classes_temp, mdr_weights_temp)
    }

if "severity_class_weight" not in globals():
    severity_classes_temp = np.array(SEVERITY_ORDER)
    severity_weights_temp = compute_class_weight(
        class_weight="balanced",
        classes=severity_classes_temp,
        y=y_train["Severity_Label"]
    )
    severity_class_weight = {
        str(cls): float(w)
        for cls, w in zip(severity_classes_temp, severity_weights_temp)
    }

severity_class_weight_numeric = {
    SEVERITY_TO_ID[k]: float(v)
    for k, v in severity_class_weight.items()
}

# Sample weights
mdr_train_weights = y_train_mdr.map(mdr_class_weight).astype(float).values
severity_train_weights = y_train_severity.map(severity_class_weight_numeric).astype(float).values

print("\nMDR Class Weight:")
print(mdr_class_weight)

print("\nSeverity Class Weight:")
print(severity_class_weight)

# ============================================================
# Baseline Model Definitions
# ============================================================

def get_baseline_models(task_type):

    models = {}

    if task_type == "binary":

        models["LogisticRegression"] = LogisticRegression(
            max_iter=2000,
            random_state=SEED
        )

        models["RandomForest"] = RandomForestClassifier(
            n_estimators=300,
            random_state=SEED,
            n_jobs=-1
        )

        if HAS_XGB:
            models["XGBoost"] = XGBClassifier(
                n_estimators=300,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.90,
                colsample_bytree=0.90,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1
            )

        if HAS_LGBM:
            models["LightGBM"] = LGBMClassifier(
                n_estimators=300,
                learning_rate=0.05,
                objective="binary",
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )

        if HAS_CATBOOST:
            models["CatBoost"] = CatBoostClassifier(
                iterations=300,
                learning_rate=0.05,
                depth=6,
                loss_function="Logloss",
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False
            )

    elif task_type == "multiclass":

        models["LogisticRegression"] = LogisticRegression(
            max_iter=3000,
            random_state=SEED
        )

        models["RandomForest"] = RandomForestClassifier(
            n_estimators=300,
            random_state=SEED,
            n_jobs=-1
        )

        if HAS_XGB:
            models["XGBoost"] = XGBClassifier(
                n_estimators=300,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.90,
                colsample_bytree=0.90,
                objective="multi:softprob",
                num_class=4,
                eval_metric="mlogloss",
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1
            )

        if HAS_LGBM:
            models["LightGBM"] = LGBMClassifier(
                n_estimators=300,
                learning_rate=0.05,
                objective="multiclass",
                num_class=4,
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )

        if HAS_CATBOOST:
            models["CatBoost"] = CatBoostClassifier(
                iterations=300,
                learning_rate=0.05,
                depth=6,
                loss_function="MultiClass",
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False
            )

    return models

# ============================================================
# Utility Functions
# ============================================================

def safe_name(text):
    return str(text).replace(" ", "_").replace("/", "_").replace("-", "_")


def create_task_model_dirs(task_name, model_name):

    task_dir = os.path.join(BLOCK09_DIR, task_name)
    model_task_dir = os.path.join(task_dir, model_name)

    dirs = {
        "task": task_dir,
        "model": model_task_dir,
        "models": os.path.join(model_task_dir, "models"),
        "metrics": os.path.join(model_task_dir, "metrics"),
        "reports": os.path.join(model_task_dir, "classification_reports"),
        "predictions": os.path.join(model_task_dir, "predictions"),
        "figures": os.path.join(model_task_dir, "figures")
    }

    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    return dirs


def fit_model(model, X, y, sample_weight=None):

    try:
        model.fit(X, y, sample_weight=sample_weight)
    except TypeError:
        model.fit(X, y)

    return model


def get_aligned_proba(model, X, expected_classes):

    pred = np.ravel(model.predict(X))

    if hasattr(model, "predict_proba"):

        proba = model.predict_proba(X)

        if proba.ndim == 1:
            proba = proba.reshape(-1, 1)

        model_classes = list(getattr(model, "classes_", expected_classes))

        aligned = np.zeros((len(X), len(expected_classes)))

        for i, cls in enumerate(model_classes):
            if cls in expected_classes:
                aligned[:, expected_classes.index(cls)] = proba[:, i]

        row_sum = aligned.sum(axis=1)

        valid_rows = row_sum > 0

        aligned[valid_rows] = aligned[valid_rows] / row_sum[valid_rows].reshape(-1, 1)

        if (~valid_rows).sum() > 0:
            for row_idx in np.where(~valid_rows)[0]:
                pred_class = pred[row_idx]
                if pred_class in expected_classes:
                    aligned[row_idx, expected_classes.index(pred_class)] = 1.0

        return aligned

    else:

        aligned = np.zeros((len(X), len(expected_classes)))

        for i, p in enumerate(pred):
            if p in expected_classes:
                aligned[i, expected_classes.index(p)] = 1.0

        return aligned


def macro_specificity_score(y_true, y_pred, classes):

    specificities = []

    for cls in classes:

        y_true_binary = (np.array(y_true) == cls).astype(int)
        y_pred_binary = (np.array(y_pred) == cls).astype(int)

        cm = confusion_matrix(
            y_true_binary,
            y_pred_binary,
            labels=[0, 1]
        )

        tn, fp, fn, tp = cm.ravel()

        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

        specificities.append(specificity)

    return float(np.mean(specificities))


def multiclass_brier_score(y_true, proba, classes):

    y_bin = label_binarize(
        y_true,
        classes=classes
    )

    return float(
        np.mean(
            np.sum(
                (proba - y_bin) ** 2,
                axis=1
            )
        )
    )


# ============================================================
# Metric Calculation
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
    proba,
    task_type,
    classes,
    class_names
):

    metrics = {}

    if task_type == "binary":

        positive_prob = proba[:, 1]

        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=classes
        )

        tn, fp, fn, tp = cm.ravel()

        metrics["Accuracy"] = accuracy_score(y_true, y_pred)
        metrics["Precision"] = precision_score(y_true, y_pred, zero_division=0)
        metrics["Recall_Sensitivity"] = recall_score(y_true, y_pred, zero_division=0)
        metrics["Specificity"] = tn / (tn + fp) if (tn + fp) > 0 else 0
        metrics["F1"] = f1_score(y_true, y_pred, zero_division=0)
        metrics["Balanced_Accuracy"] = balanced_accuracy_score(y_true, y_pred)
        metrics["MCC"] = matthews_corrcoef(y_true, y_pred)
        metrics["Cohen_Kappa"] = cohen_kappa_score(y_true, y_pred)

        metrics["TP"] = int(tp)
        metrics["TN"] = int(tn)
        metrics["FP"] = int(fp)
        metrics["FN"] = int(fn)

        metrics["NPV"] = tn / (tn + fn) if (tn + fn) > 0 else 0
        metrics["FPR"] = fp / (fp + tn) if (fp + tn) > 0 else 0
        metrics["FNR"] = fn / (fn + tp) if (fn + tp) > 0 else 0

        try:
            metrics["ROC_AUC"] = roc_auc_score(y_true, positive_prob)
        except Exception:
            metrics["ROC_AUC"] = np.nan

        try:
            metrics["PR_AUC"] = average_precision_score(y_true, positive_prob)
        except Exception:
            metrics["PR_AUC"] = np.nan

        try:
            metrics["Log_Loss"] = log_loss(y_true, proba, labels=classes)
        except Exception:
            metrics["Log_Loss"] = np.nan

        try:
            metrics["Brier_Score"] = brier_score_loss(y_true, positive_prob)
        except Exception:
            metrics["Brier_Score"] = np.nan

    else:

        y_bin = label_binarize(
            y_true,
            classes=classes
        )

        metrics["Accuracy"] = accuracy_score(y_true, y_pred)
        metrics["Macro_Precision"] = precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
        metrics["Weighted_Precision"] = precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
        metrics["Macro_Recall_Sensitivity"] = recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
        metrics["Weighted_Recall"] = recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
        metrics["Macro_F1"] = f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
        metrics["Weighted_F1"] = f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
        metrics["Balanced_Accuracy"] = balanced_accuracy_score(y_true, y_pred)
        metrics["Macro_Specificity"] = macro_specificity_score(
            y_true,
            y_pred,
            classes
        )
        metrics["MCC"] = matthews_corrcoef(y_true, y_pred)
        metrics["Cohen_Kappa"] = cohen_kappa_score(y_true, y_pred)

        try:
            metrics["ROC_AUC_OVR_Macro"] = roc_auc_score(
                y_true,
                proba,
                multi_class="ovr",
                average="macro",
                labels=classes
            )
        except Exception:
            metrics["ROC_AUC_OVR_Macro"] = np.nan

        try:
            metrics["ROC_AUC_OVR_Weighted"] = roc_auc_score(
                y_true,
                proba,
                multi_class="ovr",
                average="weighted",
                labels=classes
            )
        except Exception:
            metrics["ROC_AUC_OVR_Weighted"] = np.nan

        try:
            metrics["PR_AUC_Macro"] = average_precision_score(
                y_bin,
                proba,
                average="macro"
            )
        except Exception:
            metrics["PR_AUC_Macro"] = np.nan

        try:
            metrics["PR_AUC_Weighted"] = average_precision_score(
                y_bin,
                proba,
                average="weighted"
            )
        except Exception:
            metrics["PR_AUC_Weighted"] = np.nan

        try:
            metrics["Log_Loss"] = log_loss(y_true, proba, labels=classes)
        except Exception:
            metrics["Log_Loss"] = np.nan

        try:
            metrics["Multiclass_Brier_Score"] = multiclass_brier_score(
                y_true,
                proba,
                classes
            )
        except Exception:
            metrics["Multiclass_Brier_Score"] = np.nan

    return metrics


# ============================================================
# Plot Functions
# ============================================================

def plot_confusion_matrix_figure(
    y_true,
    y_pred,
    classes,
    class_names,
    title,
    save_path
):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=classes
    )

    plt.figure(figsize=(7, 6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    return cm


def plot_binary_roc_curve(
    y_true,
    positive_prob,
    title,
    save_path
):

    fpr, tpr, _ = roc_curve(
        y_true,
        positive_prob
    )

    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(7, 6))

    plt.plot(
        fpr,
        tpr,
        label=f"ROC AUC = {roc_auc:.4f}"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Random"
    )

    plt.title(title)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def plot_binary_pr_curve(
    y_true,
    positive_prob,
    title,
    save_path
):

    precision, recall, _ = precision_recall_curve(
        y_true,
        positive_prob
    )

    pr_auc = average_precision_score(
        y_true,
        positive_prob
    )

    plt.figure(figsize=(7, 6))

    plt.plot(
        recall,
        precision,
        label=f"PR AUC = {pr_auc:.4f}"
    )

    plt.title(title)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend(loc="lower left")
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def plot_binary_calibration_curve(
    y_true,
    positive_prob,
    title,
    save_path
):

    prob_true, prob_pred = calibration_curve(
        y_true,
        positive_prob,
        n_bins=10,
        strategy="uniform"
    )

    plt.figure(figsize=(7, 6))

    plt.plot(
        prob_pred,
        prob_true,
        marker="o",
        label="Model"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect Calibration"
    )

    plt.title(title)
    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Fraction of Positives")
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def plot_binary_probability_distribution(
    y_true,
    positive_prob,
    class_names,
    title,
    save_path
):

    plot_df = pd.DataFrame({
        "True_Label": y_true,
        "Positive_Probability": positive_prob
    })

    plt.figure(figsize=(8, 5))

    for cls, label in zip([0, 1], class_names):
        sns.histplot(
            plot_df.loc[
                plot_df["True_Label"] == cls,
                "Positive_Probability"
            ],
            bins=30,
            kde=True,
            stat="density",
            label=label,
            alpha=0.45
        )

    plt.title(title)
    plt.xlabel("Predicted Probability for Positive Class")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def plot_multiclass_roc_curve(
    y_true,
    proba,
    classes,
    class_names,
    title,
    save_path
):

    y_bin = label_binarize(
        y_true,
        classes=classes
    )

    plt.figure(figsize=(8, 6))

    for i, class_name in enumerate(class_names):

        if len(np.unique(y_bin[:, i])) < 2:
            continue

        fpr, tpr, _ = roc_curve(
            y_bin[:, i],
            proba[:, i]
        )

        roc_auc = auc(fpr, tpr)

        plt.plot(
            fpr,
            tpr,
            label=f"{class_name} AUC = {roc_auc:.4f}"
        )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Random"
    )

    plt.title(title)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right", fontsize=8)
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def plot_multiclass_pr_curve(
    y_true,
    proba,
    classes,
    class_names,
    title,
    save_path
):

    y_bin = label_binarize(
        y_true,
        classes=classes
    )

    plt.figure(figsize=(8, 6))

    for i, class_name in enumerate(class_names):

        if len(np.unique(y_bin[:, i])) < 2:
            continue

        precision, recall, _ = precision_recall_curve(
            y_bin[:, i],
            proba[:, i]
        )

        pr_auc = average_precision_score(
            y_bin[:, i],
            proba[:, i]
        )

        plt.plot(
            recall,
            precision,
            label=f"{class_name} AP = {pr_auc:.4f}"
        )

    plt.title(title)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend(loc="lower left", fontsize=8)
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def plot_multiclass_confidence_calibration(
    y_true,
    y_pred,
    proba,
    title,
    save_path
):

    confidence = np.max(proba, axis=1)
    correctness = (np.array(y_true) == np.array(y_pred)).astype(int)

    prob_true, prob_pred = calibration_curve(
        correctness,
        confidence,
        n_bins=10,
        strategy="uniform"
    )

    plt.figure(figsize=(7, 6))

    plt.plot(
        prob_pred,
        prob_true,
        marker="o",
        label="Model Confidence"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect Calibration"
    )

    plt.title(title)
    plt.xlabel("Mean Predicted Confidence")
    plt.ylabel("Empirical Accuracy")
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def plot_multiclass_probability_distribution(
    y_true,
    y_pred,
    proba,
    title,
    save_path
):

    confidence = np.max(proba, axis=1)
    correctness = np.where(
        np.array(y_true) == np.array(y_pred),
        "Correct",
        "Incorrect"
    )

    plot_df = pd.DataFrame({
        "Confidence": confidence,
        "Prediction_Status": correctness
    })

    plt.figure(figsize=(8, 5))

    sns.histplot(
        data=plot_df,
        x="Confidence",
        hue="Prediction_Status",
        bins=30,
        kde=True,
        stat="density",
        alpha=0.45
    )

    plt.title(title)
    plt.xlabel("Maximum Predicted Probability")
    plt.ylabel("Density")
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


# ============================================================
# Split Evaluation + Export
# ============================================================

def evaluate_split_and_export(
    model,
    X_split,
    y_true,
    split_name,
    task_name,
    task_type,
    model_name,
    classes,
    class_names,
    dirs
):

    split_key = split_name.lower()

    y_pred = np.ravel(
        model.predict(X_split)
    ).astype(int)

    proba = get_aligned_proba(
        model,
        X_split,
        classes
    )

    metrics = calculate_metrics(
        y_true=y_true,
        y_pred=y_pred,
        proba=proba,
        task_type=task_type,
        classes=classes,
        class_names=class_names
    )

    metrics["Task"] = task_name
    metrics["Model"] = model_name
    metrics["Split"] = split_name

    # --------------------------------------------------------
    # Classification Report
    # --------------------------------------------------------

    report_text = classification_report(
        y_true,
        y_pred,
        labels=classes,
        target_names=class_names,
        digits=4,
        zero_division=0
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=classes,
        target_names=class_names,
        digits=4,
        zero_division=0,
        output_dict=True
    )

    metrics_table_text = (
        pd.DataFrame([metrics])
        .drop(columns=["Task", "Model", "Split"])
        .T
        .rename(columns={0: "Value"})
        .to_string()
    )

    report_file = os.path.join(
        dirs["reports"],
        f"{safe_name(model_name)}_{split_key}_classification_report.txt"
    )

    with open(report_file, "w") as f:
        f.write("=" * 70 + "\n")
        f.write(f"{task_name} — {model_name} — {split_name} Classification Report\n")
        f.write("=" * 70 + "\n\n")
        f.write(report_text)
        f.write("\n\n")
        f.write("=" * 70 + "\n")
        f.write("Additional Metrics\n")
        f.write("=" * 70 + "\n")
        f.write(metrics_table_text)

    report_csv = os.path.join(
        dirs["reports"],
        f"{safe_name(model_name)}_{split_key}_classification_report.csv"
    )

    pd.DataFrame(report_dict).T.to_csv(report_csv)

    print("\n" + "=" * 70)
    print(f"{task_name} | {model_name} | {split_name} REPORT")
    print("=" * 70)
    print(report_text)

    # --------------------------------------------------------
    # Confusion Matrix
    # --------------------------------------------------------

    cm = plot_confusion_matrix_figure(
        y_true=y_true,
        y_pred=y_pred,
        classes=classes,
        class_names=class_names,
        title=f"{task_name} — {model_name} — {split_name} Confusion Matrix",
        save_path=os.path.join(
            dirs["figures"],
            f"{safe_name(model_name)}_{split_key}_confusion_matrix.png"
        )
    )

    pd.DataFrame(
        cm,
        index=[f"True_{c}" for c in class_names],
        columns=[f"Pred_{c}" for c in class_names]
    ).to_csv(
        os.path.join(
            dirs["metrics"],
            f"{safe_name(model_name)}_{split_key}_confusion_matrix.csv"
        )
    )

    # --------------------------------------------------------
    # ROC, PR, Calibration, Probability Distribution
    # --------------------------------------------------------

    if task_type == "binary":

        positive_prob = proba[:, 1]

        plot_binary_roc_curve(
            y_true,
            positive_prob,
            f"{task_name} — {model_name} — {split_name} ROC Curve",
            os.path.join(
                dirs["figures"],
                f"{safe_name(model_name)}_{split_key}_roc_curve.png"
            )
        )

        plot_binary_pr_curve(
            y_true,
            positive_prob,
            f"{task_name} — {model_name} — {split_name} Precision-Recall Curve",
            os.path.join(
                dirs["figures"],
                f"{safe_name(model_name)}_{split_key}_precision_recall_curve.png"
            )
        )

        plot_binary_calibration_curve(
            y_true,
            positive_prob,
            f"{task_name} — {model_name} — {split_name} Calibration Curve",
            os.path.join(
                dirs["figures"],
                f"{safe_name(model_name)}_{split_key}_calibration_curve.png"
            )
        )

        plot_binary_probability_distribution(
            y_true,
            positive_prob,
            class_names,
            f"{task_name} — {model_name} — {split_name} Probability Distribution",
            os.path.join(
                dirs["figures"],
                f"{safe_name(model_name)}_{split_key}_probability_distribution.png"
            )
        )

    else:

        plot_multiclass_roc_curve(
            y_true,
            proba,
            classes,
            class_names,
            f"{task_name} — {model_name} — {split_name} Multiclass ROC Curve",
            os.path.join(
                dirs["figures"],
                f"{safe_name(model_name)}_{split_key}_roc_curve.png"
            )
        )

        plot_multiclass_pr_curve(
            y_true,
            proba,
            classes,
            class_names,
            f"{task_name} — {model_name} — {split_name} Multiclass Precision-Recall Curve",
            os.path.join(
                dirs["figures"],
                f"{safe_name(model_name)}_{split_key}_precision_recall_curve.png"
            )
        )

        plot_multiclass_confidence_calibration(
            y_true,
            y_pred,
            proba,
            f"{task_name} — {model_name} — {split_name} Confidence Calibration",
            os.path.join(
                dirs["figures"],
                f"{safe_name(model_name)}_{split_key}_calibration_curve.png"
            )
        )

        plot_multiclass_probability_distribution(
            y_true,
            y_pred,
            proba,
            f"{task_name} — {model_name} — {split_name} Probability Distribution",
            os.path.join(
                dirs["figures"],
                f"{safe_name(model_name)}_{split_key}_probability_distribution.png"
            )
        )

    # --------------------------------------------------------
    # Prediction Export
    # --------------------------------------------------------

    if task_type == "binary":

        pred_df = pd.DataFrame({
            "Sample_Index": X_split.index,
            "True_Label": y_true.values,
            "True_Label_Name": [
                class_names[i] for i in y_true.values
            ],
            "Predicted_Label": y_pred,
            "Predicted_Label_Name": [
                class_names[i] for i in y_pred
            ],
            "Probability_Non_MDR": proba[:, 0],
            "Probability_MDR": proba[:, 1],
            "Split": split_name,
            "Task": task_name,
            "Model": model_name
        })

    else:

        pred_df = pd.DataFrame({
            "Sample_Index": X_split.index,
            "True_Label_ID": y_true.values,
            "True_Label": [
                ID_TO_SEVERITY[i] for i in y_true.values
            ],
            "Predicted_Label_ID": y_pred,
            "Predicted_Label": [
                ID_TO_SEVERITY[i] for i in y_pred
            ],
            "Split": split_name,
            "Task": task_name,
            "Model": model_name
        })

        for i, class_name in enumerate(class_names):
            pred_df[f"Probability_{class_name}"] = proba[:, i]

    pred_df.to_csv(
        os.path.join(
            dirs["predictions"],
            f"{safe_name(model_name)}_{split_key}_predictions.csv"
        ),
        index=False
    )

    return metrics, pred_df


# ============================================================
# Learning Curve
# ============================================================

def plot_learning_curve_custom(
    model,
    X_train_input,
    y_train_input,
    X_val_input,
    y_val_input,
    sample_weight_input,
    task_type,
    task_name,
    model_name,
    dirs
):

    if not RUN_LEARNING_CURVES:
        return None

    fractions = [0.25, 0.50, 0.75, 1.00]

    rows = []

    for frac in fractions:

        if frac < 1.0:

            subset_positions, _ = train_test_split(
                np.arange(len(X_train_input)),
                train_size=frac,
                random_state=SEED,
                stratify=y_train_input
            )

        else:

            subset_positions = np.arange(len(X_train_input))

        X_sub = X_train_input.iloc[subset_positions]
        y_sub = y_train_input.iloc[subset_positions]
        w_sub = sample_weight_input[subset_positions]

        temp_model = clone(model)

        temp_model = fit_model(
            temp_model,
            X_sub,
            y_sub,
            sample_weight=w_sub
        )

        train_pred = np.ravel(
            temp_model.predict(X_sub)
        ).astype(int)

        val_pred = np.ravel(
            temp_model.predict(X_val_input)
        ).astype(int)

        if task_type == "binary":

            train_score = f1_score(
                y_sub,
                train_pred,
                zero_division=0
            )

            val_score = f1_score(
                y_val_input,
                val_pred,
                zero_division=0
            )

            metric_name = "F1"

        else:

            train_score = f1_score(
                y_sub,
                train_pred,
                average="macro",
                zero_division=0
            )

            val_score = f1_score(
                y_val_input,
                val_pred,
                average="macro",
                zero_division=0
            )

            metric_name = "Macro F1"

        rows.append({
            "Training_Fraction": frac,
            "Training_Samples": len(X_sub),
            "Train_Score": train_score,
            "Validation_Score": val_score,
            "Metric": metric_name
        })

    learning_df = pd.DataFrame(rows)

    learning_df.to_csv(
        os.path.join(
            dirs["metrics"],
            f"{safe_name(model_name)}_learning_curve.csv"
        ),
        index=False
    )

    plt.figure(figsize=(8, 5))

    plt.plot(
        learning_df["Training_Samples"],
        learning_df["Train_Score"],
        marker="o",
        label=f"Train {metric_name}"
    )

    plt.plot(
        learning_df["Training_Samples"],
        learning_df["Validation_Score"],
        marker="o",
        label=f"Validation {metric_name}"
    )

    plt.title(f"{task_name} — {model_name} Learning Curve")
    plt.xlabel("Training Samples")
    plt.ylabel(metric_name)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            dirs["figures"],
            f"{safe_name(model_name)}_learning_curve.png"
        ),
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    return learning_df


# ============================================================
# Main Task Runner
# ============================================================

def run_baseline_task(
    task_name,
    task_type,
    X_train_input,
    X_val_input,
    X_test_input,
    y_train_input,
    y_val_input,
    y_test_input,
    sample_weight_input,
    classes,
    class_names
):

    print("\n")
    print("=" * 70)
    print(f"RUNNING BASELINE MODELS FOR {task_name}")
    print("=" * 70)

    models = get_baseline_models(task_type)

    task_metrics = []
    task_predictions = []
    training_times = []

    for model_name, model in models.items():

        print("\n")
        print("=" * 70)
        print(f"TRAINING MODEL: {task_name} — {model_name}")
        print("=" * 70)

        dirs = create_task_model_dirs(
            task_name=task_name,
            model_name=model_name
        )

        start_time = time.time()

        trained_model = fit_model(
            model,
            X_train_input,
            y_train_input,
            sample_weight=sample_weight_input
        )

        train_time = time.time() - start_time

        training_times.append({
            "Task": task_name,
            "Model": model_name,
            "Training_Time_Seconds": round(train_time, 4)
        })

        print(f"Training Time: {train_time:.4f} seconds")

        # Save model
        joblib.dump(
            trained_model,
            os.path.join(
                dirs["models"],
                f"{safe_name(model_name)}.pkl"
            )
        )

        # Train / Validation / Test evaluation
        for split_name, X_split, y_split in [

            ("Train", X_train_input, y_train_input),
            ("Validation", X_val_input, y_val_input),
            ("Test", X_test_input, y_test_input)

        ]:

            metrics, pred_df = evaluate_split_and_export(
                model=trained_model,
                X_split=X_split,
                y_true=y_split,
                split_name=split_name,
                task_name=task_name,
                task_type=task_type,
                model_name=model_name,
                classes=classes,
                class_names=class_names,
                dirs=dirs
            )

            task_metrics.append(metrics)
            task_predictions.append(pred_df)

        # Learning curve
        plot_learning_curve_custom(
            model=trained_model,
            X_train_input=X_train_input,
            y_train_input=y_train_input,
            X_val_input=X_val_input,
            y_val_input=y_val_input,
            sample_weight_input=sample_weight_input,
            task_type=task_type,
            task_name=task_name,
            model_name=model_name,
            dirs=dirs
        )

    # Save task-level outputs
    task_metrics_df = pd.DataFrame(task_metrics)
    task_predictions_df = pd.concat(task_predictions, ignore_index=True)
    training_times_df = pd.DataFrame(training_times)

    task_metrics_df.to_csv(
        os.path.join(
            METRIC_DIR,
            f"{task_name}_baseline_all_metrics.csv"
        ),
        index=False
    )

    task_predictions_df.to_csv(
        os.path.join(
            PRED_DIR,
            f"{task_name}_baseline_all_predictions.csv"
        ),
        index=False
    )

    training_times_df.to_csv(
        os.path.join(
            METRIC_DIR,
            f"{task_name}_training_times.csv"
        ),
        index=False
    )

    return task_metrics_df, task_predictions_df, training_times_df


print("\n")
print("=" * 70)
print("BLOCK 09 PART 1 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 09
# PART 2
# Train All Baseline Models + Final Comparison Tables/Figures
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 09 PART 2 STARTED")
print("=" * 70)

# ============================================================
# Run MDR Baseline Models
# ============================================================

mdr_metrics_df, mdr_predictions_df, mdr_training_times_df = run_baseline_task(
    task_name="MDR",
    task_type="binary",
    X_train_input=X_train_scaled,
    X_val_input=X_val_scaled,
    X_test_input=X_test_scaled,
    y_train_input=y_train_mdr,
    y_val_input=y_val_mdr,
    y_test_input=y_test_mdr,
    sample_weight_input=mdr_train_weights,
    classes=MDR_CLASSES,
    class_names=MDR_CLASS_NAMES
)

# ============================================================
# Run Severity Baseline Models
# ============================================================

severity_metrics_df, severity_predictions_df, severity_training_times_df = run_baseline_task(
    task_name="Severity",
    task_type="multiclass",
    X_train_input=X_train_scaled,
    X_val_input=X_val_scaled,
    X_test_input=X_test_scaled,
    y_train_input=y_train_severity,
    y_val_input=y_val_severity,
    y_test_input=y_test_severity,
    sample_weight_input=severity_train_weights,
    classes=SEVERITY_CLASSES,
    class_names=SEVERITY_CLASS_NAMES
)

# ============================================================
# Combine All Metrics
# ============================================================

all_baseline_metrics = pd.concat(
    [
        mdr_metrics_df,
        severity_metrics_df
    ],
    ignore_index=True
)

all_training_times = pd.concat(
    [
        mdr_training_times_df,
        severity_training_times_df
    ],
    ignore_index=True
)

all_baseline_metrics.to_csv(
    os.path.join(
        METRIC_DIR,
        "block09_all_baseline_metrics.csv"
    ),
    index=False
)

all_training_times.to_csv(
    os.path.join(
        METRIC_DIR,
        "block09_all_training_times.csv"
    ),
    index=False
)

# ============================================================
# Main Journal Comparison Tables
# ============================================================

mdr_validation_table = (
    all_baseline_metrics[
        (all_baseline_metrics["Task"] == "MDR") &
        (all_baseline_metrics["Split"] == "Validation")
    ]
    .sort_values(
        by=["ROC_AUC", "F1"],
        ascending=False
    )
)

mdr_test_table = (
    all_baseline_metrics[
        (all_baseline_metrics["Task"] == "MDR") &
        (all_baseline_metrics["Split"] == "Test")
    ]
    .sort_values(
        by=["ROC_AUC", "F1"],
        ascending=False
    )
)

severity_validation_table = (
    all_baseline_metrics[
        (all_baseline_metrics["Task"] == "Severity") &
        (all_baseline_metrics["Split"] == "Validation")
    ]
    .sort_values(
        by=["Macro_F1", "Balanced_Accuracy"],
        ascending=False
    )
)

severity_test_table = (
    all_baseline_metrics[
        (all_baseline_metrics["Task"] == "Severity") &
        (all_baseline_metrics["Split"] == "Test")
    ]
    .sort_values(
        by=["Macro_F1", "Balanced_Accuracy"],
        ascending=False
    )
)

mdr_validation_table.to_csv(
    os.path.join(
        METRIC_DIR,
        "mdr_validation_baseline_comparison.csv"
    ),
    index=False
)

mdr_test_table.to_csv(
    os.path.join(
        METRIC_DIR,
        "mdr_test_baseline_comparison.csv"
    ),
    index=False
)

severity_validation_table.to_csv(
    os.path.join(
        METRIC_DIR,
        "severity_validation_baseline_comparison.csv"
    ),
    index=False
)

severity_test_table.to_csv(
    os.path.join(
        METRIC_DIR,
        "severity_test_baseline_comparison.csv"
    ),
    index=False
)

# ============================================================
# Publication-Ready Summary Tables
# ============================================================

mdr_publication_table = mdr_test_table[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall_Sensitivity",
        "Specificity",
        "F1",
        "Balanced_Accuracy",
        "MCC",
        "Cohen_Kappa",
        "ROC_AUC",
        "PR_AUC",
        "Log_Loss",
        "Brier_Score"
    ]
].copy()

severity_publication_table = severity_test_table[
    [
        "Model",
        "Accuracy",
        "Macro_Precision",
        "Weighted_Precision",
        "Macro_Recall_Sensitivity",
        "Weighted_Recall",
        "Macro_F1",
        "Weighted_F1",
        "Balanced_Accuracy",
        "Macro_Specificity",
        "MCC",
        "Cohen_Kappa",
        "ROC_AUC_OVR_Macro",
        "PR_AUC_Macro",
        "Log_Loss",
        "Multiclass_Brier_Score"
    ]
].copy()

mdr_publication_table.to_csv(
    os.path.join(
        METRIC_DIR,
        "table_09a_mdr_baseline_publication_results.csv"
    ),
    index=False
)

severity_publication_table.to_csv(
    os.path.join(
        METRIC_DIR,
        "table_09b_severity_baseline_publication_results.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("MDR TEST BASELINE COMPARISON")
print("=" * 70)

display(mdr_publication_table)

print("\n")
print("=" * 70)
print("SEVERITY TEST BASELINE COMPARISON")
print("=" * 70)

display(severity_publication_table)

# ============================================================
# Final Comparison Figures
# ============================================================

def plot_baseline_comparison_bar(
    df,
    metric,
    title,
    ylabel,
    save_path
):

    plot_df = df.copy()

    plt.figure(figsize=(9, 5))

    sns.barplot(
        data=plot_df,
        x=metric,
        y="Model",
        orient="h"
    )

    plt.title(title)
    plt.xlabel(ylabel)
    plt.ylabel("Model")
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


plot_baseline_comparison_bar(
    df=mdr_test_table,
    metric="ROC_AUC",
    title="MDR Baseline Model Comparison — Test ROC-AUC",
    ylabel="ROC-AUC",
    save_path=os.path.join(
        FIG_DIR,
        "figure_09a_mdr_test_roc_auc_comparison.png"
    )
)

plot_baseline_comparison_bar(
    df=mdr_test_table,
    metric="F1",
    title="MDR Baseline Model Comparison — Test F1-score",
    ylabel="F1-score",
    save_path=os.path.join(
        FIG_DIR,
        "figure_09b_mdr_test_f1_comparison.png"
    )
)

plot_baseline_comparison_bar(
    df=severity_test_table,
    metric="Macro_F1",
    title="Severity Baseline Model Comparison — Test Macro F1-score",
    ylabel="Macro F1-score",
    save_path=os.path.join(
        FIG_DIR,
        "figure_09c_severity_test_macro_f1_comparison.png"
    )
)

plot_baseline_comparison_bar(
    df=severity_test_table,
    metric="Balanced_Accuracy",
    title="Severity Baseline Model Comparison — Test Balanced Accuracy",
    ylabel="Balanced Accuracy",
    save_path=os.path.join(
        FIG_DIR,
        "figure_09d_severity_test_balanced_accuracy_comparison.png"
    )
)

# ============================================================
# Final Export Summary
# ============================================================

block09_summary = pd.DataFrame({
    "Output_Category": [
        "All baseline metrics",
        "MDR predictions",
        "Severity predictions",
        "Training times",
        "MDR publication table",
        "Severity publication table",
        "Per-model classification reports",
        "Per-model confusion matrices",
        "Per-model ROC curves",
        "Per-model PR curves",
        "Per-model calibration curves",
        "Per-model probability distributions",
        "Per-model learning curves",
        "Saved baseline models"
    ],
    "Saved": [
        "block09_all_baseline_metrics.csv",
        "MDR_baseline_all_predictions.csv",
        "Severity_baseline_all_predictions.csv",
        "block09_all_training_times.csv",
        "table_09a_mdr_baseline_publication_results.csv",
        "table_09b_severity_baseline_publication_results.csv",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})

block09_summary.to_csv(
    os.path.join(
        BLOCK09_DIR,
        "block09_output_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("BLOCK 09 GENERATED FILES")
print("=" * 70)

print("Main Metrics:")
print("- metrics/block09_all_baseline_metrics.csv")
print("- metrics/block09_all_training_times.csv")
print("- metrics/mdr_validation_baseline_comparison.csv")
print("- metrics/mdr_test_baseline_comparison.csv")
print("- metrics/severity_validation_baseline_comparison.csv")
print("- metrics/severity_test_baseline_comparison.csv")
print("- metrics/table_09a_mdr_baseline_publication_results.csv")
print("- metrics/table_09b_severity_baseline_publication_results.csv")

print("\nPredictions:")
print("- predictions/MDR_baseline_all_predictions.csv")
print("- predictions/Severity_baseline_all_predictions.csv")

print("\nFigures:")
print("- Train/Validation/Test confusion matrices for every model")
print("- Train/Validation/Test ROC curves for every model")
print("- Train/Validation/Test PR curves for every model")
print("- Train/Validation/Test calibration curves for every model")
print("- Train/Validation/Test probability distributions for every model")
print("- Learning curves for every model")
print("- Final baseline comparison bar plots")

print("\nModels:")
print("- Saved .pkl model for every baseline model")

print("\n")
print("=" * 70)
print("BLOCK 09 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 10
# Hyperparameter Optimization
# PART 1
# Search Space + Validation-Based Optimization Helpers
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 10 STARTED — HYPERPARAMETER OPTIMIZATION")
print("=" * 70)

import os
import time
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

from sklearn.base import clone
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False


# ============================================================
# Dependency Check from Block 09
# ============================================================

required_block09_objects = [
    "fit_model",
    "get_aligned_proba",
    "calculate_metrics",
    "evaluate_split_and_export",
    "plot_learning_curve_custom",
    "safe_name",
    "MDR_CLASSES",
    "MDR_CLASS_NAMES",
    "SEVERITY_CLASSES",
    "SEVERITY_CLASS_NAMES",
    "mdr_train_weights",
    "severity_train_weights"
]

missing_objects = [
    obj for obj in required_block09_objects
    if obj not in globals()
]

if len(missing_objects) > 0:
    raise RuntimeError(
        "Please run BLOCK 09 Part 1 before BLOCK 10. Missing objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# Output Directories
# ============================================================

BLOCK10_DIR = os.path.join(OUTPUT_DIR, "block10_hyperparameter_optimization")

BLOCK10_MODEL_DIR = os.path.join(BLOCK10_DIR, "optimized_models")
BLOCK10_METRIC_DIR = os.path.join(BLOCK10_DIR, "metrics")
BLOCK10_REPORT_DIR = os.path.join(BLOCK10_DIR, "classification_reports")
BLOCK10_PRED_DIR = os.path.join(BLOCK10_DIR, "predictions")
BLOCK10_FIG_DIR = os.path.join(BLOCK10_DIR, "figures")
BLOCK10_SEARCH_DIR = os.path.join(BLOCK10_DIR, "search_results")

for d in [
    BLOCK10_DIR,
    BLOCK10_MODEL_DIR,
    BLOCK10_METRIC_DIR,
    BLOCK10_REPORT_DIR,
    BLOCK10_PRED_DIR,
    BLOCK10_FIG_DIR,
    BLOCK10_SEARCH_DIR
]:
    os.makedirs(d, exist_ok=True)

FIG_DPI = 300


# ============================================================
# Search Configuration
# ============================================================

# Keep this moderate for Kaggle speed.
# You can increase later after checking runtime.
N_ITER_FAST = 12
N_ITER_TREE = 18
N_ITER_BOOSTING = 18

OPTIMIZATION_RANDOM_STATE = SEED


# ============================================================
# Hyperparameter Spaces
# ============================================================

def get_search_spaces(task_type):

    spaces = {}

    spaces["LogisticRegression"] = {
        "C": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
        "solver": ["lbfgs"],
        "max_iter": [2000, 3000]
    }

    spaces["RandomForest"] = {
        "n_estimators": [300, 500, 700],
        "max_depth": [None, 5, 8, 10, 15, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.7],
        "bootstrap": [True, False]
    }

    if HAS_XGB:
        spaces["XGBoost"] = {
            "n_estimators": [300, 500, 700],
            "max_depth": [3, 4, 5, 6],
            "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.10],
            "subsample": [0.75, 0.85, 0.95, 1.00],
            "colsample_bytree": [0.75, 0.85, 0.95, 1.00],
            "min_child_weight": [1, 3, 5],
            "reg_lambda": [1.0, 2.0, 5.0, 10.0],
            "reg_alpha": [0.0, 0.01, 0.1]
        }

    if HAS_LGBM:
        spaces["LightGBM"] = {
            "n_estimators": [300, 500, 700],
            "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.10],
            "num_leaves": [15, 31, 63],
            "max_depth": [-1, 5, 8, 12],
            "min_child_samples": [10, 20, 40],
            "subsample": [0.75, 0.85, 0.95, 1.00],
            "colsample_bytree": [0.75, 0.85, 0.95, 1.00],
            "reg_lambda": [0.0, 1.0, 5.0, 10.0],
            "reg_alpha": [0.0, 0.01, 0.1]
        }

    if HAS_CATBOOST:
        spaces["CatBoost"] = {
            "iterations": [300, 500, 700],
            "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.10],
            "depth": [4, 5, 6, 7, 8],
            "l2_leaf_reg": [1, 3, 5, 7, 10],
            "random_strength": [0.5, 1.0, 2.0],
            "bagging_temperature": [0.0, 0.5, 1.0]
        }

    return spaces


# ============================================================
# Model Builder
# ============================================================

def build_model(model_name, task_type, params):

    if model_name == "LogisticRegression":

        return LogisticRegression(
            random_state=SEED,
            **params
        )

    if model_name == "RandomForest":

        return RandomForestClassifier(
            random_state=SEED,
            n_jobs=-1,
            **params
        )

    if model_name == "XGBoost":

        if task_type == "binary":

            return XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1,
                **params
            )

        else:

            return XGBClassifier(
                objective="multi:softprob",
                num_class=4,
                eval_metric="mlogloss",
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1,
                **params
            )

    if model_name == "LightGBM":

        if task_type == "binary":

            return LGBMClassifier(
                objective="binary",
                random_state=SEED,
                n_jobs=-1,
                verbose=-1,
                **params
            )

        else:

            return LGBMClassifier(
                objective="multiclass",
                num_class=4,
                random_state=SEED,
                n_jobs=-1,
                verbose=-1,
                **params
            )

    if model_name == "CatBoost":

        if task_type == "binary":

            return CatBoostClassifier(
                loss_function="Logloss",
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False,
                **params
            )

        else:

            return CatBoostClassifier(
                loss_function="MultiClass",
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False,
                **params
            )

    raise ValueError(f"Unknown model: {model_name}")


# ============================================================
# Validation Search Function
# ============================================================

def validation_hyperparameter_search(
    task_name,
    task_type,
    model_name,
    X_train_input,
    y_train_input,
    X_val_input,
    y_val_input,
    sample_weight_input,
    classes,
    class_names,
    n_iter
):

    print("\n")
    print("=" * 70)
    print(f"HYPERPARAMETER SEARCH: {task_name} — {model_name}")
    print("=" * 70)

    search_space = get_search_spaces(task_type)[model_name]

    sampled_params = list(
        ParameterSampler(
            search_space,
            n_iter=n_iter,
            random_state=OPTIMIZATION_RANDOM_STATE
        )
    )

    search_rows = []

    best_score = -np.inf
    best_model = None
    best_params = None
    best_metrics = None

    for i, params in enumerate(sampled_params, start=1):

        start_time = time.time()

        try:

            model = build_model(
                model_name=model_name,
                task_type=task_type,
                params=params
            )

            model = fit_model(
                model,
                X_train_input,
                y_train_input,
                sample_weight=sample_weight_input
            )

            val_pred = np.ravel(
                model.predict(X_val_input)
            ).astype(int)

            val_proba = get_aligned_proba(
                model,
                X_val_input,
                classes
            )

            val_metrics = calculate_metrics(
                y_true=y_val_input,
                y_pred=val_pred,
                proba=val_proba,
                task_type=task_type,
                classes=classes,
                class_names=class_names
            )

            if task_type == "binary":

                primary_score = val_metrics.get("ROC_AUC", np.nan)
                secondary_score = val_metrics.get("F1", np.nan)
                tertiary_score = val_metrics.get("PR_AUC", np.nan)

            else:

                primary_score = val_metrics.get("Macro_F1", np.nan)
                secondary_score = val_metrics.get("Balanced_Accuracy", np.nan)
                tertiary_score = val_metrics.get("ROC_AUC_OVR_Macro", np.nan)

            elapsed = time.time() - start_time

            row = {
                "Task": task_name,
                "Model": model_name,
                "Iteration": i,
                "Search_Status": "Success",
                "Primary_Score": primary_score,
                "Secondary_Score": secondary_score,
                "Tertiary_Score": tertiary_score,
                "Elapsed_Seconds": round(elapsed, 4)
            }

            row.update(val_metrics)

            for k, v in params.items():
                row[f"param_{k}"] = v

            search_rows.append(row)

            score_tuple = (
                primary_score if not pd.isna(primary_score) else -np.inf,
                secondary_score if not pd.isna(secondary_score) else -np.inf,
                tertiary_score if not pd.isna(tertiary_score) else -np.inf
            )

            best_tuple = (
                best_metrics.get("Primary_Score", -np.inf)
                if best_metrics is not None else -np.inf,
                best_metrics.get("Secondary_Score", -np.inf)
                if best_metrics is not None else -np.inf,
                best_metrics.get("Tertiary_Score", -np.inf)
                if best_metrics is not None else -np.inf
            )

            if score_tuple > best_tuple:

                best_score = primary_score
                best_model = model
                best_params = params
                best_metrics = row

            print(
                f"Iteration {i:02d}/{len(sampled_params)} | "
                f"Primary={primary_score:.4f} | "
                f"Secondary={secondary_score:.4f} | "
                f"Time={elapsed:.2f}s"
            )

        except Exception as e:

            elapsed = time.time() - start_time

            failed_row = {
                "Task": task_name,
                "Model": model_name,
                "Iteration": i,
                "Search_Status": "Failed",
                "Error": str(e),
                "Elapsed_Seconds": round(elapsed, 4)
            }

            for k, v in params.items():
                failed_row[f"param_{k}"] = v

            search_rows.append(failed_row)

            print(f"Iteration {i:02d} failed: {e}")

    search_df = pd.DataFrame(search_rows)

    search_df.to_csv(
        os.path.join(
            BLOCK10_SEARCH_DIR,
            f"{task_name}_{model_name}_hyperparameter_search_results.csv"
        ),
        index=False
    )

    print("\nBest Params:")
    print(best_params)

    return best_model, best_params, best_metrics, search_df


# ============================================================
# Optimized Model Evaluation Runner
# ============================================================

def create_block10_model_dirs(task_name, model_name):

    task_dir = os.path.join(BLOCK10_DIR, task_name)
    model_task_dir = os.path.join(task_dir, model_name)

    dirs = {
        "task": task_dir,
        "model": model_task_dir,
        "models": os.path.join(model_task_dir, "models"),
        "metrics": os.path.join(model_task_dir, "metrics"),
        "reports": os.path.join(model_task_dir, "classification_reports"),
        "predictions": os.path.join(model_task_dir, "predictions"),
        "figures": os.path.join(model_task_dir, "figures")
    }

    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    return dirs


def evaluate_optimized_model(
    task_name,
    task_type,
    model_name,
    model,
    best_params,
    X_train_input,
    X_val_input,
    X_test_input,
    y_train_input,
    y_val_input,
    y_test_input,
    sample_weight_input,
    classes,
    class_names
):

    print("\n")
    print("=" * 70)
    print(f"FINAL OPTIMIZED EVALUATION: {task_name} — {model_name}")
    print("=" * 70)

    dirs = create_block10_model_dirs(
        task_name=task_name,
        model_name=model_name
    )

    # Save optimized model
    joblib.dump(
        model,
        os.path.join(
            dirs["models"],
            f"{safe_name(model_name)}_optimized.pkl"
        )
    )

    # Save best params
    pd.DataFrame(
        list(best_params.items()),
        columns=["Parameter", "Best_Value"]
    ).to_csv(
        os.path.join(
            dirs["metrics"],
            f"{safe_name(model_name)}_best_hyperparameters.csv"
        ),
        index=False
    )

    optimized_metrics = []
    optimized_predictions = []

    for split_name, X_split, y_split in [

        ("Train", X_train_input, y_train_input),
        ("Validation", X_val_input, y_val_input),
        ("Test", X_test_input, y_test_input)

    ]:

        metrics, pred_df = evaluate_split_and_export(
            model=model,
            X_split=X_split,
            y_true=y_split,
            split_name=split_name,
            task_name=f"{task_name}_Optimized",
            task_type=task_type,
            model_name=model_name,
            classes=classes,
            class_names=class_names,
            dirs=dirs
        )

        optimized_metrics.append(metrics)
        optimized_predictions.append(pred_df)

    # Learning curve
    plot_learning_curve_custom(
        model=model,
        X_train_input=X_train_input,
        y_train_input=y_train_input,
        X_val_input=X_val_input,
        y_val_input=y_val_input,
        sample_weight_input=sample_weight_input,
        task_type=task_type,
        task_name=f"{task_name}_Optimized",
        model_name=model_name,
        dirs=dirs
    )

    optimized_metrics_df = pd.DataFrame(optimized_metrics)

    optimized_predictions_df = pd.concat(
        optimized_predictions,
        ignore_index=True
    )

    optimized_metrics_df.to_csv(
        os.path.join(
            dirs["metrics"],
            f"{safe_name(model_name)}_optimized_train_validation_test_metrics.csv"
        ),
        index=False
    )

    optimized_predictions_df.to_csv(
        os.path.join(
            dirs["predictions"],
            f"{safe_name(model_name)}_optimized_train_validation_test_predictions.csv"
        ),
        index=False
    )

    return optimized_metrics_df, optimized_predictions_df


print("\n")
print("=" * 70)
print("BLOCK 10 PART 1 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 10
# PART 2
# Run Hyperparameter Optimization for MDR and Severity Models
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 10 PART 2 STARTED")
print("=" * 70)

# ============================================================
# Candidate Models
# ============================================================

MDR_OPTIMIZATION_MODELS = [
    "CatBoost",
    "XGBoost",
    "RandomForest",
    "LightGBM"
]

SEVERITY_OPTIMIZATION_MODELS = [
    "XGBoost",
    "RandomForest",
    "LightGBM",
    "CatBoost"
]

# Remove unavailable models automatically
available_search_spaces_binary = get_search_spaces("binary")
available_search_spaces_multiclass = get_search_spaces("multiclass")

MDR_OPTIMIZATION_MODELS = [
    m for m in MDR_OPTIMIZATION_MODELS
    if m in available_search_spaces_binary
]

SEVERITY_OPTIMIZATION_MODELS = [
    m for m in SEVERITY_OPTIMIZATION_MODELS
    if m in available_search_spaces_multiclass
]

print("\nMDR Optimization Models:")
print(MDR_OPTIMIZATION_MODELS)

print("\nSeverity Optimization Models:")
print(SEVERITY_OPTIMIZATION_MODELS)


# ============================================================
# MDR Optimization
# ============================================================

mdr_optimized_metrics_list = []
mdr_optimized_predictions_list = []
mdr_search_summary_list = []

for model_name in MDR_OPTIMIZATION_MODELS:

    if model_name in ["RandomForest"]:
        n_iter = N_ITER_TREE
    else:
        n_iter = N_ITER_BOOSTING

    best_model, best_params, best_metrics, search_df = validation_hyperparameter_search(
        task_name="MDR",
        task_type="binary",
        model_name=model_name,
        X_train_input=X_train_scaled,
        y_train_input=y_train_mdr,
        X_val_input=X_val_scaled,
        y_val_input=y_val_mdr,
        sample_weight_input=mdr_train_weights,
        classes=MDR_CLASSES,
        class_names=MDR_CLASS_NAMES,
        n_iter=n_iter
    )

    eval_metrics_df, eval_predictions_df = evaluate_optimized_model(
        task_name="MDR",
        task_type="binary",
        model_name=model_name,
        model=best_model,
        best_params=best_params,
        X_train_input=X_train_scaled,
        X_val_input=X_val_scaled,
        X_test_input=X_test_scaled,
        y_train_input=y_train_mdr,
        y_val_input=y_val_mdr,
        y_test_input=y_test_mdr,
        sample_weight_input=mdr_train_weights,
        classes=MDR_CLASSES,
        class_names=MDR_CLASS_NAMES
    )

    mdr_optimized_metrics_list.append(eval_metrics_df)
    mdr_optimized_predictions_list.append(eval_predictions_df)

    mdr_search_summary_list.append({
        "Task": "MDR",
        "Model": model_name,
        "Best_Params": best_params,
        "Validation_Primary_Score": best_metrics["Primary_Score"],
        "Validation_Secondary_Score": best_metrics["Secondary_Score"],
        "Validation_Tertiary_Score": best_metrics["Tertiary_Score"]
    })


# ============================================================
# Severity Optimization
# ============================================================

severity_optimized_metrics_list = []
severity_optimized_predictions_list = []
severity_search_summary_list = []

for model_name in SEVERITY_OPTIMIZATION_MODELS:

    if model_name in ["RandomForest"]:
        n_iter = N_ITER_TREE
    else:
        n_iter = N_ITER_BOOSTING

    best_model, best_params, best_metrics, search_df = validation_hyperparameter_search(
        task_name="Severity",
        task_type="multiclass",
        model_name=model_name,
        X_train_input=X_train_scaled,
        y_train_input=y_train_severity,
        X_val_input=X_val_scaled,
        y_val_input=y_val_severity,
        sample_weight_input=severity_train_weights,
        classes=SEVERITY_CLASSES,
        class_names=SEVERITY_CLASS_NAMES,
        n_iter=n_iter
    )

    eval_metrics_df, eval_predictions_df = evaluate_optimized_model(
        task_name="Severity",
        task_type="multiclass",
        model_name=model_name,
        model=best_model,
        best_params=best_params,
        X_train_input=X_train_scaled,
        X_val_input=X_val_scaled,
        X_test_input=X_test_scaled,
        y_train_input=y_train_severity,
        y_val_input=y_val_severity,
        y_test_input=y_test_severity,
        sample_weight_input=severity_train_weights,
        classes=SEVERITY_CLASSES,
        class_names=SEVERITY_CLASS_NAMES
    )

    severity_optimized_metrics_list.append(eval_metrics_df)
    severity_optimized_predictions_list.append(eval_predictions_df)

    severity_search_summary_list.append({
        "Task": "Severity",
        "Model": model_name,
        "Best_Params": best_params,
        "Validation_Primary_Score": best_metrics["Primary_Score"],
        "Validation_Secondary_Score": best_metrics["Secondary_Score"],
        "Validation_Tertiary_Score": best_metrics["Tertiary_Score"]
    })


# ============================================================
# Combine Optimized Outputs
# ============================================================

mdr_optimized_metrics_df = pd.concat(
    mdr_optimized_metrics_list,
    ignore_index=True
)

severity_optimized_metrics_df = pd.concat(
    severity_optimized_metrics_list,
    ignore_index=True
)

all_optimized_metrics_df = pd.concat(
    [
        mdr_optimized_metrics_df,
        severity_optimized_metrics_df
    ],
    ignore_index=True
)

mdr_optimized_predictions_df = pd.concat(
    mdr_optimized_predictions_list,
    ignore_index=True
)

severity_optimized_predictions_df = pd.concat(
    severity_optimized_predictions_list,
    ignore_index=True
)

all_search_summary_df = pd.DataFrame(
    mdr_search_summary_list + severity_search_summary_list
)

# Save combined files
all_optimized_metrics_df.to_csv(
    os.path.join(
        BLOCK10_METRIC_DIR,
        "block10_all_optimized_metrics.csv"
    ),
    index=False
)

mdr_optimized_metrics_df.to_csv(
    os.path.join(
        BLOCK10_METRIC_DIR,
        "mdr_optimized_metrics.csv"
    ),
    index=False
)

severity_optimized_metrics_df.to_csv(
    os.path.join(
        BLOCK10_METRIC_DIR,
        "severity_optimized_metrics.csv"
    ),
    index=False
)

mdr_optimized_predictions_df.to_csv(
    os.path.join(
        BLOCK10_PRED_DIR,
        "mdr_optimized_predictions.csv"
    ),
    index=False
)

severity_optimized_predictions_df.to_csv(
    os.path.join(
        BLOCK10_PRED_DIR,
        "severity_optimized_predictions.csv"
    ),
    index=False
)

all_search_summary_df.to_csv(
    os.path.join(
        BLOCK10_METRIC_DIR,
        "block10_best_hyperparameter_summary.csv"
    ),
    index=False
)


# ============================================================
# Optimized Comparison Tables
# ============================================================

mdr_optimized_test_table = (
    all_optimized_metrics_df[
        (all_optimized_metrics_df["Task"] == "MDR_Optimized") &
        (all_optimized_metrics_df["Split"] == "Test")
    ]
    .sort_values(
        by=["ROC_AUC", "F1"],
        ascending=False
    )
)

severity_optimized_test_table = (
    all_optimized_metrics_df[
        (all_optimized_metrics_df["Task"] == "Severity_Optimized") &
        (all_optimized_metrics_df["Split"] == "Test")
    ]
    .sort_values(
        by=["Macro_F1", "Balanced_Accuracy"],
        ascending=False
    )
)

mdr_optimized_test_table.to_csv(
    os.path.join(
        BLOCK10_METRIC_DIR,
        "mdr_optimized_test_comparison.csv"
    ),
    index=False
)

severity_optimized_test_table.to_csv(
    os.path.join(
        BLOCK10_METRIC_DIR,
        "severity_optimized_test_comparison.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("MDR OPTIMIZED TEST COMPARISON")
print("=" * 70)

display(mdr_optimized_test_table)

print("\n")
print("=" * 70)
print("SEVERITY OPTIMIZED TEST COMPARISON")
print("=" * 70)

display(severity_optimized_test_table)


# ============================================================
# Baseline vs Optimized Comparison
# ============================================================

baseline_metrics_path = os.path.join(
    OUTPUT_DIR,
    "block09_baseline_models",
    "metrics",
    "block09_all_baseline_metrics.csv"
)

if os.path.exists(baseline_metrics_path):

    baseline_metrics_df = pd.read_csv(baseline_metrics_path)

    baseline_mdr_test = baseline_metrics_df[
        (baseline_metrics_df["Task"] == "MDR") &
        (baseline_metrics_df["Split"] == "Test")
    ].copy()

    baseline_severity_test = baseline_metrics_df[
        (baseline_metrics_df["Task"] == "Severity") &
        (baseline_metrics_df["Split"] == "Test")
    ].copy()

    baseline_mdr_test["Model_Type"] = "Baseline"
    baseline_severity_test["Model_Type"] = "Baseline"

    optimized_mdr_test = mdr_optimized_test_table.copy()
    optimized_severity_test = severity_optimized_test_table.copy()

    optimized_mdr_test["Model_Type"] = "Optimized"
    optimized_severity_test["Model_Type"] = "Optimized"

    mdr_baseline_vs_optimized = pd.concat(
        [
            baseline_mdr_test,
            optimized_mdr_test
        ],
        ignore_index=True
    )

    severity_baseline_vs_optimized = pd.concat(
        [
            baseline_severity_test,
            optimized_severity_test
        ],
        ignore_index=True
    )

    mdr_baseline_vs_optimized.to_csv(
        os.path.join(
            BLOCK10_METRIC_DIR,
            "mdr_baseline_vs_optimized_comparison.csv"
        ),
        index=False
    )

    severity_baseline_vs_optimized.to_csv(
        os.path.join(
            BLOCK10_METRIC_DIR,
            "severity_baseline_vs_optimized_comparison.csv"
        ),
        index=False
    )

else:

    print("Baseline metrics file not found. Skipping baseline vs optimized comparison.")


# ============================================================
# Publication Figures
# ============================================================

def plot_optimized_metric_comparison(
    df,
    metric,
    title,
    save_path
):

    plot_df = df.copy()

    plt.figure(figsize=(9, 5))

    sns.barplot(
        data=plot_df,
        x=metric,
        y="Model",
        orient="h"
    )

    plt.title(title)
    plt.xlabel(metric)
    plt.ylabel("Model")
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


plot_optimized_metric_comparison(
    df=mdr_optimized_test_table,
    metric="ROC_AUC",
    title="Optimized MDR Models — Test ROC-AUC",
    save_path=os.path.join(
        BLOCK10_FIG_DIR,
        "figure_10a_optimized_mdr_roc_auc.png"
    )
)

plot_optimized_metric_comparison(
    df=mdr_optimized_test_table,
    metric="F1",
    title="Optimized MDR Models — Test F1-score",
    save_path=os.path.join(
        BLOCK10_FIG_DIR,
        "figure_10b_optimized_mdr_f1.png"
    )
)

plot_optimized_metric_comparison(
    df=severity_optimized_test_table,
    metric="Macro_F1",
    title="Optimized Severity Models — Test Macro F1-score",
    save_path=os.path.join(
        BLOCK10_FIG_DIR,
        "figure_10c_optimized_severity_macro_f1.png"
    )
)

plot_optimized_metric_comparison(
    df=severity_optimized_test_table,
    metric="Balanced_Accuracy",
    title="Optimized Severity Models — Test Balanced Accuracy",
    save_path=os.path.join(
        BLOCK10_FIG_DIR,
        "figure_10d_optimized_severity_balanced_accuracy.png"
    )
)


# ============================================================
# Final Summary
# ============================================================

block10_summary = pd.DataFrame({
    "Output_Category": [
        "All optimized metrics",
        "MDR optimized metrics",
        "Severity optimized metrics",
        "Best hyperparameter summary",
        "MDR optimized predictions",
        "Severity optimized predictions",
        "Individual search results",
        "Optimized model PKL files",
        "Train/Validation/Test reports",
        "Train/Validation/Test confusion matrices",
        "Train/Validation/Test ROC curves",
        "Train/Validation/Test PR curves",
        "Train/Validation/Test calibration curves",
        "Train/Validation/Test probability distributions",
        "Learning curves",
        "Baseline vs optimized comparison"
    ],
    "Saved": [
        "block10_all_optimized_metrics.csv",
        "mdr_optimized_metrics.csv",
        "severity_optimized_metrics.csv",
        "block10_best_hyperparameter_summary.csv",
        "mdr_optimized_predictions.csv",
        "severity_optimized_predictions.csv",
        "search_results/*.csv",
        "optimized_models/*.pkl and per-model folders",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes, if Block 09 metrics exists"
    ]
})

block10_summary.to_csv(
    os.path.join(
        BLOCK10_DIR,
        "block10_output_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("BLOCK 10 GENERATED FILES")
print("=" * 70)

print("- metrics/block10_all_optimized_metrics.csv")
print("- metrics/mdr_optimized_metrics.csv")
print("- metrics/severity_optimized_metrics.csv")
print("- metrics/block10_best_hyperparameter_summary.csv")
print("- metrics/mdr_optimized_test_comparison.csv")
print("- metrics/severity_optimized_test_comparison.csv")
print("- predictions/mdr_optimized_predictions.csv")
print("- predictions/severity_optimized_predictions.csv")
print("- search_results/*_hyperparameter_search_results.csv")
print("- optimized model .pkl files")
print("- reports, figures, confusion matrices, ROC, PR, calibration, probability distributions, learning curves")

print("\n")
print("=" * 70)
print("BLOCK 10 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 11
# Feature Selection
# PART 1
# Feature Importance + Consensus Feature Ranking
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 11 STARTED — FEATURE SELECTION")
print("=" * 70)

import os
import ast
import time
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score
)
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False


# ============================================================
# Dependency Check
# ============================================================

required_objects = [
    "X_train_scaled",
    "X_val_scaled",
    "X_test_scaled",
    "y_train_mdr",
    "y_val_mdr",
    "y_test_mdr",
    "y_train_severity",
    "y_val_severity",
    "y_test_severity",
    "mdr_train_weights",
    "severity_train_weights",
    "fit_model",
    "get_aligned_proba",
    "calculate_metrics",
    "evaluate_split_and_export",
    "plot_learning_curve_custom",
    "safe_name",
    "MDR_CLASSES",
    "MDR_CLASS_NAMES",
    "SEVERITY_CLASSES",
    "SEVERITY_CLASS_NAMES"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if len(missing_objects) > 0:
    raise RuntimeError(
        "Please run previous blocks before BLOCK 11. Missing: "
        + ", ".join(missing_objects)
    )


# ============================================================
# Output Directories
# ============================================================

BLOCK11_DIR = os.path.join(OUTPUT_DIR, "block11_feature_selection")

BLOCK11_MODEL_DIR = os.path.join(BLOCK11_DIR, "models")
BLOCK11_METRIC_DIR = os.path.join(BLOCK11_DIR, "metrics")
BLOCK11_REPORT_DIR = os.path.join(BLOCK11_DIR, "classification_reports")
BLOCK11_PRED_DIR = os.path.join(BLOCK11_DIR, "predictions")
BLOCK11_FIG_DIR = os.path.join(BLOCK11_DIR, "figures")
BLOCK11_RANK_DIR = os.path.join(BLOCK11_DIR, "feature_rankings")
BLOCK11_SELECTED_DIR = os.path.join(BLOCK11_DIR, "selected_features")

for d in [
    BLOCK11_DIR,
    BLOCK11_MODEL_DIR,
    BLOCK11_METRIC_DIR,
    BLOCK11_REPORT_DIR,
    BLOCK11_PRED_DIR,
    BLOCK11_FIG_DIR,
    BLOCK11_RANK_DIR,
    BLOCK11_SELECTED_DIR
]:
    os.makedirs(d, exist_ok=True)

FIG_DPI = 300


# ============================================================
# Feature Names
# ============================================================

feature_names = list(X_train_scaled.columns)

print("\nNumber of Input Features:", len(feature_names))
print("Features:")
print(feature_names)


# ============================================================
# Helper Functions
# ============================================================

def normalize_score(values):

    values = np.array(values, dtype=float)

    if np.all(np.isnan(values)):
        return np.zeros_like(values)

    values = np.nan_to_num(values, nan=0.0)

    min_val = np.min(values)
    max_val = np.max(values)

    if max_val - min_val == 0:
        return np.ones_like(values)

    return (values - min_val) / (max_val - min_val)


def get_importance_model(model_name, task_type):

    if model_name == "RandomForest":

        return RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            random_state=SEED,
            n_jobs=-1
        )

    if model_name == "XGBoost" and HAS_XGB:

        if task_type == "binary":

            return XGBClassifier(
                n_estimators=500,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.90,
                colsample_bytree=0.90,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1
            )

        return XGBClassifier(
            n_estimators=500,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.90,
            colsample_bytree=0.90,
            objective="multi:softprob",
            num_class=4,
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1
        )

    if model_name == "LightGBM" and HAS_LGBM:

        if task_type == "binary":

            return LGBMClassifier(
                n_estimators=500,
                learning_rate=0.05,
                objective="binary",
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )

        return LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            objective="multiclass",
            num_class=4,
            random_state=SEED,
            n_jobs=-1,
            verbose=-1
        )

    if model_name == "CatBoost" and HAS_CATBOOST:

        if task_type == "binary":

            return CatBoostClassifier(
                iterations=500,
                learning_rate=0.05,
                depth=6,
                loss_function="Logloss",
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False
            )

        return CatBoostClassifier(
            iterations=500,
            learning_rate=0.05,
            depth=6,
            loss_function="MultiClass",
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False
        )

    return None


def get_model_feature_importance(model, model_name):

    if hasattr(model, "feature_importances_"):
        return np.array(model.feature_importances_, dtype=float)

    if model_name == "CatBoost":
        return np.array(model.get_feature_importance(), dtype=float)

    return np.zeros(len(feature_names))


def permutation_scoring_name(task_type):

    if task_type == "binary":
        return "roc_auc"

    return "f1_macro"


def plot_feature_ranking(
    ranking_df,
    score_column,
    title,
    save_path,
    top_n=20
):

    plot_df = ranking_df.head(top_n).copy()

    plt.figure(figsize=(9, 7))

    sns.barplot(
        data=plot_df,
        x=score_column,
        y="Feature",
        orient="h"
    )

    plt.title(title)
    plt.xlabel(score_column)
    plt.ylabel("Feature")
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


# ============================================================
# Feature Ranking Function
# ============================================================

def generate_feature_ranking(
    task_name,
    task_type,
    X_train_input,
    y_train_input,
    X_val_input,
    y_val_input,
    sample_weight_input
):

    print("\n")
    print("=" * 70)
    print(f"GENERATING FEATURE RANKING: {task_name}")
    print("=" * 70)

    ranking_df = pd.DataFrame({
        "Feature": feature_names
    })

    # --------------------------------------------------------
    # Mutual Information
    # --------------------------------------------------------

    mi_scores = mutual_info_classif(
        X_train_input,
        y_train_input,
        discrete_features=False,
        random_state=SEED
    )

    ranking_df["Mutual_Information"] = mi_scores
    ranking_df["Mutual_Information_Normalized"] = normalize_score(mi_scores)

    # --------------------------------------------------------
    # Model-Based Importance
    # --------------------------------------------------------

    importance_models = ["RandomForest"]

    if HAS_XGB:
        importance_models.append("XGBoost")

    if HAS_LGBM:
        importance_models.append("LightGBM")

    if HAS_CATBOOST:
        importance_models.append("CatBoost")

    for model_name in importance_models:

        print(f"Training importance model: {task_name} — {model_name}")

        model = get_importance_model(
            model_name=model_name,
            task_type=task_type
        )

        model = fit_model(
            model,
            X_train_input,
            y_train_input,
            sample_weight=sample_weight_input
        )

        importance_values = get_model_feature_importance(
            model,
            model_name
        )

        ranking_df[f"{model_name}_Importance"] = importance_values
        ranking_df[f"{model_name}_Importance_Normalized"] = normalize_score(
            importance_values
        )

        joblib.dump(
            model,
            os.path.join(
                BLOCK11_MODEL_DIR,
                f"{task_name}_{model_name}_feature_importance_model.pkl"
            )
        )

    # --------------------------------------------------------
    # Permutation Importance
    # --------------------------------------------------------

    print(f"Calculating permutation importance: {task_name}")

    if task_type == "binary":
        permutation_model_name = "LightGBM" if HAS_LGBM else "RandomForest"
    else:
        permutation_model_name = "XGBoost" if HAS_XGB else "RandomForest"

    permutation_model = get_importance_model(
        model_name=permutation_model_name,
        task_type=task_type
    )

    permutation_model = fit_model(
        permutation_model,
        X_train_input,
        y_train_input,
        sample_weight=sample_weight_input
    )

    perm_result = permutation_importance(
        permutation_model,
        X_val_input,
        y_val_input,
        scoring=permutation_scoring_name(task_type),
        n_repeats=10,
        random_state=SEED,
        n_jobs=-1
    )

    ranking_df["Permutation_Importance_Mean"] = perm_result.importances_mean
    ranking_df["Permutation_Importance_STD"] = perm_result.importances_std
    ranking_df["Permutation_Importance_Normalized"] = normalize_score(
        perm_result.importances_mean
    )

    # --------------------------------------------------------
    # Consensus Score
    # --------------------------------------------------------

    normalized_columns = [
        col for col in ranking_df.columns
        if col.endswith("_Normalized")
    ]

    ranking_df["Consensus_Feature_Score"] = ranking_df[
        normalized_columns
    ].mean(axis=1)

    ranking_df["Consensus_Rank"] = ranking_df[
        "Consensus_Feature_Score"
    ].rank(
        ascending=False,
        method="dense"
    ).astype(int)

    ranking_df = ranking_df.sort_values(
        by="Consensus_Feature_Score",
        ascending=False
    ).reset_index(drop=True)

    ranking_df.to_csv(
        os.path.join(
            BLOCK11_RANK_DIR,
            f"{task_name}_feature_ranking.csv"
        ),
        index=False
    )

    # --------------------------------------------------------
    # Plot Rankings
    # --------------------------------------------------------

    plot_feature_ranking(
        ranking_df=ranking_df,
        score_column="Consensus_Feature_Score",
        title=f"{task_name} Consensus Feature Importance Ranking",
        save_path=os.path.join(
            BLOCK11_FIG_DIR,
            f"figure_11_{task_name}_consensus_feature_ranking.png"
        ),
        top_n=min(20, len(feature_names))
    )

    plot_feature_ranking(
        ranking_df=ranking_df,
        score_column="Permutation_Importance_Mean",
        title=f"{task_name} Permutation Feature Importance",
        save_path=os.path.join(
            BLOCK11_FIG_DIR,
            f"figure_11_{task_name}_permutation_importance.png"
        ),
        top_n=min(20, len(feature_names))
    )

    print("\nTop 10 Features:")
    display(
        ranking_df[
            [
                "Feature",
                "Consensus_Feature_Score",
                "Consensus_Rank"
            ]
        ].head(10)
    )

    return ranking_df


# ============================================================
# Generate MDR and Severity Feature Rankings
# ============================================================

mdr_feature_ranking = generate_feature_ranking(
    task_name="MDR",
    task_type="binary",
    X_train_input=X_train_scaled,
    y_train_input=y_train_mdr,
    X_val_input=X_val_scaled,
    y_val_input=y_val_mdr,
    sample_weight_input=mdr_train_weights
)

severity_feature_ranking = generate_feature_ranking(
    task_name="Severity",
    task_type="multiclass",
    X_train_input=X_train_scaled,
    y_train_input=y_train_severity,
    X_val_input=X_val_scaled,
    y_val_input=y_val_severity,
    sample_weight_input=severity_train_weights
)


# ============================================================
# Save Selected Feature Lists
# ============================================================

def save_top_feature_sets(task_name, ranking_df):

    candidate_k_values = [
        5,
        8,
        10,
        12,
        15,
        len(feature_names)
    ]

    candidate_k_values = sorted(
        list(
            set(
                [
                    k for k in candidate_k_values
                    if k <= len(feature_names)
                ]
            )
        )
    )

    selected_rows = []

    for k in candidate_k_values:

        selected_features = ranking_df["Feature"].head(k).tolist()

        pd.DataFrame({
            "Selected_Feature": selected_features
        }).to_csv(
            os.path.join(
                BLOCK11_SELECTED_DIR,
                f"{task_name}_top_{k}_features.csv"
            ),
            index=False
        )

        selected_rows.append({
            "Task": task_name,
            "Top_K": k,
            "Selected_Features": selected_features
        })

    return pd.DataFrame(selected_rows)


mdr_selected_feature_sets = save_top_feature_sets(
    task_name="MDR",
    ranking_df=mdr_feature_ranking
)

severity_selected_feature_sets = save_top_feature_sets(
    task_name="Severity",
    ranking_df=severity_feature_ranking
)

all_selected_feature_sets = pd.concat(
    [
        mdr_selected_feature_sets,
        severity_selected_feature_sets
    ],
    ignore_index=True
)

all_selected_feature_sets.to_csv(
    os.path.join(
        BLOCK11_SELECTED_DIR,
        "all_candidate_selected_feature_sets.csv"
    ),
    index=False
)


print("\n")
print("=" * 70)
print("BLOCK 11 PART 1 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 11
# Feature Selection
# PART 2
# Selected Feature Evaluation + Final Selected-Feature Models
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 11 PART 2 STARTED")
print("=" * 70)


# ============================================================
# Model Builder for Feature Selection Evaluation
# ============================================================

def build_feature_selection_model(model_name, task_type):

    if model_name == "RandomForest":

        return RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            random_state=SEED,
            n_jobs=-1
        )

    if model_name == "XGBoost" and HAS_XGB:

        if task_type == "binary":

            return XGBClassifier(
                n_estimators=500,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.90,
                colsample_bytree=0.90,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1
            )

        return XGBClassifier(
            n_estimators=500,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.90,
            colsample_bytree=0.90,
            objective="multi:softprob",
            num_class=4,
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1
        )

    if model_name == "LightGBM" and HAS_LGBM:

        if task_type == "binary":

            return LGBMClassifier(
                n_estimators=700,
                learning_rate=0.01,
                num_leaves=15,
                objective="binary",
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )

        return LGBMClassifier(
            n_estimators=700,
            learning_rate=0.05,
            num_leaves=15,
            objective="multiclass",
            num_class=4,
            random_state=SEED,
            n_jobs=-1,
            verbose=-1
        )

    if model_name == "CatBoost" and HAS_CATBOOST:

        if task_type == "binary":

            return CatBoostClassifier(
                iterations=500,
                learning_rate=0.03,
                depth=7,
                l2_leaf_reg=3,
                loss_function="Logloss",
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False
            )

        return CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=8,
            l2_leaf_reg=3,
            loss_function="MultiClass",
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False
        )

    raise ValueError(f"Model not available or unknown: {model_name}")


# ============================================================
# Quick Validation Evaluation
# ============================================================

def quick_selected_feature_validation(
    task_name,
    task_type,
    ranking_df,
    candidate_models,
    X_train_input,
    X_val_input,
    y_train_input,
    y_val_input,
    sample_weight_input,
    classes
):

    print("\n")
    print("=" * 70)
    print(f"SELECTED FEATURE VALIDATION SEARCH: {task_name}")
    print("=" * 70)

    candidate_k_values = [
        5,
        8,
        10,
        12,
        15,
        len(feature_names)
    ]

    candidate_k_values = sorted(
        list(
            set(
                [
                    k for k in candidate_k_values
                    if k <= len(feature_names)
                ]
            )
        )
    )

    rows = []

    best_tuple = (-np.inf, -np.inf, -np.inf)
    best_model = None
    best_model_name = None
    best_k = None
    best_features = None

    for k in candidate_k_values:

        selected_features = ranking_df["Feature"].head(k).tolist()

        X_train_selected = X_train_input[selected_features]
        X_val_selected = X_val_input[selected_features]

        for model_name in candidate_models:

            try:

                model = build_feature_selection_model(
                    model_name=model_name,
                    task_type=task_type
                )

                start_time = time.time()

                model = fit_model(
                    model,
                    X_train_selected,
                    y_train_input,
                    sample_weight=sample_weight_input
                )

                elapsed = time.time() - start_time

                val_pred = np.ravel(
                    model.predict(X_val_selected)
                ).astype(int)

                val_proba = get_aligned_proba(
                    model,
                    X_val_selected,
                    classes
                )

                if task_type == "binary":

                    primary_score = roc_auc_score(
                        y_val_input,
                        val_proba[:, 1]
                    )

                    secondary_score = f1_score(
                        y_val_input,
                        val_pred,
                        zero_division=0
                    )

                    tertiary_score = average_precision_score(
                        y_val_input,
                        val_proba[:, 1]
                    )

                else:

                    primary_score = f1_score(
                        y_val_input,
                        val_pred,
                        average="macro",
                        zero_division=0
                    )

                    secondary_score = balanced_accuracy_score(
                        y_val_input,
                        val_pred
                    )

                    tertiary_score = np.nan

                row = {
                    "Task": task_name,
                    "Model": model_name,
                    "Top_K": k,
                    "Number_of_Features": len(selected_features),
                    "Selected_Features": selected_features,
                    "Primary_Score": primary_score,
                    "Secondary_Score": secondary_score,
                    "Tertiary_Score": tertiary_score,
                    "Training_Time_Seconds": round(elapsed, 4)
                }

                rows.append(row)

                score_tuple = (
                    primary_score,
                    secondary_score,
                    tertiary_score if not pd.isna(tertiary_score) else -np.inf
                )

                if score_tuple > best_tuple:

                    best_tuple = score_tuple
                    best_model = model
                    best_model_name = model_name
                    best_k = k
                    best_features = selected_features

                print(
                    f"{task_name} | {model_name} | Top-{k} | "
                    f"Primary={primary_score:.4f} | "
                    f"Secondary={secondary_score:.4f}"
                )

            except Exception as e:

                rows.append({
                    "Task": task_name,
                    "Model": model_name,
                    "Top_K": k,
                    "Search_Status": "Failed",
                    "Error": str(e)
                })

                print(f"FAILED: {task_name} | {model_name} | Top-{k} | {e}")

    results_df = pd.DataFrame(rows)

    results_df.to_csv(
        os.path.join(
            BLOCK11_METRIC_DIR,
            f"{task_name}_selected_feature_validation_results.csv"
        ),
        index=False
    )

    best_summary = {
        "Task": task_name,
        "Best_Model": best_model_name,
        "Best_Top_K": best_k,
        "Best_Number_of_Features": len(best_features),
        "Best_Selected_Features": best_features,
        "Best_Primary_Score": best_tuple[0],
        "Best_Secondary_Score": best_tuple[1],
        "Best_Tertiary_Score": best_tuple[2]
    }

    print("\nBest Selected Feature Model:")
    print(best_summary)

    return results_df, best_summary, best_model


# ============================================================
# Candidate Models
# ============================================================

MDR_FS_MODELS = []

if HAS_LGBM:
    MDR_FS_MODELS.append("LightGBM")

MDR_FS_MODELS.append("RandomForest")

if HAS_XGB:
    MDR_FS_MODELS.append("XGBoost")

if HAS_CATBOOST:
    MDR_FS_MODELS.append("CatBoost")


SEVERITY_FS_MODELS = []

if HAS_XGB:
    SEVERITY_FS_MODELS.append("XGBoost")

if HAS_CATBOOST:
    SEVERITY_FS_MODELS.append("CatBoost")

if HAS_LGBM:
    SEVERITY_FS_MODELS.append("LightGBM")

SEVERITY_FS_MODELS.append("RandomForest")


print("\nMDR Feature Selection Models:")
print(MDR_FS_MODELS)

print("\nSeverity Feature Selection Models:")
print(SEVERITY_FS_MODELS)


# ============================================================
# Run Selected Feature Search
# ============================================================

mdr_fs_validation_results, mdr_fs_best_summary, mdr_fs_best_model = quick_selected_feature_validation(
    task_name="MDR",
    task_type="binary",
    ranking_df=mdr_feature_ranking,
    candidate_models=MDR_FS_MODELS,
    X_train_input=X_train_scaled,
    X_val_input=X_val_scaled,
    y_train_input=y_train_mdr,
    y_val_input=y_val_mdr,
    sample_weight_input=mdr_train_weights,
    classes=MDR_CLASSES
)

severity_fs_validation_results, severity_fs_best_summary, severity_fs_best_model = quick_selected_feature_validation(
    task_name="Severity",
    task_type="multiclass",
    ranking_df=severity_feature_ranking,
    candidate_models=SEVERITY_FS_MODELS,
    X_train_input=X_train_scaled,
    X_val_input=X_val_scaled,
    y_train_input=y_train_severity,
    y_val_input=y_val_severity,
    sample_weight_input=severity_train_weights,
    classes=SEVERITY_CLASSES
)


# ============================================================
# Plot Feature Subset Search Results
# ============================================================

def plot_feature_subset_search(
    df,
    task_name,
    metric,
    save_path
):

    plt.figure(figsize=(10, 6))

    sns.lineplot(
        data=df,
        x="Top_K",
        y=metric,
        hue="Model",
        marker="o"
    )

    plt.title(f"{task_name} Feature Selection Validation — {metric}")
    plt.xlabel("Number of Selected Features")
    plt.ylabel(metric)
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


plot_feature_subset_search(
    df=mdr_fs_validation_results,
    task_name="MDR",
    metric="Primary_Score",
    save_path=os.path.join(
        BLOCK11_FIG_DIR,
        "figure_11_mdr_feature_subset_validation_roc_auc.png"
    )
)

plot_feature_subset_search(
    df=mdr_fs_validation_results,
    task_name="MDR",
    metric="Secondary_Score",
    save_path=os.path.join(
        BLOCK11_FIG_DIR,
        "figure_11_mdr_feature_subset_validation_f1.png"
    )
)

plot_feature_subset_search(
    df=severity_fs_validation_results,
    task_name="Severity",
    metric="Primary_Score",
    save_path=os.path.join(
        BLOCK11_FIG_DIR,
        "figure_11_severity_feature_subset_validation_macro_f1.png"
    )
)

plot_feature_subset_search(
    df=severity_fs_validation_results,
    task_name="Severity",
    metric="Secondary_Score",
    save_path=os.path.join(
        BLOCK11_FIG_DIR,
        "figure_11_severity_feature_subset_validation_balanced_accuracy.png"
    )
)


# ============================================================
# Final Full Evaluation for Best Selected-Feature Models
# ============================================================

def create_block11_model_dirs(task_name, model_name):

    task_dir = os.path.join(BLOCK11_DIR, task_name)
    model_task_dir = os.path.join(task_dir, model_name)

    dirs = {
        "task": task_dir,
        "model": model_task_dir,
        "models": os.path.join(model_task_dir, "models"),
        "metrics": os.path.join(model_task_dir, "metrics"),
        "reports": os.path.join(model_task_dir, "classification_reports"),
        "predictions": os.path.join(model_task_dir, "predictions"),
        "figures": os.path.join(model_task_dir, "figures")
    }

    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    return dirs


def full_evaluate_selected_feature_model(
    task_name,
    task_type,
    best_summary,
    model,
    X_train_input,
    X_val_input,
    X_test_input,
    y_train_input,
    y_val_input,
    y_test_input,
    sample_weight_input,
    classes,
    class_names
):

    model_name = best_summary["Best_Model"]
    selected_features = best_summary["Best_Selected_Features"]

    print("\n")
    print("=" * 70)
    print(f"FINAL SELECTED-FEATURE EVALUATION: {task_name} — {model_name}")
    print("=" * 70)

    print("Selected Features:")
    print(selected_features)

    dirs = create_block11_model_dirs(
        task_name=task_name,
        model_name=model_name
    )

    X_train_selected = X_train_input[selected_features]
    X_val_selected = X_val_input[selected_features]
    X_test_selected = X_test_input[selected_features]

    # Refit final model on selected features
    final_model = build_feature_selection_model(
        model_name=model_name,
        task_type=task_type
    )

    final_model = fit_model(
        final_model,
        X_train_selected,
        y_train_input,
        sample_weight=sample_weight_input
    )

    joblib.dump(
        final_model,
        os.path.join(
            dirs["models"],
            f"{safe_name(model_name)}_selected_features.pkl"
        )
    )

    pd.DataFrame({
        "Selected_Feature": selected_features
    }).to_csv(
        os.path.join(
            dirs["metrics"],
            f"{safe_name(model_name)}_final_selected_features.csv"
        ),
        index=False
    )

    pd.DataFrame([best_summary]).to_csv(
        os.path.join(
            dirs["metrics"],
            f"{safe_name(model_name)}_selected_feature_summary.csv"
        ),
        index=False
    )

    metrics_list = []
    predictions_list = []

    for split_name, X_split, y_split in [

        ("Train", X_train_selected, y_train_input),
        ("Validation", X_val_selected, y_val_input),
        ("Test", X_test_selected, y_test_input)

    ]:

        metrics, pred_df = evaluate_split_and_export(
            model=final_model,
            X_split=X_split,
            y_true=y_split,
            split_name=split_name,
            task_name=f"{task_name}_Feature_Selected",
            task_type=task_type,
            model_name=model_name,
            classes=classes,
            class_names=class_names,
            dirs=dirs
        )

        metrics["Selected_Feature_Count"] = len(selected_features)
        metrics["Selected_Features"] = selected_features

        metrics_list.append(metrics)
        predictions_list.append(pred_df)

    plot_learning_curve_custom(
        model=final_model,
        X_train_input=X_train_selected,
        y_train_input=y_train_input,
        X_val_input=X_val_selected,
        y_val_input=y_val_input,
        sample_weight_input=sample_weight_input,
        task_type=task_type,
        task_name=f"{task_name}_Feature_Selected",
        model_name=model_name,
        dirs=dirs
    )

    metrics_df = pd.DataFrame(metrics_list)

    predictions_df = pd.concat(
        predictions_list,
        ignore_index=True
    )

    metrics_df.to_csv(
        os.path.join(
            dirs["metrics"],
            f"{safe_name(model_name)}_feature_selected_train_validation_test_metrics.csv"
        ),
        index=False
    )

    predictions_df.to_csv(
        os.path.join(
            dirs["predictions"],
            f"{safe_name(model_name)}_feature_selected_predictions.csv"
        ),
        index=False
    )

    return metrics_df, predictions_df, final_model


mdr_fs_metrics_df, mdr_fs_predictions_df, mdr_fs_final_model = full_evaluate_selected_feature_model(
    task_name="MDR",
    task_type="binary",
    best_summary=mdr_fs_best_summary,
    model=mdr_fs_best_model,
    X_train_input=X_train_scaled,
    X_val_input=X_val_scaled,
    X_test_input=X_test_scaled,
    y_train_input=y_train_mdr,
    y_val_input=y_val_mdr,
    y_test_input=y_test_mdr,
    sample_weight_input=mdr_train_weights,
    classes=MDR_CLASSES,
    class_names=MDR_CLASS_NAMES
)

severity_fs_metrics_df, severity_fs_predictions_df, severity_fs_final_model = full_evaluate_selected_feature_model(
    task_name="Severity",
    task_type="multiclass",
    best_summary=severity_fs_best_summary,
    model=severity_fs_best_model,
    X_train_input=X_train_scaled,
    X_val_input=X_val_scaled,
    X_test_input=X_test_scaled,
    y_train_input=y_train_severity,
    y_val_input=y_val_severity,
    y_test_input=y_test_severity,
    sample_weight_input=severity_train_weights,
    classes=SEVERITY_CLASSES,
    class_names=SEVERITY_CLASS_NAMES
)


# ============================================================
# Combine and Save Final Block 11 Outputs
# ============================================================

block11_all_metrics = pd.concat(
    [
        mdr_fs_metrics_df,
        severity_fs_metrics_df
    ],
    ignore_index=True
)

block11_all_predictions = pd.concat(
    [
        mdr_fs_predictions_df,
        severity_fs_predictions_df
    ],
    ignore_index=True
)

block11_best_feature_summary = pd.DataFrame(
    [
        mdr_fs_best_summary,
        severity_fs_best_summary
    ]
)

block11_all_metrics.to_csv(
    os.path.join(
        BLOCK11_METRIC_DIR,
        "block11_all_feature_selected_metrics.csv"
    ),
    index=False
)

block11_all_predictions.to_csv(
    os.path.join(
        BLOCK11_PRED_DIR,
        "block11_all_feature_selected_predictions.csv"
    ),
    index=False
)

block11_best_feature_summary.to_csv(
    os.path.join(
        BLOCK11_METRIC_DIR,
        "block11_best_selected_feature_summary.csv"
    ),
    index=False
)


# ============================================================
# Test Comparison Tables
# ============================================================

mdr_fs_test_table = block11_all_metrics[
    (block11_all_metrics["Task"] == "MDR_Feature_Selected") &
    (block11_all_metrics["Split"] == "Test")
].copy()

severity_fs_test_table = block11_all_metrics[
    (block11_all_metrics["Task"] == "Severity_Feature_Selected") &
    (block11_all_metrics["Split"] == "Test")
].copy()

mdr_fs_test_table.to_csv(
    os.path.join(
        BLOCK11_METRIC_DIR,
        "mdr_feature_selected_test_result.csv"
    ),
    index=False
)

severity_fs_test_table.to_csv(
    os.path.join(
        BLOCK11_METRIC_DIR,
        "severity_feature_selected_test_result.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("MDR FEATURE-SELECTED TEST RESULT")
print("=" * 70)

display(mdr_fs_test_table)

print("\n")
print("=" * 70)
print("SEVERITY FEATURE-SELECTED TEST RESULT")
print("=" * 70)

display(severity_fs_test_table)


# ============================================================
# Block 11 Summary
# ============================================================

block11_summary = pd.DataFrame({
    "Output_Category": [
        "MDR feature ranking",
        "Severity feature ranking",
        "Candidate selected feature sets",
        "MDR selected feature validation",
        "Severity selected feature validation",
        "Best selected feature summary",
        "Final feature-selected metrics",
        "Final feature-selected predictions",
        "Feature-selected model PKL files",
        "Train/Validation/Test reports",
        "Train/Validation/Test confusion matrices",
        "Train/Validation/Test ROC curves",
        "Train/Validation/Test PR curves",
        "Train/Validation/Test calibration curves",
        "Train/Validation/Test probability distributions",
        "Learning curves"
    ],
    "Saved": [
        "feature_rankings/MDR_feature_ranking.csv",
        "feature_rankings/Severity_feature_ranking.csv",
        "selected_features/*.csv",
        "metrics/MDR_selected_feature_validation_results.csv",
        "metrics/Severity_selected_feature_validation_results.csv",
        "metrics/block11_best_selected_feature_summary.csv",
        "metrics/block11_all_feature_selected_metrics.csv",
        "predictions/block11_all_feature_selected_predictions.csv",
        "models and per-task folders",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})

block11_summary.to_csv(
    os.path.join(
        BLOCK11_DIR,
        "block11_output_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("BLOCK 11 GENERATED FILES")
print("=" * 70)

print("- feature_rankings/MDR_feature_ranking.csv")
print("- feature_rankings/Severity_feature_ranking.csv")
print("- selected_features/*.csv")
print("- metrics/MDR_selected_feature_validation_results.csv")
print("- metrics/Severity_selected_feature_validation_results.csv")
print("- metrics/block11_best_selected_feature_summary.csv")
print("- metrics/block11_all_feature_selected_metrics.csv")
print("- predictions/block11_all_feature_selected_predictions.csv")
print("- Feature ranking plots")
print("- Feature subset validation plots")
print("- Train/Validation/Test reports, confusion matrices, ROC, PR, calibration, probability distributions")
print("- Learning curves")
print("- Selected-feature model .pkl files")

print("\n")
print("=" * 70)
print("BLOCK 11 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 12
# SHAP Explainability / XAI
# PART 1
# Final XAI Models + SHAP Value Computation
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 12 STARTED — SHAP EXPLAINABILITY / XAI")
print("=" * 70)

import os
import time
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ============================================================
# SHAP Import
# ============================================================

try:
    import shap
    HAS_SHAP = True
    shap.initjs()
except Exception as e:
    HAS_SHAP = False
    print("SHAP is not available:", e)

if not HAS_SHAP:
    raise RuntimeError(
        "SHAP package is not installed. Please enable/install SHAP before running Block 12."
    )

# ============================================================
# Model Imports
# ============================================================

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False


# ============================================================
# Dependency Check
# ============================================================

required_objects = [
    "X_train_scaled",
    "X_val_scaled",
    "X_test_scaled",
    "y_train_mdr",
    "y_val_mdr",
    "y_test_mdr",
    "y_train_severity",
    "y_val_severity",
    "y_test_severity",
    "mdr_train_weights",
    "severity_train_weights",
    "MDR_CLASSES",
    "MDR_CLASS_NAMES",
    "SEVERITY_CLASSES",
    "SEVERITY_CLASS_NAMES",
    "SEVERITY_ORDER",
    "ID_TO_SEVERITY",
    "SEED"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if len(missing_objects) > 0:
    raise RuntimeError(
        "Please run previous blocks before BLOCK 12. Missing: "
        + ", ".join(missing_objects)
    )


# ============================================================
# Output Directories
# ============================================================

BLOCK12_DIR = os.path.join(OUTPUT_DIR, "block12_shap_xai")

BLOCK12_MODEL_DIR = os.path.join(BLOCK12_DIR, "models")
BLOCK12_VALUE_DIR = os.path.join(BLOCK12_DIR, "shap_values")
BLOCK12_TABLE_DIR = os.path.join(BLOCK12_DIR, "tables")
BLOCK12_FIG_DIR = os.path.join(BLOCK12_DIR, "figures")
BLOCK12_LOCAL_DIR = os.path.join(BLOCK12_DIR, "local_explanations")

for d in [
    BLOCK12_DIR,
    BLOCK12_MODEL_DIR,
    BLOCK12_VALUE_DIR,
    BLOCK12_TABLE_DIR,
    BLOCK12_FIG_DIR,
    BLOCK12_LOCAL_DIR
]:
    os.makedirs(d, exist_ok=True)

FIG_DPI = 300


# ============================================================
# SHAP Runtime Settings
# ============================================================

# To keep Kaggle runtime manageable.
SHAP_SAMPLE_SIZE = 1000
LOCAL_EXPLANATION_SAMPLES = 5
TOP_N_FEATURES = 15
TOP_N_DEPENDENCE = 5

rng = np.random.default_rng(SEED)


# ============================================================
# Feature Selection for XAI
# ============================================================

# MDR: Block 11 showed full 19 features are better/needed.
MDR_XAI_FEATURES = list(X_train_scaled.columns)

# Severity: Block 11 selected compact 10-feature subset.
if "severity_fs_best_summary" in globals():
    SEVERITY_XAI_FEATURES = severity_fs_best_summary["Best_Selected_Features"]
else:
    SEVERITY_XAI_FEATURES = [
        "Mean_MIC",
        "Broth_Dilution_Count",
        "Antibiotics_Tested",
        "Median_MIC",
        "Log_Mean_MIC",
        "Std_MIC",
        "Missing_Testing_Standard_Ratio",
        "Coverage_Ratio",
        "EUCAST_Count",
        "Disk_Diffusion_Count"
    ]

print("\nMDR XAI Features:", len(MDR_XAI_FEATURES))
print(MDR_XAI_FEATURES)

print("\nSeverity XAI Features:", len(SEVERITY_XAI_FEATURES))
print(SEVERITY_XAI_FEATURES)


# ============================================================
# Final XAI Model Builders
# ============================================================

def build_mdr_xai_model():

    # Prefer LightGBM because Block 10/11 showed very strong MDR performance
    # and SHAP TreeExplainer works efficiently with LightGBM.
    if HAS_LGBM:

        return LGBMClassifier(
            objective="binary",
            n_estimators=700,
            learning_rate=0.01,
            num_leaves=15,
            max_depth=12,
            min_child_samples=20,
            subsample=0.95,
            colsample_bytree=1.0,
            reg_lambda=0.0,
            reg_alpha=0.01,
            random_state=SEED,
            n_jobs=-1,
            verbose=-1
        )

    if HAS_XGB:

        return XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            n_estimators=300,
            max_depth=6,
            learning_rate=0.10,
            subsample=0.95,
            colsample_bytree=0.85,
            min_child_weight=1,
            reg_lambda=1.0,
            reg_alpha=0.0,
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1
        )

    if HAS_CATBOOST:

        return CatBoostClassifier(
            loss_function="Logloss",
            iterations=500,
            learning_rate=0.03,
            depth=7,
            l2_leaf_reg=3,
            random_strength=1.0,
            bagging_temperature=0.0,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False
        )

    raise RuntimeError("No suitable MDR XAI model is available.")


def build_severity_xai_model():

    # Prefer XGBoost because Block 09/10/11 consistently showed strongest severity performance.
    if HAS_XGB:

        return XGBClassifier(
            objective="multi:softprob",
            num_class=4,
            eval_metric="mlogloss",
            n_estimators=500,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.90,
            colsample_bytree=0.90,
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1
        )

    if HAS_LGBM:

        return LGBMClassifier(
            objective="multiclass",
            num_class=4,
            n_estimators=700,
            learning_rate=0.05,
            num_leaves=15,
            random_state=SEED,
            n_jobs=-1,
            verbose=-1
        )

    if HAS_CATBOOST:

        return CatBoostClassifier(
            loss_function="MultiClass",
            iterations=300,
            learning_rate=0.05,
            depth=8,
            l2_leaf_reg=3,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False
        )

    raise RuntimeError("No suitable Severity XAI model is available.")


# ============================================================
# Prepare XAI Datasets
# ============================================================

X_train_mdr_xai = X_train_scaled[MDR_XAI_FEATURES].copy()
X_val_mdr_xai = X_val_scaled[MDR_XAI_FEATURES].copy()
X_test_mdr_xai = X_test_scaled[MDR_XAI_FEATURES].copy()

X_train_severity_xai = X_train_scaled[SEVERITY_XAI_FEATURES].copy()
X_val_severity_xai = X_val_scaled[SEVERITY_XAI_FEATURES].copy()
X_test_severity_xai = X_test_scaled[SEVERITY_XAI_FEATURES].copy()


# ============================================================
# Train Final XAI Models
# ============================================================

print("\n")
print("=" * 70)
print("TRAINING FINAL XAI MODELS")
print("=" * 70)

start_time = time.time()

mdr_xai_model = build_mdr_xai_model()

try:
    mdr_xai_model.fit(
        X_train_mdr_xai,
        y_train_mdr,
        sample_weight=mdr_train_weights
    )
except TypeError:
    mdr_xai_model.fit(
        X_train_mdr_xai,
        y_train_mdr
    )

mdr_train_time = time.time() - start_time

print(f"MDR XAI model trained in {mdr_train_time:.4f} seconds")

start_time = time.time()

severity_xai_model = build_severity_xai_model()

try:
    severity_xai_model.fit(
        X_train_severity_xai,
        y_train_severity,
        sample_weight=severity_train_weights
    )
except TypeError:
    severity_xai_model.fit(
        X_train_severity_xai,
        y_train_severity
    )

severity_train_time = time.time() - start_time

print(f"Severity XAI model trained in {severity_train_time:.4f} seconds")


# ============================================================
# Save XAI Models
# ============================================================

joblib.dump(
    mdr_xai_model,
    os.path.join(
        BLOCK12_MODEL_DIR,
        "mdr_final_xai_model.pkl"
    )
)

joblib.dump(
    severity_xai_model,
    os.path.join(
        BLOCK12_MODEL_DIR,
        "severity_final_xai_model.pkl"
    )
)

pd.DataFrame({
    "Task": ["MDR", "Severity"],
    "Model_Type": [
        type(mdr_xai_model).__name__,
        type(severity_xai_model).__name__
    ],
    "Training_Time_Seconds": [
        round(mdr_train_time, 4),
        round(severity_train_time, 4)
    ],
    "Number_of_Features": [
        len(MDR_XAI_FEATURES),
        len(SEVERITY_XAI_FEATURES)
    ]
}).to_csv(
    os.path.join(
        BLOCK12_TABLE_DIR,
        "xai_model_training_summary.csv"
    ),
    index=False
)


# ============================================================
# SHAP Sample Selection
# ============================================================

def sample_for_shap(X, y, sample_size, task_name):

    if len(X) <= sample_size:
        X_sample = X.copy()
        y_sample = y.copy()
    else:
        sample_idx, _ = train_test_split(
            np.arange(len(X)),
            train_size=sample_size,
            random_state=SEED,
            stratify=y
        )

        X_sample = X.iloc[sample_idx].copy()
        y_sample = y.iloc[sample_idx].copy()

    print(f"{task_name} SHAP Sample Shape:", X_sample.shape)

    return X_sample, y_sample


from sklearn.model_selection import train_test_split

X_mdr_shap, y_mdr_shap = sample_for_shap(
    X_test_mdr_xai,
    y_test_mdr,
    SHAP_SAMPLE_SIZE,
    "MDR"
)

X_severity_shap, y_severity_shap = sample_for_shap(
    X_test_severity_xai,
    y_test_severity,
    SHAP_SAMPLE_SIZE,
    "Severity"
)


# ============================================================
# SHAP Value Computation
# ============================================================

print("\n")
print("=" * 70)
print("COMPUTING SHAP VALUES")
print("=" * 70)

# MDR SHAP
start_time = time.time()

mdr_explainer = shap.TreeExplainer(
    mdr_xai_model
)

mdr_shap_values_raw = mdr_explainer.shap_values(
    X_mdr_shap
)

mdr_shap_time = time.time() - start_time

print(f"MDR SHAP values computed in {mdr_shap_time:.4f} seconds")


# Severity SHAP
start_time = time.time()

severity_explainer = shap.TreeExplainer(
    severity_xai_model
)

severity_shap_values_raw = severity_explainer.shap_values(
    X_severity_shap
)

severity_shap_time = time.time() - start_time

print(f"Severity SHAP values computed in {severity_shap_time:.4f} seconds")


# ============================================================
# SHAP Extraction Helpers
# ============================================================

def extract_binary_positive_shap(shap_values_raw):

    if isinstance(shap_values_raw, list):

        if len(shap_values_raw) == 2:
            return np.array(shap_values_raw[1])

        return np.array(shap_values_raw[0])

    arr = np.array(shap_values_raw)

    if arr.ndim == 3:

        if arr.shape[2] == 2:
            return arr[:, :, 1]

        if arr.shape[0] == 2:
            return arr[1, :, :]

    return arr


def extract_multiclass_shap(shap_values_raw, n_classes):

    if isinstance(shap_values_raw, list):

        return [
            np.array(shap_values_raw[i])
            for i in range(n_classes)
        ]

    arr = np.array(shap_values_raw)

    if arr.ndim == 3:

        if arr.shape[2] == n_classes:

            return [
                arr[:, :, i]
                for i in range(n_classes)
            ]

        if arr.shape[0] == n_classes:

            return [
                arr[i, :, :]
                for i in range(n_classes)
            ]

    raise RuntimeError(
        f"Unexpected multiclass SHAP shape: {arr.shape}"
    )


mdr_shap_values = extract_binary_positive_shap(
    mdr_shap_values_raw
)

severity_shap_values_list = extract_multiclass_shap(
    severity_shap_values_raw,
    n_classes=len(SEVERITY_CLASSES)
)

print("\nMDR SHAP shape:", mdr_shap_values.shape)

for i, class_name in enumerate(SEVERITY_CLASS_NAMES):
    print(f"Severity SHAP shape for {class_name}:", severity_shap_values_list[i].shape)


# ============================================================
# Save Raw SHAP Arrays
# ============================================================

np.save(
    os.path.join(
        BLOCK12_VALUE_DIR,
        "mdr_shap_values_positive_class.npy"
    ),
    mdr_shap_values
)

for i, class_name in enumerate(SEVERITY_CLASS_NAMES):

    np.save(
        os.path.join(
            BLOCK12_VALUE_DIR,
            f"severity_shap_values_{safe_name(class_name)}.npy"
        ),
        severity_shap_values_list[i]
    )

X_mdr_shap.to_csv(
    os.path.join(
        BLOCK12_VALUE_DIR,
        "mdr_shap_sample_features.csv"
    ),
    index=False
)

X_severity_shap.to_csv(
    os.path.join(
        BLOCK12_VALUE_DIR,
        "severity_shap_sample_features.csv"
    ),
    index=False
)


# ============================================================
# SHAP Computation Summary
# ============================================================

shap_computation_summary = pd.DataFrame({
    "Task": [
        "MDR",
        "Severity"
    ],
    "Samples_Explained": [
        len(X_mdr_shap),
        len(X_severity_shap)
    ],
    "Features_Explained": [
        X_mdr_shap.shape[1],
        X_severity_shap.shape[1]
    ],
    "SHAP_Time_Seconds": [
        round(mdr_shap_time, 4),
        round(severity_shap_time, 4)
    ],
    "Model_Type": [
        type(mdr_xai_model).__name__,
        type(severity_xai_model).__name__
    ]
})

shap_computation_summary.to_csv(
    os.path.join(
        BLOCK12_TABLE_DIR,
        "shap_computation_summary.csv"
    ),
    index=False
)

display(shap_computation_summary)


print("\n")
print("=" * 70)
print("BLOCK 12 PART 1 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 12
# SHAP Explainability / XAI
# PART 2
# Global and Local SHAP Explanations
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 12 PART 2 STARTED")
print("=" * 70)


# ============================================================
# Global SHAP Importance Tables
# ============================================================

def make_global_shap_importance(
    shap_values,
    X_sample,
    task_name,
    class_name=None
):

    mean_abs_shap = np.mean(
        np.abs(shap_values),
        axis=0
    )

    importance_df = pd.DataFrame({
        "Feature": X_sample.columns,
        "Mean_Absolute_SHAP": mean_abs_shap
    })

    importance_df = importance_df.sort_values(
        by="Mean_Absolute_SHAP",
        ascending=False
    ).reset_index(drop=True)

    importance_df["Rank"] = np.arange(
        1,
        len(importance_df) + 1
    )

    if class_name is not None:
        importance_df["Class"] = class_name

    importance_df["Task"] = task_name

    return importance_df


mdr_shap_importance = make_global_shap_importance(
    shap_values=mdr_shap_values,
    X_sample=X_mdr_shap,
    task_name="MDR",
    class_name="MDR"
)

severity_class_importance_tables = []

for i, class_name in enumerate(SEVERITY_CLASS_NAMES):

    class_importance = make_global_shap_importance(
        shap_values=severity_shap_values_list[i],
        X_sample=X_severity_shap,
        task_name="Severity",
        class_name=class_name
    )

    severity_class_importance_tables.append(class_importance)

severity_shap_importance_by_class = pd.concat(
    severity_class_importance_tables,
    ignore_index=True
)

# Overall multiclass importance
severity_abs_stack = np.stack(
    [
        np.abs(values)
        for values in severity_shap_values_list
    ],
    axis=2
)

severity_overall_values = np.mean(
    severity_abs_stack,
    axis=2
)

severity_overall_importance = make_global_shap_importance(
    shap_values=severity_overall_values,
    X_sample=X_severity_shap,
    task_name="Severity",
    class_name="Overall"
)

# Save tables
mdr_shap_importance.to_csv(
    os.path.join(
        BLOCK12_TABLE_DIR,
        "mdr_global_shap_importance.csv"
    ),
    index=False
)

severity_shap_importance_by_class.to_csv(
    os.path.join(
        BLOCK12_TABLE_DIR,
        "severity_classwise_global_shap_importance.csv"
    ),
    index=False
)

severity_overall_importance.to_csv(
    os.path.join(
        BLOCK12_TABLE_DIR,
        "severity_overall_global_shap_importance.csv"
    ),
    index=False
)


print("\n")
print("=" * 70)
print("MDR GLOBAL SHAP IMPORTANCE")
print("=" * 70)

display(mdr_shap_importance.head(15))

print("\n")
print("=" * 70)
print("SEVERITY OVERALL GLOBAL SHAP IMPORTANCE")
print("=" * 70)

display(severity_overall_importance.head(15))


# ============================================================
# Plot Functions
# ============================================================

def save_shap_summary_bar(
    shap_values,
    X_sample,
    title,
    save_path
):

    plt.figure(figsize=(9, 7))

    shap.summary_plot(
        shap_values,
        X_sample,
        plot_type="bar",
        show=False,
        max_display=min(TOP_N_FEATURES, X_sample.shape[1])
    )

    plt.title(title)
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def save_shap_beeswarm(
    shap_values,
    X_sample,
    title,
    save_path
):

    plt.figure(figsize=(9, 7))

    shap.summary_plot(
        shap_values,
        X_sample,
        show=False,
        max_display=min(TOP_N_FEATURES, X_sample.shape[1])
    )

    plt.title(title)
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def save_custom_shap_bar(
    importance_df,
    title,
    save_path,
    top_n=15
):

    plot_df = importance_df.head(top_n).copy()

    plt.figure(figsize=(9, 7))

    sns.barplot(
        data=plot_df,
        x="Mean_Absolute_SHAP",
        y="Feature",
        orient="h"
    )

    plt.title(title)
    plt.xlabel("Mean Absolute SHAP Value")
    plt.ylabel("Feature")
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


def save_dependence_plots(
    shap_values,
    X_sample,
    importance_df,
    task_name,
    class_name
):

    top_features = importance_df["Feature"].head(
        min(TOP_N_DEPENDENCE, len(importance_df))
    ).tolist()

    for feature in top_features:

        try:

            plt.figure(figsize=(7, 5))

            shap.dependence_plot(
                feature,
                shap_values,
                X_sample,
                show=False
            )

            plt.title(
                f"{task_name} — {class_name} — SHAP Dependence: {feature}"
            )

            plt.tight_layout()

            plt.savefig(
                os.path.join(
                    BLOCK12_FIG_DIR,
                    f"{task_name}_{safe_name(class_name)}_dependence_{safe_name(feature)}.png"
                ),
                dpi=FIG_DPI,
                bbox_inches="tight"
            )

            plt.show()
            plt.close()

        except Exception as e:

            print(f"Dependence plot skipped for {task_name} | {class_name} | {feature}: {e}")


# ============================================================
# MDR SHAP Plots
# ============================================================

save_custom_shap_bar(
    importance_df=mdr_shap_importance,
    title="MDR Prediction — Global SHAP Feature Importance",
    save_path=os.path.join(
        BLOCK12_FIG_DIR,
        "figure_12a_mdr_global_shap_importance_bar.png"
    ),
    top_n=TOP_N_FEATURES
)

save_shap_summary_bar(
    shap_values=mdr_shap_values,
    X_sample=X_mdr_shap,
    title="MDR Prediction — SHAP Summary Bar Plot",
    save_path=os.path.join(
        BLOCK12_FIG_DIR,
        "figure_12b_mdr_shap_summary_bar.png"
    )
)

save_shap_beeswarm(
    shap_values=mdr_shap_values,
    X_sample=X_mdr_shap,
    title="MDR Prediction — SHAP Beeswarm Plot",
    save_path=os.path.join(
        BLOCK12_FIG_DIR,
        "figure_12c_mdr_shap_beeswarm.png"
    )
)

save_dependence_plots(
    shap_values=mdr_shap_values,
    X_sample=X_mdr_shap,
    importance_df=mdr_shap_importance,
    task_name="MDR",
    class_name="MDR"
)


# ============================================================
# Severity SHAP Plots
# ============================================================

save_custom_shap_bar(
    importance_df=severity_overall_importance,
    title="Severity Prediction — Overall Global SHAP Feature Importance",
    save_path=os.path.join(
        BLOCK12_FIG_DIR,
        "figure_12d_severity_overall_global_shap_importance_bar.png"
    ),
    top_n=TOP_N_FEATURES
)

save_shap_summary_bar(
    shap_values=severity_overall_values,
    X_sample=X_severity_shap,
    title="Severity Prediction — Overall SHAP Summary Bar Plot",
    save_path=os.path.join(
        BLOCK12_FIG_DIR,
        "figure_12e_severity_overall_shap_summary_bar.png"
    )
)

save_shap_beeswarm(
    shap_values=severity_overall_values,
    X_sample=X_severity_shap,
    title="Severity Prediction — Overall SHAP Beeswarm Plot",
    save_path=os.path.join(
        BLOCK12_FIG_DIR,
        "figure_12f_severity_overall_shap_beeswarm.png"
    )
)

for i, class_name in enumerate(SEVERITY_CLASS_NAMES):

    class_importance_df = severity_shap_importance_by_class[
        severity_shap_importance_by_class["Class"] == class_name
    ].copy()

    save_custom_shap_bar(
        importance_df=class_importance_df,
        title=f"Severity Prediction — {class_name} Class SHAP Importance",
        save_path=os.path.join(
            BLOCK12_FIG_DIR,
            f"figure_12_severity_{safe_name(class_name)}_class_shap_importance_bar.png"
        ),
        top_n=TOP_N_FEATURES
    )

    save_shap_summary_bar(
        shap_values=severity_shap_values_list[i],
        X_sample=X_severity_shap,
        title=f"Severity Prediction — {class_name} Class SHAP Summary Bar",
        save_path=os.path.join(
            BLOCK12_FIG_DIR,
            f"figure_12_severity_{safe_name(class_name)}_summary_bar.png"
        )
    )

    save_dependence_plots(
        shap_values=severity_shap_values_list[i],
        X_sample=X_severity_shap,
        importance_df=class_importance_df,
        task_name="Severity",
        class_name=class_name
    )


# ============================================================
# Local SHAP Explanation Tables
# ============================================================

def save_local_explanations_binary(
    model,
    X_sample,
    y_sample,
    shap_values,
    task_name,
    class_names
):

    proba = model.predict_proba(X_sample)
    pred = np.argmax(proba, axis=1)

    sample_count = min(
        LOCAL_EXPLANATION_SAMPLES,
        len(X_sample)
    )

    local_summary_rows = []

    for i in range(sample_count):

        local_df = pd.DataFrame({
            "Feature": X_sample.columns,
            "Feature_Value": X_sample.iloc[i].values,
            "SHAP_Value": shap_values[i],
            "Absolute_SHAP": np.abs(shap_values[i])
        }).sort_values(
            by="Absolute_SHAP",
            ascending=False
        )

        local_df.to_csv(
            os.path.join(
                BLOCK12_LOCAL_DIR,
                f"{task_name}_local_explanation_sample_{i}.csv"
            ),
            index=False
        )

        top_local = local_df.head(10).copy()

        plt.figure(figsize=(8, 5))

        sns.barplot(
            data=top_local,
            x="SHAP_Value",
            y="Feature",
            orient="h"
        )

        plt.title(
            f"{task_name} Local SHAP Explanation — Sample {i}"
        )

        plt.xlabel("SHAP Value")
        plt.ylabel("Feature")
        plt.tight_layout()

        plt.savefig(
            os.path.join(
                BLOCK12_LOCAL_DIR,
                f"{task_name}_local_explanation_sample_{i}.png"
            ),
            dpi=FIG_DPI,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

        local_summary_rows.append({
            "Task": task_name,
            "Sample_Number": i,
            "True_Label": int(y_sample.iloc[i]),
            "True_Label_Name": class_names[int(y_sample.iloc[i])],
            "Predicted_Label": int(pred[i]),
            "Predicted_Label_Name": class_names[int(pred[i])],
            "Predicted_Probability_Positive": proba[i, 1],
            "Top_Contributing_Feature": local_df.iloc[0]["Feature"],
            "Top_Contributing_SHAP": local_df.iloc[0]["SHAP_Value"]
        })

    local_summary = pd.DataFrame(local_summary_rows)

    local_summary.to_csv(
        os.path.join(
            BLOCK12_LOCAL_DIR,
            f"{task_name}_local_explanation_summary.csv"
        ),
        index=False
    )

    return local_summary


def save_local_explanations_multiclass(
    model,
    X_sample,
    y_sample,
    shap_values_list,
    task_name,
    class_names
):

    proba = model.predict_proba(X_sample)
    pred = np.argmax(proba, axis=1)

    sample_count = min(
        LOCAL_EXPLANATION_SAMPLES,
        len(X_sample)
    )

    local_summary_rows = []

    for i in range(sample_count):

        predicted_class = int(pred[i])
        predicted_class_name = class_names[predicted_class]

        shap_values_for_predicted_class = shap_values_list[predicted_class][i]

        local_df = pd.DataFrame({
            "Feature": X_sample.columns,
            "Feature_Value": X_sample.iloc[i].values,
            "SHAP_Value": shap_values_for_predicted_class,
            "Absolute_SHAP": np.abs(shap_values_for_predicted_class)
        }).sort_values(
            by="Absolute_SHAP",
            ascending=False
        )

        local_df.to_csv(
            os.path.join(
                BLOCK12_LOCAL_DIR,
                f"{task_name}_local_explanation_sample_{i}_{safe_name(predicted_class_name)}.csv"
            ),
            index=False
        )

        top_local = local_df.head(10).copy()

        plt.figure(figsize=(8, 5))

        sns.barplot(
            data=top_local,
            x="SHAP_Value",
            y="Feature",
            orient="h"
        )

        plt.title(
            f"{task_name} Local SHAP — Sample {i} — Predicted {predicted_class_name}"
        )

        plt.xlabel("SHAP Value")
        plt.ylabel("Feature")
        plt.tight_layout()

        plt.savefig(
            os.path.join(
                BLOCK12_LOCAL_DIR,
                f"{task_name}_local_explanation_sample_{i}_{safe_name(predicted_class_name)}.png"
            ),
            dpi=FIG_DPI,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

        local_summary_rows.append({
            "Task": task_name,
            "Sample_Number": i,
            "True_Label_ID": int(y_sample.iloc[i]),
            "True_Label": class_names[int(y_sample.iloc[i])],
            "Predicted_Label_ID": predicted_class,
            "Predicted_Label": predicted_class_name,
            "Predicted_Class_Probability": proba[i, predicted_class],
            "Top_Contributing_Feature": local_df.iloc[0]["Feature"],
            "Top_Contributing_SHAP": local_df.iloc[0]["SHAP_Value"]
        })

    local_summary = pd.DataFrame(local_summary_rows)

    local_summary.to_csv(
        os.path.join(
            BLOCK12_LOCAL_DIR,
            f"{task_name}_local_explanation_summary.csv"
        ),
        index=False
    )

    return local_summary


mdr_local_summary = save_local_explanations_binary(
    model=mdr_xai_model,
    X_sample=X_mdr_shap,
    y_sample=y_mdr_shap,
    shap_values=mdr_shap_values,
    task_name="MDR",
    class_names=MDR_CLASS_NAMES
)

severity_local_summary = save_local_explanations_multiclass(
    model=severity_xai_model,
    X_sample=X_severity_shap,
    y_sample=y_severity_shap,
    shap_values_list=severity_shap_values_list,
    task_name="Severity",
    class_names=SEVERITY_CLASS_NAMES
)


# ============================================================
# Final XAI Interpretation Tables
# ============================================================

xai_key_findings = pd.DataFrame({
    "Task": [
        "MDR",
        "MDR",
        "MDR",
        "Severity",
        "Severity",
        "Severity"
    ],
    "Finding_Type": [
        "Top SHAP Feature",
        "Second SHAP Feature",
        "Third SHAP Feature",
        "Top SHAP Feature",
        "Second SHAP Feature",
        "Third SHAP Feature"
    ],
    "Feature": [
        mdr_shap_importance.iloc[0]["Feature"],
        mdr_shap_importance.iloc[1]["Feature"],
        mdr_shap_importance.iloc[2]["Feature"],
        severity_overall_importance.iloc[0]["Feature"],
        severity_overall_importance.iloc[1]["Feature"],
        severity_overall_importance.iloc[2]["Feature"]
    ],
    "Mean_Absolute_SHAP": [
        mdr_shap_importance.iloc[0]["Mean_Absolute_SHAP"],
        mdr_shap_importance.iloc[1]["Mean_Absolute_SHAP"],
        mdr_shap_importance.iloc[2]["Mean_Absolute_SHAP"],
        severity_overall_importance.iloc[0]["Mean_Absolute_SHAP"],
        severity_overall_importance.iloc[1]["Mean_Absolute_SHAP"],
        severity_overall_importance.iloc[2]["Mean_Absolute_SHAP"]
    ]
})

xai_key_findings.to_csv(
    os.path.join(
        BLOCK12_TABLE_DIR,
        "xai_key_findings_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("XAI KEY FINDINGS")
print("=" * 70)

display(xai_key_findings)


# ============================================================
# Block 12 Summary
# ============================================================

block12_summary = pd.DataFrame({
    "Output_Category": [
        "Final XAI models",
        "Raw MDR SHAP values",
        "Raw Severity SHAP values",
        "MDR global SHAP importance",
        "Severity classwise SHAP importance",
        "Severity overall SHAP importance",
        "SHAP summary bar plots",
        "SHAP beeswarm plots",
        "SHAP dependence plots",
        "Local explanation tables",
        "Local explanation figures",
        "XAI key findings table",
        "SHAP computation summary"
    ],
    "Saved": [
        "models/*.pkl",
        "shap_values/mdr_shap_values_positive_class.npy",
        "shap_values/severity_shap_values_*.npy",
        "tables/mdr_global_shap_importance.csv",
        "tables/severity_classwise_global_shap_importance.csv",
        "tables/severity_overall_global_shap_importance.csv",
        "figures/*summary_bar*.png",
        "figures/*beeswarm*.png",
        "figures/*dependence*.png",
        "local_explanations/*.csv",
        "local_explanations/*.png",
        "tables/xai_key_findings_summary.csv",
        "tables/shap_computation_summary.csv"
    ]
})

block12_summary.to_csv(
    os.path.join(
        BLOCK12_DIR,
        "block12_output_summary.csv"
    ),
    index=False
)


print("\n")
print("=" * 70)
print("BLOCK 12 GENERATED FILES")
print("=" * 70)

print("- models/mdr_final_xai_model.pkl")
print("- models/severity_final_xai_model.pkl")
print("- shap_values/*.npy")
print("- tables/mdr_global_shap_importance.csv")
print("- tables/severity_classwise_global_shap_importance.csv")
print("- tables/severity_overall_global_shap_importance.csv")
print("- tables/xai_key_findings_summary.csv")
print("- figures/SHAP summary bar plots")
print("- figures/SHAP beeswarm plots")
print("- figures/SHAP dependence plots")
print("- local_explanations/*.csv")
print("- local_explanations/*.png")

print("\n")
print("=" * 70)
print("BLOCK 12 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 13
# Stacking Ensemble
# PART 1
# Setup + Out-of-Fold Meta-Feature Generation
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 13 STARTED — STACKING ENSEMBLE")
print("=" * 70)

import os
import time
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

from sklearn.ensemble import RandomForestClassifier


# ============================================================
# Dependency Check
# ============================================================

required_objects = [
    "X_train_scaled",
    "X_val_scaled",
    "X_test_scaled",
    "y_train_mdr",
    "y_val_mdr",
    "y_test_mdr",
    "y_train_severity",
    "y_val_severity",
    "y_test_severity",
    "mdr_train_weights",
    "severity_train_weights",
    "fit_model",
    "get_aligned_proba",
    "evaluate_split_and_export",
    "plot_learning_curve_custom",
    "safe_name",
    "MDR_CLASSES",
    "MDR_CLASS_NAMES",
    "SEVERITY_CLASSES",
    "SEVERITY_CLASS_NAMES",
    "SEED"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if len(missing_objects) > 0:
    raise RuntimeError(
        "Please run previous blocks before BLOCK 13. Missing: "
        + ", ".join(missing_objects)
    )


# ============================================================
# Output Directories
# ============================================================

BLOCK13_DIR = os.path.join(OUTPUT_DIR, "block13_stacking_ensemble")

BLOCK13_MODEL_DIR = os.path.join(BLOCK13_DIR, "models")
BLOCK13_METRIC_DIR = os.path.join(BLOCK13_DIR, "metrics")
BLOCK13_REPORT_DIR = os.path.join(BLOCK13_DIR, "classification_reports")
BLOCK13_PRED_DIR = os.path.join(BLOCK13_DIR, "predictions")
BLOCK13_FIG_DIR = os.path.join(BLOCK13_DIR, "figures")
BLOCK13_META_DIR = os.path.join(BLOCK13_DIR, "meta_features")

for d in [
    BLOCK13_DIR,
    BLOCK13_MODEL_DIR,
    BLOCK13_METRIC_DIR,
    BLOCK13_REPORT_DIR,
    BLOCK13_PRED_DIR,
    BLOCK13_FIG_DIR,
    BLOCK13_META_DIR
]:
    os.makedirs(d, exist_ok=True)

FIG_DPI = 300
N_STACKING_FOLDS = 5


# ============================================================
# Base Model Definitions
# ============================================================

def get_stacking_base_models(task_type):

    models = {}

    if task_type == "binary":

        if HAS_LGBM:
            models["LightGBM"] = LGBMClassifier(
                objective="binary",
                n_estimators=700,
                learning_rate=0.01,
                num_leaves=15,
                max_depth=12,
                min_child_samples=20,
                subsample=0.95,
                colsample_bytree=1.0,
                reg_lambda=0.0,
                reg_alpha=0.01,
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )

        if HAS_XGB:
            models["XGBoost"] = XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                n_estimators=300,
                max_depth=6,
                learning_rate=0.10,
                subsample=0.95,
                colsample_bytree=0.85,
                min_child_weight=1,
                reg_lambda=1.0,
                reg_alpha=0.0,
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1
            )

        models["RandomForest"] = RandomForestClassifier(
            n_estimators=700,
            max_depth=10,
            min_samples_split=10,
            min_samples_leaf=2,
            max_features=0.7,
            bootstrap=False,
            random_state=SEED,
            n_jobs=-1
        )

        if HAS_CATBOOST:
            models["CatBoost"] = CatBoostClassifier(
                loss_function="Logloss",
                iterations=500,
                learning_rate=0.03,
                depth=7,
                l2_leaf_reg=3,
                random_strength=1.0,
                bagging_temperature=0.0,
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False
            )

    else:

        if HAS_XGB:
            models["XGBoost"] = XGBClassifier(
                objective="multi:softprob",
                num_class=4,
                eval_metric="mlogloss",
                n_estimators=300,
                max_depth=6,
                learning_rate=0.10,
                subsample=0.95,
                colsample_bytree=0.85,
                min_child_weight=1,
                reg_lambda=1.0,
                reg_alpha=0.0,
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1
            )

        models["RandomForest"] = RandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=1,
            max_features=0.7,
            bootstrap=False,
            random_state=SEED,
            n_jobs=-1
        )

        if HAS_LGBM:
            models["LightGBM"] = LGBMClassifier(
                objective="multiclass",
                num_class=4,
                n_estimators=700,
                learning_rate=0.05,
                num_leaves=15,
                max_depth=-1,
                min_child_samples=20,
                subsample=1.0,
                colsample_bytree=0.95,
                reg_lambda=5.0,
                reg_alpha=0.01,
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )

        if HAS_CATBOOST:
            models["CatBoost"] = CatBoostClassifier(
                loss_function="MultiClass",
                iterations=300,
                learning_rate=0.05,
                depth=8,
                l2_leaf_reg=3,
                random_strength=0.5,
                bagging_temperature=1.0,
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False
            )

    return models


def get_meta_feature_columns(base_model_names, task_type, class_names):

    columns = []

    if task_type == "binary":

        for model_name in base_model_names:
            columns.append(f"{model_name}_Probability_Positive")

    else:

        for model_name in base_model_names:

            for class_name in class_names:
                columns.append(
                    f"{model_name}_Probability_{safe_name(class_name)}"
                )

    return columns


# ============================================================
# OOF Meta-Feature Generation
# ============================================================

def generate_oof_and_holdout_meta_features(
    task_name,
    task_type,
    base_models,
    X_train_input,
    X_val_input,
    X_test_input,
    y_train_input,
    sample_weight_input,
    classes,
    class_names
):

    print("\n")
    print("=" * 70)
    print(f"GENERATING STACKING META-FEATURES: {task_name}")
    print("=" * 70)

    base_model_names = list(base_models.keys())

    meta_columns = get_meta_feature_columns(
        base_model_names,
        task_type,
        class_names
    )

    X_train_meta = pd.DataFrame(
        np.zeros(
            (
                len(X_train_input),
                len(meta_columns)
            )
        ),
        columns=meta_columns,
        index=X_train_input.index
    )

    X_val_meta_parts = []
    X_test_meta_parts = []

    fitted_base_models = {}

    skf = StratifiedKFold(
        n_splits=N_STACKING_FOLDS,
        shuffle=True,
        random_state=SEED
    )

    # --------------------------------------------------------
    # OOF Features for Training Meta-Learner
    # --------------------------------------------------------

    for model_name, base_model in base_models.items():

        print("\n")
        print("-" * 70)
        print(f"OOF training base model: {task_name} — {model_name}")
        print("-" * 70)

        if task_type == "binary":
            model_meta_columns = [
                f"{model_name}_Probability_Positive"
            ]
        else:
            model_meta_columns = [
                f"{model_name}_Probability_{safe_name(class_name)}"
                for class_name in class_names
            ]

        for fold, (train_idx, oof_idx) in enumerate(
            skf.split(X_train_input, y_train_input),
            start=1
        ):

            X_fold_train = X_train_input.iloc[train_idx]
            y_fold_train = y_train_input.iloc[train_idx]

            X_fold_oof = X_train_input.iloc[oof_idx]

            fold_weights = sample_weight_input[train_idx]

            fold_model = clone(base_model)

            start_time = time.time()

            fold_model = fit_model(
                fold_model,
                X_fold_train,
                y_fold_train,
                sample_weight=fold_weights
            )

            fold_time = time.time() - start_time

            oof_proba = get_aligned_proba(
                fold_model,
                X_fold_oof,
                classes
            )

            if task_type == "binary":

                X_train_meta.loc[
                    X_fold_oof.index,
                    model_meta_columns[0]
                ] = oof_proba[:, 1]

            else:

                for class_idx, col in enumerate(model_meta_columns):
                    X_train_meta.loc[
                        X_fold_oof.index,
                        col
                    ] = oof_proba[:, class_idx]

            print(
                f"Fold {fold}/{N_STACKING_FOLDS} completed "
                f"in {fold_time:.2f}s"
            )

        # ----------------------------------------------------
        # Fit Full Base Model for Validation/Test Meta-Features
        # ----------------------------------------------------

        full_model = clone(base_model)

        start_time = time.time()

        full_model = fit_model(
            full_model,
            X_train_input,
            y_train_input,
            sample_weight=sample_weight_input
        )

        full_time = time.time() - start_time

        fitted_base_models[model_name] = full_model

        val_proba = get_aligned_proba(
            full_model,
            X_val_input,
            classes
        )

        test_proba = get_aligned_proba(
            full_model,
            X_test_input,
            classes
        )

        if task_type == "binary":

            X_val_meta_parts.append(
                pd.DataFrame(
                    {
                        model_meta_columns[0]: val_proba[:, 1]
                    },
                    index=X_val_input.index
                )
            )

            X_test_meta_parts.append(
                pd.DataFrame(
                    {
                        model_meta_columns[0]: test_proba[:, 1]
                    },
                    index=X_test_input.index
                )
            )

        else:

            val_part = pd.DataFrame(
                val_proba,
                columns=model_meta_columns,
                index=X_val_input.index
            )

            test_part = pd.DataFrame(
                test_proba,
                columns=model_meta_columns,
                index=X_test_input.index
            )

            X_val_meta_parts.append(val_part)
            X_test_meta_parts.append(test_part)

        print(
            f"Full {model_name} fitted in {full_time:.2f}s"
        )

    X_val_meta = pd.concat(
        X_val_meta_parts,
        axis=1
    )[meta_columns]

    X_test_meta = pd.concat(
        X_test_meta_parts,
        axis=1
    )[meta_columns]

    # Save meta-features
    X_train_meta.to_csv(
        os.path.join(
            BLOCK13_META_DIR,
            f"{task_name}_train_oof_meta_features.csv"
        ),
        index=True
    )

    X_val_meta.to_csv(
        os.path.join(
            BLOCK13_META_DIR,
            f"{task_name}_validation_meta_features.csv"
        ),
        index=True
    )

    X_test_meta.to_csv(
        os.path.join(
            BLOCK13_META_DIR,
            f"{task_name}_test_meta_features.csv"
        ),
        index=True
    )

    print("\nMeta-feature shapes:")
    print("Train:", X_train_meta.shape)
    print("Validation:", X_val_meta.shape)
    print("Test:", X_test_meta.shape)

    return X_train_meta, X_val_meta, X_test_meta, fitted_base_models


# ============================================================
# Stacking Evaluation Helper
# ============================================================

def create_block13_task_dirs(task_name):

    task_dir = os.path.join(BLOCK13_DIR, task_name)

    dirs = {
        "task": task_dir,
        "models": os.path.join(task_dir, "models"),
        "metrics": os.path.join(task_dir, "metrics"),
        "reports": os.path.join(task_dir, "classification_reports"),
        "predictions": os.path.join(task_dir, "predictions"),
        "figures": os.path.join(task_dir, "figures")
    }

    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    return dirs


def train_and_evaluate_stacking_ensemble(
    task_name,
    task_type,
    X_train_meta,
    X_val_meta,
    X_test_meta,
    y_train_input,
    y_val_input,
    y_test_input,
    sample_weight_input,
    fitted_base_models,
    classes,
    class_names
):

    print("\n")
    print("=" * 70)
    print(f"TRAINING STACKING META-LEARNER: {task_name}")
    print("=" * 70)

    dirs = create_block13_task_dirs(task_name)

    meta_model = LogisticRegression(
        max_iter=3000,
        random_state=SEED
    )

    start_time = time.time()

    meta_model = fit_model(
        meta_model,
        X_train_meta,
        y_train_input,
        sample_weight=sample_weight_input
    )

    meta_training_time = time.time() - start_time

    print(f"Meta-learner trained in {meta_training_time:.4f} seconds")

    model_bundle = {
        "task_name": task_name,
        "task_type": task_type,
        "base_models": fitted_base_models,
        "meta_model": meta_model,
        "meta_feature_columns": list(X_train_meta.columns),
        "classes": classes,
        "class_names": class_names
    }

    joblib.dump(
        model_bundle,
        os.path.join(
            dirs["models"],
            f"{task_name}_stacking_ensemble_bundle.pkl"
        )
    )

    pd.DataFrame({
        "Task": [task_name],
        "Meta_Learner": [type(meta_model).__name__],
        "Number_of_Base_Models": [len(fitted_base_models)],
        "Base_Models": [list(fitted_base_models.keys())],
        "Number_of_Meta_Features": [X_train_meta.shape[1]],
        "Meta_Training_Time_Seconds": [round(meta_training_time, 4)]
    }).to_csv(
        os.path.join(
            dirs["metrics"],
            f"{task_name}_stacking_training_summary.csv"
        ),
        index=False
    )

    metrics_list = []
    predictions_list = []

    for split_name, X_split_meta, y_split in [

        ("Train", X_train_meta, y_train_input),
        ("Validation", X_val_meta, y_val_input),
        ("Test", X_test_meta, y_test_input)

    ]:

        metrics, pred_df = evaluate_split_and_export(
            model=meta_model,
            X_split=X_split_meta,
            y_true=y_split,
            split_name=split_name,
            task_name=f"{task_name}_Stacking",
            task_type=task_type,
            model_name="StackingEnsemble",
            classes=classes,
            class_names=class_names,
            dirs=dirs
        )

        metrics["Meta_Feature_Count"] = X_train_meta.shape[1]
        metrics["Base_Models"] = list(fitted_base_models.keys())

        metrics_list.append(metrics)
        predictions_list.append(pred_df)

    # Meta-level learning curve
    plot_learning_curve_custom(
        model=meta_model,
        X_train_input=X_train_meta,
        y_train_input=y_train_input,
        X_val_input=X_val_meta,
        y_val_input=y_val_input,
        sample_weight_input=sample_weight_input,
        task_type=task_type,
        task_name=f"{task_name}_Stacking",
        model_name="StackingEnsemble",
        dirs=dirs
    )

    metrics_df = pd.DataFrame(metrics_list)

    predictions_df = pd.concat(
        predictions_list,
        ignore_index=True
    )

    metrics_df.to_csv(
        os.path.join(
            dirs["metrics"],
            f"{task_name}_stacking_train_validation_test_metrics.csv"
        ),
        index=False
    )

    predictions_df.to_csv(
        os.path.join(
            dirs["predictions"],
            f"{task_name}_stacking_train_validation_test_predictions.csv"
        ),
        index=False
    )

    return metrics_df, predictions_df, model_bundle


print("\n")
print("=" * 70)
print("BLOCK 13 PART 1 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 13
# Stacking Ensemble
# PART 2
# Run MDR and Severity Stacking Ensembles
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 13 PART 2 STARTED")
print("=" * 70)


# ============================================================
# MDR Stacking Ensemble
# ============================================================

mdr_stacking_base_models = get_stacking_base_models(
    task_type="binary"
)

print("\nMDR Stacking Base Models:")
print(list(mdr_stacking_base_models.keys()))

mdr_train_meta, mdr_val_meta, mdr_test_meta, mdr_fitted_base_models = generate_oof_and_holdout_meta_features(
    task_name="MDR",
    task_type="binary",
    base_models=mdr_stacking_base_models,
    X_train_input=X_train_scaled,
    X_val_input=X_val_scaled,
    X_test_input=X_test_scaled,
    y_train_input=y_train_mdr,
    sample_weight_input=mdr_train_weights,
    classes=MDR_CLASSES,
    class_names=MDR_CLASS_NAMES
)

mdr_stacking_metrics_df, mdr_stacking_predictions_df, mdr_stacking_model_bundle = train_and_evaluate_stacking_ensemble(
    task_name="MDR",
    task_type="binary",
    X_train_meta=mdr_train_meta,
    X_val_meta=mdr_val_meta,
    X_test_meta=mdr_test_meta,
    y_train_input=y_train_mdr,
    y_val_input=y_val_mdr,
    y_test_input=y_test_mdr,
    sample_weight_input=mdr_train_weights,
    fitted_base_models=mdr_fitted_base_models,
    classes=MDR_CLASSES,
    class_names=MDR_CLASS_NAMES
)


# ============================================================
# Severity Stacking Ensemble
# ============================================================

severity_stacking_base_models = get_stacking_base_models(
    task_type="multiclass"
)

print("\nSeverity Stacking Base Models:")
print(list(severity_stacking_base_models.keys()))

severity_train_meta, severity_val_meta, severity_test_meta, severity_fitted_base_models = generate_oof_and_holdout_meta_features(
    task_name="Severity",
    task_type="multiclass",
    base_models=severity_stacking_base_models,
    X_train_input=X_train_scaled,
    X_val_input=X_val_scaled,
    X_test_input=X_test_scaled,
    y_train_input=y_train_severity,
    sample_weight_input=severity_train_weights,
    classes=SEVERITY_CLASSES,
    class_names=SEVERITY_CLASS_NAMES
)

severity_stacking_metrics_df, severity_stacking_predictions_df, severity_stacking_model_bundle = train_and_evaluate_stacking_ensemble(
    task_name="Severity",
    task_type="multiclass",
    X_train_meta=severity_train_meta,
    X_val_meta=severity_val_meta,
    X_test_meta=severity_test_meta,
    y_train_input=y_train_severity,
    y_val_input=y_val_severity,
    y_test_input=y_test_severity,
    sample_weight_input=severity_train_weights,
    fitted_base_models=severity_fitted_base_models,
    classes=SEVERITY_CLASSES,
    class_names=SEVERITY_CLASS_NAMES
)


# ============================================================
# Combined Stacking Metrics and Predictions
# ============================================================

block13_all_metrics = pd.concat(
    [
        mdr_stacking_metrics_df,
        severity_stacking_metrics_df
    ],
    ignore_index=True
)

block13_all_predictions = pd.concat(
    [
        mdr_stacking_predictions_df,
        severity_stacking_predictions_df
    ],
    ignore_index=True
)

block13_all_metrics.to_csv(
    os.path.join(
        BLOCK13_METRIC_DIR,
        "block13_all_stacking_metrics.csv"
    ),
    index=False
)

block13_all_predictions.to_csv(
    os.path.join(
        BLOCK13_PRED_DIR,
        "block13_all_stacking_predictions.csv"
    ),
    index=False
)


# ============================================================
# Test Comparison Tables
# ============================================================

mdr_stacking_test_table = block13_all_metrics[
    (block13_all_metrics["Task"] == "MDR_Stacking") &
    (block13_all_metrics["Split"] == "Test")
].copy()

severity_stacking_test_table = block13_all_metrics[
    (block13_all_metrics["Task"] == "Severity_Stacking") &
    (block13_all_metrics["Split"] == "Test")
].copy()

mdr_stacking_test_table.to_csv(
    os.path.join(
        BLOCK13_METRIC_DIR,
        "mdr_stacking_test_result.csv"
    ),
    index=False
)

severity_stacking_test_table.to_csv(
    os.path.join(
        BLOCK13_METRIC_DIR,
        "severity_stacking_test_result.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("MDR STACKING TEST RESULT")
print("=" * 70)

display(mdr_stacking_test_table)

print("\n")
print("=" * 70)
print("SEVERITY STACKING TEST RESULT")
print("=" * 70)

display(severity_stacking_test_table)


# ============================================================
# Compare Baseline, Optimized, Feature-Selected, Stacking
# ============================================================

comparison_frames = []

# Block 09 baseline
block09_path = os.path.join(
    OUTPUT_DIR,
    "block09_baseline_models",
    "metrics",
    "block09_all_baseline_metrics.csv"
)

if os.path.exists(block09_path):

    block09_metrics = pd.read_csv(block09_path)
    block09_test = block09_metrics[block09_metrics["Split"] == "Test"].copy()
    block09_test["Stage"] = "Baseline"
    comparison_frames.append(block09_test)

# Block 10 optimized
block10_path = os.path.join(
    OUTPUT_DIR,
    "block10_hyperparameter_optimization",
    "metrics",
    "block10_all_optimized_metrics.csv"
)

if os.path.exists(block10_path):

    block10_metrics = pd.read_csv(block10_path)
    block10_test = block10_metrics[block10_metrics["Split"] == "Test"].copy()
    block10_test["Stage"] = "Optimized"
    comparison_frames.append(block10_test)

# Block 11 feature selected
block11_path = os.path.join(
    OUTPUT_DIR,
    "block11_feature_selection",
    "metrics",
    "block11_all_feature_selected_metrics.csv"
)

if os.path.exists(block11_path):

    block11_metrics = pd.read_csv(block11_path)
    block11_test = block11_metrics[block11_metrics["Split"] == "Test"].copy()
    block11_test["Stage"] = "Feature_Selected"
    comparison_frames.append(block11_test)

# Block 13 stacking
block13_test = block13_all_metrics[block13_all_metrics["Split"] == "Test"].copy()
block13_test["Stage"] = "Stacking_Ensemble"
comparison_frames.append(block13_test)

if len(comparison_frames) > 0:

    block13_model_stage_comparison = pd.concat(
        comparison_frames,
        ignore_index=True
    )

    block13_model_stage_comparison.to_csv(
        os.path.join(
            BLOCK13_METRIC_DIR,
            "block13_model_stage_comparison.csv"
        ),
        index=False
    )

else:

    block13_model_stage_comparison = pd.DataFrame()


# ============================================================
# Publication Comparison Figures
# ============================================================

def plot_stage_comparison(
    df,
    task_keyword,
    metric,
    title,
    save_path
):

    plot_df = df[
        df["Task"].astype(str).str.contains(task_keyword, case=False, regex=False)
    ].copy()

    if len(plot_df) == 0:
        print(f"No data found for {task_keyword} — {metric}")
        return

    plot_df["Model_Label"] = plot_df["Stage"].astype(str) + " | " + plot_df["Model"].astype(str)

    plot_df = plot_df.sort_values(
        by=metric,
        ascending=False
    )

    plt.figure(figsize=(11, 6))

    sns.barplot(
        data=plot_df,
        x=metric,
        y="Model_Label",
        orient="h"
    )

    plt.title(title)
    plt.xlabel(metric)
    plt.ylabel("Model Stage | Model")
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


if len(block13_model_stage_comparison) > 0:

    plot_stage_comparison(
        df=block13_model_stage_comparison,
        task_keyword="MDR",
        metric="ROC_AUC",
        title="MDR Model Stage Comparison — Test ROC-AUC",
        save_path=os.path.join(
            BLOCK13_FIG_DIR,
            "figure_13a_mdr_stage_comparison_roc_auc.png"
        )
    )

    plot_stage_comparison(
        df=block13_model_stage_comparison,
        task_keyword="MDR",
        metric="F1",
        title="MDR Model Stage Comparison — Test F1-score",
        save_path=os.path.join(
            BLOCK13_FIG_DIR,
            "figure_13b_mdr_stage_comparison_f1.png"
        )
    )

    plot_stage_comparison(
        df=block13_model_stage_comparison,
        task_keyword="Severity",
        metric="Macro_F1",
        title="Severity Model Stage Comparison — Test Macro F1-score",
        save_path=os.path.join(
            BLOCK13_FIG_DIR,
            "figure_13c_severity_stage_comparison_macro_f1.png"
        )
    )

    plot_stage_comparison(
        df=block13_model_stage_comparison,
        task_keyword="Severity",
        metric="Balanced_Accuracy",
        title="Severity Model Stage Comparison — Test Balanced Accuracy",
        save_path=os.path.join(
            BLOCK13_FIG_DIR,
            "figure_13d_severity_stage_comparison_balanced_accuracy.png"
        )
    )


# ============================================================
# Stacking Summary
# ============================================================

block13_summary = pd.DataFrame({
    "Output_Category": [
        "OOF meta-features",
        "Validation meta-features",
        "Test meta-features",
        "Stacking model bundles",
        "MDR stacking metrics",
        "Severity stacking metrics",
        "Combined stacking metrics",
        "Combined stacking predictions",
        "Train/Validation/Test reports",
        "Train/Validation/Test confusion matrices",
        "Train/Validation/Test ROC curves",
        "Train/Validation/Test PR curves",
        "Train/Validation/Test calibration curves",
        "Train/Validation/Test probability distributions",
        "Learning curves",
        "Stage comparison table",
        "Stage comparison figures"
    ],
    "Saved": [
        "meta_features/*_train_oof_meta_features.csv",
        "meta_features/*_validation_meta_features.csv",
        "meta_features/*_test_meta_features.csv",
        "models and per-task model bundles",
        "MDR/metrics/MDR_stacking_train_validation_test_metrics.csv",
        "Severity/metrics/Severity_stacking_train_validation_test_metrics.csv",
        "metrics/block13_all_stacking_metrics.csv",
        "predictions/block13_all_stacking_predictions.csv",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "metrics/block13_model_stage_comparison.csv",
        "figures/figure_13*.png"
    ]
})

block13_summary.to_csv(
    os.path.join(
        BLOCK13_DIR,
        "block13_output_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("BLOCK 13 GENERATED FILES")
print("=" * 70)

print("- meta_features/*_train_oof_meta_features.csv")
print("- meta_features/*_validation_meta_features.csv")
print("- meta_features/*_test_meta_features.csv")
print("- models/*_stacking_ensemble_bundle.pkl")
print("- metrics/block13_all_stacking_metrics.csv")
print("- predictions/block13_all_stacking_predictions.csv")
print("- metrics/mdr_stacking_test_result.csv")
print("- metrics/severity_stacking_test_result.csv")
print("- metrics/block13_model_stage_comparison.csv")
print("- Train/Validation/Test reports, confusion matrices, ROC, PR, calibration, probability distributions")
print("- Meta-level learning curves")
print("- Stage comparison figures")

print("\n")
print("=" * 70)
print("BLOCK 13 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 14
# Voting Ensemble
# PART 1
# Setup + Voting Ensemble Helper Functions
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 14 STARTED — VOTING ENSEMBLE")
print("=" * 70)

import os
import time
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    balanced_accuracy_score
)
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False


# ============================================================
# Dependency Check
# ============================================================

required_objects = [
    "X_train_scaled",
    "X_val_scaled",
    "X_test_scaled",
    "y_train_mdr",
    "y_val_mdr",
    "y_test_mdr",
    "y_train_severity",
    "y_val_severity",
    "y_test_severity",
    "mdr_train_weights",
    "severity_train_weights",
    "fit_model",
    "get_aligned_proba",
    "evaluate_split_and_export",
    "safe_name",
    "MDR_CLASSES",
    "MDR_CLASS_NAMES",
    "SEVERITY_CLASSES",
    "SEVERITY_CLASS_NAMES",
    "SEED"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if len(missing_objects) > 0:
    raise RuntimeError(
        "Please run previous blocks before BLOCK 14. Missing: "
        + ", ".join(missing_objects)
    )


# ============================================================
# Output Directories
# ============================================================

BLOCK14_DIR = os.path.join(OUTPUT_DIR, "block14_voting_ensemble")

BLOCK14_MODEL_DIR = os.path.join(BLOCK14_DIR, "models")
BLOCK14_METRIC_DIR = os.path.join(BLOCK14_DIR, "metrics")
BLOCK14_REPORT_DIR = os.path.join(BLOCK14_DIR, "classification_reports")
BLOCK14_PRED_DIR = os.path.join(BLOCK14_DIR, "predictions")
BLOCK14_FIG_DIR = os.path.join(BLOCK14_DIR, "figures")
BLOCK14_WEIGHT_DIR = os.path.join(BLOCK14_DIR, "voting_weights")

for d in [
    BLOCK14_DIR,
    BLOCK14_MODEL_DIR,
    BLOCK14_METRIC_DIR,
    BLOCK14_REPORT_DIR,
    BLOCK14_PRED_DIR,
    BLOCK14_FIG_DIR,
    BLOCK14_WEIGHT_DIR
]:
    os.makedirs(d, exist_ok=True)

FIG_DPI = 300


# ============================================================
# Base Model Definitions
# ============================================================

def get_voting_base_models(task_type):

    models = {}

    if task_type == "binary":

        if HAS_LGBM:
            models["LightGBM"] = LGBMClassifier(
                objective="binary",
                n_estimators=700,
                learning_rate=0.01,
                num_leaves=15,
                max_depth=12,
                min_child_samples=20,
                subsample=0.95,
                colsample_bytree=1.0,
                reg_lambda=0.0,
                reg_alpha=0.01,
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )

        if HAS_XGB:
            models["XGBoost"] = XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                n_estimators=300,
                max_depth=6,
                learning_rate=0.10,
                subsample=0.95,
                colsample_bytree=0.85,
                min_child_weight=1,
                reg_lambda=1.0,
                reg_alpha=0.0,
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1
            )

        models["RandomForest"] = RandomForestClassifier(
            n_estimators=700,
            max_depth=10,
            min_samples_split=10,
            min_samples_leaf=2,
            max_features=0.7,
            bootstrap=False,
            random_state=SEED,
            n_jobs=-1
        )

        if HAS_CATBOOST:
            models["CatBoost"] = CatBoostClassifier(
                loss_function="Logloss",
                iterations=500,
                learning_rate=0.03,
                depth=7,
                l2_leaf_reg=3,
                random_strength=1.0,
                bagging_temperature=0.0,
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False
            )

    else:

        if HAS_XGB:
            models["XGBoost"] = XGBClassifier(
                objective="multi:softprob",
                num_class=4,
                eval_metric="mlogloss",
                n_estimators=300,
                max_depth=6,
                learning_rate=0.10,
                subsample=0.95,
                colsample_bytree=0.85,
                min_child_weight=1,
                reg_lambda=1.0,
                reg_alpha=0.0,
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1
            )

        models["RandomForest"] = RandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=1,
            max_features=0.7,
            bootstrap=False,
            random_state=SEED,
            n_jobs=-1
        )

        if HAS_LGBM:
            models["LightGBM"] = LGBMClassifier(
                objective="multiclass",
                num_class=4,
                n_estimators=700,
                learning_rate=0.05,
                num_leaves=15,
                max_depth=-1,
                min_child_samples=20,
                subsample=1.0,
                colsample_bytree=0.95,
                reg_lambda=5.0,
                reg_alpha=0.01,
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )

        if HAS_CATBOOST:
            models["CatBoost"] = CatBoostClassifier(
                loss_function="MultiClass",
                iterations=300,
                learning_rate=0.05,
                depth=8,
                l2_leaf_reg=3,
                random_strength=0.5,
                bagging_temperature=1.0,
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False
            )

    return models


# ============================================================
# Voting Ensemble Predictor
# ============================================================

class FittedSoftVotingEnsemble:

    def __init__(
        self,
        fitted_models,
        model_weights,
        classes
    ):

        self.fitted_models = fitted_models
        self.model_weights = model_weights
        self.classes_ = np.array(classes)
        self.classes = list(classes)

    def predict_proba(self, X):

        weighted_probabilities = []

        weight_values = []

        for model_name, model in self.fitted_models.items():

            proba = get_aligned_proba(
                model,
                X,
                self.classes
            )

            weight = float(
                self.model_weights.get(
                    model_name,
                    1.0
                )
            )

            weighted_probabilities.append(
                proba * weight
            )

            weight_values.append(weight)

        total_weight = np.sum(weight_values)

        if total_weight == 0:
            total_weight = len(weight_values)

        ensemble_proba = np.sum(
            weighted_probabilities,
            axis=0
        ) / total_weight

        ensemble_proba = ensemble_proba / ensemble_proba.sum(
            axis=1,
            keepdims=True
        )

        return ensemble_proba

    def predict(self, X):

        proba = self.predict_proba(X)

        return self.classes_[
            np.argmax(
                proba,
                axis=1
            )
        ]


# ============================================================
# Model Training + Weight Calculation
# ============================================================

def train_base_models_for_voting(
    task_name,
    task_type,
    X_train_input,
    y_train_input,
    sample_weight_input
):

    print("\n")
    print("=" * 70)
    print(f"TRAINING BASE MODELS FOR VOTING: {task_name}")
    print("=" * 70)

    base_models = get_voting_base_models(
        task_type=task_type
    )

    fitted_models = {}

    training_rows = []

    for model_name, model in base_models.items():

        print(f"Training {task_name} — {model_name}")

        start_time = time.time()

        model = fit_model(
            model,
            X_train_input,
            y_train_input,
            sample_weight=sample_weight_input
        )

        elapsed = time.time() - start_time

        fitted_models[model_name] = model

        training_rows.append({
            "Task": task_name,
            "Model": model_name,
            "Training_Time_Seconds": round(elapsed, 4)
        })

        print(f"{model_name} trained in {elapsed:.4f} seconds")

    training_df = pd.DataFrame(training_rows)

    training_df.to_csv(
        os.path.join(
            BLOCK14_METRIC_DIR,
            f"{task_name}_voting_base_model_training_times.csv"
        ),
        index=False
    )

    return fitted_models, training_df


def calculate_validation_weights(
    task_name,
    task_type,
    fitted_models,
    X_val_input,
    y_val_input,
    classes
):

    print("\n")
    print("=" * 70)
    print(f"CALCULATING VALIDATION-BASED VOTING WEIGHTS: {task_name}")
    print("=" * 70)

    weight_rows = []

    for model_name, model in fitted_models.items():

        val_proba = get_aligned_proba(
            model,
            X_val_input,
            classes
        )

        val_pred = np.argmax(
            val_proba,
            axis=1
        )

        if task_type == "binary":

            primary_score = roc_auc_score(
                y_val_input,
                val_proba[:, 1]
            )

            secondary_score = f1_score(
                y_val_input,
                val_pred,
                zero_division=0
            )

            tertiary_score = average_precision_score(
                y_val_input,
                val_proba[:, 1]
            )

        else:

            primary_score = f1_score(
                y_val_input,
                val_pred,
                average="macro",
                zero_division=0
            )

            secondary_score = balanced_accuracy_score(
                y_val_input,
                val_pred
            )

            tertiary_score = np.nan

        weight_rows.append({
            "Task": task_name,
            "Model": model_name,
            "Primary_Score": primary_score,
            "Secondary_Score": secondary_score,
            "Tertiary_Score": tertiary_score
        })

    weights_df = pd.DataFrame(weight_rows)

    raw_scores = weights_df["Primary_Score"].values.astype(float)

    raw_scores = np.nan_to_num(
        raw_scores,
        nan=0.0
    )

    if raw_scores.sum() == 0:

        normalized_weights = np.ones_like(raw_scores) / len(raw_scores)

    else:

        normalized_weights = raw_scores / raw_scores.sum()

    weights_df["Normalized_Weight"] = normalized_weights

    weights = {
        row["Model"]: row["Normalized_Weight"]
        for _, row in weights_df.iterrows()
    }

    weights_df.to_csv(
        os.path.join(
            BLOCK14_WEIGHT_DIR,
            f"{task_name}_validation_voting_weights.csv"
        ),
        index=False
    )

    print(weights_df)

    return weights, weights_df


# ============================================================
# Directory Helper
# ============================================================

def create_block14_task_dirs(task_name, ensemble_name):

    task_dir = os.path.join(
        BLOCK14_DIR,
        task_name,
        ensemble_name
    )

    dirs = {
        "task": task_dir,
        "models": os.path.join(task_dir, "models"),
        "metrics": os.path.join(task_dir, "metrics"),
        "reports": os.path.join(task_dir, "classification_reports"),
        "predictions": os.path.join(task_dir, "predictions"),
        "figures": os.path.join(task_dir, "figures")
    }

    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    return dirs


# ============================================================
# Voting Learning Curve
# ============================================================

def plot_voting_learning_curve(
    task_name,
    task_type,
    ensemble_name,
    X_train_input,
    y_train_input,
    X_val_input,
    y_val_input,
    sample_weight_input,
    classes,
    weights_mode,
    dirs
):

    fractions = [0.25, 0.50, 0.75, 1.00]

    rows = []

    for frac in fractions:

        if frac < 1.0:

            from sklearn.model_selection import train_test_split

            subset_positions, _ = train_test_split(
                np.arange(len(X_train_input)),
                train_size=frac,
                random_state=SEED,
                stratify=y_train_input
            )

        else:

            subset_positions = np.arange(len(X_train_input))

        X_sub = X_train_input.iloc[subset_positions]
        y_sub = y_train_input.iloc[subset_positions]
        w_sub = sample_weight_input[subset_positions]

        fitted_models, _ = train_base_models_for_voting(
            task_name=f"{task_name}_LearningCurve_{frac}",
            task_type=task_type,
            X_train_input=X_sub,
            y_train_input=y_sub,
            sample_weight_input=w_sub
        )

        if weights_mode == "validation_weighted":

            model_weights, _ = calculate_validation_weights(
                task_name=f"{task_name}_LearningCurve_{frac}",
                task_type=task_type,
                fitted_models=fitted_models,
                X_val_input=X_val_input,
                y_val_input=y_val_input,
                classes=classes
            )

        else:

            model_weights = {
                model_name: 1.0
                for model_name in fitted_models.keys()
            }

        voting_model = FittedSoftVotingEnsemble(
            fitted_models=fitted_models,
            model_weights=model_weights,
            classes=classes
        )

        train_pred = voting_model.predict(X_sub)
        val_pred = voting_model.predict(X_val_input)

        if task_type == "binary":

            train_score = f1_score(
                y_sub,
                train_pred,
                zero_division=0
            )

            val_score = f1_score(
                y_val_input,
                val_pred,
                zero_division=0
            )

            metric_name = "F1"

        else:

            train_score = f1_score(
                y_sub,
                train_pred,
                average="macro",
                zero_division=0
            )

            val_score = f1_score(
                y_val_input,
                val_pred,
                average="macro",
                zero_division=0
            )

            metric_name = "Macro F1"

        rows.append({
            "Training_Fraction": frac,
            "Training_Samples": len(X_sub),
            "Train_Score": train_score,
            "Validation_Score": val_score,
            "Metric": metric_name
        })

    learning_df = pd.DataFrame(rows)

    learning_df.to_csv(
        os.path.join(
            dirs["metrics"],
            f"{task_name}_{ensemble_name}_learning_curve.csv"
        ),
        index=False
    )

    plt.figure(figsize=(8, 5))

    plt.plot(
        learning_df["Training_Samples"],
        learning_df["Train_Score"],
        marker="o",
        label=f"Train {metric_name}"
    )

    plt.plot(
        learning_df["Training_Samples"],
        learning_df["Validation_Score"],
        marker="o",
        label=f"Validation {metric_name}"
    )

    plt.title(f"{task_name} — {ensemble_name} Learning Curve")
    plt.xlabel("Training Samples")
    plt.ylabel(metric_name)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            dirs["figures"],
            f"{task_name}_{ensemble_name}_learning_curve.png"
        ),
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    return learning_df


# ============================================================
# Voting Ensemble Evaluation
# ============================================================

def evaluate_voting_ensemble(
    task_name,
    task_type,
    ensemble_name,
    voting_model,
    X_train_input,
    X_val_input,
    X_test_input,
    y_train_input,
    y_val_input,
    y_test_input,
    sample_weight_input,
    classes,
    class_names,
    weights_mode
):

    print("\n")
    print("=" * 70)
    print(f"EVALUATING VOTING ENSEMBLE: {task_name} — {ensemble_name}")
    print("=" * 70)

    dirs = create_block14_task_dirs(
        task_name=task_name,
        ensemble_name=ensemble_name
    )

    # Save model bundle
    joblib.dump(
        voting_model,
        os.path.join(
            dirs["models"],
            f"{task_name}_{ensemble_name}_bundle.pkl"
        )
    )

    # Save voting weights
    pd.DataFrame({
        "Model": list(voting_model.model_weights.keys()),
        "Weight": list(voting_model.model_weights.values())
    }).to_csv(
        os.path.join(
            dirs["metrics"],
            f"{task_name}_{ensemble_name}_weights.csv"
        ),
        index=False
    )

    metrics_list = []
    predictions_list = []

    for split_name, X_split, y_split in [

        ("Train", X_train_input, y_train_input),
        ("Validation", X_val_input, y_val_input),
        ("Test", X_test_input, y_test_input)

    ]:

        metrics, pred_df = evaluate_split_and_export(
            model=voting_model,
            X_split=X_split,
            y_true=y_split,
            split_name=split_name,
            task_name=f"{task_name}_{ensemble_name}",
            task_type=task_type,
            model_name=ensemble_name,
            classes=classes,
            class_names=class_names,
            dirs=dirs
        )

        metrics["Voting_Type"] = ensemble_name
        metrics["Number_of_Base_Models"] = len(voting_model.fitted_models)
        metrics["Base_Models"] = list(voting_model.fitted_models.keys())
        metrics["Weights_Mode"] = weights_mode

        metrics_list.append(metrics)
        predictions_list.append(pred_df)

    plot_voting_learning_curve(
        task_name=task_name,
        task_type=task_type,
        ensemble_name=ensemble_name,
        X_train_input=X_train_input,
        y_train_input=y_train_input,
        X_val_input=X_val_input,
        y_val_input=y_val_input,
        sample_weight_input=sample_weight_input,
        classes=classes,
        weights_mode=weights_mode,
        dirs=dirs
    )

    metrics_df = pd.DataFrame(metrics_list)

    predictions_df = pd.concat(
        predictions_list,
        ignore_index=True
    )

    metrics_df.to_csv(
        os.path.join(
            dirs["metrics"],
            f"{task_name}_{ensemble_name}_train_validation_test_metrics.csv"
        ),
        index=False
    )

    predictions_df.to_csv(
        os.path.join(
            dirs["predictions"],
            f"{task_name}_{ensemble_name}_predictions.csv"
        ),
        index=False
    )

    return metrics_df, predictions_df


print("\n")
print("=" * 70)
print("BLOCK 14 PART 1 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 14
# Voting Ensemble
# PART 2
# Run Equal-Weighted and Validation-Weighted Voting Ensembles
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 14 PART 2 STARTED")
print("=" * 70)


# ============================================================
# Main Voting Runner
# ============================================================

def run_voting_task(
    task_name,
    task_type,
    X_train_input,
    X_val_input,
    X_test_input,
    y_train_input,
    y_val_input,
    y_test_input,
    sample_weight_input,
    classes,
    class_names
):

    print("\n")
    print("=" * 70)
    print(f"RUNNING VOTING ENSEMBLES FOR {task_name}")
    print("=" * 70)

    fitted_models, training_df = train_base_models_for_voting(
        task_name=task_name,
        task_type=task_type,
        X_train_input=X_train_input,
        y_train_input=y_train_input,
        sample_weight_input=sample_weight_input
    )

    equal_weights = {
        model_name: 1.0
        for model_name in fitted_models.keys()
    }

    validation_weights, weights_df = calculate_validation_weights(
        task_name=task_name,
        task_type=task_type,
        fitted_models=fitted_models,
        X_val_input=X_val_input,
        y_val_input=y_val_input,
        classes=classes
    )

    equal_voting_model = FittedSoftVotingEnsemble(
        fitted_models=fitted_models,
        model_weights=equal_weights,
        classes=classes
    )

    weighted_voting_model = FittedSoftVotingEnsemble(
        fitted_models=fitted_models,
        model_weights=validation_weights,
        classes=classes
    )

    equal_metrics_df, equal_predictions_df = evaluate_voting_ensemble(
        task_name=task_name,
        task_type=task_type,
        ensemble_name="EqualSoftVoting",
        voting_model=equal_voting_model,
        X_train_input=X_train_input,
        X_val_input=X_val_input,
        X_test_input=X_test_input,
        y_train_input=y_train_input,
        y_val_input=y_val_input,
        y_test_input=y_test_input,
        sample_weight_input=sample_weight_input,
        classes=classes,
        class_names=class_names,
        weights_mode="equal"
    )

    weighted_metrics_df, weighted_predictions_df = evaluate_voting_ensemble(
        task_name=task_name,
        task_type=task_type,
        ensemble_name="ValidationWeightedSoftVoting",
        voting_model=weighted_voting_model,
        X_train_input=X_train_input,
        X_val_input=X_val_input,
        X_test_input=X_test_input,
        y_train_input=y_train_input,
        y_val_input=y_val_input,
        y_test_input=y_test_input,
        sample_weight_input=sample_weight_input,
        classes=classes,
        class_names=class_names,
        weights_mode="validation_weighted"
    )

    task_metrics_df = pd.concat(
        [
            equal_metrics_df,
            weighted_metrics_df
        ],
        ignore_index=True
    )

    task_predictions_df = pd.concat(
        [
            equal_predictions_df,
            weighted_predictions_df
        ],
        ignore_index=True
    )

    return task_metrics_df, task_predictions_df, training_df, weights_df


# ============================================================
# MDR Voting
# ============================================================

mdr_voting_metrics_df, mdr_voting_predictions_df, mdr_voting_training_df, mdr_voting_weights_df = run_voting_task(
    task_name="MDR",
    task_type="binary",
    X_train_input=X_train_scaled,
    X_val_input=X_val_scaled,
    X_test_input=X_test_scaled,
    y_train_input=y_train_mdr,
    y_val_input=y_val_mdr,
    y_test_input=y_test_mdr,
    sample_weight_input=mdr_train_weights,
    classes=MDR_CLASSES,
    class_names=MDR_CLASS_NAMES
)


# ============================================================
# Severity Voting
# ============================================================

severity_voting_metrics_df, severity_voting_predictions_df, severity_voting_training_df, severity_voting_weights_df = run_voting_task(
    task_name="Severity",
    task_type="multiclass",
    X_train_input=X_train_scaled,
    X_val_input=X_val_scaled,
    X_test_input=X_test_scaled,
    y_train_input=y_train_severity,
    y_val_input=y_val_severity,
    y_test_input=y_test_severity,
    sample_weight_input=severity_train_weights,
    classes=SEVERITY_CLASSES,
    class_names=SEVERITY_CLASS_NAMES
)


# ============================================================
# Combine Results
# ============================================================

block14_all_metrics = pd.concat(
    [
        mdr_voting_metrics_df,
        severity_voting_metrics_df
    ],
    ignore_index=True
)

block14_all_predictions = pd.concat(
    [
        mdr_voting_predictions_df,
        severity_voting_predictions_df
    ],
    ignore_index=True
)

block14_training_summary = pd.concat(
    [
        mdr_voting_training_df,
        severity_voting_training_df
    ],
    ignore_index=True
)

block14_weights_summary = pd.concat(
    [
        mdr_voting_weights_df,
        severity_voting_weights_df
    ],
    ignore_index=True
)

block14_all_metrics.to_csv(
    os.path.join(
        BLOCK14_METRIC_DIR,
        "block14_all_voting_metrics.csv"
    ),
    index=False
)

block14_all_predictions.to_csv(
    os.path.join(
        BLOCK14_PRED_DIR,
        "block14_all_voting_predictions.csv"
    ),
    index=False
)

block14_training_summary.to_csv(
    os.path.join(
        BLOCK14_METRIC_DIR,
        "block14_voting_training_summary.csv"
    ),
    index=False
)

block14_weights_summary.to_csv(
    os.path.join(
        BLOCK14_WEIGHT_DIR,
        "block14_validation_weight_summary.csv"
    ),
    index=False
)


# ============================================================
# Test Result Tables
# ============================================================

mdr_voting_test_table = block14_all_metrics[
    (block14_all_metrics["Task"].astype(str).str.contains("MDR")) &
    (block14_all_metrics["Split"] == "Test")
].copy()

severity_voting_test_table = block14_all_metrics[
    (block14_all_metrics["Task"].astype(str).str.contains("Severity")) &
    (block14_all_metrics["Split"] == "Test")
].copy()

mdr_voting_test_table = mdr_voting_test_table.sort_values(
    by=["ROC_AUC", "F1"],
    ascending=False
)

severity_voting_test_table = severity_voting_test_table.sort_values(
    by=["Macro_F1", "Balanced_Accuracy"],
    ascending=False
)

mdr_voting_test_table.to_csv(
    os.path.join(
        BLOCK14_METRIC_DIR,
        "mdr_voting_test_result.csv"
    ),
    index=False
)

severity_voting_test_table.to_csv(
    os.path.join(
        BLOCK14_METRIC_DIR,
        "severity_voting_test_result.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("MDR VOTING TEST RESULT")
print("=" * 70)

display(mdr_voting_test_table)

print("\n")
print("=" * 70)
print("SEVERITY VOTING TEST RESULT")
print("=" * 70)

display(severity_voting_test_table)


# ============================================================
# Compare All Stages
# ============================================================

comparison_frames = []

# Block 09 baseline
block09_path = os.path.join(
    OUTPUT_DIR,
    "block09_baseline_models",
    "metrics",
    "block09_all_baseline_metrics.csv"
)

if os.path.exists(block09_path):

    block09_metrics = pd.read_csv(block09_path)
    block09_test = block09_metrics[block09_metrics["Split"] == "Test"].copy()
    block09_test["Stage"] = "Baseline"
    comparison_frames.append(block09_test)

# Block 10 optimized
block10_path = os.path.join(
    OUTPUT_DIR,
    "block10_hyperparameter_optimization",
    "metrics",
    "block10_all_optimized_metrics.csv"
)

if os.path.exists(block10_path):

    block10_metrics = pd.read_csv(block10_path)
    block10_test = block10_metrics[block10_metrics["Split"] == "Test"].copy()
    block10_test["Stage"] = "Optimized"
    comparison_frames.append(block10_test)

# Block 11 feature selected
block11_path = os.path.join(
    OUTPUT_DIR,
    "block11_feature_selection",
    "metrics",
    "block11_all_feature_selected_metrics.csv"
)

if os.path.exists(block11_path):

    block11_metrics = pd.read_csv(block11_path)
    block11_test = block11_metrics[block11_metrics["Split"] == "Test"].copy()
    block11_test["Stage"] = "Feature_Selected"
    comparison_frames.append(block11_test)

# Block 13 stacking
block13_path = os.path.join(
    OUTPUT_DIR,
    "block13_stacking_ensemble",
    "metrics",
    "block13_all_stacking_metrics.csv"
)

if os.path.exists(block13_path):

    block13_metrics = pd.read_csv(block13_path)
    block13_test = block13_metrics[block13_metrics["Split"] == "Test"].copy()
    block13_test["Stage"] = "Stacking_Ensemble"
    comparison_frames.append(block13_test)

# Block 14 voting
block14_test = block14_all_metrics[
    block14_all_metrics["Split"] == "Test"
].copy()

block14_test["Stage"] = "Voting_Ensemble"
comparison_frames.append(block14_test)

full_stage_comparison = pd.concat(
    comparison_frames,
    ignore_index=True
)

full_stage_comparison.to_csv(
    os.path.join(
        BLOCK14_METRIC_DIR,
        "block14_full_model_stage_comparison.csv"
    ),
    index=False
)


# ============================================================
# Publication Figures
# ============================================================

def plot_voting_stage_comparison(
    df,
    task_keyword,
    metric,
    title,
    save_path
):

    plot_df = df[
        df["Task"].astype(str).str.contains(
            task_keyword,
            case=False,
            regex=False
        )
    ].copy()

    if len(plot_df) == 0:
        print(f"No data for {task_keyword} — {metric}")
        return

    plot_df["Model_Label"] = (
        plot_df["Stage"].astype(str)
        + " | "
        + plot_df["Model"].astype(str)
    )

    plot_df = plot_df.sort_values(
        by=metric,
        ascending=False
    ).head(15)

    plt.figure(figsize=(11, 7))

    sns.barplot(
        data=plot_df,
        x=metric,
        y="Model_Label",
        orient="h"
    )

    plt.title(title)
    plt.xlabel(metric)
    plt.ylabel("Stage | Model")
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


plot_voting_stage_comparison(
    df=full_stage_comparison,
    task_keyword="MDR",
    metric="ROC_AUC",
    title="MDR Model Stage Comparison — Test ROC-AUC",
    save_path=os.path.join(
        BLOCK14_FIG_DIR,
        "figure_14a_mdr_full_stage_comparison_roc_auc.png"
    )
)

plot_voting_stage_comparison(
    df=full_stage_comparison,
    task_keyword="MDR",
    metric="F1",
    title="MDR Model Stage Comparison — Test F1-score",
    save_path=os.path.join(
        BLOCK14_FIG_DIR,
        "figure_14b_mdr_full_stage_comparison_f1.png"
    )
)

plot_voting_stage_comparison(
    df=full_stage_comparison,
    task_keyword="Severity",
    metric="Macro_F1",
    title="Severity Model Stage Comparison — Test Macro F1-score",
    save_path=os.path.join(
        BLOCK14_FIG_DIR,
        "figure_14c_severity_full_stage_comparison_macro_f1.png"
    )
)

plot_voting_stage_comparison(
    df=full_stage_comparison,
    task_keyword="Severity",
    metric="Balanced_Accuracy",
    title="Severity Model Stage Comparison — Test Balanced Accuracy",
    save_path=os.path.join(
        BLOCK14_FIG_DIR,
        "figure_14d_severity_full_stage_comparison_balanced_accuracy.png"
    )
)


# ============================================================
# Final Summary
# ============================================================

block14_summary = pd.DataFrame({
    "Output_Category": [
        "Voting ensemble metrics",
        "Voting ensemble predictions",
        "Voting training summary",
        "Validation weight summary",
        "MDR voting test result",
        "Severity voting test result",
        "Full model stage comparison",
        "Voting model bundles",
        "Train/Validation/Test reports",
        "Train/Validation/Test confusion matrices",
        "Train/Validation/Test ROC curves",
        "Train/Validation/Test PR curves",
        "Train/Validation/Test calibration curves",
        "Train/Validation/Test probability distributions",
        "Learning curves",
        "Publication comparison figures"
    ],
    "Saved": [
        "metrics/block14_all_voting_metrics.csv",
        "predictions/block14_all_voting_predictions.csv",
        "metrics/block14_voting_training_summary.csv",
        "voting_weights/block14_validation_weight_summary.csv",
        "metrics/mdr_voting_test_result.csv",
        "metrics/severity_voting_test_result.csv",
        "metrics/block14_full_model_stage_comparison.csv",
        "models and per-task folders",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "figures/figure_14*.png"
    ]
})

block14_summary.to_csv(
    os.path.join(
        BLOCK14_DIR,
        "block14_output_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("BLOCK 14 GENERATED FILES")
print("=" * 70)

print("- metrics/block14_all_voting_metrics.csv")
print("- predictions/block14_all_voting_predictions.csv")
print("- metrics/mdr_voting_test_result.csv")
print("- metrics/severity_voting_test_result.csv")
print("- metrics/block14_full_model_stage_comparison.csv")
print("- voting_weights/block14_validation_weight_summary.csv")
print("- voting ensemble model bundles")
print("- Train/Validation/Test reports, confusion matrices, ROC, PR, calibration, probability distributions")
print("- Voting learning curves")
print("- Full stage comparison figures")

print("\n")
print("=" * 70)
print("BLOCK 14 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 15
# Statistical Analysis and Significance Testing
# PART 1
# Load Results + Define Statistical Helper Functions
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 15 STARTED — STATISTICAL ANALYSIS")
print("=" * 70)

import os
import glob
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    cohen_kappa_score
)

# ============================================================
# Dependency Check
# ============================================================

required_objects = [
    "OUTPUT_DIR",
    "SEED",
    "MDR_CLASSES",
    "MDR_CLASS_NAMES",
    "SEVERITY_CLASSES",
    "SEVERITY_CLASS_NAMES",
    "safe_name"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if len(missing_objects) > 0:
    raise RuntimeError(
        "Please run previous blocks before BLOCK 15. Missing: "
        + ", ".join(missing_objects)
    )


# ============================================================
# Output Directories
# ============================================================

BLOCK15_DIR = os.path.join(OUTPUT_DIR, "block15_statistical_analysis")

BLOCK15_TABLE_DIR = os.path.join(BLOCK15_DIR, "tables")
BLOCK15_FIG_DIR = os.path.join(BLOCK15_DIR, "figures")
BLOCK15_TEST_DIR = os.path.join(BLOCK15_DIR, "statistical_tests")
BLOCK15_CI_DIR = os.path.join(BLOCK15_DIR, "confidence_intervals")

for d in [
    BLOCK15_DIR,
    BLOCK15_TABLE_DIR,
    BLOCK15_FIG_DIR,
    BLOCK15_TEST_DIR,
    BLOCK15_CI_DIR
]:
    os.makedirs(d, exist_ok=True)

FIG_DPI = 300
N_BOOTSTRAP = 1000
rng = np.random.default_rng(SEED)


# ============================================================
# Stage Paths
# ============================================================

stage_paths = {
    "Baseline": os.path.join(
        OUTPUT_DIR,
        "block09_baseline_models"
    ),
    "Optimized": os.path.join(
        OUTPUT_DIR,
        "block10_hyperparameter_optimization"
    ),
    "Feature_Selected": os.path.join(
        OUTPUT_DIR,
        "block11_feature_selection"
    ),
    "Stacking_Ensemble": os.path.join(
        OUTPUT_DIR,
        "block13_stacking_ensemble"
    ),
    "Voting_Ensemble": os.path.join(
        OUTPUT_DIR,
        "block14_voting_ensemble"
    )
}


# ============================================================
# Load Metrics
# ============================================================

known_metric_files = {
    "Baseline": os.path.join(
        stage_paths["Baseline"],
        "metrics",
        "block09_all_baseline_metrics.csv"
    ),
    "Optimized": os.path.join(
        stage_paths["Optimized"],
        "metrics",
        "block10_all_optimized_metrics.csv"
    ),
    "Feature_Selected": os.path.join(
        stage_paths["Feature_Selected"],
        "metrics",
        "block11_all_feature_selected_metrics.csv"
    ),
    "Stacking_Ensemble": os.path.join(
        stage_paths["Stacking_Ensemble"],
        "metrics",
        "block13_all_stacking_metrics.csv"
    ),
    "Voting_Ensemble": os.path.join(
        stage_paths["Voting_Ensemble"],
        "metrics",
        "block14_all_voting_metrics.csv"
    )
}

metric_frames = []

for stage, path in known_metric_files.items():

    if os.path.exists(path):

        df_stage = pd.read_csv(path)
        df_stage["Stage"] = stage
        df_stage["Source_File"] = path

        metric_frames.append(df_stage)

        print(f"Loaded metrics: {stage} -> {path}")

    else:

        print(f"Missing metrics file: {stage} -> {path}")

if len(metric_frames) == 0:
    raise RuntimeError("No metric files found from previous blocks.")

all_stage_metrics = pd.concat(
    metric_frames,
    ignore_index=True
)

all_stage_metrics.to_csv(
    os.path.join(
        BLOCK15_TABLE_DIR,
        "all_stage_metrics_loaded.csv"
    ),
    index=False
)

print("\nAll loaded metric rows:", len(all_stage_metrics))


# ============================================================
# Load Predictions
# ============================================================

def load_prediction_files():

    prediction_frames = []

    known_prediction_files = {
        "Baseline": [
            os.path.join(
                stage_paths["Baseline"],
                "predictions",
                "block09_all_baseline_predictions.csv"
            )
        ],
        "Optimized": [
            os.path.join(
                stage_paths["Optimized"],
                "predictions",
                "block10_all_optimized_predictions.csv"
            ),
            os.path.join(
                stage_paths["Optimized"],
                "predictions",
                "mdr_optimized_predictions.csv"
            ),
            os.path.join(
                stage_paths["Optimized"],
                "predictions",
                "severity_optimized_predictions.csv"
            )
        ],
        "Feature_Selected": [
            os.path.join(
                stage_paths["Feature_Selected"],
                "predictions",
                "block11_all_feature_selected_predictions.csv"
            )
        ],
        "Stacking_Ensemble": [
            os.path.join(
                stage_paths["Stacking_Ensemble"],
                "predictions",
                "block13_all_stacking_predictions.csv"
            )
        ],
        "Voting_Ensemble": [
            os.path.join(
                stage_paths["Voting_Ensemble"],
                "predictions",
                "block14_all_voting_predictions.csv"
            )
        ]
    }

    for stage, paths in known_prediction_files.items():

        loaded_any = False

        for path in paths:

            if os.path.exists(path):

                pred_df = pd.read_csv(path)
                pred_df["Stage"] = stage
                pred_df["Source_File"] = path

                prediction_frames.append(pred_df)

                loaded_any = True

                print(f"Loaded predictions: {stage} -> {path}")

        if not loaded_any:

            recursive_files = glob.glob(
                os.path.join(
                    stage_paths[stage],
                    "**",
                    "*predictions*.csv"
                ),
                recursive=True
            )

            for path in recursive_files:

                try:

                    pred_df = pd.read_csv(path)

                    if len(pred_df) == 0:
                        continue

                    pred_df["Stage"] = stage
                    pred_df["Source_File"] = path

                    prediction_frames.append(pred_df)

                    print(f"Loaded fallback predictions: {stage} -> {path}")

                except Exception as e:

                    print(f"Skipped prediction file: {path} | {e}")

    if len(prediction_frames) == 0:
        raise RuntimeError("No prediction files found from previous blocks.")

    combined_predictions = pd.concat(
        prediction_frames,
        ignore_index=True
    )

    return combined_predictions


all_stage_predictions = load_prediction_files()

all_stage_predictions.to_csv(
    os.path.join(
        BLOCK15_TABLE_DIR,
        "all_stage_predictions_loaded.csv"
    ),
    index=False
)

print("\nAll loaded prediction rows:", len(all_stage_predictions))
print("\nPrediction columns:")
print(list(all_stage_predictions.columns))


# ============================================================
# Standardize Prediction Columns
# ============================================================

def find_first_existing_column(df, candidates):

    for col in candidates:

        if col in df.columns:
            return col

    return None


TRUE_COL = find_first_existing_column(
    all_stage_predictions,
    [
        "True_Label",
        "y_true",
        "True",
        "Actual",
        "Actual_Label",
        "Ground_Truth"
    ]
)

PRED_COL = find_first_existing_column(
    all_stage_predictions,
    [
        "Predicted_Label",
        "y_pred",
        "Prediction",
        "Predicted",
        "Predicted_Class"
    ]
)

TASK_COL = find_first_existing_column(
    all_stage_predictions,
    [
        "Task"
    ]
)

SPLIT_COL = find_first_existing_column(
    all_stage_predictions,
    [
        "Split"
    ]
)

MODEL_COL = find_first_existing_column(
    all_stage_predictions,
    [
        "Model"
    ]
)

INDEX_COL = find_first_existing_column(
    all_stage_predictions,
    [
        "Index",
        "Sample_Index",
        "Original_Index",
        "Genome_Index"
    ]
)

if TRUE_COL is None or PRED_COL is None or TASK_COL is None or SPLIT_COL is None or MODEL_COL is None:

    raise RuntimeError(
        "Could not detect required prediction columns. "
        f"Detected TRUE_COL={TRUE_COL}, PRED_COL={PRED_COL}, "
        f"TASK_COL={TASK_COL}, SPLIT_COL={SPLIT_COL}, MODEL_COL={MODEL_COL}"
    )

print("\nDetected Columns:")
print("TRUE_COL :", TRUE_COL)
print("PRED_COL :", PRED_COL)
print("TASK_COL :", TASK_COL)
print("SPLIT_COL:", SPLIT_COL)
print("MODEL_COL:", MODEL_COL)
print("INDEX_COL:", INDEX_COL)


# ============================================================
# Probability Column Detection
# ============================================================

def get_probability_columns(df, task_type):

    numeric_cols = df.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    exclude_cols = {
        TRUE_COL,
        PRED_COL,
        "Stage",
        "Fold",
        "Run",
        "Seed"
    }

    if INDEX_COL is not None:
        exclude_cols.add(INDEX_COL)

    prob_cols = []

    for col in df.columns:

        col_lower = str(col).lower()

        if (
            "prob" in col_lower
            or "proba" in col_lower
            or "probability" in col_lower
        ):

            if col not in exclude_cols:
                prob_cols.append(col)

    prob_cols = [
        col for col in prob_cols
        if col in numeric_cols
    ]

    if task_type == "binary":

        positive_candidates = [
            col for col in prob_cols
            if (
                "positive" in col.lower()
                or "mdr" in col.lower()
                or col.lower().endswith("_1")
                or col.lower().endswith("1")
            )
        ]

        if len(positive_candidates) > 0:
            return positive_candidates[:1]

        if len(prob_cols) >= 2:
            return [prob_cols[-1]]

        return prob_cols[:1]

    return prob_cols


# ============================================================
# Filter Test Predictions
# ============================================================

test_predictions = all_stage_predictions[
    all_stage_predictions[SPLIT_COL].astype(str).str.lower() == "test"
].copy()

test_predictions["Task_Normalized"] = test_predictions[TASK_COL].astype(str)

test_predictions.to_csv(
    os.path.join(
        BLOCK15_TABLE_DIR,
        "test_predictions_for_statistical_analysis.csv"
    ),
    index=False
)

print("\nTest prediction rows:", len(test_predictions))


# ============================================================
# Utility: Clean Model Key
# ============================================================

def make_model_key(row):

    return (
        str(row["Stage"])
        + " | "
        + str(row[TASK_COL])
        + " | "
        + str(row[MODEL_COL])
    )


test_predictions["Model_Key"] = test_predictions.apply(
    make_model_key,
    axis=1
)


# ============================================================
# Bootstrap Metric Confidence Interval
# ============================================================

def bootstrap_metric_ci(
    y_true,
    y_pred,
    y_proba,
    task_type,
    metric_name,
    n_bootstrap=N_BOOTSTRAP,
    alpha=0.05
):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if y_proba is not None:
        y_proba = np.array(y_proba)

    n = len(y_true)

    scores = []

    for _ in range(n_bootstrap):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        y_true_b = y_true[idx]
        y_pred_b = y_pred[idx]

        try:

            if metric_name == "Accuracy":

                score = accuracy_score(
                    y_true_b,
                    y_pred_b
                )

            elif metric_name == "F1":

                if task_type == "binary":

                    score = f1_score(
                        y_true_b,
                        y_pred_b,
                        zero_division=0
                    )

                else:

                    score = f1_score(
                        y_true_b,
                        y_pred_b,
                        average="macro",
                        zero_division=0
                    )

            elif metric_name == "Balanced_Accuracy":

                score = balanced_accuracy_score(
                    y_true_b,
                    y_pred_b
                )

            elif metric_name == "MCC":

                score = matthews_corrcoef(
                    y_true_b,
                    y_pred_b
                )

            elif metric_name == "Cohen_Kappa":

                score = cohen_kappa_score(
                    y_true_b,
                    y_pred_b
                )

            elif metric_name == "ROC_AUC" and y_proba is not None:

                y_proba_b = y_proba[idx]

                if task_type == "binary":

                    score = roc_auc_score(
                        y_true_b,
                        y_proba_b
                    )

                else:

                    if y_proba_b.ndim == 2 and y_proba_b.shape[1] > 1:

                        score = roc_auc_score(
                            y_true_b,
                            y_proba_b,
                            multi_class="ovr",
                            average="macro"
                        )

                    else:

                        continue

            else:

                continue

            if not np.isnan(score):
                scores.append(score)

        except Exception:
            continue

    if len(scores) == 0:

        return {
            "Mean": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "Bootstrap_N": 0
        }

    lower = np.percentile(
        scores,
        100 * alpha / 2
    )

    upper = np.percentile(
        scores,
        100 * (1 - alpha / 2)
    )

    return {
        "Mean": float(np.mean(scores)),
        "CI_Lower": float(lower),
        "CI_Upper": float(upper),
        "Bootstrap_N": len(scores)
    }


# ============================================================
# Paired Bootstrap Difference Test
# ============================================================

def paired_bootstrap_difference_test(
    y_true,
    pred_a,
    pred_b,
    task_type,
    metric_name,
    n_bootstrap=N_BOOTSTRAP
):

    y_true = np.array(y_true)
    pred_a = np.array(pred_a)
    pred_b = np.array(pred_b)

    n = len(y_true)

    diffs = []

    for _ in range(n_bootstrap):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        yt = y_true[idx]
        pa = pred_a[idx]
        pb = pred_b[idx]

        try:

            if metric_name == "Accuracy":

                score_a = accuracy_score(yt, pa)
                score_b = accuracy_score(yt, pb)

            elif metric_name == "F1":

                if task_type == "binary":

                    score_a = f1_score(
                        yt,
                        pa,
                        zero_division=0
                    )

                    score_b = f1_score(
                        yt,
                        pb,
                        zero_division=0
                    )

                else:

                    score_a = f1_score(
                        yt,
                        pa,
                        average="macro",
                        zero_division=0
                    )

                    score_b = f1_score(
                        yt,
                        pb,
                        average="macro",
                        zero_division=0
                    )

            elif metric_name == "Balanced_Accuracy":

                score_a = balanced_accuracy_score(yt, pa)
                score_b = balanced_accuracy_score(yt, pb)

            elif metric_name == "MCC":

                score_a = matthews_corrcoef(yt, pa)
                score_b = matthews_corrcoef(yt, pb)

            else:

                continue

            diffs.append(score_a - score_b)

        except Exception:
            continue

    diffs = np.array(diffs)

    if len(diffs) == 0:

        return {
            "Mean_Difference": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "P_Value_Two_Sided": np.nan,
            "Bootstrap_N": 0
        }

    ci_lower = np.percentile(diffs, 2.5)
    ci_upper = np.percentile(diffs, 97.5)

    p_value = 2 * min(
        np.mean(diffs <= 0),
        np.mean(diffs >= 0)
    )

    p_value = min(float(p_value), 1.0)

    return {
        "Mean_Difference": float(np.mean(diffs)),
        "CI_Lower": float(ci_lower),
        "CI_Upper": float(ci_upper),
        "P_Value_Two_Sided": p_value,
        "Bootstrap_N": len(diffs)
    }


# ============================================================
# McNemar Test on Correct/Incorrect Pairs
# ============================================================

def paired_mcnemar_test(
    y_true,
    pred_a,
    pred_b
):

    y_true = np.array(y_true)
    pred_a = np.array(pred_a)
    pred_b = np.array(pred_b)

    correct_a = pred_a == y_true
    correct_b = pred_b == y_true

    both_correct = np.sum(correct_a & correct_b)
    a_correct_b_wrong = np.sum(correct_a & ~correct_b)
    a_wrong_b_correct = np.sum(~correct_a & correct_b)
    both_wrong = np.sum(~correct_a & ~correct_b)

    table = [
        [both_correct, a_correct_b_wrong],
        [a_wrong_b_correct, both_wrong]
    ]

    try:

        result = mcnemar(
            table,
            exact=False,
            correction=True
        )

        statistic = float(result.statistic)
        p_value = float(result.pvalue)

    except Exception:

        statistic = np.nan
        p_value = np.nan

    return {
        "Both_Correct": int(both_correct),
        "A_Correct_B_Wrong": int(a_correct_b_wrong),
        "A_Wrong_B_Correct": int(a_wrong_b_correct),
        "Both_Wrong": int(both_wrong),
        "McNemar_Statistic": statistic,
        "McNemar_P_Value": p_value
    }


print("\n")
print("=" * 70)
print("BLOCK 15 PART 1 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 15
# PART 2 FAST RECOVERY VERSION
# Statistical Analysis + Ranking + Reduced Pairwise Tests
# ============================================================

print("=" * 70)
print("BLOCK 15 PART 2 FAST RECOVERY STARTED")
print("=" * 70)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score
)

from statsmodels.stats.contingency_tables import mcnemar

sns.set_theme(style="whitegrid")

# ============================================================
# Safety Check
# ============================================================

required_fast_objects = [
    "all_stage_metrics",
    "test_predictions",
    "TRUE_COL",
    "PRED_COL",
    "TASK_COL",
    "SPLIT_COL",
    "MODEL_COL",
    "INDEX_COL",
    "BLOCK15_DIR",
    "BLOCK15_TABLE_DIR",
    "BLOCK15_TEST_DIR",
    "BLOCK15_CI_DIR",
    "BLOCK15_FIG_DIR",
    "SEED"
]

missing_fast_objects = [
    obj for obj in required_fast_objects
    if obj not in globals()
]

if len(missing_fast_objects) > 0:
    raise RuntimeError(
        "Block 15 Part 1 must be active in memory. Missing: "
        + ", ".join(missing_fast_objects)
    )

rng = np.random.default_rng(SEED)
N_BOOTSTRAP_FAST = 300
FIG_DPI = 300


# ============================================================
# Helper Functions
# ============================================================

def infer_task_type_fast(task_name):

    task_name_lower = str(task_name).lower()

    if "mdr" in task_name_lower:
        return "binary"

    if "severity" in task_name_lower:
        return "multiclass"

    return "unknown"


def metric_score_fast(y_true, y_pred, task_type, metric_name):

    if metric_name == "Accuracy":
        return accuracy_score(y_true, y_pred)

    if metric_name == "F1":

        if task_type == "binary":
            return f1_score(y_true, y_pred, zero_division=0)

        return f1_score(y_true, y_pred, average="macro", zero_division=0)

    if metric_name == "Balanced_Accuracy":
        return balanced_accuracy_score(y_true, y_pred)

    if metric_name == "MCC":
        return matthews_corrcoef(y_true, y_pred)

    if metric_name == "Cohen_Kappa":
        return cohen_kappa_score(y_true, y_pred)

    return np.nan


def bootstrap_ci_fast(y_true, y_pred, task_type, metric_name):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    n = len(y_true)
    scores = []

    for _ in range(N_BOOTSTRAP_FAST):

        idx = rng.integers(0, n, size=n)

        try:
            score = metric_score_fast(
                y_true[idx],
                y_pred[idx],
                task_type,
                metric_name
            )

            if not np.isnan(score):
                scores.append(score)

        except Exception:
            continue

    if len(scores) == 0:

        return {
            "Mean": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "Bootstrap_N": 0
        }

    return {
        "Mean": float(np.mean(scores)),
        "CI_Lower": float(np.percentile(scores, 2.5)),
        "CI_Upper": float(np.percentile(scores, 97.5)),
        "Bootstrap_N": len(scores)
    }


def paired_bootstrap_diff_fast(
    y_true,
    pred_best,
    pred_compare,
    task_type,
    metric_name
):

    y_true = np.array(y_true)
    pred_best = np.array(pred_best)
    pred_compare = np.array(pred_compare)

    n = len(y_true)
    diffs = []

    for _ in range(N_BOOTSTRAP_FAST):

        idx = rng.integers(0, n, size=n)

        try:

            score_best = metric_score_fast(
                y_true[idx],
                pred_best[idx],
                task_type,
                metric_name
            )

            score_compare = metric_score_fast(
                y_true[idx],
                pred_compare[idx],
                task_type,
                metric_name
            )

            diffs.append(score_best - score_compare)

        except Exception:
            continue

    if len(diffs) == 0:

        return {
            "Mean_Difference": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "P_Value_Two_Sided": np.nan
        }

    diffs = np.array(diffs)

    p_value = 2 * min(
        np.mean(diffs <= 0),
        np.mean(diffs >= 0)
    )

    return {
        "Mean_Difference": float(np.mean(diffs)),
        "CI_Lower": float(np.percentile(diffs, 2.5)),
        "CI_Upper": float(np.percentile(diffs, 97.5)),
        "P_Value_Two_Sided": min(float(p_value), 1.0)
    }


def mcnemar_fast(y_true, pred_best, pred_compare):

    y_true = np.array(y_true)
    pred_best = np.array(pred_best)
    pred_compare = np.array(pred_compare)

    best_correct = pred_best == y_true
    compare_correct = pred_compare == y_true

    both_correct = np.sum(best_correct & compare_correct)
    best_correct_compare_wrong = np.sum(best_correct & ~compare_correct)
    best_wrong_compare_correct = np.sum(~best_correct & compare_correct)
    both_wrong = np.sum(~best_correct & ~compare_correct)

    table = [
        [both_correct, best_correct_compare_wrong],
        [best_wrong_compare_correct, both_wrong]
    ]

    try:
        result = mcnemar(table, exact=False, correction=True)
        stat = float(result.statistic)
        pval = float(result.pvalue)

    except Exception:
        stat = np.nan
        pval = np.nan

    return {
        "Both_Correct": int(both_correct),
        "Best_Correct_Compared_Wrong": int(best_correct_compare_wrong),
        "Best_Wrong_Compared_Correct": int(best_wrong_compare_correct),
        "Both_Wrong": int(both_wrong),
        "McNemar_Statistic": stat,
        "McNemar_P_Value": pval
    }


# ============================================================
# Test Metrics Ranking
# ============================================================

test_metrics = all_stage_metrics[
    all_stage_metrics["Split"].astype(str).str.lower() == "test"
].copy()

test_metrics["Task_Type"] = test_metrics["Task"].apply(
    infer_task_type_fast
)

mdr_test_metrics = test_metrics[
    test_metrics["Task_Type"] == "binary"
].copy()

severity_test_metrics = test_metrics[
    test_metrics["Task_Type"] == "multiclass"
].copy()

mdr_test_ranking = mdr_test_metrics.sort_values(
    by=["ROC_AUC", "F1", "Accuracy"],
    ascending=False
)

severity_test_ranking = severity_test_metrics.sort_values(
    by=["Macro_F1", "Balanced_Accuracy", "Accuracy"],
    ascending=False
)

mdr_test_ranking.to_csv(
    os.path.join(
        BLOCK15_TABLE_DIR,
        "mdr_test_performance_ranking.csv"
    ),
    index=False
)

severity_test_ranking.to_csv(
    os.path.join(
        BLOCK15_TABLE_DIR,
        "severity_test_performance_ranking.csv"
    ),
    index=False
)

mdr_best_row = mdr_test_ranking.iloc[0]
severity_best_row = severity_test_ranking.iloc[0]

best_model_summary = pd.DataFrame([
    {
        "Task": "MDR",
        "Best_Stage": mdr_best_row["Stage"],
        "Best_Model": mdr_best_row["Model"],
        "Best_Task_Name": mdr_best_row["Task"],
        "Accuracy": mdr_best_row.get("Accuracy", np.nan),
        "F1": mdr_best_row.get("F1", np.nan),
        "Balanced_Accuracy": mdr_best_row.get("Balanced_Accuracy", np.nan),
        "ROC_AUC": mdr_best_row.get("ROC_AUC", np.nan),
        "MCC": mdr_best_row.get("MCC", np.nan)
    },
    {
        "Task": "Severity",
        "Best_Stage": severity_best_row["Stage"],
        "Best_Model": severity_best_row["Model"],
        "Best_Task_Name": severity_best_row["Task"],
        "Accuracy": severity_best_row.get("Accuracy", np.nan),
        "Macro_F1": severity_best_row.get("Macro_F1", np.nan),
        "Balanced_Accuracy": severity_best_row.get("Balanced_Accuracy", np.nan),
        "ROC_AUC_OVR_Macro": severity_best_row.get("ROC_AUC_OVR_Macro", np.nan),
        "MCC": severity_best_row.get("MCC", np.nan)
    }
])

best_model_summary.to_csv(
    os.path.join(
        BLOCK15_TABLE_DIR,
        "best_model_summary_from_test_metrics.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("BEST MODEL SUMMARY FROM TEST METRICS")
print("=" * 70)
display(best_model_summary)


print("\n")
print("=" * 70)
print("MDR TEST PERFORMANCE RANKING")
print("=" * 70)
display(mdr_test_ranking.head(15))


print("\n")
print("=" * 70)
print("SEVERITY TEST PERFORMANCE RANKING")
print("=" * 70)
display(severity_test_ranking.head(15))


# ============================================================
# Prepare Deduplicated Test Predictions
# ============================================================

test_predictions_fast = test_predictions.copy()

dedup_cols = [
    "Stage",
    TASK_COL,
    MODEL_COL,
    SPLIT_COL,
    TRUE_COL,
    PRED_COL
]

if INDEX_COL is not None:
    dedup_cols.append(INDEX_COL)

test_predictions_fast = test_predictions_fast.drop_duplicates(
    subset=dedup_cols
).copy()

print("\nTest predictions after fast dedup:", len(test_predictions_fast))


# ============================================================
# Build Model Keys
# ============================================================

test_predictions_fast["Task_Type"] = test_predictions_fast[TASK_COL].apply(
    infer_task_type_fast
)

test_predictions_fast["Model_Key"] = (
    test_predictions_fast["Stage"].astype(str)
    + " | "
    + test_predictions_fast[TASK_COL].astype(str)
    + " | "
    + test_predictions_fast[MODEL_COL].astype(str)
)


# ============================================================
# Keep Only Top Models for Faster Statistics
# ============================================================

top_mdr_keys = (
    mdr_test_ranking
    .head(8)
    .apply(
        lambda r:
        str(r["Stage"]) + " | " + str(r["Task"]) + " | " + str(r["Model"]),
        axis=1
    )
    .tolist()
)

top_severity_keys = (
    severity_test_ranking
    .head(8)
    .apply(
        lambda r:
        str(r["Stage"]) + " | " + str(r["Task"]) + " | " + str(r["Model"]),
        axis=1
    )
    .tolist()
)

top_model_keys = list(set(top_mdr_keys + top_severity_keys))

top_predictions = test_predictions_fast[
    test_predictions_fast["Model_Key"].isin(top_model_keys)
].copy()

print("Top model prediction rows:", len(top_predictions))


# ============================================================
# Confidence Intervals for Top Models
# ============================================================

ci_rows = []

for model_key, group_df in top_predictions.groupby("Model_Key"):

    task_type = group_df["Task_Type"].iloc[0]
    stage = group_df["Stage"].iloc[0]
    task = group_df[TASK_COL].iloc[0]
    model = group_df[MODEL_COL].iloc[0]

    y_true = group_df[TRUE_COL].values
    y_pred = group_df[PRED_COL].values

    if task_type == "binary":

        metrics_list = [
            "Accuracy",
            "F1",
            "Balanced_Accuracy",
            "MCC",
            "Cohen_Kappa"
        ]

    else:

        metrics_list = [
            "Accuracy",
            "F1",
            "Balanced_Accuracy",
            "MCC",
            "Cohen_Kappa"
        ]

    for metric_name in metrics_list:

        ci = bootstrap_ci_fast(
            y_true=y_true,
            y_pred=y_pred,
            task_type=task_type,
            metric_name=metric_name
        )

        ci_rows.append({
            "Stage": stage,
            "Task": task,
            "Model": model,
            "Model_Key": model_key,
            "Task_Type": task_type,
            "Metric": metric_name,
            "Mean": ci["Mean"],
            "CI_Lower": ci["CI_Lower"],
            "CI_Upper": ci["CI_Upper"],
            "Bootstrap_N": ci["Bootstrap_N"],
            "N_Test_Samples": len(group_df)
        })

confidence_interval_df_fast = pd.DataFrame(ci_rows)

confidence_interval_df_fast.to_csv(
    os.path.join(
        BLOCK15_CI_DIR,
        "bootstrap_confidence_intervals_fast.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("FAST BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 70)
display(confidence_interval_df_fast.head(20))


# ============================================================
# Pairwise Tests Against Best Top Model
# ============================================================

def get_group_by_key(pred_df, key):

    temp = pred_df[
        pred_df["Model_Key"] == key
    ].copy()

    if len(temp) == 0:
        return None

    if INDEX_COL is not None and INDEX_COL in temp.columns:
        temp = temp.sort_values(INDEX_COL).reset_index(drop=True)
    else:
        temp = temp.reset_index(drop=True)

    return temp


mdr_best_key = (
    str(mdr_best_row["Stage"])
    + " | "
    + str(mdr_best_row["Task"])
    + " | "
    + str(mdr_best_row["Model"])
)

severity_best_key = (
    str(severity_best_row["Stage"])
    + " | "
    + str(severity_best_row["Task"])
    + " | "
    + str(severity_best_row["Model"])
)

pairwise_rows = []

for task_label, task_type, best_key, compare_keys in [
    ("MDR", "binary", mdr_best_key, top_mdr_keys),
    ("Severity", "multiclass", severity_best_key, top_severity_keys)
]:

    best_group = get_group_by_key(
        top_predictions,
        best_key
    )

    if best_group is None:
        print("Best group missing:", best_key)
        continue

    for compare_key in compare_keys:

        if compare_key == best_key:
            continue

        compare_group = get_group_by_key(
            top_predictions,
            compare_key
        )

        if compare_group is None:
            continue

        min_len = min(
            len(best_group),
            len(compare_group)
        )

        bg = best_group.iloc[:min_len].copy()
        cg = compare_group.iloc[:min_len].copy()

        y_true = bg[TRUE_COL].values
        pred_best = bg[PRED_COL].values
        pred_compare = cg[PRED_COL].values

        for metric_name in [
            "Accuracy",
            "F1",
            "Balanced_Accuracy",
            "MCC"
        ]:

            diff = paired_bootstrap_diff_fast(
                y_true=y_true,
                pred_best=pred_best,
                pred_compare=pred_compare,
                task_type=task_type,
                metric_name=metric_name
            )

            mcn = mcnemar_fast(
                y_true=y_true,
                pred_best=pred_best,
                pred_compare=pred_compare
            )

            pairwise_rows.append({
                "Task": task_label,
                "Best_Model_Key": best_key,
                "Compared_Model_Key": compare_key,
                "Metric": metric_name,
                "Mean_Difference_Best_Minus_Compared": diff["Mean_Difference"],
                "Difference_CI_Lower": diff["CI_Lower"],
                "Difference_CI_Upper": diff["CI_Upper"],
                "Bootstrap_P_Value_Two_Sided": diff["P_Value_Two_Sided"],
                "McNemar_Statistic": mcn["McNemar_Statistic"],
                "McNemar_P_Value": mcn["McNemar_P_Value"],
                "Both_Correct": mcn["Both_Correct"],
                "Best_Correct_Compared_Wrong": mcn["Best_Correct_Compared_Wrong"],
                "Best_Wrong_Compared_Correct": mcn["Best_Wrong_Compared_Correct"],
                "Both_Wrong": mcn["Both_Wrong"],
                "N_Compared": min_len
            })

pairwise_tests_fast = pd.DataFrame(pairwise_rows)

pairwise_tests_fast.to_csv(
    os.path.join(
        BLOCK15_TEST_DIR,
        "pairwise_tests_against_best_model_fast.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("FAST PAIRWISE TESTS AGAINST BEST MODEL")
print("=" * 70)
display(pairwise_tests_fast.head(40))


# ============================================================
# CI Figures
# ============================================================

def plot_ci_fast(
    ci_df,
    task_type,
    metric,
    title,
    save_path
):

    plot_df = ci_df[
        (ci_df["Task_Type"] == task_type) &
        (ci_df["Metric"] == metric)
    ].copy()

    if len(plot_df) == 0:
        print("No data for:", title)
        return

    plot_df["Label"] = (
        plot_df["Stage"].astype(str)
        + " | "
        + plot_df["Model"].astype(str)
    )

    plot_df = plot_df.sort_values(
        by="Mean",
        ascending=False
    )

    y_pos = np.arange(len(plot_df))

    plt.figure(figsize=(11, 7))

    plt.errorbar(
        plot_df["Mean"],
        y_pos,
        xerr=[
            plot_df["Mean"] - plot_df["CI_Lower"],
            plot_df["CI_Upper"] - plot_df["Mean"]
        ],
        fmt="o",
        capsize=4
    )

    plt.yticks(y_pos, plot_df["Label"])
    plt.xlabel(f"{metric} with 95% Bootstrap CI")
    plt.ylabel("Model")
    plt.title(title)
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


plot_ci_fast(
    ci_df=confidence_interval_df_fast,
    task_type="binary",
    metric="F1",
    title="MDR — F1 Bootstrap Confidence Intervals",
    save_path=os.path.join(
        BLOCK15_FIG_DIR,
        "figure_15a_mdr_f1_bootstrap_ci_fast.png"
    )
)

plot_ci_fast(
    ci_df=confidence_interval_df_fast,
    task_type="multiclass",
    metric="F1",
    title="Severity — Macro F1 Bootstrap Confidence Intervals",
    save_path=os.path.join(
        BLOCK15_FIG_DIR,
        "figure_15b_severity_macro_f1_bootstrap_ci_fast.png"
    )
)


# ============================================================
# Summary
# ============================================================

block15_summary_fast = pd.DataFrame({
    "Output": [
        "MDR performance ranking",
        "Severity performance ranking",
        "Best model summary",
        "Fast bootstrap confidence intervals",
        "Fast pairwise tests",
        "CI figures"
    ],
    "File": [
        "tables/mdr_test_performance_ranking.csv",
        "tables/severity_test_performance_ranking.csv",
        "tables/best_model_summary_from_test_metrics.csv",
        "confidence_intervals/bootstrap_confidence_intervals_fast.csv",
        "statistical_tests/pairwise_tests_against_best_model_fast.csv",
        "figures/figure_15*_fast.png"
    ]
})

block15_summary_fast.to_csv(
    os.path.join(
        BLOCK15_DIR,
        "block15_fast_recovery_output_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("BLOCK 15 FAST RECOVERY GENERATED FILES")
print("=" * 70)

print("- tables/mdr_test_performance_ranking.csv")
print("- tables/severity_test_performance_ranking.csv")
print("- tables/best_model_summary_from_test_metrics.csv")
print("- confidence_intervals/bootstrap_confidence_intervals_fast.csv")
print("- statistical_tests/pairwise_tests_against_best_model_fast.csv")
print("- figures/figure_15a_mdr_f1_bootstrap_ci_fast.png")
print("- figures/figure_15b_severity_macro_f1_bootstrap_ci_fast.png")

print("\n")
print("=" * 70)
print("BLOCK 15 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 15B
# MDR Statistical Analysis Fix
# Fixes NaN bootstrap CI caused by mixed/string MDR labels
# ============================================================

print("=" * 70)
print("BLOCK 15B STARTED — MDR STATISTICAL FIX")
print("=" * 70)

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score
)

from statsmodels.stats.contingency_tables import mcnemar

rng = np.random.default_rng(SEED)
N_BOOTSTRAP_FIX = 500


# ============================================================
# Robust Label Converter
# ============================================================

def normalize_mdr_label_array(values):

    converted = []

    for v in values:

        s = str(v).strip().lower()

        if s in ["1", "1.0", "mdr", "resistant", "positive", "true"]:
            converted.append(1)

        elif s in ["0", "0.0", "non-mdr", "non_mdr", "non mdr", "susceptible", "negative", "false"]:
            converted.append(0)

        else:
            converted.append(np.nan)

    return np.array(converted, dtype=float)


def get_clean_mdr_arrays(group_df):

    if "True_Label_ID" in group_df.columns and "Predicted_Label_ID" in group_df.columns:

        y_true = pd.to_numeric(
            group_df["True_Label_ID"],
            errors="coerce"
        ).values

        y_pred = pd.to_numeric(
            group_df["Predicted_Label_ID"],
            errors="coerce"
        ).values

    else:

        y_true = normalize_mdr_label_array(
            group_df[TRUE_COL].values
        )

        y_pred = normalize_mdr_label_array(
            group_df[PRED_COL].values
        )

    valid_mask = (
        ~np.isnan(y_true)
        &
        ~np.isnan(y_pred)
    )

    y_true = y_true[valid_mask].astype(int)
    y_pred = y_pred[valid_mask].astype(int)

    return y_true, y_pred


# ============================================================
# Metric Functions
# ============================================================

def mdr_metric_score(y_true, y_pred, metric):

    if metric == "Accuracy":
        return accuracy_score(y_true, y_pred)

    if metric == "F1":
        return f1_score(y_true, y_pred, zero_division=0)

    if metric == "Balanced_Accuracy":
        return balanced_accuracy_score(y_true, y_pred)

    if metric == "MCC":
        return matthews_corrcoef(y_true, y_pred)

    if metric == "Cohen_Kappa":
        return cohen_kappa_score(y_true, y_pred)

    return np.nan


def bootstrap_mdr_ci(y_true, y_pred, metric):

    n = len(y_true)
    scores = []

    for _ in range(N_BOOTSTRAP_FIX):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        try:

            score = mdr_metric_score(
                y_true[idx],
                y_pred[idx],
                metric
            )

            if not np.isnan(score):
                scores.append(score)

        except Exception:
            continue

    if len(scores) == 0:

        return {
            "Mean": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "Bootstrap_N": 0
        }

    return {
        "Mean": float(np.mean(scores)),
        "CI_Lower": float(np.percentile(scores, 2.5)),
        "CI_Upper": float(np.percentile(scores, 97.5)),
        "Bootstrap_N": len(scores)
    }


def paired_mdr_diff(y_true, best_pred, compare_pred, metric):

    n = len(y_true)
    diffs = []

    for _ in range(N_BOOTSTRAP_FIX):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        try:

            score_best = mdr_metric_score(
                y_true[idx],
                best_pred[idx],
                metric
            )

            score_compare = mdr_metric_score(
                y_true[idx],
                compare_pred[idx],
                metric
            )

            diffs.append(score_best - score_compare)

        except Exception:
            continue

    if len(diffs) == 0:

        return {
            "Mean_Difference": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "P_Value_Two_Sided": np.nan
        }

    diffs = np.array(diffs)

    p_value = 2 * min(
        np.mean(diffs <= 0),
        np.mean(diffs >= 0)
    )

    return {
        "Mean_Difference": float(np.mean(diffs)),
        "CI_Lower": float(np.percentile(diffs, 2.5)),
        "CI_Upper": float(np.percentile(diffs, 97.5)),
        "P_Value_Two_Sided": min(float(p_value), 1.0)
    }


def mcnemar_mdr(y_true, best_pred, compare_pred):

    best_correct = best_pred == y_true
    compare_correct = compare_pred == y_true

    both_correct = np.sum(best_correct & compare_correct)
    best_correct_compare_wrong = np.sum(best_correct & ~compare_correct)
    best_wrong_compare_correct = np.sum(~best_correct & compare_correct)
    both_wrong = np.sum(~best_correct & ~compare_correct)

    table = [
        [both_correct, best_correct_compare_wrong],
        [best_wrong_compare_correct, both_wrong]
    ]

    try:
        result = mcnemar(table, exact=False, correction=True)
        stat = float(result.statistic)
        pval = float(result.pvalue)

    except Exception:
        stat = np.nan
        pval = np.nan

    return {
        "Both_Correct": int(both_correct),
        "Best_Correct_Compared_Wrong": int(best_correct_compare_wrong),
        "Best_Wrong_Compared_Correct": int(best_wrong_compare_correct),
        "Both_Wrong": int(both_wrong),
        "McNemar_Statistic": stat,
        "McNemar_P_Value": pval
    }


# ============================================================
# Prepare MDR Test Predictions
# ============================================================

mdr_predictions_fix = test_predictions.copy()

mdr_predictions_fix = mdr_predictions_fix[
    mdr_predictions_fix[TASK_COL].astype(str).str.lower().str.contains("mdr")
].copy()

mdr_predictions_fix = mdr_predictions_fix[
    mdr_predictions_fix[SPLIT_COL].astype(str).str.lower() == "test"
].copy()

dedup_cols = [
    "Stage",
    TASK_COL,
    MODEL_COL,
    SPLIT_COL
]

if INDEX_COL is not None:
    dedup_cols.append(INDEX_COL)

mdr_predictions_fix = mdr_predictions_fix.drop_duplicates(
    subset=dedup_cols
).copy()

mdr_predictions_fix["Model_Key"] = (
    mdr_predictions_fix["Stage"].astype(str)
    + " | "
    + mdr_predictions_fix[TASK_COL].astype(str)
    + " | "
    + mdr_predictions_fix[MODEL_COL].astype(str)
)

print("MDR test prediction rows after fix dedup:", len(mdr_predictions_fix))


# ============================================================
# Top MDR Models From Ranking
# ============================================================

top_mdr_keys_fix = (
    mdr_test_ranking
    .head(8)
    .apply(
        lambda r:
        str(r["Stage"]) + " | " + str(r["Task"]) + " | " + str(r["Model"]),
        axis=1
    )
    .tolist()
)

top_mdr_predictions_fix = mdr_predictions_fix[
    mdr_predictions_fix["Model_Key"].isin(top_mdr_keys_fix)
].copy()

print("Top MDR model prediction rows:", len(top_mdr_predictions_fix))


# ============================================================
# MDR Bootstrap CI
# ============================================================

mdr_ci_rows_fix = []

for model_key, group_df in top_mdr_predictions_fix.groupby("Model_Key"):

    stage = group_df["Stage"].iloc[0]
    task = group_df[TASK_COL].iloc[0]
    model = group_df[MODEL_COL].iloc[0]

    y_true, y_pred = get_clean_mdr_arrays(group_df)

    for metric in [
        "Accuracy",
        "F1",
        "Balanced_Accuracy",
        "MCC",
        "Cohen_Kappa"
    ]:

        ci = bootstrap_mdr_ci(
            y_true=y_true,
            y_pred=y_pred,
            metric=metric
        )

        mdr_ci_rows_fix.append({
            "Stage": stage,
            "Task": task,
            "Model": model,
            "Model_Key": model_key,
            "Task_Type": "binary",
            "Metric": metric,
            "Mean": ci["Mean"],
            "CI_Lower": ci["CI_Lower"],
            "CI_Upper": ci["CI_Upper"],
            "Bootstrap_N": ci["Bootstrap_N"],
            "N_Test_Samples": len(y_true)
        })

mdr_bootstrap_ci_fixed = pd.DataFrame(mdr_ci_rows_fix)

mdr_bootstrap_ci_fixed.to_csv(
    os.path.join(
        BLOCK15_CI_DIR,
        "mdr_bootstrap_confidence_intervals_fixed.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("MDR FIXED BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 70)

display(mdr_bootstrap_ci_fixed.head(30))


# ============================================================
# MDR Pairwise Tests Against Voting and ROC-AUC Best
# ============================================================

# For final decision, use Voting as practical best by Accuracy/F1/MCC.
voting_candidates = [
    key for key in top_mdr_keys_fix
    if "Voting_Ensemble" in key and "EqualSoftVoting" in key
]

if len(voting_candidates) > 0:
    mdr_best_key_for_pairwise = voting_candidates[0]
else:
    mdr_best_key_for_pairwise = top_mdr_keys_fix[0]

best_group = top_mdr_predictions_fix[
    top_mdr_predictions_fix["Model_Key"] == mdr_best_key_for_pairwise
].copy()

if INDEX_COL is not None:
    best_group = best_group.sort_values(INDEX_COL).reset_index(drop=True)
else:
    best_group = best_group.reset_index(drop=True)

best_y_true, best_pred = get_clean_mdr_arrays(best_group)

pairwise_rows_fix = []

for compare_key in top_mdr_keys_fix:

    if compare_key == mdr_best_key_for_pairwise:
        continue

    compare_group = top_mdr_predictions_fix[
        top_mdr_predictions_fix["Model_Key"] == compare_key
    ].copy()

    if len(compare_group) == 0:
        continue

    if INDEX_COL is not None:
        compare_group = compare_group.sort_values(INDEX_COL).reset_index(drop=True)
    else:
        compare_group = compare_group.reset_index(drop=True)

    compare_y_true, compare_pred = get_clean_mdr_arrays(compare_group)

    min_len = min(
        len(best_y_true),
        len(compare_y_true),
        len(best_pred),
        len(compare_pred)
    )

    y_true_aligned = best_y_true[:min_len]
    best_pred_aligned = best_pred[:min_len]
    compare_pred_aligned = compare_pred[:min_len]

    for metric in [
        "Accuracy",
        "F1",
        "Balanced_Accuracy",
        "MCC"
    ]:

        diff = paired_mdr_diff(
            y_true=y_true_aligned,
            best_pred=best_pred_aligned,
            compare_pred=compare_pred_aligned,
            metric=metric
        )

        mcn = mcnemar_mdr(
            y_true=y_true_aligned,
            best_pred=best_pred_aligned,
            compare_pred=compare_pred_aligned
        )

        pairwise_rows_fix.append({
            "Task": "MDR",
            "Best_Model_Key": mdr_best_key_for_pairwise,
            "Compared_Model_Key": compare_key,
            "Metric": metric,
            "Mean_Difference_Best_Minus_Compared": diff["Mean_Difference"],
            "Difference_CI_Lower": diff["CI_Lower"],
            "Difference_CI_Upper": diff["CI_Upper"],
            "Bootstrap_P_Value_Two_Sided": diff["P_Value_Two_Sided"],
            "McNemar_Statistic": mcn["McNemar_Statistic"],
            "McNemar_P_Value": mcn["McNemar_P_Value"],
            "Both_Correct": mcn["Both_Correct"],
            "Best_Correct_Compared_Wrong": mcn["Best_Correct_Compared_Wrong"],
            "Best_Wrong_Compared_Correct": mcn["Best_Wrong_Compared_Correct"],
            "Both_Wrong": mcn["Both_Wrong"],
            "N_Compared": min_len
        })

mdr_pairwise_tests_fixed = pd.DataFrame(pairwise_rows_fix)

mdr_pairwise_tests_fixed.to_csv(
    os.path.join(
        BLOCK15_TEST_DIR,
        "mdr_pairwise_tests_fixed.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("MDR FIXED PAIRWISE TESTS")
print("=" * 70)

display(mdr_pairwise_tests_fixed.head(40))


# ============================================================
# Final Fixed Summary
# ============================================================

print("\n")
print("=" * 70)
print("BLOCK 15B GENERATED FILES")
print("=" * 70)

print("- confidence_intervals/mdr_bootstrap_confidence_intervals_fixed.csv")
print("- statistical_tests/mdr_pairwise_tests_fixed.csv")

print("\n")
print("=" * 70)
print("BLOCK 15B COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 15C
# Robust MDR Statistical Fix
# Handles mixed MDR label columns safely
# ============================================================

print("=" * 70)
print("BLOCK 15C STARTED — ROBUST MDR STATISTICAL FIX")
print("=" * 70)

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score
)

from statsmodels.stats.contingency_tables import mcnemar

rng = np.random.default_rng(SEED)
N_BOOTSTRAP_FIX = 500


# ============================================================
# Robust MDR Label Cleaning
# ============================================================

def convert_mdr_labels(series):

    s = series.copy()

    numeric = pd.to_numeric(
        s,
        errors="coerce"
    )

    if numeric.notna().sum() > 0:
        return numeric

    s = (
        s.astype(str)
        .str.strip()
        .str.lower()
        .str.replace("_", "-", regex=False)
    )

    mapping = {
        "1": 1,
        "1.0": 1,
        "mdr": 1,
        "resistant": 1,
        "positive": 1,
        "true": 1,

        "0": 0,
        "0.0": 0,
        "non-mdr": 0,
        "non mdr": 0,
        "nonmdr": 0,
        "susceptible": 0,
        "negative": 0,
        "false": 0
    }

    return s.map(mapping)


def extract_clean_mdr_arrays(group_df):

    true_candidates = [
        "True_Label_ID",
        "True_Label",
        "True_Label_Name"
    ]

    pred_candidates = [
        "Predicted_Label_ID",
        "Predicted_Label",
        "Predicted_Label_Name"
    ]

    best_true = None
    best_pred = None
    best_n = -1

    for true_col in true_candidates:

        if true_col not in group_df.columns:
            continue

        y_true_temp = convert_mdr_labels(
            group_df[true_col]
        )

        for pred_col in pred_candidates:

            if pred_col not in group_df.columns:
                continue

            y_pred_temp = convert_mdr_labels(
                group_df[pred_col]
            )

            valid_mask = (
                y_true_temp.notna()
                &
                y_pred_temp.notna()
            )

            valid_n = int(valid_mask.sum())

            if valid_n > best_n:

                best_n = valid_n
                best_true = y_true_temp[valid_mask].astype(int).values
                best_pred = y_pred_temp[valid_mask].astype(int).values
                best_true_col = true_col
                best_pred_col = pred_col

    print(
        f"Selected columns: true={best_true_col}, pred={best_pred_col}, valid_n={best_n}"
    )

    return best_true, best_pred, best_true_col, best_pred_col


# ============================================================
# Metric Helpers
# ============================================================

def metric_score(y_true, y_pred, metric):

    if metric == "Accuracy":
        return accuracy_score(y_true, y_pred)

    if metric == "F1":
        return f1_score(y_true, y_pred, zero_division=0)

    if metric == "Balanced_Accuracy":
        return balanced_accuracy_score(y_true, y_pred)

    if metric == "MCC":
        return matthews_corrcoef(y_true, y_pred)

    if metric == "Cohen_Kappa":
        return cohen_kappa_score(y_true, y_pred)

    return np.nan


def bootstrap_ci(y_true, y_pred, metric):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    n = len(y_true)

    if n == 0:

        return {
            "Mean": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "Bootstrap_N": 0
        }

    scores = []

    for _ in range(N_BOOTSTRAP_FIX):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        try:
            score = metric_score(
                y_true[idx],
                y_pred[idx],
                metric
            )

            if not np.isnan(score):
                scores.append(score)

        except Exception:
            continue

    if len(scores) == 0:

        return {
            "Mean": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "Bootstrap_N": 0
        }

    return {
        "Mean": float(np.mean(scores)),
        "CI_Lower": float(np.percentile(scores, 2.5)),
        "CI_Upper": float(np.percentile(scores, 97.5)),
        "Bootstrap_N": len(scores)
    }


def paired_bootstrap_diff(y_true, best_pred, compare_pred, metric):

    y_true = np.array(y_true)
    best_pred = np.array(best_pred)
    compare_pred = np.array(compare_pred)

    n = len(y_true)

    if n == 0:

        return {
            "Mean_Difference": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "P_Value_Two_Sided": np.nan
        }

    diffs = []

    for _ in range(N_BOOTSTRAP_FIX):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        try:

            best_score = metric_score(
                y_true[idx],
                best_pred[idx],
                metric
            )

            compare_score = metric_score(
                y_true[idx],
                compare_pred[idx],
                metric
            )

            diffs.append(best_score - compare_score)

        except Exception:
            continue

    if len(diffs) == 0:

        return {
            "Mean_Difference": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "P_Value_Two_Sided": np.nan
        }

    diffs = np.array(diffs)

    p_value = 2 * min(
        np.mean(diffs <= 0),
        np.mean(diffs >= 0)
    )

    return {
        "Mean_Difference": float(np.mean(diffs)),
        "CI_Lower": float(np.percentile(diffs, 2.5)),
        "CI_Upper": float(np.percentile(diffs, 97.5)),
        "P_Value_Two_Sided": min(float(p_value), 1.0)
    }


def mcnemar_test_fixed(y_true, best_pred, compare_pred):

    best_correct = best_pred == y_true
    compare_correct = compare_pred == y_true

    both_correct = np.sum(best_correct & compare_correct)
    best_correct_compare_wrong = np.sum(best_correct & ~compare_correct)
    best_wrong_compare_correct = np.sum(~best_correct & compare_correct)
    both_wrong = np.sum(~best_correct & ~compare_correct)

    table = [
        [both_correct, best_correct_compare_wrong],
        [best_wrong_compare_correct, both_wrong]
    ]

    try:
        result = mcnemar(
            table,
            exact=False,
            correction=True
        )

        statistic = float(result.statistic)
        p_value = float(result.pvalue)

    except Exception:
        statistic = np.nan
        p_value = np.nan

    return {
        "Both_Correct": int(both_correct),
        "Best_Correct_Compared_Wrong": int(best_correct_compare_wrong),
        "Best_Wrong_Compared_Correct": int(best_wrong_compare_correct),
        "Both_Wrong": int(both_wrong),
        "McNemar_Statistic": statistic,
        "McNemar_P_Value": p_value
    }


# ============================================================
# Prepare MDR Test Predictions
# ============================================================

mdr_pred = test_predictions.copy()

mdr_pred = mdr_pred[
    mdr_pred[TASK_COL].astype(str).str.lower().str.contains("mdr")
].copy()

mdr_pred = mdr_pred[
    mdr_pred[SPLIT_COL].astype(str).str.lower() == "test"
].copy()

dedup_cols = [
    "Stage",
    TASK_COL,
    MODEL_COL,
    SPLIT_COL
]

if INDEX_COL is not None:
    dedup_cols.append(INDEX_COL)

mdr_pred = mdr_pred.drop_duplicates(
    subset=dedup_cols
).copy()

mdr_pred["Model_Key"] = (
    mdr_pred["Stage"].astype(str)
    + " | "
    + mdr_pred[TASK_COL].astype(str)
    + " | "
    + mdr_pred[MODEL_COL].astype(str)
)

print("MDR test rows after dedup:", len(mdr_pred))

print("\nMDR available Model Keys:")
print(mdr_pred["Model_Key"].value_counts().head(20))


# ============================================================
# Top MDR Keys
# ============================================================

top_mdr_keys = (
    mdr_test_ranking
    .head(8)
    .apply(
        lambda r:
        str(r["Stage"]) + " | " + str(r["Task"]) + " | " + str(r["Model"]),
        axis=1
    )
    .tolist()
)

top_mdr_pred = mdr_pred[
    mdr_pred["Model_Key"].isin(top_mdr_keys)
].copy()

print("\nTop MDR prediction rows:", len(top_mdr_pred))


# ============================================================
# Bootstrap CI
# ============================================================

ci_rows = []

for model_key, group_df in top_mdr_pred.groupby("Model_Key"):

    stage = group_df["Stage"].iloc[0]
    task = group_df[TASK_COL].iloc[0]
    model = group_df[MODEL_COL].iloc[0]

    y_true, y_pred, true_col, pred_col = extract_clean_mdr_arrays(
        group_df
    )

    print(
        model_key,
        "N =",
        len(y_true),
        "True counts:",
        pd.Series(y_true).value_counts().to_dict(),
        "Pred counts:",
        pd.Series(y_pred).value_counts().to_dict()
    )

    for metric in [
        "Accuracy",
        "F1",
        "Balanced_Accuracy",
        "MCC",
        "Cohen_Kappa"
    ]:

        ci = bootstrap_ci(
            y_true=y_true,
            y_pred=y_pred,
            metric=metric
        )

        ci_rows.append({
            "Stage": stage,
            "Task": task,
            "Model": model,
            "Model_Key": model_key,
            "Metric": metric,
            "Mean": ci["Mean"],
            "CI_Lower": ci["CI_Lower"],
            "CI_Upper": ci["CI_Upper"],
            "Bootstrap_N": ci["Bootstrap_N"],
            "N_Test_Samples": len(y_true),
            "True_Column_Used": true_col,
            "Pred_Column_Used": pred_col
        })

mdr_bootstrap_ci_robust = pd.DataFrame(ci_rows)

mdr_bootstrap_ci_robust.to_csv(
    os.path.join(
        BLOCK15_CI_DIR,
        "mdr_bootstrap_confidence_intervals_robust.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("MDR ROBUST BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 70)

display(mdr_bootstrap_ci_robust.head(40))


# ============================================================
# Pairwise Tests Using Voting as Practical Final MDR Model
# ============================================================

voting_keys = [
    key for key in top_mdr_keys
    if "Voting_Ensemble" in key and "EqualSoftVoting" in key
]

if len(voting_keys) > 0:
    best_key = voting_keys[0]
else:
    best_key = top_mdr_keys[0]

print("\nMDR pairwise reference model:")
print(best_key)


def get_model_group(df, model_key):

    g = df[
        df["Model_Key"] == model_key
    ].copy()

    if INDEX_COL is not None:
        g = g.sort_values(INDEX_COL).reset_index(drop=True)
    else:
        g = g.reset_index(drop=True)

    y_true, y_pred, _, _ = extract_clean_mdr_arrays(g)

    return g, y_true, y_pred


best_group, best_true, best_pred = get_model_group(
    top_mdr_pred,
    best_key
)

pairwise_rows = []

for compare_key in top_mdr_keys:

    if compare_key == best_key:
        continue

    compare_group, compare_true, compare_pred = get_model_group(
        top_mdr_pred,
        compare_key
    )

    min_len = min(
        len(best_true),
        len(compare_true),
        len(best_pred),
        len(compare_pred)
    )

    if min_len == 0:
        continue

    y_true = best_true[:min_len]
    pred_best = best_pred[:min_len]
    pred_compare = compare_pred[:min_len]

    for metric in [
        "Accuracy",
        "F1",
        "Balanced_Accuracy",
        "MCC"
    ]:

        diff = paired_bootstrap_diff(
            y_true=y_true,
            best_pred=pred_best,
            compare_pred=pred_compare,
            metric=metric
        )

        mcn = mcnemar_test_fixed(
            y_true=y_true,
            best_pred=pred_best,
            compare_pred=pred_compare
        )

        pairwise_rows.append({
            "Task": "MDR",
            "Best_Model_Key": best_key,
            "Compared_Model_Key": compare_key,
            "Metric": metric,
            "Mean_Difference_Best_Minus_Compared": diff["Mean_Difference"],
            "Difference_CI_Lower": diff["CI_Lower"],
            "Difference_CI_Upper": diff["CI_Upper"],
            "Bootstrap_P_Value_Two_Sided": diff["P_Value_Two_Sided"],
            "McNemar_Statistic": mcn["McNemar_Statistic"],
            "McNemar_P_Value": mcn["McNemar_P_Value"],
            "Both_Correct": mcn["Both_Correct"],
            "Best_Correct_Compared_Wrong": mcn["Best_Correct_Compared_Wrong"],
            "Best_Wrong_Compared_Correct": mcn["Best_Wrong_Compared_Correct"],
            "Both_Wrong": mcn["Both_Wrong"],
            "N_Compared": min_len
        })

mdr_pairwise_tests_robust = pd.DataFrame(pairwise_rows)

mdr_pairwise_tests_robust.to_csv(
    os.path.join(
        BLOCK15_TEST_DIR,
        "mdr_pairwise_tests_robust.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("MDR ROBUST PAIRWISE TESTS")
print("=" * 70)

display(mdr_pairwise_tests_robust.head(40))


print("\n")
print("=" * 70)
print("BLOCK 15C GENERATED FILES")
print("=" * 70)

print("- confidence_intervals/mdr_bootstrap_confidence_intervals_robust.csv")
print("- statistical_tests/mdr_pairwise_tests_robust.csv")

print("\n")
print("=" * 70)
print("BLOCK 15C COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 16
# Final Model Comparison, Proposed Model Selection,
# and Publication-Ready Tables
# FINAL JOURNAL VERSION
# ============================================================

print("=" * 70)
print("BLOCK 16 STARTED — FINAL MODEL SELECTION")
print("=" * 70)

import os
import ast
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


# ============================================================
# Dependency Check
# ============================================================

required_objects = [
    "OUTPUT_DIR",
    "SEED",
    "safe_name"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if len(missing_objects) > 0:
    raise RuntimeError(
        "Please run previous blocks before BLOCK 16. Missing: "
        + ", ".join(missing_objects)
    )


# ============================================================
# Output Directories
# ============================================================

BLOCK16_DIR = os.path.join(
    OUTPUT_DIR,
    "block16_final_model_selection"
)

BLOCK16_TABLE_DIR = os.path.join(
    BLOCK16_DIR,
    "publication_tables"
)

BLOCK16_FIG_DIR = os.path.join(
    BLOCK16_DIR,
    "publication_figures"
)

BLOCK16_TEXT_DIR = os.path.join(
    BLOCK16_DIR,
    "paper_ready_text"
)

for d in [
    BLOCK16_DIR,
    BLOCK16_TABLE_DIR,
    BLOCK16_FIG_DIR,
    BLOCK16_TEXT_DIR
]:
    os.makedirs(d, exist_ok=True)

FIG_DPI = 300


# ============================================================
# File Paths
# ============================================================

block15_table_dir = os.path.join(
    OUTPUT_DIR,
    "block15_statistical_analysis",
    "tables"
)

block15_ci_dir = os.path.join(
    OUTPUT_DIR,
    "block15_statistical_analysis",
    "confidence_intervals"
)

block15_test_dir = os.path.join(
    OUTPUT_DIR,
    "block15_statistical_analysis",
    "statistical_tests"
)

block12_table_dir = os.path.join(
    OUTPUT_DIR,
    "block12_shap_xai",
    "tables"
)

mdr_ranking_path = os.path.join(
    block15_table_dir,
    "mdr_test_performance_ranking.csv"
)

severity_ranking_path = os.path.join(
    block15_table_dir,
    "severity_test_performance_ranking.csv"
)

best_summary_path = os.path.join(
    block15_table_dir,
    "best_model_summary_from_test_metrics.csv"
)

mdr_ci_path = os.path.join(
    block15_ci_dir,
    "mdr_bootstrap_confidence_intervals_robust.csv"
)

severity_ci_path = os.path.join(
    block15_ci_dir,
    "bootstrap_confidence_intervals_fast.csv"
)

mdr_pairwise_path = os.path.join(
    block15_test_dir,
    "mdr_pairwise_tests_robust.csv"
)

severity_pairwise_path = os.path.join(
    block15_test_dir,
    "pairwise_tests_against_best_model_fast.csv"
)

mdr_shap_path = os.path.join(
    block12_table_dir,
    "mdr_global_shap_importance.csv"
)

severity_shap_path = os.path.join(
    block12_table_dir,
    "severity_overall_global_shap_importance.csv"
)

xai_key_findings_path = os.path.join(
    block12_table_dir,
    "xai_key_findings_summary.csv"
)


# ============================================================
# Load Ranking Tables
# ============================================================

if not os.path.exists(mdr_ranking_path):
    raise FileNotFoundError(
        f"Missing MDR ranking file: {mdr_ranking_path}"
    )

if not os.path.exists(severity_ranking_path):
    raise FileNotFoundError(
        f"Missing Severity ranking file: {severity_ranking_path}"
    )

mdr_ranking = pd.read_csv(mdr_ranking_path)
severity_ranking = pd.read_csv(severity_ranking_path)

print("\nLoaded MDR ranking:", mdr_ranking.shape)
print("Loaded Severity ranking:", severity_ranking.shape)


# ============================================================
# Helper Functions
# ============================================================

def select_first_matching_model(df, stage_keyword, task_keyword, model_keyword):

    mask = (
        df["Stage"].astype(str).str.contains(
            stage_keyword,
            case=False,
            regex=False
        )
        &
        df["Task"].astype(str).str.contains(
            task_keyword,
            case=False,
            regex=False
        )
        &
        df["Model"].astype(str).str.contains(
            model_keyword,
            case=False,
            regex=False
        )
    )

    matched = df[mask].copy()

    if len(matched) == 0:
        return None

    return matched.iloc[0].copy()


def get_numeric(row, col):

    if col in row.index:

        try:
            return float(row[col])
        except Exception:
            return np.nan

    return np.nan


def format_ci(mean, lower, upper, decimals=3):

    if pd.isna(mean) or pd.isna(lower) or pd.isna(upper):
        return "NA"

    return (
        f"{mean:.{decimals}f} "
        f"({lower:.{decimals}f}–{upper:.{decimals}f})"
    )


def safe_float(value):

    try:
        return float(value)
    except Exception:
        return np.nan


def get_ci_for_model(ci_df, model_key, metric):

    if ci_df is None or len(ci_df) == 0:
        return np.nan, np.nan, np.nan

    temp = ci_df[
        (ci_df["Model_Key"].astype(str) == str(model_key))
        &
        (ci_df["Metric"].astype(str) == str(metric))
    ].copy()

    if len(temp) == 0:
        return np.nan, np.nan, np.nan

    row = temp.iloc[0]

    return (
        safe_float(row.get("Mean", np.nan)),
        safe_float(row.get("CI_Lower", np.nan)),
        safe_float(row.get("CI_Upper", np.nan))
    )


# ============================================================
# Select Final Proposed Models
# ============================================================

# MDR:
# CatBoost has highest ROC-AUC in Block 15 ranking,
# but Voting ensemble has best classification performance
# by Accuracy/F1/MCC. For final deployable proposed framework,
# choose Voting EqualSoftVoting.

mdr_final_row = select_first_matching_model(
    df=mdr_ranking,
    stage_keyword="Voting_Ensemble",
    task_keyword="MDR",
    model_keyword="EqualSoftVoting"
)

if mdr_final_row is None:

    print("MDR Voting EqualSoftVoting not found. Falling back to top ranked MDR row.")
    mdr_final_row = mdr_ranking.iloc[0].copy()

# Severity:
# Voting EqualSoftVoting is top in Block 15 ranking.
severity_final_row = select_first_matching_model(
    df=severity_ranking,
    stage_keyword="Voting_Ensemble",
    task_keyword="Severity",
    model_keyword="EqualSoftVoting"
)

if severity_final_row is None:

    print("Severity Voting EqualSoftVoting not found. Falling back to top ranked Severity row.")
    severity_final_row = severity_ranking.iloc[0].copy()


# ============================================================
# Final Proposed Model Summary
# ============================================================

final_model_selection_summary = pd.DataFrame([
    {
        "Task": "MDR Prediction",
        "Final_Proposed_Stage": mdr_final_row["Stage"],
        "Final_Proposed_Model": mdr_final_row["Model"],
        "Final_Task_Name": mdr_final_row["Task"],
        "Primary_Selection_Criterion": "Best practical classification performance by Accuracy, F1, Balanced Accuracy, and MCC",
        "Accuracy": get_numeric(mdr_final_row, "Accuracy"),
        "F1_or_Macro_F1": get_numeric(mdr_final_row, "F1"),
        "Balanced_Accuracy": get_numeric(mdr_final_row, "Balanced_Accuracy"),
        "MCC": get_numeric(mdr_final_row, "MCC"),
        "ROC_AUC": get_numeric(mdr_final_row, "ROC_AUC"),
        "PR_AUC": get_numeric(mdr_final_row, "PR_AUC"),
        "Interpretation": "Selected as final MDR model because it achieved the strongest classification-oriented test performance, although CatBoost showed marginally higher ROC-AUC."
    },
    {
        "Task": "Resistance Severity Assessment",
        "Final_Proposed_Stage": severity_final_row["Stage"],
        "Final_Proposed_Model": severity_final_row["Model"],
        "Final_Task_Name": severity_final_row["Task"],
        "Primary_Selection_Criterion": "Best test Macro F1, Accuracy, Balanced Accuracy, and MCC among severity models",
        "Accuracy": get_numeric(severity_final_row, "Accuracy"),
        "F1_or_Macro_F1": get_numeric(severity_final_row, "Macro_F1"),
        "Balanced_Accuracy": get_numeric(severity_final_row, "Balanced_Accuracy"),
        "MCC": get_numeric(severity_final_row, "MCC"),
        "ROC_AUC": get_numeric(severity_final_row, "ROC_AUC_OVR_Macro"),
        "PR_AUC": get_numeric(severity_final_row, "PR_AUC_Macro"),
        "Interpretation": "Selected as final severity model because it achieved the strongest overall multiclass performance."
    }
])

final_model_selection_summary.to_csv(
    os.path.join(
        BLOCK16_TABLE_DIR,
        "table_16_final_proposed_model_selection.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("FINAL PROPOSED MODEL SELECTION")
print("=" * 70)

display(final_model_selection_summary)


# ============================================================
# Load Confidence Intervals
# ============================================================

mdr_ci_df = pd.read_csv(mdr_ci_path) if os.path.exists(mdr_ci_path) else pd.DataFrame()
severity_ci_df = pd.read_csv(severity_ci_path) if os.path.exists(severity_ci_path) else pd.DataFrame()

print("\nMDR CI rows:", len(mdr_ci_df))
print("Severity CI rows:", len(severity_ci_df))


mdr_model_key = (
    str(mdr_final_row["Stage"])
    + " | "
    + str(mdr_final_row["Task"])
    + " | "
    + str(mdr_final_row["Model"])
)

severity_model_key = (
    str(severity_final_row["Stage"])
    + " | "
    + str(severity_final_row["Task"])
    + " | "
    + str(severity_final_row["Model"])
)


# ============================================================
# Final Metrics With CI Table
# ============================================================

metric_ci_rows = []

for task_label, model_key, ci_df, task_type in [
    ("MDR Prediction", mdr_model_key, mdr_ci_df, "binary"),
    ("Resistance Severity Assessment", severity_model_key, severity_ci_df, "multiclass")
]:

    if task_type == "binary":

        metrics = [
            "Accuracy",
            "F1",
            "Balanced_Accuracy",
            "MCC",
            "Cohen_Kappa"
        ]

    else:

        metrics = [
            "Accuracy",
            "F1",
            "Balanced_Accuracy",
            "MCC",
            "Cohen_Kappa"
        ]

    for metric in metrics:

        mean, lower, upper = get_ci_for_model(
            ci_df=ci_df,
            model_key=model_key,
            metric=metric
        )

        metric_ci_rows.append({
            "Task": task_label,
            "Model_Key": model_key,
            "Metric": metric if not (task_type == "multiclass" and metric == "F1") else "Macro_F1",
            "Mean": mean,
            "CI_Lower": lower,
            "CI_Upper": upper,
            "Formatted_95CI": format_ci(mean, lower, upper)
        })

final_metrics_with_ci = pd.DataFrame(metric_ci_rows)

final_metrics_with_ci.to_csv(
    os.path.join(
        BLOCK16_TABLE_DIR,
        "table_16_final_metrics_with_bootstrap_ci.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("FINAL METRICS WITH BOOTSTRAP 95% CI")
print("=" * 70)

display(final_metrics_with_ci)


# ============================================================
# Publication Performance Tables
# ============================================================

mdr_publication_cols = [
    "Stage",
    "Task",
    "Model",
    "Accuracy",
    "Precision",
    "Recall_Sensitivity",
    "Specificity",
    "F1",
    "Balanced_Accuracy",
    "MCC",
    "Cohen_Kappa",
    "ROC_AUC",
    "PR_AUC",
    "Log_Loss",
    "Brier_Score"
]

severity_publication_cols = [
    "Stage",
    "Task",
    "Model",
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall_Sensitivity",
    "Macro_F1",
    "Weighted_F1",
    "Balanced_Accuracy",
    "Macro_Specificity",
    "MCC",
    "Cohen_Kappa",
    "ROC_AUC_OVR_Macro",
    "PR_AUC_Macro",
    "Log_Loss",
    "Multiclass_Brier_Score"
]

mdr_available_cols = [
    col for col in mdr_publication_cols
    if col in mdr_ranking.columns
]

severity_available_cols = [
    col for col in severity_publication_cols
    if col in severity_ranking.columns
]

mdr_publication_table = mdr_ranking[
    mdr_available_cols
].copy()

severity_publication_table = severity_ranking[
    severity_available_cols
].copy()

# Keep top 12 for paper table
mdr_publication_table_top = mdr_publication_table.head(12).copy()
severity_publication_table_top = severity_publication_table.head(12).copy()

mdr_publication_table_top.to_csv(
    os.path.join(
        BLOCK16_TABLE_DIR,
        "table_16_mdr_test_performance_publication.csv"
    ),
    index=False
)

severity_publication_table_top.to_csv(
    os.path.join(
        BLOCK16_TABLE_DIR,
        "table_16_severity_test_performance_publication.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("MDR PUBLICATION PERFORMANCE TABLE")
print("=" * 70)

display(mdr_publication_table_top)

print("\n")
print("=" * 70)
print("SEVERITY PUBLICATION PERFORMANCE TABLE")
print("=" * 70)

display(severity_publication_table_top)


# ============================================================
# XAI Summary for Final Paper
# ============================================================

xai_rows = []

if os.path.exists(mdr_shap_path):

    mdr_shap = pd.read_csv(mdr_shap_path)

    for _, row in mdr_shap.head(10).iterrows():

        xai_rows.append({
            "Task": "MDR Prediction",
            "Feature": row["Feature"],
            "SHAP_Rank": row.get("Rank", np.nan),
            "Mean_Absolute_SHAP": row.get("Mean_Absolute_SHAP", np.nan)
        })

if os.path.exists(severity_shap_path):

    severity_shap = pd.read_csv(severity_shap_path)

    for _, row in severity_shap.head(10).iterrows():

        xai_rows.append({
            "Task": "Resistance Severity Assessment",
            "Feature": row["Feature"],
            "SHAP_Rank": row.get("Rank", np.nan),
            "Mean_Absolute_SHAP": row.get("Mean_Absolute_SHAP", np.nan)
        })

xai_publication_summary = pd.DataFrame(xai_rows)

xai_publication_summary.to_csv(
    os.path.join(
        BLOCK16_TABLE_DIR,
        "table_16_xai_feature_importance_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("XAI FEATURE IMPORTANCE SUMMARY")
print("=" * 70)

display(xai_publication_summary)


# ============================================================
# Framework Component Table
# ============================================================

framework_components = pd.DataFrame({
    "Framework_Component": [
        "Data audit and cleaning",
        "Qualified genome selection",
        "MDR label construction",
        "Resistance severity assessment",
        "Feature engineering",
        "Baseline model benchmarking",
        "Hyperparameter optimization",
        "Feature selection",
        "SHAP explainability",
        "Stacking ensemble",
        "Voting ensemble",
        "Statistical validation",
        "Final proposed model selection"
    ],
    "Purpose": [
        "Remove duplicates, standardize phenotype/MIC/lab fields, and assess missingness",
        "Retain genomes with observed phenotype evidence",
        "Create binary MDR label using resistance across antibiotic classes",
        "Create multiclass severity labels using resistant antibiotic and class burden",
        "Generate genome-level MIC, testing, coverage, and missingness features",
        "Establish reference performance across standard ML algorithms",
        "Tune tree-based learners for stronger candidate models",
        "Assess compact feature subsets and model stability",
        "Explain global and local model decisions",
        "Combine base learners using meta-learning",
        "Combine probabilistic outputs using soft voting",
        "Estimate uncertainty and compare paired model performance",
        "Select final MDR and severity models for publication"
    ],
    "Block": [
        "00–03",
        "04",
        "05",
        "06",
        "07",
        "09",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15/15C",
        "16"
    ]
})

framework_components.to_csv(
    os.path.join(
        BLOCK16_TABLE_DIR,
        "table_16_framework_components.csv"
    ),
    index=False
)


# ============================================================
# Publication Figures
# ============================================================

def plot_top_model_comparison(
    df,
    metric,
    title,
    save_path,
    top_n=10
):

    if metric not in df.columns:
        print(f"Metric not found: {metric}")
        return

    plot_df = df.copy()

    plot_df["Model_Label"] = (
        plot_df["Stage"].astype(str)
        + " | "
        + plot_df["Model"].astype(str)
    )

    plot_df = plot_df.sort_values(
        by=metric,
        ascending=False
    ).head(top_n)

    plt.figure(figsize=(11, 7))

    sns.barplot(
        data=plot_df,
        x=metric,
        y="Model_Label",
        orient="h"
    )

    plt.title(title)
    plt.xlabel(metric)
    plt.ylabel("Model")
    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=FIG_DPI,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


plot_top_model_comparison(
    df=mdr_ranking,
    metric="F1",
    title="MDR Prediction — Test F1 Comparison",
    save_path=os.path.join(
        BLOCK16_FIG_DIR,
        "figure_16a_mdr_test_f1_comparison.png"
    )
)

plot_top_model_comparison(
    df=mdr_ranking,
    metric="MCC",
    title="MDR Prediction — Test MCC Comparison",
    save_path=os.path.join(
        BLOCK16_FIG_DIR,
        "figure_16b_mdr_test_mcc_comparison.png"
    )
)

plot_top_model_comparison(
    df=severity_ranking,
    metric="Macro_F1",
    title="Severity Assessment — Test Macro F1 Comparison",
    save_path=os.path.join(
        BLOCK16_FIG_DIR,
        "figure_16c_severity_test_macro_f1_comparison.png"
    )
)

plot_top_model_comparison(
    df=severity_ranking,
    metric="MCC",
    title="Severity Assessment — Test MCC Comparison",
    save_path=os.path.join(
        BLOCK16_FIG_DIR,
        "figure_16d_severity_test_mcc_comparison.png"
    )
)


# ============================================================
# Final Paper-Ready Text
# ============================================================

mdr_acc = get_numeric(mdr_final_row, "Accuracy")
mdr_f1 = get_numeric(mdr_final_row, "F1")
mdr_bal = get_numeric(mdr_final_row, "Balanced_Accuracy")
mdr_mcc = get_numeric(mdr_final_row, "MCC")
mdr_auc = get_numeric(mdr_final_row, "ROC_AUC")

sev_acc = get_numeric(severity_final_row, "Accuracy")
sev_macro_f1 = get_numeric(severity_final_row, "Macro_F1")
sev_bal = get_numeric(severity_final_row, "Balanced_Accuracy")
sev_mcc = get_numeric(severity_final_row, "MCC")
sev_auc = get_numeric(severity_final_row, "ROC_AUC_OVR_Macro")

paper_results_text = f"""
Final model selection identified soft voting ensembles as the most suitable practical models for the proposed framework. 
For MDR prediction, the selected voting ensemble achieved an accuracy of {mdr_acc:.4f}, F1-score of {mdr_f1:.4f}, balanced accuracy of {mdr_bal:.4f}, and MCC of {mdr_mcc:.4f} on the independent test set. 
Although the baseline CatBoost model achieved a marginally higher ROC-AUC, the voting ensemble showed stronger classification-oriented performance, including accuracy, F1-score, balanced accuracy, and MCC.

For resistance severity assessment, the selected soft voting ensemble achieved an accuracy of {sev_acc:.4f}, macro F1-score of {sev_macro_f1:.4f}, balanced accuracy of {sev_bal:.4f}, MCC of {sev_mcc:.4f}, and macro one-vs-rest ROC-AUC of {sev_auc:.4f}. 
This model was therefore selected as the final severity classifier.

SHAP-based explainability showed that MDR prediction was mainly influenced by antibiotic testing coverage, broth dilution evidence, MIC-data completeness, and testing-standard completeness. 
In contrast, severity assessment was primarily driven by MIC magnitude and MIC distribution features, particularly mean MIC, median MIC, antibiotic testing coverage, and MIC variability.
"""

with open(
    os.path.join(
        BLOCK16_TEXT_DIR,
        "paper_ready_results_summary.txt"
    ),
    "w"
) as f:
    f.write(paper_results_text.strip())


paper_limitations_text = """
The proposed framework is a phenotype-informed genome-level machine learning framework rather than a raw whole-genome-sequence model. 
The input features are derived from genome-level antibiotic testing, MIC summaries, laboratory methods, testing standards, coverage, and missingness patterns. 
Therefore, claims should be framed as genome-level MDR and severity prediction using curated resistance phenotype evidence, not as direct sequence-based AMR prediction.

The current implementation treats MDR prediction and severity assessment as related predictive tasks within a unified framework, but the models are trained separately. 
If the study is framed as true multi-task learning, an additional shared-representation model with joint loss should be included. 
Otherwise, the title should use multi-objective or unified explainable machine learning framework terminology.
"""

with open(
    os.path.join(
        BLOCK16_TEXT_DIR,
        "paper_ready_limitations_and_positioning.txt"
    ),
    "w"
) as f:
    f.write(paper_limitations_text.strip())


# ============================================================
# Final Output Summary
# ============================================================

block16_output_summary = pd.DataFrame({
    "Output_Category": [
        "Final proposed model selection",
        "Final metrics with bootstrap CI",
        "MDR publication performance table",
        "Severity publication performance table",
        "XAI feature importance summary",
        "Framework component table",
        "MDR F1 comparison figure",
        "MDR MCC comparison figure",
        "Severity Macro F1 comparison figure",
        "Severity MCC comparison figure",
        "Paper-ready results summary",
        "Paper-ready limitations and positioning"
    ],
    "Saved_File": [
        "publication_tables/table_16_final_proposed_model_selection.csv",
        "publication_tables/table_16_final_metrics_with_bootstrap_ci.csv",
        "publication_tables/table_16_mdr_test_performance_publication.csv",
        "publication_tables/table_16_severity_test_performance_publication.csv",
        "publication_tables/table_16_xai_feature_importance_summary.csv",
        "publication_tables/table_16_framework_components.csv",
        "publication_figures/figure_16a_mdr_test_f1_comparison.png",
        "publication_figures/figure_16b_mdr_test_mcc_comparison.png",
        "publication_figures/figure_16c_severity_test_macro_f1_comparison.png",
        "publication_figures/figure_16d_severity_test_mcc_comparison.png",
        "paper_ready_text/paper_ready_results_summary.txt",
        "paper_ready_text/paper_ready_limitations_and_positioning.txt"
    ]
})

block16_output_summary.to_csv(
    os.path.join(
        BLOCK16_DIR,
        "block16_output_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("BLOCK 16 GENERATED FILES")
print("=" * 70)

print("- publication_tables/table_16_final_proposed_model_selection.csv")
print("- publication_tables/table_16_final_metrics_with_bootstrap_ci.csv")
print("- publication_tables/table_16_mdr_test_performance_publication.csv")
print("- publication_tables/table_16_severity_test_performance_publication.csv")
print("- publication_tables/table_16_xai_feature_importance_summary.csv")
print("- publication_tables/table_16_framework_components.csv")
print("- publication_figures/figure_16a_mdr_test_f1_comparison.png")
print("- publication_figures/figure_16b_mdr_test_mcc_comparison.png")
print("- publication_figures/figure_16c_severity_test_macro_f1_comparison.png")
print("- publication_figures/figure_16d_severity_test_mcc_comparison.png")
print("- paper_ready_text/paper_ready_results_summary.txt")
print("- paper_ready_text/paper_ready_limitations_and_positioning.txt")

print("\n")
print("=" * 70)
print("BLOCK 16 COMPLETED SUCCESSFULLY")
print("=" * 70)

In [ ]:
# ============================================================
# BLOCK 17 — CLEAN ABLATION STUDY
# Feature-Family Ablation + Ensemble Component Ablation
# Paste this whole file into ONE Kaggle Code cell and run.
# ============================================================

print("=" * 70)
print("BLOCK 17 STARTED — CLEAN ABLATION STUDY")
print("=" * 70)

import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score,
    confusion_matrix, roc_auc_score, average_precision_score,
    log_loss, brier_score_loss
)
from sklearn.utils.class_weight import compute_class_weight

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None

# ------------------------------------------------------------
# Check required previous-block variables
# ------------------------------------------------------------

required_vars = [
    "OUTPUT_DIR", "SEED", "feature_columns",
    "X_train", "X_test", "y_train", "y_test"
]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError("Run previous blocks first. Missing: " + ", ".join(missing_vars))

# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

BLOCK17_DIR = os.path.join(OUTPUT_DIR, "block17_ablation_study")
BLOCK17_TABLE_DIR = os.path.join(BLOCK17_DIR, "tables")
BLOCK17_FIG_DIR = os.path.join(BLOCK17_DIR, "figures")
BLOCK17_TEXT_DIR = os.path.join(BLOCK17_DIR, "paper_ready_text")

for d in [BLOCK17_DIR, BLOCK17_TABLE_DIR, BLOCK17_FIG_DIR, BLOCK17_TEXT_DIR]:
    os.makedirs(d, exist_ok=True)

# ------------------------------------------------------------
# Targets
# ------------------------------------------------------------

y_train_mdr = y_train["MDR_Label"].astype(int).values
y_test_mdr = y_test["MDR_Label"].astype(int).values

severity_order = ["Low", "Moderate", "High", "Critical"]
severity_to_id = {label: i for i, label in enumerate(severity_order)}

y_train_sev = y_train["Severity_Label"].astype(str).values
y_test_sev = y_test["Severity_Label"].astype(str).values
y_train_sev_enc = pd.Series(y_train_sev).map(severity_to_id).astype(int).values
y_test_sev_enc = pd.Series(y_test_sev).map(severity_to_id).astype(int).values
n_severity_classes = len(severity_order)

print("MDR train distribution:", pd.Series(y_train_mdr).value_counts().to_dict())
print("MDR test distribution :", pd.Series(y_test_mdr).value_counts().to_dict())
print("Severity train distribution:", pd.Series(y_train_sev).value_counts().to_dict())
print("Severity test distribution :", pd.Series(y_test_sev).value_counts().to_dict())

# ------------------------------------------------------------
# Feature groups
# ------------------------------------------------------------

all_features = list(feature_columns)

mic_features = [
    "Mean_MIC", "Median_MIC", "Max_MIC", "Min_MIC",
    "Std_MIC", "MIC_Range", "Log_Mean_MIC"
]
coverage_features = ["Antibiotics_Tested", "Total_Records", "Coverage_Ratio"]
lab_method_features = [
    "Broth_Dilution_Count", "Disk_Diffusion_Count",
    "Agar_Dilution_Count", "MIC_Method_Count"
]
testing_standard_features = ["CLSI_Count", "EUCAST_Count", "Missing_Testing_Standard_Ratio"]
missingness_features = [
    "Missing_MIC_Ratio", "Missing_Testing_Standard_Ratio",
    "Missing_Laboratory_Method_Ratio"
]

mdr_top_shap_features = [
    "Antibiotics_Tested", "Broth_Dilution_Count", "Missing_MIC_Ratio",
    "Missing_Testing_Standard_Ratio", "Mean_MIC", "Median_MIC",
    "CLSI_Count", "Disk_Diffusion_Count", "EUCAST_Count", "Std_MIC"
]
severity_top_shap_features = [
    "Mean_MIC", "Median_MIC", "Antibiotics_Tested",
    "Missing_Testing_Standard_Ratio", "Std_MIC", "Broth_Dilution_Count",
    "EUCAST_Count", "Disk_Diffusion_Count", "Coverage_Ratio", "Log_Mean_MIC"
]

def keep_existing(features):
    return [f for f in features if f in all_features]

def remove_features(features_to_remove):
    remove_set = set(keep_existing(features_to_remove))
    return [f for f in all_features if f not in remove_set]

mdr_ablation_sets = {
    "Full_All_Features": all_features,
    "Without_MIC_Features": remove_features(mic_features),
    "Without_Coverage_Features": remove_features(coverage_features),
    "Without_Lab_Method_Features": remove_features(lab_method_features),
    "Without_Testing_Standard_Features": remove_features(testing_standard_features),
    "Without_Missingness_Features": remove_features(missingness_features),
    "Without_Top3_SHAP_Features": remove_features(mdr_top_shap_features[:3]),
    "Top5_SHAP_Only": keep_existing(mdr_top_shap_features[:5]),
    "Top10_SHAP_Only": keep_existing(mdr_top_shap_features[:10])
}

severity_ablation_sets = {
    "Full_All_Features": all_features,
    "Without_MIC_Features": remove_features(mic_features),
    "Without_Coverage_Features": remove_features(coverage_features),
    "Without_Lab_Method_Features": remove_features(lab_method_features),
    "Without_Testing_Standard_Features": remove_features(testing_standard_features),
    "Without_Missingness_Features": remove_features(missingness_features),
    "Without_Top3_SHAP_Features": remove_features(severity_top_shap_features[:3]),
    "Top5_SHAP_Only": keep_existing(severity_top_shap_features[:5]),
    "Top10_SHAP_Only": keep_existing(severity_top_shap_features[:10])
}

print("\nMDR ablation settings:")
for k, v in mdr_ablation_sets.items():
    print(k, "->", len(v), "features")

print("\nSeverity ablation settings:")
for k, v in severity_ablation_sets.items():
    print(k, "->", len(v), "features")

# ------------------------------------------------------------
# Class weights
# ------------------------------------------------------------

mdr_neg = np.sum(y_train_mdr == 0)
mdr_pos = np.sum(y_train_mdr == 1)
mdr_scale_pos_weight = mdr_neg / max(mdr_pos, 1)

severity_classes = np.array([0, 1, 2, 3])
severity_weights = compute_class_weight(
    class_weight="balanced",
    classes=severity_classes,
    y=y_train_sev_enc
)
severity_class_weight_dict = {
    int(c): float(w) for c, w in zip(severity_classes, severity_weights)
}
severity_sample_weight = np.array([
    severity_class_weight_dict[int(y)] for y in y_train_sev_enc
])

# ------------------------------------------------------------
# Model builders
# ------------------------------------------------------------

def build_mdr_models():
    models = {}
    if LGBMClassifier is not None:
        models["LightGBM"] = LGBMClassifier(
            n_estimators=250, learning_rate=0.05, num_leaves=31,
            subsample=0.9, colsample_bytree=0.9,
            class_weight="balanced", random_state=SEED,
            n_jobs=-1, verbosity=-1
        )
    if XGBClassifier is not None:
        models["XGBoost"] = XGBClassifier(
            n_estimators=250, learning_rate=0.05, max_depth=4,
            subsample=0.9, colsample_bytree=0.9,
            objective="binary:logistic", eval_metric="logloss",
            scale_pos_weight=mdr_scale_pos_weight,
            random_state=SEED, n_jobs=-1, tree_method="hist"
        )
    models["RandomForest"] = RandomForestClassifier(
        n_estimators=250, class_weight="balanced",
        random_state=SEED, n_jobs=-1
    )
    if CatBoostClassifier is not None:
        models["CatBoost"] = CatBoostClassifier(
            iterations=250, learning_rate=0.05, depth=6,
            loss_function="Logloss", random_seed=SEED,
            auto_class_weights="Balanced", verbose=False
        )
    return models

def build_severity_models():
    models = {}
    if LGBMClassifier is not None:
        models["LightGBM"] = LGBMClassifier(
            n_estimators=250, learning_rate=0.05, num_leaves=31,
            subsample=0.9, colsample_bytree=0.9,
            objective="multiclass", num_class=n_severity_classes,
            class_weight=severity_class_weight_dict,
            random_state=SEED, n_jobs=-1, verbosity=-1
        )
    if XGBClassifier is not None:
        models["XGBoost"] = XGBClassifier(
            n_estimators=250, learning_rate=0.05, max_depth=4,
            subsample=0.9, colsample_bytree=0.9,
            objective="multi:softprob", num_class=n_severity_classes,
            eval_metric="mlogloss", random_state=SEED,
            n_jobs=-1, tree_method="hist"
        )
    models["RandomForest"] = RandomForestClassifier(
        n_estimators=250, class_weight=severity_class_weight_dict,
        random_state=SEED, n_jobs=-1
    )
    if CatBoostClassifier is not None:
        cat_weights = [severity_class_weight_dict[i] for i in range(n_severity_classes)]
        models["CatBoost"] = CatBoostClassifier(
            iterations=250, learning_rate=0.05, depth=6,
            loss_function="MultiClass", random_seed=SEED,
            class_weights=cat_weights, verbose=False
        )
    return models

# ------------------------------------------------------------
# Probability helpers
# ------------------------------------------------------------

def get_binary_probability(model, X_data):
    proba = model.predict_proba(X_data)
    classes = getattr(model, "classes_", None)
    if classes is not None:
        classes = list(classes)
        if 1 in classes:
            return proba[:, classes.index(1)]
    if proba.ndim == 2 and proba.shape[1] >= 2:
        return proba[:, 1]
    return proba.ravel()

def get_multiclass_probability(model, X_data):
    proba = model.predict_proba(X_data)
    if proba.shape[1] == n_severity_classes:
        return proba
    aligned = np.zeros((len(X_data), n_severity_classes))
    classes = getattr(model, "classes_", None)
    if classes is not None:
        for i, cls in enumerate(classes):
            cls_int = int(cls)
            if cls_int < n_severity_classes:
                aligned[:, cls_int] = proba[:, i]
    else:
        aligned[:, :proba.shape[1]] = proba
    return aligned

# ------------------------------------------------------------
# Evaluation functions
# ------------------------------------------------------------

def evaluate_mdr(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    out = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall_Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity,
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Cohen_Kappa": cohen_kappa_score(y_true, y_pred),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn)
    }
    try:
        out["ROC_AUC"] = roc_auc_score(y_true, y_prob)
    except Exception:
        out["ROC_AUC"] = np.nan
    try:
        out["PR_AUC"] = average_precision_score(y_true, y_prob)
    except Exception:
        out["PR_AUC"] = np.nan
    try:
        out["Log_Loss"] = log_loss(y_true, np.column_stack([1 - y_prob, y_prob]), labels=[0, 1])
    except Exception:
        out["Log_Loss"] = np.nan
    try:
        out["Brier_Score"] = brier_score_loss(y_true, y_prob)
    except Exception:
        out["Brier_Score"] = np.nan
    return out

def macro_specificity(y_true, y_pred):
    specs = []
    for cls in range(n_severity_classes):
        yt = (y_true == cls).astype(int)
        yp = (y_pred == cls).astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
    return float(np.nanmean(specs))

def multiclass_brier_score(y_true, y_prob):
    one_hot = np.eye(n_severity_classes)[y_true]
    return float(np.mean(np.sum((y_prob - one_hot) ** 2, axis=1)))

def evaluate_severity(y_true, y_pred, y_prob):
    out = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro_Recall_Sensitivity": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted_F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro_Specificity": macro_specificity(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Cohen_Kappa": cohen_kappa_score(y_true, y_pred)
    }
    try:
        out["ROC_AUC_OVR_Macro"] = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        out["ROC_AUC_OVR_Macro"] = np.nan
    try:
        out["PR_AUC_Macro"] = average_precision_score(np.eye(n_severity_classes)[y_true], y_prob, average="macro")
    except Exception:
        out["PR_AUC_Macro"] = np.nan
    try:
        out["Log_Loss"] = log_loss(y_true, y_prob, labels=list(range(n_severity_classes)))
    except Exception:
        out["Log_Loss"] = np.nan
    try:
        out["Multiclass_Brier_Score"] = multiclass_brier_score(y_true, y_prob)
    except Exception:
        out["Multiclass_Brier_Score"] = np.nan
    return out

# ------------------------------------------------------------
# Voting runners
# ------------------------------------------------------------

def run_mdr_voting(features, model_subset=None):
    if model_subset is None:
        model_subset = ["LightGBM", "XGBoost", "RandomForest", "CatBoost"]
    selected = {k: v for k, v in build_mdr_models().items() if k in model_subset}
    Xtr = X_train[features].copy()
    Xte = X_test[features].copy()
    probs = []
    used = []
    for name, model in selected.items():
        try:
            model.fit(Xtr, y_train_mdr)
            probs.append(get_binary_probability(model, Xte))
            used.append(name)
        except Exception as e:
            print("Skipped MDR model:", name, "|", str(e))
    if not probs:
        raise RuntimeError("All MDR models failed.")
    final_prob = np.mean(np.column_stack(probs), axis=1)
    final_pred = (final_prob >= 0.5).astype(int)
    return final_pred, final_prob, used

def run_severity_voting(features, model_subset=None):
    if model_subset is None:
        model_subset = ["LightGBM", "XGBoost", "RandomForest", "CatBoost"]
    selected = {k: v for k, v in build_severity_models().items() if k in model_subset}
    Xtr = X_train[features].copy()
    Xte = X_test[features].copy()
    probs = []
    used = []
    for name, model in selected.items():
        try:
            if name == "XGBoost":
                model.fit(Xtr, y_train_sev_enc, sample_weight=severity_sample_weight)
            else:
                model.fit(Xtr, y_train_sev_enc)
            probs.append(get_multiclass_probability(model, Xte))
            used.append(name)
        except Exception as e:
            print("Skipped Severity model:", name, "|", str(e))
    if not probs:
        raise RuntimeError("All severity models failed.")
    final_prob = np.mean(np.stack(probs, axis=0), axis=0)
    final_pred = np.argmax(final_prob, axis=1)
    return final_pred, final_prob, used

# ------------------------------------------------------------
# Run feature ablation
# ------------------------------------------------------------

mdr_feature_rows = []
print("\n" + "=" * 70)
print("MDR FEATURE-FAMILY ABLATION")
print("=" * 70)
for setting, feats in mdr_ablation_sets.items():
    start = time.time()
    pred, prob, used = run_mdr_voting(feats)
    metrics = evaluate_mdr(y_test_mdr, pred, prob)
    row = {"Task": "MDR", "Ablation_Type": "Feature_Family", "Ablation_Setting": setting,
           "Feature_Count": len(feats), "Features_Used": str(feats), "Models_Used": str(used),
           "Runtime_Seconds": time.time() - start}
    row.update(metrics)
    mdr_feature_rows.append(row)
    print(setting, "| Features:", len(feats), "| F1:", round(metrics["F1"], 4), "| MCC:", round(metrics["MCC"], 4))

severity_feature_rows = []
print("\n" + "=" * 70)
print("SEVERITY FEATURE-FAMILY ABLATION")
print("=" * 70)
for setting, feats in severity_ablation_sets.items():
    start = time.time()
    pred, prob, used = run_severity_voting(feats)
    metrics = evaluate_severity(y_test_sev_enc, pred, prob)
    row = {"Task": "Severity", "Ablation_Type": "Feature_Family", "Ablation_Setting": setting,
           "Feature_Count": len(feats), "Features_Used": str(feats), "Models_Used": str(used),
           "Runtime_Seconds": time.time() - start}
    row.update(metrics)
    severity_feature_rows.append(row)
    print(setting, "| Features:", len(feats), "| Macro F1:", round(metrics["Macro_F1"], 4), "| MCC:", round(metrics["MCC"], 4))

mdr_feature_ablation = pd.DataFrame(mdr_feature_rows)
severity_feature_ablation = pd.DataFrame(severity_feature_rows)

# ------------------------------------------------------------
# Delta columns
# ------------------------------------------------------------

def add_delta_columns(df, baseline_setting, metrics):
    df = df.copy()
    baseline = df[df["Ablation_Setting"] == baseline_setting].iloc[0]
    for metric in metrics:
        if metric in df.columns:
            df["Delta_" + metric + "_vs_Full"] = df[metric] - baseline[metric]
    return df

mdr_metrics = ["Accuracy", "F1", "Balanced_Accuracy", "MCC", "ROC_AUC", "PR_AUC"]
severity_metrics = ["Accuracy", "Macro_F1", "Balanced_Accuracy", "MCC", "ROC_AUC_OVR_Macro", "PR_AUC_Macro"]

mdr_feature_ablation = add_delta_columns(mdr_feature_ablation, "Full_All_Features", mdr_metrics)
severity_feature_ablation = add_delta_columns(severity_feature_ablation, "Full_All_Features", severity_metrics)

mdr_feature_ablation.to_csv(os.path.join(BLOCK17_TABLE_DIR, "table_17_mdr_feature_family_ablation.csv"), index=False)
severity_feature_ablation.to_csv(os.path.join(BLOCK17_TABLE_DIR, "table_17_severity_feature_family_ablation.csv"), index=False)

print("\n" + "=" * 70)
print("MDR FEATURE-FAMILY ABLATION WITH DELTA")
print("=" * 70)
display(mdr_feature_ablation)

print("\n" + "=" * 70)
print("SEVERITY FEATURE-FAMILY ABLATION WITH DELTA")
print("=" * 70)
display(severity_feature_ablation)

# ------------------------------------------------------------
# Ensemble component ablation
# ------------------------------------------------------------

component_sets = {
    "Full_4_Model_Voting": ["LightGBM", "XGBoost", "RandomForest", "CatBoost"],
    "Without_LightGBM": ["XGBoost", "RandomForest", "CatBoost"],
    "Without_XGBoost": ["LightGBM", "RandomForest", "CatBoost"],
    "Without_RandomForest": ["LightGBM", "XGBoost", "CatBoost"],
    "Without_CatBoost": ["LightGBM", "XGBoost", "RandomForest"]
}

mdr_component_rows = []
print("\n" + "=" * 70)
print("MDR ENSEMBLE COMPONENT ABLATION")
print("=" * 70)
for setting, subset in component_sets.items():
    start = time.time()
    pred, prob, used = run_mdr_voting(all_features, model_subset=subset)
    metrics = evaluate_mdr(y_test_mdr, pred, prob)
    row = {"Task": "MDR", "Ablation_Type": "Ensemble_Component", "Ablation_Setting": setting,
           "Model_Count": len(used), "Models_Used": str(used), "Feature_Count": len(all_features),
           "Runtime_Seconds": time.time() - start}
    row.update(metrics)
    mdr_component_rows.append(row)
    print(setting, "| Models:", used, "| F1:", round(metrics["F1"], 4), "| MCC:", round(metrics["MCC"], 4))

severity_component_rows = []
print("\n" + "=" * 70)
print("SEVERITY ENSEMBLE COMPONENT ABLATION")
print("=" * 70)
for setting, subset in component_sets.items():
    start = time.time()
    pred, prob, used = run_severity_voting(all_features, model_subset=subset)
    metrics = evaluate_severity(y_test_sev_enc, pred, prob)
    row = {"Task": "Severity", "Ablation_Type": "Ensemble_Component", "Ablation_Setting": setting,
           "Model_Count": len(used), "Models_Used": str(used), "Feature_Count": len(all_features),
           "Runtime_Seconds": time.time() - start}
    row.update(metrics)
    severity_component_rows.append(row)
    print(setting, "| Models:", used, "| Macro F1:", round(metrics["Macro_F1"], 4), "| MCC:", round(metrics["MCC"], 4))

mdr_component_ablation = pd.DataFrame(mdr_component_rows)
severity_component_ablation = pd.DataFrame(severity_component_rows)
mdr_component_ablation = add_delta_columns(mdr_component_ablation, "Full_4_Model_Voting", mdr_metrics)
severity_component_ablation = add_delta_columns(severity_component_ablation, "Full_4_Model_Voting", severity_metrics)

mdr_component_ablation.to_csv(os.path.join(BLOCK17_TABLE_DIR, "table_17_mdr_ensemble_component_ablation.csv"), index=False)
severity_component_ablation.to_csv(os.path.join(BLOCK17_TABLE_DIR, "table_17_severity_ensemble_component_ablation.csv"), index=False)

print("\n" + "=" * 70)
print("MDR ENSEMBLE COMPONENT ABLATION WITH DELTA")
print("=" * 70)
display(mdr_component_ablation)

print("\n" + "=" * 70)
print("SEVERITY ENSEMBLE COMPONENT ABLATION WITH DELTA")
print("=" * 70)
display(severity_component_ablation)

# ------------------------------------------------------------
# Save combined and publication tables
# ------------------------------------------------------------

combined_ablation_results = pd.concat(
    [mdr_feature_ablation, severity_feature_ablation, mdr_component_ablation, severity_component_ablation],
    ignore_index=True,
    sort=False
)
combined_ablation_results.to_csv(os.path.join(BLOCK17_TABLE_DIR, "table_17_combined_ablation_results.csv"), index=False)

mdr_pub_cols = ["Task", "Ablation_Type", "Ablation_Setting", "Feature_Count", "Accuracy", "F1", "Balanced_Accuracy", "MCC", "ROC_AUC", "PR_AUC", "Delta_F1_vs_Full", "Delta_MCC_vs_Full"]
sev_pub_cols = ["Task", "Ablation_Type", "Ablation_Setting", "Feature_Count", "Accuracy", "Macro_F1", "Balanced_Accuracy", "MCC", "ROC_AUC_OVR_Macro", "PR_AUC_Macro", "Delta_Macro_F1_vs_Full", "Delta_MCC_vs_Full"]

mdr_feature_ablation[[c for c in mdr_pub_cols if c in mdr_feature_ablation.columns]].to_csv(os.path.join(BLOCK17_TABLE_DIR, "table_17_mdr_ablation_publication_summary.csv"), index=False)
severity_feature_ablation[[c for c in sev_pub_cols if c in severity_feature_ablation.columns]].to_csv(os.path.join(BLOCK17_TABLE_DIR, "table_17_severity_ablation_publication_summary.csv"), index=False)

# ------------------------------------------------------------
# Figures
# ------------------------------------------------------------

def plot_delta_bar(df, delta_col, title, output_filename, baseline_setting):
    if delta_col not in df.columns:
        print("Skipping figure. Missing column:", delta_col)
        return
    plot_df = df[df["Ablation_Setting"] != baseline_setting].copy()
    plot_df = plot_df.sort_values(by=delta_col, ascending=True)
    plt.figure(figsize=(10, 6))
    plt.barh(plot_df["Ablation_Setting"], plot_df[delta_col])
    plt.axvline(0, linewidth=1)
    plt.title(title)
    plt.xlabel("Delta compared with full model")
    plt.ylabel("Ablation setting")
    plt.tight_layout()
    plt.savefig(os.path.join(BLOCK17_FIG_DIR, output_filename), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

plot_delta_bar(mdr_feature_ablation, "Delta_F1_vs_Full", "MDR Feature-Family Ablation — Delta F1", "figure_17a_mdr_feature_ablation_delta_f1.png", "Full_All_Features")
plot_delta_bar(mdr_feature_ablation, "Delta_MCC_vs_Full", "MDR Feature-Family Ablation — Delta MCC", "figure_17b_mdr_feature_ablation_delta_mcc.png", "Full_All_Features")
plot_delta_bar(severity_feature_ablation, "Delta_Macro_F1_vs_Full", "Severity Feature-Family Ablation — Delta Macro F1", "figure_17c_severity_feature_ablation_delta_macro_f1.png", "Full_All_Features")
plot_delta_bar(severity_feature_ablation, "Delta_MCC_vs_Full", "Severity Feature-Family Ablation — Delta MCC", "figure_17d_severity_feature_ablation_delta_mcc.png", "Full_All_Features")
plot_delta_bar(mdr_component_ablation, "Delta_F1_vs_Full", "MDR Ensemble Component Ablation — Delta F1", "figure_17e_mdr_component_ablation_delta_f1.png", "Full_4_Model_Voting")
plot_delta_bar(severity_component_ablation, "Delta_Macro_F1_vs_Full", "Severity Ensemble Component Ablation — Delta Macro F1", "figure_17f_severity_component_ablation_delta_macro_f1.png", "Full_4_Model_Voting")

# ------------------------------------------------------------
# Paper-ready text
# ------------------------------------------------------------

mdr_full = mdr_feature_ablation[mdr_feature_ablation["Ablation_Setting"] == "Full_All_Features"].iloc[0]
severity_full = severity_feature_ablation[severity_feature_ablation["Ablation_Setting"] == "Full_All_Features"].iloc[0]
mdr_worst = mdr_feature_ablation[mdr_feature_ablation["Ablation_Setting"] != "Full_All_Features"].sort_values(by="Delta_F1_vs_Full", ascending=True).iloc[0]
severity_worst = severity_feature_ablation[severity_feature_ablation["Ablation_Setting"] != "Full_All_Features"].sort_values(by="Delta_Macro_F1_vs_Full", ascending=True).iloc[0]
mdr_best_compact = mdr_feature_ablation[mdr_feature_ablation["Ablation_Setting"].astype(str).str.contains("Only")].sort_values(by="F1", ascending=False).iloc[0]
severity_best_compact = severity_feature_ablation[severity_feature_ablation["Ablation_Setting"].astype(str).str.contains("Only")].sort_values(by="Macro_F1", ascending=False).iloc[0]

ablation_text = (
    "Ablation analysis was performed to quantify the contribution of feature families and ensemble components. "
    f"For MDR prediction, the full-feature voting ensemble achieved an F1-score of {mdr_full['F1']:.4f} and MCC of {mdr_full['MCC']:.4f}. "
    f"The largest MDR performance decrease was observed for {mdr_worst['Ablation_Setting']}, with F1 changing by {mdr_worst['Delta_F1_vs_Full']:.4f} relative to the full model. "
    f"Among compact MDR configurations, {mdr_best_compact['Ablation_Setting']} achieved an F1-score of {mdr_best_compact['F1']:.4f} using {int(mdr_best_compact['Feature_Count'])} features. "
    f"For resistance severity assessment, the full-feature voting ensemble achieved a macro F1-score of {severity_full['Macro_F1']:.4f} and MCC of {severity_full['MCC']:.4f}. "
    f"The largest severity performance decrease was observed for {severity_worst['Ablation_Setting']}, with macro F1 changing by {severity_worst['Delta_Macro_F1_vs_Full']:.4f} relative to the full model. "
    f"Among compact severity configurations, {severity_best_compact['Ablation_Setting']} achieved a macro F1-score of {severity_best_compact['Macro_F1']:.4f} using {int(severity_best_compact['Feature_Count'])} features. "
    "These ablation results support the importance of MIC-derived variables, antibiotic testing coverage, laboratory testing indicators, and missingness-aware features in the proposed genome-level MDR and severity prediction framework."
)

with open(os.path.join(BLOCK17_TEXT_DIR, "paper_ready_ablation_study_summary.txt"), "w", encoding="utf-8") as f:
    f.write(ablation_text)

print("\n" + "=" * 70)
print("PAPER-READY ABLATION TEXT")
print("=" * 70)
print(ablation_text)

# ------------------------------------------------------------
# Output summary
# ------------------------------------------------------------

block17_output_summary = pd.DataFrame({
    "Output_Category": [
        "MDR feature-family ablation",
        "Severity feature-family ablation",
        "MDR ensemble component ablation",
        "Severity ensemble component ablation",
        "Combined ablation results",
        "MDR ablation publication summary",
        "Severity ablation publication summary",
        "Ablation figures",
        "Paper-ready ablation text"
    ],
    "Saved_File": [
        "tables/table_17_mdr_feature_family_ablation.csv",
        "tables/table_17_severity_feature_family_ablation.csv",
        "tables/table_17_mdr_ensemble_component_ablation.csv",
        "tables/table_17_severity_ensemble_component_ablation.csv",
        "tables/table_17_combined_ablation_results.csv",
        "tables/table_17_mdr_ablation_publication_summary.csv",
        "tables/table_17_severity_ablation_publication_summary.csv",
        "figures/figure_17*.png",
        "paper_ready_text/paper_ready_ablation_study_summary.txt"
    ]
})
block17_output_summary.to_csv(os.path.join(BLOCK17_DIR, "block17_output_summary.csv"), index=False)

print("\n" + "=" * 70)
print("BLOCK 17 GENERATED FILES")
print("=" * 70)
print("- tables/table_17_mdr_feature_family_ablation.csv")
print("- tables/table_17_severity_feature_family_ablation.csv")
print("- tables/table_17_mdr_ensemble_component_ablation.csv")
print("- tables/table_17_severity_ensemble_component_ablation.csv")
print("- tables/table_17_combined_ablation_results.csv")
print("- tables/table_17_mdr_ablation_publication_summary.csv")
print("- tables/table_17_severity_ablation_publication_summary.csv")
print("- figures/figure_17a_mdr_feature_ablation_delta_f1.png")
print("- figures/figure_17b_mdr_feature_ablation_delta_mcc.png")
print("- figures/figure_17c_severity_feature_ablation_delta_macro_f1.png")
print("- figures/figure_17d_severity_feature_ablation_delta_mcc.png")
print("- figures/figure_17e_mdr_component_ablation_delta_f1.png")
print("- figures/figure_17f_severity_component_ablation_delta_macro_f1.png")
print("- paper_ready_text/paper_ready_ablation_study_summary.txt")
print("\n" + "=" * 70)
print("BLOCK 17 COMPLETED SUCCESSFULLY")
print("=" * 70)


In [ ]:
# ============================================================
# BLOCK 18
# FINAL PUBLICATION PACKAGE
# Final Tables + Final Figures Index + Paper-Ready Result Text
# Journal Submission Support Version
# ============================================================

print("=" * 70)
print("BLOCK 18 STARTED — FINAL PUBLICATION PACKAGE")
print("=" * 70)

import os
import json
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ============================================================
# Required Variable Check
# ============================================================

required_vars = [
    "OUTPUT_DIR"
]

missing_vars = [
    v for v in required_vars
    if v not in globals()
]

if len(missing_vars) > 0:
    raise RuntimeError(
        "Missing required variables from previous blocks: "
        + ", ".join(missing_vars)
    )


# ============================================================
# Output Directories
# ============================================================

BLOCK18_DIR = os.path.join(
    OUTPUT_DIR,
    "block18_final_publication_package"
)

BLOCK18_TABLE_DIR = os.path.join(
    BLOCK18_DIR,
    "final_tables"
)

BLOCK18_FIG_DIR = os.path.join(
    BLOCK18_DIR,
    "final_figures"
)

BLOCK18_TEXT_DIR = os.path.join(
    BLOCK18_DIR,
    "paper_ready_text"
)

BLOCK18_EXPORT_DIR = os.path.join(
    BLOCK18_DIR,
    "export_package"
)

for d in [
    BLOCK18_DIR,
    BLOCK18_TABLE_DIR,
    BLOCK18_FIG_DIR,
    BLOCK18_TEXT_DIR,
    BLOCK18_EXPORT_DIR
]:
    os.makedirs(d, exist_ok=True)


# ============================================================
# Helper Functions
# ============================================================

def read_csv_if_exists(path):

    if os.path.exists(path):
        return pd.read_csv(path)

    print("Missing file:", path)
    return pd.DataFrame()


def copy_if_exists(src, dst_dir):

    if os.path.exists(src):

        os.makedirs(dst_dir, exist_ok=True)

        dst = os.path.join(
            dst_dir,
            os.path.basename(src)
        )

        shutil.copy2(src, dst)

        return dst

    return None


def safe_float(value):

    try:
        return float(value)
    except Exception:
        return np.nan


def format_metric(value, decimals=4):

    value = safe_float(value)

    if np.isnan(value):
        return "NA"

    return f"{value:.{decimals}f}"


def format_delta(value, decimals=4):

    value = safe_float(value)

    if np.isnan(value):
        return "NA"

    sign = "+" if value > 0 else ""

    return f"{sign}{value:.{decimals}f}"


def first_existing(paths):

    for p in paths:

        if os.path.exists(p):
            return p

    return None


# ============================================================
# Source Paths
# ============================================================

block16_dir = os.path.join(
    OUTPUT_DIR,
    "block16_final_model_selection"
)

block16_table_dir = os.path.join(
    block16_dir,
    "publication_tables"
)

block16_text_dir = os.path.join(
    block16_dir,
    "paper_ready_text"
)

block16_fig_dir = os.path.join(
    block16_dir,
    "publication_figures"
)

block17_dir = os.path.join(
    OUTPUT_DIR,
    "block17_ablation_study"
)

block17_table_dir = os.path.join(
    block17_dir,
    "tables"
)

block17_text_dir = os.path.join(
    block17_dir,
    "paper_ready_text"
)

block17_fig_dir = os.path.join(
    block17_dir,
    "figures"
)

block15_dir = os.path.join(
    OUTPUT_DIR,
    "block15_statistical_analysis"
)

block15_ci_dir = os.path.join(
    block15_dir,
    "confidence_intervals"
)

block15_test_dir = os.path.join(
    block15_dir,
    "statistical_tests"
)

block12_dir = os.path.join(
    OUTPUT_DIR,
    "block12_shap_xai"
)

block12_table_dir = os.path.join(
    block12_dir,
    "tables"
)

block05_dir = os.path.join(
    OUTPUT_DIR,
    "block05_mdr_label_generation"
)

block06_dir = os.path.join(
    OUTPUT_DIR,
    "block06_severity_modeling"
)


# ============================================================
# Load Final Tables
# ============================================================

final_selection = read_csv_if_exists(
    os.path.join(
        block16_table_dir,
        "table_16_final_proposed_model_selection.csv"
    )
)

final_ci = read_csv_if_exists(
    os.path.join(
        block16_table_dir,
        "table_16_final_metrics_with_bootstrap_ci.csv"
    )
)

mdr_perf = read_csv_if_exists(
    os.path.join(
        block16_table_dir,
        "table_16_mdr_test_performance_publication.csv"
    )
)

severity_perf = read_csv_if_exists(
    os.path.join(
        block16_table_dir,
        "table_16_severity_test_performance_publication.csv"
    )
)

xai_summary = read_csv_if_exists(
    os.path.join(
        block16_table_dir,
        "table_16_xai_feature_importance_summary.csv"
    )
)

framework_components = read_csv_if_exists(
    os.path.join(
        block16_table_dir,
        "table_16_framework_components.csv"
    )
)

mdr_ablation = read_csv_if_exists(
    first_existing([
        os.path.join(block17_table_dir, "table_17_mdr_ablation_publication_summary.csv"),
        os.path.join(block17_table_dir, "table_17_mdr_feature_family_ablation.csv"),
        os.path.join(block17_table_dir, "table_17_mdr_feature_ablation.csv")
    ]) or ""
)

severity_ablation = read_csv_if_exists(
    first_existing([
        os.path.join(block17_table_dir, "table_17_severity_ablation_publication_summary.csv"),
        os.path.join(block17_table_dir, "table_17_severity_feature_family_ablation.csv"),
        os.path.join(block17_table_dir, "table_17_severity_feature_ablation.csv")
    ]) or ""
)

mdr_component_ablation = read_csv_if_exists(
    os.path.join(
        block17_table_dir,
        "table_17_mdr_ensemble_component_ablation.csv"
    )
)

severity_component_ablation = read_csv_if_exists(
    os.path.join(
        block17_table_dir,
        "table_17_severity_ensemble_component_ablation.csv"
    )
)

combined_ablation = read_csv_if_exists(
    os.path.join(
        block17_table_dir,
        "table_17_combined_ablation_results.csv"
    )
)

mdr_ci_robust = read_csv_if_exists(
    os.path.join(
        block15_ci_dir,
        "mdr_bootstrap_confidence_intervals_robust.csv"
    )
)

mdr_pairwise = read_csv_if_exists(
    os.path.join(
        block15_test_dir,
        "mdr_pairwise_tests_robust.csv"
    )
)

severity_pairwise = read_csv_if_exists(
    os.path.join(
        block15_test_dir,
        "pairwise_tests_against_best_model_fast.csv"
    )
)


# ============================================================
# Table 1: Dataset and Framework Summary
# ============================================================

dataset_summary_rows = []

if "model_df" in globals():

    dataset_summary_rows.append({
        "Item": "Final modeling genomes",
        "Value": len(model_df)
    })

if "X_train" in globals() and "X_test" in globals():

    dataset_summary_rows.extend([
        {
            "Item": "Training samples",
            "Value": len(X_train)
        },
        {
            "Item": "Test samples",
            "Value": len(X_test)
        }
    ])

if "feature_columns" in globals():

    dataset_summary_rows.append({
        "Item": "Genome-level engineered features",
        "Value": len(feature_columns)
    })

if "y_train" in globals() and "y_test" in globals():

    if "MDR_Label" in y_train.columns:

        train_mdr_counts = y_train["MDR_Label"].value_counts().to_dict()
        test_mdr_counts = y_test["MDR_Label"].value_counts().to_dict()

        dataset_summary_rows.append({
            "Item": "Training MDR distribution",
            "Value": str(train_mdr_counts)
        })

        dataset_summary_rows.append({
            "Item": "Test MDR distribution",
            "Value": str(test_mdr_counts)
        })

    if "Severity_Label" in y_train.columns:

        train_sev_counts = y_train["Severity_Label"].value_counts().to_dict()
        test_sev_counts = y_test["Severity_Label"].value_counts().to_dict()

        dataset_summary_rows.append({
            "Item": "Training severity distribution",
            "Value": str(train_sev_counts)
        })

        dataset_summary_rows.append({
            "Item": "Test severity distribution",
            "Value": str(test_sev_counts)
        })

dataset_framework_summary = pd.DataFrame(dataset_summary_rows)

dataset_framework_summary.to_csv(
    os.path.join(
        BLOCK18_TABLE_DIR,
        "table_1_dataset_and_framework_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("TABLE 1 — DATASET AND FRAMEWORK SUMMARY")
print("=" * 70)
display(dataset_framework_summary)


# ============================================================
# Table 2: Final Proposed Model Performance
# ============================================================

final_model_performance_rows = []

if len(final_selection) > 0:

    for _, row in final_selection.iterrows():

        task = row.get("Task", "Unknown")

        final_model_performance_rows.append({
            "Task": task,
            "Final Model": str(row.get("Final_Proposed_Stage", "")) + " / " + str(row.get("Final_Proposed_Model", "")),
            "Accuracy": format_metric(row.get("Accuracy", np.nan)),
            "F1 or Macro F1": format_metric(row.get("F1_or_Macro_F1", np.nan)),
            "Balanced Accuracy": format_metric(row.get("Balanced_Accuracy", np.nan)),
            "MCC": format_metric(row.get("MCC", np.nan)),
            "ROC-AUC": format_metric(row.get("ROC_AUC", np.nan)),
            "PR-AUC": format_metric(row.get("PR_AUC", np.nan)),
            "Selection Rationale": row.get("Primary_Selection_Criterion", "")
        })

final_model_performance = pd.DataFrame(final_model_performance_rows)

final_model_performance.to_csv(
    os.path.join(
        BLOCK18_TABLE_DIR,
        "table_2_final_proposed_model_performance.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("TABLE 2 — FINAL PROPOSED MODEL PERFORMANCE")
print("=" * 70)
display(final_model_performance)


# ============================================================
# Table 3: Final Metrics With Bootstrap CI
# ============================================================

final_ci_table = final_ci.copy()

final_ci_table.to_csv(
    os.path.join(
        BLOCK18_TABLE_DIR,
        "table_3_bootstrap_confidence_intervals.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("TABLE 3 — BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 70)
display(final_ci_table)


# ============================================================
# Table 4 and 5: Test Performance Rankings
# ============================================================

mdr_perf.to_csv(
    os.path.join(
        BLOCK18_TABLE_DIR,
        "table_4_mdr_test_model_comparison.csv"
    ),
    index=False
)

severity_perf.to_csv(
    os.path.join(
        BLOCK18_TABLE_DIR,
        "table_5_severity_test_model_comparison.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("TABLE 4 — MDR TEST MODEL COMPARISON")
print("=" * 70)
display(mdr_perf)

print("\n")
print("=" * 70)
print("TABLE 5 — SEVERITY TEST MODEL COMPARISON")
print("=" * 70)
display(severity_perf)


# ============================================================
# Table 6: XAI Summary
# ============================================================

xai_summary.to_csv(
    os.path.join(
        BLOCK18_TABLE_DIR,
        "table_6_xai_feature_importance_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("TABLE 6 — XAI FEATURE IMPORTANCE SUMMARY")
print("=" * 70)
display(xai_summary)


# ============================================================
# Table 7: Ablation Study Summary
# ============================================================

ablation_summary_rows = []

if len(mdr_ablation) > 0:

    mdr_full = mdr_ablation[
        mdr_ablation["Ablation_Setting"] == "Full_All_Features"
    ]

    if len(mdr_full) > 0:

        mdr_full = mdr_full.iloc[0]

        ablation_summary_rows.append({
            "Task": "MDR Prediction",
            "Ablation Setting": "Full_All_Features",
            "Feature Count": int(mdr_full.get("Feature_Count", 0)),
            "Primary Metric": "F1",
            "Primary Metric Value": format_metric(mdr_full.get("F1", np.nan)),
            "MCC": format_metric(mdr_full.get("MCC", np.nan)),
            "Delta vs Full": "Reference"
        })

    if "Delta_F1_vs_Full" in mdr_ablation.columns:

        mdr_worst = mdr_ablation[
            mdr_ablation["Ablation_Setting"] != "Full_All_Features"
        ].sort_values(
            by="Delta_F1_vs_Full",
            ascending=True
        ).iloc[0]

        ablation_summary_rows.append({
            "Task": "MDR Prediction",
            "Ablation Setting": mdr_worst.get("Ablation_Setting", ""),
            "Feature Count": int(mdr_worst.get("Feature_Count", 0)),
            "Primary Metric": "F1",
            "Primary Metric Value": format_metric(mdr_worst.get("F1", np.nan)),
            "MCC": format_metric(mdr_worst.get("MCC", np.nan)),
            "Delta vs Full": format_delta(mdr_worst.get("Delta_F1_vs_Full", np.nan))
        })

        compact_mdr = mdr_ablation[
            mdr_ablation["Ablation_Setting"].astype(str).str.contains("Only")
        ]

        if len(compact_mdr) > 0:

            compact_mdr = compact_mdr.sort_values(
                by="F1",
                ascending=False
            ).iloc[0]

            ablation_summary_rows.append({
                "Task": "MDR Prediction",
                "Ablation Setting": compact_mdr.get("Ablation_Setting", ""),
                "Feature Count": int(compact_mdr.get("Feature_Count", 0)),
                "Primary Metric": "F1",
                "Primary Metric Value": format_metric(compact_mdr.get("F1", np.nan)),
                "MCC": format_metric(compact_mdr.get("MCC", np.nan)),
                "Delta vs Full": format_delta(compact_mdr.get("Delta_F1_vs_Full", np.nan))
            })

if len(severity_ablation) > 0:

    sev_full = severity_ablation[
        severity_ablation["Ablation_Setting"] == "Full_All_Features"
    ]

    if len(sev_full) > 0:

        sev_full = sev_full.iloc[0]

        ablation_summary_rows.append({
            "Task": "Resistance Severity Assessment",
            "Ablation Setting": "Full_All_Features",
            "Feature Count": int(sev_full.get("Feature_Count", 0)),
            "Primary Metric": "Macro F1",
            "Primary Metric Value": format_metric(sev_full.get("Macro_F1", np.nan)),
            "MCC": format_metric(sev_full.get("MCC", np.nan)),
            "Delta vs Full": "Reference"
        })

    if "Delta_Macro_F1_vs_Full" in severity_ablation.columns:

        sev_worst = severity_ablation[
            severity_ablation["Ablation_Setting"] != "Full_All_Features"
        ].sort_values(
            by="Delta_Macro_F1_vs_Full",
            ascending=True
        ).iloc[0]

        ablation_summary_rows.append({
            "Task": "Resistance Severity Assessment",
            "Ablation Setting": sev_worst.get("Ablation_Setting", ""),
            "Feature Count": int(sev_worst.get("Feature_Count", 0)),
            "Primary Metric": "Macro F1",
            "Primary Metric Value": format_metric(sev_worst.get("Macro_F1", np.nan)),
            "MCC": format_metric(sev_worst.get("MCC", np.nan)),
            "Delta vs Full": format_delta(sev_worst.get("Delta_Macro_F1_vs_Full", np.nan))
        })

        compact_sev = severity_ablation[
            severity_ablation["Ablation_Setting"].astype(str).str.contains("Only")
        ]

        if len(compact_sev) > 0:

            compact_sev = compact_sev.sort_values(
                by="Macro_F1",
                ascending=False
            ).iloc[0]

            ablation_summary_rows.append({
                "Task": "Resistance Severity Assessment",
                "Ablation Setting": compact_sev.get("Ablation_Setting", ""),
                "Feature Count": int(compact_sev.get("Feature_Count", 0)),
                "Primary Metric": "Macro F1",
                "Primary Metric Value": format_metric(compact_sev.get("Macro_F1", np.nan)),
                "MCC": format_metric(compact_sev.get("MCC", np.nan)),
                "Delta vs Full": format_delta(compact_sev.get("Delta_Macro_F1_vs_Full", np.nan))
            })

ablation_summary_table = pd.DataFrame(ablation_summary_rows)

ablation_summary_table.to_csv(
    os.path.join(
        BLOCK18_TABLE_DIR,
        "table_7_ablation_study_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("TABLE 7 — ABLATION STUDY SUMMARY")
print("=" * 70)
display(ablation_summary_table)


# ============================================================
# Table 8: Ensemble Component Ablation Summary
# ============================================================

component_summary_rows = []

if len(mdr_component_ablation) > 0:

    temp = mdr_component_ablation.copy()

    cols = [
        "Task",
        "Ablation_Setting",
        "Model_Count",
        "Models_Used",
        "Accuracy",
        "F1",
        "MCC",
        "ROC_AUC",
        "Delta_F1_vs_Full",
        "Delta_MCC_vs_Full"
    ]

    for _, row in temp.iterrows():

        component_summary_rows.append({
            "Task": "MDR Prediction",
            "Ablation Setting": row.get("Ablation_Setting", ""),
            "Model Count": int(row.get("Model_Count", 0)),
            "Accuracy": format_metric(row.get("Accuracy", np.nan)),
            "F1 or Macro F1": format_metric(row.get("F1", np.nan)),
            "MCC": format_metric(row.get("MCC", np.nan)),
            "ROC-AUC": format_metric(row.get("ROC_AUC", np.nan)),
            "Delta Primary Metric": format_delta(row.get("Delta_F1_vs_Full", np.nan))
        })

if len(severity_component_ablation) > 0:

    temp = severity_component_ablation.copy()

    for _, row in temp.iterrows():

        component_summary_rows.append({
            "Task": "Resistance Severity Assessment",
            "Ablation Setting": row.get("Ablation_Setting", ""),
            "Model Count": int(row.get("Model_Count", 0)),
            "Accuracy": format_metric(row.get("Accuracy", np.nan)),
            "F1 or Macro F1": format_metric(row.get("Macro_F1", np.nan)),
            "MCC": format_metric(row.get("MCC", np.nan)),
            "ROC-AUC": format_metric(row.get("ROC_AUC_OVR_Macro", np.nan)),
            "Delta Primary Metric": format_delta(row.get("Delta_Macro_F1_vs_Full", np.nan))
        })

ensemble_component_summary = pd.DataFrame(component_summary_rows)

ensemble_component_summary.to_csv(
    os.path.join(
        BLOCK18_TABLE_DIR,
        "table_8_ensemble_component_ablation_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("TABLE 8 — ENSEMBLE COMPONENT ABLATION SUMMARY")
print("=" * 70)
display(ensemble_component_summary)


# ============================================================
# Table 9: Statistical Testing Summary
# ============================================================

statistical_summary_rows = []

if len(mdr_pairwise) > 0:

    focus_mdr = mdr_pairwise[
        mdr_pairwise["Metric"].isin(["Accuracy", "F1", "MCC", "Balanced_Accuracy"])
    ].copy()

    if len(focus_mdr) > 0:

        for _, row in focus_mdr.head(12).iterrows():

            statistical_summary_rows.append({
                "Task": "MDR Prediction",
                "Best Model": row.get("Best_Model_Key", ""),
                "Compared Model": row.get("Compared_Model_Key", ""),
                "Metric": row.get("Metric", ""),
                "Mean Difference": format_delta(row.get("Mean_Difference_Best_Minus_Compared", np.nan)),
                "95% Difference CI Lower": format_metric(row.get("Difference_CI_Lower", np.nan)),
                "95% Difference CI Upper": format_metric(row.get("Difference_CI_Upper", np.nan)),
                "Bootstrap p-value": format_metric(row.get("Bootstrap_P_Value_Two_Sided", np.nan)),
                "McNemar p-value": format_metric(row.get("McNemar_P_Value", np.nan))
            })

if len(severity_pairwise) > 0:

    focus_sev = severity_pairwise[
        severity_pairwise["Task"].astype(str).str.lower().str.contains("severity")
        &
        severity_pairwise["Metric"].isin(["Accuracy", "F1", "MCC", "Balanced_Accuracy"])
    ].copy()

    if len(focus_sev) > 0:

        for _, row in focus_sev.head(12).iterrows():

            statistical_summary_rows.append({
                "Task": "Resistance Severity Assessment",
                "Best Model": row.get("Best_Model_Key", ""),
                "Compared Model": row.get("Compared_Model_Key", ""),
                "Metric": row.get("Metric", ""),
                "Mean Difference": format_delta(row.get("Mean_Difference_Best_Minus_Compared", np.nan)),
                "95% Difference CI Lower": format_metric(row.get("Difference_CI_Lower", np.nan)),
                "95% Difference CI Upper": format_metric(row.get("Difference_CI_Upper", np.nan)),
                "Bootstrap p-value": format_metric(row.get("Bootstrap_P_Value_Two_Sided", np.nan)),
                "McNemar p-value": format_metric(row.get("McNemar_P_Value", np.nan))
            })

statistical_testing_summary = pd.DataFrame(statistical_summary_rows)

statistical_testing_summary.to_csv(
    os.path.join(
        BLOCK18_TABLE_DIR,
        "table_9_statistical_testing_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("TABLE 9 — STATISTICAL TESTING SUMMARY")
print("=" * 70)
display(statistical_testing_summary)


# ============================================================
# Table 10: Manuscript Table Index
# ============================================================

manuscript_table_index = pd.DataFrame({
    "Table_Number": [
        "Table 1",
        "Table 2",
        "Table 3",
        "Table 4",
        "Table 5",
        "Table 6",
        "Table 7",
        "Table 8",
        "Table 9"
    ],
    "Title": [
        "Dataset and framework summary",
        "Final proposed model performance",
        "Bootstrap confidence intervals for final models",
        "MDR test model comparison",
        "Resistance severity test model comparison",
        "SHAP-based feature importance summary",
        "Ablation study summary",
        "Ensemble component ablation summary",
        "Statistical testing summary"
    ],
    "File": [
        "table_1_dataset_and_framework_summary.csv",
        "table_2_final_proposed_model_performance.csv",
        "table_3_bootstrap_confidence_intervals.csv",
        "table_4_mdr_test_model_comparison.csv",
        "table_5_severity_test_model_comparison.csv",
        "table_6_xai_feature_importance_summary.csv",
        "table_7_ablation_study_summary.csv",
        "table_8_ensemble_component_ablation_summary.csv",
        "table_9_statistical_testing_summary.csv"
    ]
})

manuscript_table_index.to_csv(
    os.path.join(
        BLOCK18_TABLE_DIR,
        "table_10_manuscript_table_index.csv"
    ),
    index=False
)


# ============================================================
# Copy Key Figures
# ============================================================

figure_sources = [
    os.path.join(block16_fig_dir, "figure_16a_mdr_test_f1_comparison.png"),
    os.path.join(block16_fig_dir, "figure_16b_mdr_test_mcc_comparison.png"),
    os.path.join(block16_fig_dir, "figure_16c_severity_test_macro_f1_comparison.png"),
    os.path.join(block16_fig_dir, "figure_16d_severity_test_mcc_comparison.png"),
    os.path.join(block17_fig_dir, "figure_17a_mdr_feature_ablation_delta_f1.png"),
    os.path.join(block17_fig_dir, "figure_17b_mdr_feature_ablation_delta_mcc.png"),
    os.path.join(block17_fig_dir, "figure_17c_severity_feature_ablation_delta_macro_f1.png"),
    os.path.join(block17_fig_dir, "figure_17d_severity_feature_ablation_delta_mcc.png"),
    os.path.join(block17_fig_dir, "figure_17e_mdr_component_ablation_delta_f1.png"),
    os.path.join(block17_fig_dir, "figure_17f_severity_component_ablation_delta_macro_f1.png")
]

copied_figures = []

for src in figure_sources:

    copied = copy_if_exists(
        src,
        BLOCK18_FIG_DIR
    )

    if copied is not None:

        copied_figures.append({
            "Figure_File": os.path.basename(copied),
            "Source_Path": src,
            "Copied_To": copied
        })

figure_index = pd.DataFrame(copied_figures)

figure_index.to_csv(
    os.path.join(
        BLOCK18_FIG_DIR,
        "figure_index.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("FINAL FIGURE INDEX")
print("=" * 70)
display(figure_index)


# ============================================================
# Paper-Ready Result Text
# ============================================================

def get_final_metric(task_keyword, metric):

    if len(final_selection) == 0:
        return np.nan

    temp = final_selection[
        final_selection["Task"].astype(str).str.lower().str.contains(
            task_keyword.lower()
        )
    ]

    if len(temp) == 0:
        return np.nan

    return safe_float(temp.iloc[0].get(metric, np.nan))


mdr_acc = get_final_metric("MDR", "Accuracy")
mdr_f1 = get_final_metric("MDR", "F1_or_Macro_F1")
mdr_ba = get_final_metric("MDR", "Balanced_Accuracy")
mdr_mcc = get_final_metric("MDR", "MCC")
mdr_auc = get_final_metric("MDR", "ROC_AUC")
mdr_pr = get_final_metric("MDR", "PR_AUC")

sev_acc = get_final_metric("Severity", "Accuracy")
sev_f1 = get_final_metric("Severity", "F1_or_Macro_F1")
sev_ba = get_final_metric("Severity", "Balanced_Accuracy")
sev_mcc = get_final_metric("Severity", "MCC")
sev_auc = get_final_metric("Severity", "ROC_AUC")
sev_pr = get_final_metric("Severity", "PR_AUC")

results_text = (
    "Final model selection identified soft voting ensembles as the most suitable "
    "models for the proposed genome-level MDR and resistance severity prediction "
    "framework. For MDR prediction, the selected voting ensemble achieved an "
    f"accuracy of {mdr_acc:.4f}, F1-score of {mdr_f1:.4f}, balanced accuracy of "
    f"{mdr_ba:.4f}, MCC of {mdr_mcc:.4f}, ROC-AUC of {mdr_auc:.4f}, and PR-AUC "
    f"of {mdr_pr:.4f} on the independent test set. For resistance severity "
    f"assessment, the final voting ensemble achieved an accuracy of {sev_acc:.4f}, "
    f"macro F1-score of {sev_f1:.4f}, balanced accuracy of {sev_ba:.4f}, MCC of "
    f"{sev_mcc:.4f}, macro ROC-AUC of {sev_auc:.4f}, and macro PR-AUC of "
    f"{sev_pr:.4f}. "
)

if len(xai_summary) > 0:

    mdr_xai = xai_summary[
        xai_summary["Task"].astype(str).str.lower().str.contains("mdr")
    ].head(5)

    sev_xai = xai_summary[
        xai_summary["Task"].astype(str).str.lower().str.contains("severity")
    ].head(5)

    mdr_xai_features = ", ".join(mdr_xai["Feature"].astype(str).tolist())
    sev_xai_features = ", ".join(sev_xai["Feature"].astype(str).tolist())

    xai_text = (
        f"SHAP analysis indicated that the most influential MDR predictors were "
        f"{mdr_xai_features}. For severity assessment, the most influential "
        f"features were {sev_xai_features}. "
    )

else:
    xai_text = ""

if len(ablation_summary_table) > 0:

    ablation_text = (
        "Ablation analysis showed that removing MIC-derived features produced "
        "the largest performance degradation, supporting the central contribution "
        "of MIC magnitude and MIC distribution features to both MDR prediction "
        "and severity assessment."
    )

else:
    ablation_text = ""

positioning_text = (
    "The framework should be described as a phenotype-informed genome-level "
    "machine learning framework because the predictors are derived from curated "
    "antibiotic testing records, MIC summaries, laboratory method indicators, "
    "testing standard information, coverage features, and missingness patterns. "
    "It should not be described as a raw whole-genome-sequence or k-mer based "
    "prediction model unless sequence-derived features are added."
)

complete_paper_text = (
    results_text
    + "\n\n"
    + xai_text
    + "\n\n"
    + ablation_text
    + "\n\n"
    + positioning_text
)

with open(
    os.path.join(
        BLOCK18_TEXT_DIR,
        "final_paper_ready_results_discussion.txt"
    ),
    "w",
    encoding="utf-8"
) as f:
    f.write(complete_paper_text)


# ============================================================
# Final Abstract-Style Summary
# ============================================================

abstract_text = (
    "This study developed an explainable genome-level machine learning framework "
    "for multidrug resistance prediction and resistance severity assessment in "
    "Escherichia coli. After phenotype curation, MDR label construction, severity "
    "index generation, feature engineering, baseline benchmarking, hyperparameter "
    "optimization, feature selection, explainability analysis, ensemble modeling, "
    "statistical validation, and ablation study, soft voting ensembles were selected "
    "as the final models. The final MDR model achieved "
    f"accuracy={mdr_acc:.4f}, F1={mdr_f1:.4f}, MCC={mdr_mcc:.4f}, and "
    f"ROC-AUC={mdr_auc:.4f}. The final severity model achieved "
    f"accuracy={sev_acc:.4f}, macro F1={sev_f1:.4f}, MCC={sev_mcc:.4f}, and "
    f"macro ROC-AUC={sev_auc:.4f}. SHAP and ablation analyses highlighted the "
    "importance of MIC-derived variables, antibiotic testing coverage, laboratory "
    "testing indicators, and missingness-aware features."
)

with open(
    os.path.join(
        BLOCK18_TEXT_DIR,
        "final_abstract_style_summary.txt"
    ),
    "w",
    encoding="utf-8"
) as f:
    f.write(abstract_text)


# ============================================================
# Final Limitations and Recommended Title
# ============================================================

title_and_limitations = (
    "Recommended title:\n"
    "Explainable Multi-Objective Machine Learning Framework for Genome-Level "
    "Multidrug Resistance Prediction and Resistance Severity Assessment in "
    "Escherichia coli\n\n"
    "Recommended terminology:\n"
    "- Use 'genome-level' rather than 'whole-genome-sequence-based' unless "
    "sequence-derived features are added.\n"
    "- Use 'multi-objective' or 'unified framework' unless a true shared "
    "multi-task neural architecture is included.\n"
    "- Describe the input features as phenotype-informed engineered features "
    "derived from MIC, testing coverage, laboratory method, testing standard, "
    "and missingness evidence.\n\n"
    "Main limitation:\n"
    "The current framework does not use raw genome sequences, k-mers, SNPs, "
    "AMR genes, plasmid markers, or pangenome features. The study should "
    "therefore avoid overstating direct genomic sequence prediction."
)

with open(
    os.path.join(
        BLOCK18_TEXT_DIR,
        "recommended_title_positioning_and_limitations.txt"
    ),
    "w",
    encoding="utf-8"
) as f:
    f.write(title_and_limitations)


# ============================================================
# Copy All Final Tables/Figures/Text to Export Package
# ============================================================

for folder_name, source_dir in [
    ("tables", BLOCK18_TABLE_DIR),
    ("figures", BLOCK18_FIG_DIR),
    ("text", BLOCK18_TEXT_DIR)
]:

    target_dir = os.path.join(
        BLOCK18_EXPORT_DIR,
        folder_name
    )

    os.makedirs(
        target_dir,
        exist_ok=True
    )

    for file_name in os.listdir(source_dir):

        source_path = os.path.join(
            source_dir,
            file_name
        )

        if os.path.isfile(source_path):

            shutil.copy2(
                source_path,
                os.path.join(
                    target_dir,
                    file_name
                )
            )


# ============================================================
# Final Output Summary
# ============================================================

block18_output_summary = pd.DataFrame({
    "Output_Category": [
        "Dataset and framework summary",
        "Final proposed model performance",
        "Bootstrap confidence intervals",
        "MDR test comparison",
        "Severity test comparison",
        "XAI feature importance summary",
        "Ablation study summary",
        "Ensemble component ablation summary",
        "Statistical testing summary",
        "Manuscript table index",
        "Figure index",
        "Paper-ready results discussion",
        "Abstract-style summary",
        "Recommended title and limitations",
        "Export package"
    ],
    "Saved_File": [
        "final_tables/table_1_dataset_and_framework_summary.csv",
        "final_tables/table_2_final_proposed_model_performance.csv",
        "final_tables/table_3_bootstrap_confidence_intervals.csv",
        "final_tables/table_4_mdr_test_model_comparison.csv",
        "final_tables/table_5_severity_test_model_comparison.csv",
        "final_tables/table_6_xai_feature_importance_summary.csv",
        "final_tables/table_7_ablation_study_summary.csv",
        "final_tables/table_8_ensemble_component_ablation_summary.csv",
        "final_tables/table_9_statistical_testing_summary.csv",
        "final_tables/table_10_manuscript_table_index.csv",
        "final_figures/figure_index.csv",
        "paper_ready_text/final_paper_ready_results_discussion.txt",
        "paper_ready_text/final_abstract_style_summary.txt",
        "paper_ready_text/recommended_title_positioning_and_limitations.txt",
        "export_package/"
    ]
})

block18_output_summary.to_csv(
    os.path.join(
        BLOCK18_DIR,
        "block18_output_summary.csv"
    ),
    index=False
)

print("\n")
print("=" * 70)
print("FINAL PAPER-READY RESULTS TEXT")
print("=" * 70)
print(complete_paper_text)

print("\n")
print("=" * 70)
print("FINAL ABSTRACT-STYLE SUMMARY")
print("=" * 70)
print(abstract_text)

print("\n")
print("=" * 70)
print("BLOCK 18 GENERATED FILES")
print("=" * 70)

print("- final_tables/table_1_dataset_and_framework_summary.csv")
print("- final_tables/table_2_final_proposed_model_performance.csv")
print("- final_tables/table_3_bootstrap_confidence_intervals.csv")
print("- final_tables/table_4_mdr_test_model_comparison.csv")
print("- final_tables/table_5_severity_test_model_comparison.csv")
print("- final_tables/table_6_xai_feature_importance_summary.csv")
print("- final_tables/table_7_ablation_study_summary.csv")
print("- final_tables/table_8_ensemble_component_ablation_summary.csv")
print("- final_tables/table_9_statistical_testing_summary.csv")
print("- final_tables/table_10_manuscript_table_index.csv")
print("- final_figures/figure_index.csv")
print("- paper_ready_text/final_paper_ready_results_discussion.txt")
print("- paper_ready_text/final_abstract_style_summary.txt")
print("- paper_ready_text/recommended_title_positioning_and_limitations.txt")
print("- export_package/")

print("\n")
print("=" * 70)
print("BLOCK 18 COMPLETED SUCCESSFULLY")
print("=" * 70)
